In [ ]:
# !kill 1572997
!nvidia-smi -l 1

In [ ]:
%pip install transformer_lens
%pip install circuitsvis
%pip install deeptime
%mamba install -c conda-forge pygpcca
%mamba install infomap
%pip install geomloss[full]
%pip install pydiffmap
%mamba install joblib
%pip install fast-langdetect

In [ ]:
import ipykernel
from IPython.display import display, HTML

# Get the exact absolute path of the current active kernel
conn_file = ipykernel.connect.get_connection_file()
cmd = f"jupyter console --existing {conn_file}"

# Generate a UI button to copy the command
html = f"""
<div style="background: #1e1e1e; color: #d4d4d4; padding: 12px; border-radius: 4px; font-family: monospace; border: 1px solid #333;">
    <span id="jupyter_cmd">{cmd}</span><br><br>
    <button onclick="navigator.clipboard.writeText(document.getElementById('jupyter_cmd').innerText);" 
            style="background: #007acc; color: white; border: none; padding: 6px 1144/62px; border-radius: 3px; cursor: pointer;">
        Copy to VSCode Terminal
    </button>
</div>
"""
display(HTML(html))

In [ ]:
import sys
sys.path.append('/home/galk/LanguageDynamics/src') 

from datetime import datetime
from tqdm import tqdm
import os
import gc # garbage collector
import math
from typing import Dict, List, Tuple, Set, Any, Optional, Union
import heapq
import time
import itertools
import random

import numpy as np
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from torch.distributions.kl import kl_divergence
from torch.distributions.categorical import Categorical
from torch.distributions.multivariate_normal import MultivariateNormal

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, PreTrainedTokenizer

# import interpretability stuff
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, FactoredMatrix
from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
import circuitsvis as cv
import einops

# RepE utils
from fast_langdetect import detect, LangDetectConfig, LangDetector

import scipy.sparse as sp
import scipy.linalg as la
import scipy.sparse.linalg as spla
import scipy.sparse.csgraph as csgraph

# sklearn regressions
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Network kinetics analysis
import deeptime.markov.msm as msm
DEEPTIME_AVAILABLE = True
from pygpcca import GPCCA
from infomap import Infomap

from geomloss import SamplesLoss

# multi-gpu utils
from concurrent.futures import ProcessPoolExecutor
from joblib import Parallel, delayed

models_path = '/home/galk/LanguageDynamics/models/linguistic_flip_flop'

### Defining Function

In [ ]:
### General Helper Functions

def moving_mean(data, window_size):
    weights = np.ones(window_size) / window_size
    return np.convolve(data, weights, mode='same')

def plot_trajectory_pca(
    trajectory: Union[np.ndarray, torch.Tensor],
    pca: Optional[PCA] = None,
    n_components: Optional[int] = 3,
    title: Optional[str] = "Trajectory",
    cmap: str = "viridis",
    s: int = 30,
    alpha: float = 0.6,
    figsize: Tuple[int, int] = (16, 7),
    show_plot: bool = True,
) -> Tuple[np.ndarray, PCA]:
    """
    Computes (or applies) PCA on a high-dimensional trajectory and creates side-by-side
    2D and 3D scatter plots colored by trajectory position index, styled consistently
    with the analysis notebooks.

    Parameters:
        trajectory (Union[np.ndarray, torch.Tensor]):
            Input trajectory of shape (T, D) or (B, T, D). If torch.Tensor, it is detached and moved to CPU.
        pca (Optional[PCA]):
            Pre-fitted PCA object. If provided, the trajectory is transformed using this PCA.
            If None, a new PCA model is fitted on the trajectory.
        n_components (Optional[int]):
            Number of components to fit if pca is None. Defaults to 3 (minimum required for 3D plot).
        title (Optional[str]):
            Base title for the plots (e.g. 'Head Outputs' or 'Trajectory').
        cmap (str):
            Colormap for the position coloring. Default: 'viridis'.
        s (int):
            Marker size for scatter plot. Default: 30.
        alpha (float):
            Opacity for scatter points. Default: 0.6.
        figsize (Tuple[int, int]):
            Figure size (width, height). Default: (16, 7).
        show_plot (bool):
            Whether to call plt.show(). Default: True.

    Returns:
        pca_coords (np.ndarray):
            PCA-transformed coordinates of shape (T, n_components).
        pca (PCA):
            Fitted or provided PCA object.
    """
    # Convert torch.Tensor to numpy array if necessary
    if isinstance(trajectory, torch.Tensor):
        trajectory_np = trajectory.detach().cpu().numpy()
    else:
        trajectory_np = np.asarray(trajectory)

    # Flatten batch dimension if 3D (B, T, D) -> (B * T, D)
    if trajectory_np.ndim == 3:
        b, t, d = trajectory_np.shape
        trajectory_np = trajectory_np.reshape(b * t, d)
    elif trajectory_np.ndim != 2:
        raise ValueError(f"Expected trajectory of shape (T, D) or (B, T, D), got shape {trajectory_np.shape}")

    # Fit or transform PCA
    if pca is None:
        n_comp = n_components if n_components is not None else min(trajectory_np.shape[0], trajectory_np.shape[1])
        pca = PCA(n_components=n_comp)
        pca_coords = pca.fit_transform(trajectory_np)
    else:
        pca_coords = pca.transform(trajectory_np)

    if pca_coords.shape[1] < 2:
        raise ValueError(f"PCA produces fewer than 2 components ({pca_coords.shape[1]}). Cannot plot 2D/3D.")

    positions = np.arange(pca_coords.shape[0])

    # Extract explained variance ratios if available
    if hasattr(pca, "explained_variance_ratio_") and pca.explained_variance_ratio_ is not None:
        evr = pca.explained_variance_ratio_
        pc1_label = f"PC1 ({evr[0]*100:.1f}%)" if len(evr) > 0 else "PC1"
        pc2_label = f"PC2 ({evr[1]*100:.1f}%)" if len(evr) > 1 else "PC2"
        pc3_label = f"PC3 ({evr[2]*100:.1f}%)" if len(evr) > 2 else "PC3"
    else:
        pc1_label, pc2_label, pc3_label = "PC1", "PC2", "PC3"

    # Create figure with side-by-side 2D and 3D subplots
    fig = plt.figure(figsize=figsize)

    # 2D PCA plot
    ax_2d = fig.add_subplot(121)
    scatter_2d = ax_2d.scatter(
        pca_coords[:, 0],
        pca_coords[:, 1],
        c=positions,
        cmap=cmap,
        s=s,
        alpha=alpha,
    )
    ax_2d.set_xlabel(pc1_label)
    ax_2d.set_ylabel(pc2_label)
    title_suffix = f" of {title}" if title else ""
    ax_2d.set_title(f"2D PCA{title_suffix}")
    cbar_2d = plt.colorbar(scatter_2d, ax=ax_2d)
    cbar_2d.set_label("Position Index")
    ax_2d.grid(True, alpha=0.3)

    # 3D PCA plot (only if at least 3 components exist)
    if pca_coords.shape[1] >= 3:
        ax_3d = fig.add_subplot(122, projection="3d")
        scatter_3d = ax_3d.scatter(
            pca_coords[:, 0],
            pca_coords[:, 1],
            pca_coords[:, 2],
            c=positions,
            cmap=cmap,
            s=s,
            alpha=alpha,
        )
        ax_3d.set_xlabel(pc1_label)
        ax_3d.set_ylabel(pc2_label)
        ax_3d.set_zlabel(pc3_label)
        ax_3d.set_title(f"3D PCA{title_suffix}")
        cbar_3d = plt.colorbar(scatter_3d, ax=ax_3d, pad=0.1)
        cbar_3d.set_label("Position Index")

    plt.tight_layout()
    if show_plot:
        plt.show()

    return pca_coords, pca

def plot_attention_pattern(
    cache: Dict[str, torch.Tensor],
    layer: int,
    head: int,
    seq_len: Optional[int] = None
):
    plt.figure(figsize=(10,8))
    attention_pattern = cache["pattern", layer][0]  # shape: [n_heads, seq_len, seq_len]
    if seq_len is None:
        seq_len = attention_pattern.shape[-1]
    plt.imshow(attention_pattern[head][:seq_len,   :seq_len].cpu().numpy(), interpolation="none", cmap="Reds", norm="linear")
    plt.colorbar()
    plt.title(f"Head {layer}.{head} (Linear Scale)")

    plt.figure(figsize=(10,8))
    plt.imshow(attention_pattern[head][:seq_len,   :seq_len].cpu().numpy(), interpolation="none", cmap="Reds", norm="log")
    plt.colorbar()
    plt.title(f"Head {layer}.{head} (Log Scale)")

    plt.show()

def plot_attention_pattern_simple(
    attention_pattern: torch.Tensor,
    layer: int,
    head: int,
    seq_len: Optional[int] = None
):
    plt.figure(figsize=(10,8))
    if seq_len is None:
        seq_len = attention_pattern.shape[-1]
    plt.imshow(attention_pattern[:seq_len,   :seq_len].cpu().numpy(), interpolation="none", cmap="Reds", norm="linear")
    plt.colorbar()
    plt.title(f"Head {layer}.{head} (Linear Scale)")

    plt.figure(figsize=(10,8))
    plt.imshow(attention_pattern[:seq_len,   :seq_len].cpu().numpy(), interpolation="none", cmap="Reds", norm="log")
    plt.colorbar()
    plt.title(f"Head {layer}.{head} (Log Scale)")

    plt.show()

#### Activation Patching Functions

In [ ]:
## Activation Patching Functions

from functools import partial

def patch_head_hook(corrupted_hook, hook, clean_cache, head_idx, position=None, knockout=False):
    """
    The hook function that surgically overwrites the corrupted activations 
    with the clean activations for a specific head and position.
    
    corrupted_hook shape: [batch, sequence_length, n_heads, d_head]
    """
    # Fetch the clean activations from the cache
    clean_hook = clean_cache[hook.name]
    if knockout:
        clean_hook = torch.zeros_like(clean_hook)
    
    if position is None:
        # Wide Patching: Patch this head at ALL sequence positions
        corrupted_hook[:, :, head_idx, :] = clean_hook[:, :, head_idx, :]
    else:
        # Position-Specific Patching: Patch this head at a SINGLE position
        corrupted_hook[:, position, head_idx, :] = clean_hook[:, position, head_idx, :]
        
    return corrupted_hook


def run_attention_patching(
    model: HookedTransformer,
    clean_prompt: str,
    corrupt_prompt: str,
    clean_answer: str,
    corrupt_answer: str,
    layer: int,
    head_idx: int,
    position: int = None
):
    """
    Runs the full causal tracing experiment and returns logits and the recovery score.
    """
    # 1. Ensure prompts are tokenized to the same length for clean position mapping
    clean_tokens = model.to_tokens(clean_prompt)
    corrupt_tokens = model.to_tokens(corrupt_prompt)
    
    if clean_tokens.shape[1] != corrupt_tokens.shape[1]:
        print("Warning: Prompts have different token lengths. Patching by position may behave unexpectedly.")

    # Get token IDs for the expected answers to calculate our metric
    clean_answer_id = model.to_single_token(clean_answer)
    corrupt_answer_id = model.to_single_token(corrupt_answer)

    # 2. Run the CLEAN prompt and cache all activations
    clean_logits, clean_cache = model.run_with_cache(clean_tokens)

    # 3. Run the CORRUPT prompt to get baseline corrupted logits
    corrupt_logits = model(corrupt_tokens)

    # 4. Run the PATCHED prompt
    # We hook into the 'z' activation (the mixed values right before the W_O projection)
    hook_name = f"blocks.{layer}.attn.hook_z"
    
    # Use functools.partial to freeze our specific arguments into the hook function
    hook_fn = partial(
        patch_head_hook, 
        clean_cache=clean_cache, 
        head_idx=head_idx, 
        position=position
    )
    
    patched_logits = model.run_with_hooks(
        corrupt_tokens,
        fwd_hooks=[(hook_name, hook_fn)]
    )

    # 5. Calculate Metrics (Logit Difference at the final token position)
    def get_logit_diff(logits):
        # Look at the final token's prediction [batch_idx=0, pos_idx=-1]
        final_logits = logits[0, -1, :]
        return (final_logits[clean_answer_id] - final_logits[corrupt_answer_id]).item()

    clean_diff = get_logit_diff(clean_logits)
    corrupt_diff = get_logit_diff(corrupt_logits)
    patched_diff = get_logit_diff(patched_logits)

    # Recovery Score: 0% means it acts like the corrupted model, 100% means it acts like the clean model
    recovery_score = (patched_diff - corrupt_diff) / (clean_diff - corrupt_diff)

    return {
        "clean_logits": clean_logits,
        "corrupt_logits": corrupt_logits,
        "patched_logits": patched_logits,
        "clean_diff": clean_diff,
        "corrupt_diff": corrupt_diff,
        "patched_diff": patched_diff,
        "recovery_score": recovery_score
    }

def get_logit_diff(logits, clean_id, corrupt_id):
    """Calculates Logit(Clean) - Logit(Corrupt) for the final token."""
    final_logits = logits[0, -1, :]
    return (final_logits[clean_id] - final_logits[corrupt_id]).item()

def get_probability(logits, target_id):
    """Calculates the softmax probability of a specific target token."""
    final_logits = logits[0, -1, :]
    probs = F.softmax(final_logits, dim=-1)
    return probs[target_id].item()

def zero_head_hook(z, hook, head_idx, min_position=0):
    """
    An ablation hook that completely zeroes out a specific head's output.
    Because this runs during generation with KV caching, 'z' might be the 
    activations for the whole prompt, or just the single newest token.
    Using `:, :, head_idx, :` handles both cases perfectly.
    
    z shape: [batch, sequence_length, n_heads, d_head]
    """
    # Set all activations for this specific head to 0
    if z.shape[1] > min_position:
        z[:, min_position:, head_idx, :] = 0.0
    else:
        z[:, :, head_idx, :] = 0.0
    return z

def remove_pos_embed_hook(pos_embed, hook):
    return torch.zeros_like(pos_embed)

def generate_with_knockout(model, prompt: str, layer: int, head_idx: int, max_tokens: int = 20, knockout_positional=False, do_sample=False, top_p=0.9, prepend_bos=True):
    """
    Generates text while knocking out a specific head, ensuring safe hook removal.
    """
    hook_name = f"blocks.{layer}.attn.hook_z"
    hook_fn = partial(zero_head_hook, head_idx=head_idx)

    hooks = [(hook_name, hook_fn)]

    if knockout_positional:
        positional_hook_name = "hook_pos_embed"
        positional_hook_fn = remove_pos_embed_hook
        hooks += [(positional_hook_name, positional_hook_fn)]
    
    print(f"--- Intervening: Knocking out L{layer}H{head_idx} ---")
    
    # The context manager ensures hooks exist ONLY inside this indented block
    with model.hooks(fwd_hooks=hooks):
        
        # You can use the standard generate method exactly as normal
        patched_output = model.generate(
            prompt,
            max_new_tokens=max_tokens,
            do_sample=do_sample,
            temperature=1.0,
            top_p=top_p,
            prepend_bos=prepend_bos 
        )
    
    return patched_output

def compute_pairwise_cosine_similarity(
    patched_outputs: torch.Tensor
) -> torch.Tensor:
    """
    Computes the pairwise cosine similarity of the head's output across all 
    patching positions, batched for every probing query.
    
    Args:
        patched_outputs: Tensor of shape [n_queries, n_positions, d_head]
                         representing the attention output per query and position.
                         
    Returns:
        cos_sim_matrices: Tensor of shape [n_queries, n_positions, n_positions]
                          where the [i, j, k] element is the cosine similarity 
                          between position j and position k for query i.
    """
    # 1. Normalize the outputs along the d_head dimension
    # Shape remains [n_queries, n_positions, d_head]
    normalized_outputs = F.normalize(patched_outputs, p=2, dim=-1)
    
    # 2. Compute pairwise dot products (cosine similarities for normalized vectors)
    # [n_queries, n_positions, d_head] @ [n_queries, d_head, n_positions]
    # Resulting Shape: [n_queries, n_positions, n_positions]
    cos_sim_matrices = torch.matmul(
        normalized_outputs, 
        normalized_outputs.transpose(-1, -2)
    )
    
    return cos_sim_matrices

def compute_projected_cosine_similarity(
    patched_outputs: torch.Tensor,
    model: HookedTransformer,
    layer: int,
    head_idx: int
) -> torch.Tensor:
    """
    Projects the patched z outputs through the head's specific W_O matrix into 
    the residual stream, then computes pairwise cosine similarity.
    
    Args:
        patched_outputs: Tensor of shape [n_queries, n_positions, d_head]
        model: The TransformerLens model.
        layer: The transformer layer index.
        head_idx: The attention head index.
        
    Returns:
        cos_sim_matrices: Tensor of shape [n_queries, n_positions, n_positions]
    """
    # 1. Get the Output Projection weight matrix for this specific head
    # Shape of model.W_O is usually [n_layers, n_heads, d_head, d_model]
    # We extract shape: [d_head, d_model]
    W_O = model.W_O[layer, head_idx] 
    
    # 2. Project into the residual stream space
    # [n_queries, n_positions, d_head] @ [d_head, d_model] -> [n_queries, n_positions, d_model]
    projected_outputs = torch.matmul(patched_outputs, W_O)
    
    # 3. Normalize along the new d_model dimension
    normalized_outputs = F.normalize(projected_outputs, p=2, dim=-1)
    
    # 4. Compute batched pairwise dot products
    # Resulting Shape: [n_queries, n_positions, n_positions]
    cos_sim_matrices = torch.matmul(
        normalized_outputs, 
        normalized_outputs.transpose(-1, -2)
    )
    
    return cos_sim_matrices

def compute_pairwise_l2_distance(
    patched_outputs: torch.Tensor
) -> torch.Tensor:
    """
    Computes the pairwise L2 (Euclidean) distance of the head's output 
    across all patching positions.
    
    Args:
        patched_outputs: Tensor of shape [n_queries, n_positions, d_head]
        
    Returns:
        l2_matrices: Tensor of shape [n_queries, n_positions, n_positions]
                     Lower values indicate the vectors are geometrically closer.
    """
    # torch.cdist computes batched pairwise p-norm distances cleanly
    # Input shape: [batch(n_queries), P(n_positions), M(d_head)]
    # Output shape: [batch(n_queries), P(n_positions), P(n_positions)]
    l2_matrices = torch.cdist(patched_outputs, patched_outputs, p=2.0)
    
    return l2_matrices

def compute_logit_lens_jsd(
    patched_outputs: torch.Tensor,
    model: HookedTransformer,
    layer: int,
    head_idx: int
) -> torch.Tensor:
    """
    Projects patched outputs to vocabulary logits, converts to probabilities, 
    and computes the pairwise Jensen-Shannon Divergence.
    
    Args:
        patched_outputs: Tensor of shape [n_queries, n_positions, d_head]
        model: The TransformerLens model.
        layer: The transformer layer index.
        head_idx: The attention head index.
        
    Returns:
        jsd_matrices: Tensor of shape [n_queries, n_positions, n_positions]
                      0.0 means identical probability distributions.
    """
    n_queries, n_positions, _ = patched_outputs.shape
    
    # 1. Project through W_O
    # Shape: [n_queries, n_positions, d_model]
    W_O = model.W_O[layer, head_idx]
    projected_acts = torch.matmul(patched_outputs, W_O)
    
    # 2. Project through Unembedding (W_U) to get logits
    # Shape: [n_queries, n_positions, d_vocab]
    logits = torch.matmul(projected_acts, model.W_U)
    
    # 3. Convert logits to probability distributions
    # Shape: [n_queries, n_positions, d_vocab]
    probs = F.softmax(logits, dim=-1)
    
    # 4. Prepare tensors for pairwise comparison via broadcasting
    # P shape: [n_queries, n_positions, 1, d_vocab]
    # Q shape: [n_queries, 1, n_positions, d_vocab]
    P = probs.unsqueeze(2)
    Q = probs.unsqueeze(1)
    
    # Calculate the midpoint distribution M
    M = 0.5 * (P + Q)
    
    # 5. Compute KL Divergence manually for efficiency and stability
    # Clamp to avoid log(0)
    log_P = torch.log(P.clamp(min=1e-10))
    log_Q = torch.log(Q.clamp(min=1e-10))
    log_M = torch.log(M.clamp(min=1e-10))
    
    # KL(P || M) = Sum(P * log(P/M)) = Sum(P * (log_P - log_M))
    # Summing over the d_vocab dimension (dim=-1)
    kl_pm = (P * (log_P - log_M)).sum(dim=-1) # Shape: [n_queries, n_positions, n_positions]
    kl_qm = (Q * (log_Q - log_M)).sum(dim=-1) # Shape: [n_queries, n_positions, n_positions]
    
    # JSD is the average of the two KL divergences
    jsd_matrices = 0.5 * (kl_pm + kl_qm)
    
    # Clamp lower bound to 0 to eliminate any floating point underflow
    return jsd_matrices.clamp(min=0.0)

def plot_cosine_similarities(
    cos_sim_matrices: torch.Tensor,
    patch_positions: list[int] | None = None,
    query_labels: list[str] | None = None,
    v_max = None,
    v_min = None,
    figsize_per_plot: int = 6
) -> None:
    """
    Plots a grid of heatmaps showing the pairwise cosine similarity of head 
    outputs across positions, one heatmap per probing query.
    
    Args:
        cos_sim_matrices: Tensor of shape [n_queries, n_positions, n_positions]
        patch_positions: Optional list of trajectory sequence indices used. 
                         Used for axis labeling.
        query_labels: Optional list of strings describing each probing query.
        figsize_per_plot: Size multiplier for the resulting grid.
    """



    # Convert to CPU numpy for matplotlib/seaborn
    cos_sim_np = cos_sim_matrices.detach().cpu().numpy()
    n_queries = cos_sim_np.shape[0]
    n_positions = cos_sim_np.shape[1]
    
    # Determine grid layout (try to make it somewhat square)
    cols = math.ceil(math.sqrt(n_queries))
    rows = math.ceil(n_queries / cols)
    
    fig, axes = plt.subplots(
        rows, cols, 
        figsize=(cols * figsize_per_plot, rows * figsize_per_plot),
        squeeze=False # Ensures axes is always a 2D array
    )
    
    # Define tick labels if patch positions are provided
    tick_labels = patch_positions if patch_positions is not None else range(n_positions)
    # To avoid overcrowded axes on long trajectories, show every 5th label (or adjust as needed)
    tick_interval = max(1, len(tick_labels) // 10) 
    
    for idx in range(n_queries):
        r = idx // cols
        c = idx % cols
        ax = axes[r, c]
        
        # Plot heatmap
        sns.heatmap(
            cos_sim_np[idx], 
            ax=ax, 
            cmap="coolwarm", 
            vmin=min(-1.0,np.min(cos_sim_np)) if v_min is None else v_min, 
            vmax=max(1.0,np.max(cos_sim_np)) if v_max is None else v_max, 
            square=True,
            cbar=(c == cols - 1),     # Correctly turn the colorbar on/off here
            cbar_kws={"shrink": 0.8}  # Always pass a dictionary
        )
        
        title = query_labels[idx] if query_labels else f"Probing Query {idx}"
        ax.set_title(title, fontsize=12)
        ax.set_xlabel("Trajectory Position")
        ax.set_ylabel("Trajectory Position")
        
        # Format ticks
        ax.set_xticks(range(0, n_positions, tick_interval))
        ax.set_yticks(range(0, n_positions, tick_interval))
        ax.set_xticklabels([tick_labels[i] for i in range(0, n_positions, tick_interval)], rotation=45)
        ax.set_yticklabels([tick_labels[i] for i in range(0, n_positions, tick_interval)], rotation=0)

    # Hide any unused subplots in the grid
    for idx in range(n_queries, rows * cols):
        r = idx // cols
        c = idx % cols
        fig.delaxes(axes[r, c])
        
    plt.tight_layout()
    plt.show()

#### Optimization and Black Box Functions

In [ ]:
## Optimization and Black Box Functions
def JSD(pi1, pi2):
    M = 0.5 * (pi1 + pi2)
    jsd = 0.5 * (F.kl_div(  
        input=M.log(), 
        target=pi1.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M.log(), 
        target=pi2.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )
    return jsd

def compute_aligned_jsd(vals_1: torch.Tensor, keys_1: torch.Tensor, 
                        vals_2: torch.Tensor, keys_2: torch.Tensor) -> torch.Tensor:
    """
    Computes Jensen-Shannon Divergence between two sparse distributions 
    with potentially mismatched K-gram supports by dynamically uniting them.
    """
    # Find the Union Support
    cat_keys = torch.cat([keys_1, keys_2], dim=0)
    unique_keys, inverse_indices = torch.unique(cat_keys, dim=0, return_inverse=True)
    N_unique = unique_keys.size(0)
    
    # Split inverse indices back to original distributions
    idx_1 = inverse_indices[:keys_1.size(0)]
    idx_2 = inverse_indices[keys_1.size(0):]
    
    # Scatter values onto the unified support (Missing keys get eps)
    aligned_1 = torch.ones(N_unique, dtype=vals_1.dtype, device=vals_1.device) * 1e-10
    aligned_2 = torch.ones(N_unique, dtype=vals_2.dtype, device=vals_2.device) * 1e-10
    
    aligned_1.scatter_(0, idx_1, vals_1)
    aligned_2.scatter_(0, idx_2, vals_2)

    aligned_1 = aligned_1 / aligned_1.sum()
    aligned_2 = aligned_2 / aligned_2.sum()
    
    # Compute standard JSD
    # M = 0.5 * (aligned_1 + aligned_2)
    # kl_1 = torch.sum(aligned_1 * torch.log((aligned_1 + 1e-11) / (M + 1e-11)))
    # kl_2 = torch.sum(aligned_2 * torch.log((aligned_2 + 1e-11) / (M + 1e-11)))

    # loss = 0.5 * kl_1 + 0.5 * kl_2

    # print(f"keys 1 shape: {keys_1.shape}\nkeys_2 shape: {keys_2.shape}")
    # print(f"N_unique: {N_unique}")
    # print(f"unique_keys: {unique_keys}")
    # print(f"aligned_1 shape: {aligned_1.shape}")
    # print(f"idx_1 shape: {idx_1.shape}")

    loss = JSD(aligned_1.unsqueeze(0), aligned_2.unsqueeze(0)) # unsqueeze to create batch dimension
    # loss = criterion(aligned_1, aligned_2)
    
    return loss


def measure_branching_factor(
    model: HookedTransformer,
    pi: torch.Tensor,
    N_generations: int,
    N_context: int,
    N_generate: int,
    temperature: float = 1.0,
    top_p: float = 1.0,
    fwd_hooks: Optional[list] = [],
    context_estimator=None,
    jsd_threshold: float = 0.5,
    S_init: int = 1,
    generation_batch_size: Optional[int] = None,
    prepend_bos: bool = True,
    normalize_jsd: bool = True,
    seed: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Measures the BRANCHING FACTOR of a macroscopic state `pi`: how many distinct
    attractors the model falls into when it is restarted, again and again, from
    *statistically identical* contexts drawn from the same `pi`.

    The procedure:
      1. Draw `N_generations` independent contexts of `N_context` tokens from `pi`
         (`context_from_pi`, i.e. i.i.d. multinomial sampling -- every context has the
         same macroscopic state but a different realization).
      2. Continue each of them for `N_generate` tokens in one batched `model.generate`
         call, under `fwd_hooks` and the given sampling parameters.
      3. Re-estimate the macroscopic state from the *generated* tokens only, with
         `context_estimator` -- this is the state the trajectory landed in, uncontaminated
         by the prompt it started from.
      4. Build the full pairwise JSD distance matrix between those final states
         (`compute_aligned_jsd`, so supports may differ).
      5. Cluster with DBSCAN(metric="precomputed", eps=jsd_threshold, min_samples=1).
         `min_samples=1` means nothing is labelled noise: a lone trajectory is its own
         cluster, which is exactly what a branch is. The number of clusters is the
         branching factor.

    Args:
        model: HookedTransformer.
        pi: [d_vocab] dense probability vector over tokens (the macroscopic state to
            restart from). Not modified -- it is cloned before `context_from_pi`, which
            thresholds in place.
        N_generations: number of independent restarts.
        N_context: tokens sampled from `pi` to form each prompt.
        N_generate: new tokens generated per restart.
        temperature, top_p: sampling parameters handed to `model.generate`.
        fwd_hooks: list of (hook_name, hook_fn) applied during generation -- e.g. the
            `zero_head_hook` / `remove_pos_embed_hook` pair used everywhere else in this
            notebook. `None` means the unmodified model.
        context_estimator: a `ContextDistributionEstimator` (cell 103). Applied to the
            generated tokens only. Required.
        jsd_threshold: DBSCAN epsilon, in the same units as the returned distances (see
            `normalize_jsd`). Two generations closer than this are the same branch.
        S_init: key width demanded by the estimator, `sum(K_list) - len(K_list) + 1`.
        generation_batch_size: split the `N_generations` prompts into chunks of this size
            for generation (VRAM knob). `None` runs them all in one batch.
        prepend_bos: prepend the tokenizer's BOS to every sampled prompt.
        normalize_jsd: divide the JSD by ln(2) so distances live in [0, 1] and
            `jsd_threshold` is a fraction of the maximum possible divergence
            (disjoint supports). Set False to get raw nats.
        seed: seeds torch's global RNG for both context sampling and generation.
        verbose: tqdm progress bars.

    Returns a dict with:
        "texts":            List[str], the full decoded generation (prompt + continuation).
        "continuation_texts": List[str], the generated part only.
        "context_tokens":   [N_generations, N_context] the sampled prompts.
        "generated_tokens": [N_generations, N_generate] the new tokens only.
        "distributions":    List[(vals [N_c], keys [N_c, S_init])], one per generation.
        "branching_factor": int, the number of DBSCAN clusters.
        "jsd_matrix":       [N_generations, N_generations] symmetric, zero diagonal.
        "cluster_labels":   [N_generations] np.ndarray of cluster indices (0-based).
        "cluster_sizes":    np.ndarray, population of each cluster.
    """
    from sklearn.cluster import DBSCAN

    if context_estimator is None:
        raise ValueError("measure_branching_factor requires a ContextDistributionEstimator.")

    device = model.cfg.device
    if seed is not None:
        torch.manual_seed(seed)

    # ---- 1. Sample N_generations independent contexts from pi -----------------------
    pi_dense = pi.detach().flatten().float().to(device)
    pi_dense = pi_dense / pi_dense.sum().clamp(min=1e-12)

    # context_from_pi zeroes out small entries IN PLACE, so hand it a fresh clone each time.
    context_tokens = torch.stack(
        [context_from_pi(pi_dense.clone(), N=N_context).to(device) for _ in range(N_generations)]
    )                                                                   # [N_gen, N_context]

    prompt_tokens = context_tokens
    if prepend_bos:
        bos_id = getattr(getattr(model, "tokenizer", None), "bos_token_id", None)
        if bos_id is None or bos_id < 0:
            bos_id = 0
        bos_column = torch.full((N_generations, 1), int(bos_id), dtype=torch.long, device=device)
        prompt_tokens = torch.cat([bos_column, context_tokens], dim=1)  # [N_gen, 1 + N_context]

    # ---- 2. Batched generation under the hooks --------------------------------------
    chunk = generation_batch_size or N_generations

    completions = []
    with torch.no_grad():
        with model.hooks(fwd_hooks=fwd_hooks):
            for start in tqdm(
                range(0, N_generations, chunk),
                desc="Generating",
                disable=not verbose,
            ):
                batch = prompt_tokens[start:start + chunk]
                out = model.generate(
                    batch,
                    max_new_tokens=N_generate,
                    do_sample=True,
                    temperature=temperature,
                    prepend_bos=prepend_bos,
                    top_p=top_p,
                    # Fixed-length continuations: an early EOS would leave the batch
                    # padded and make "the last N_generate tokens" mean different
                    # things for different rows.
                    stop_at_eos=False,
                    return_type="tokens",
                    verbose=False,
                )
                completions.append(out)

    full_tokens = torch.cat(completions, dim=0)                         # [N_gen, prompt + N_generate]
    generated_tokens = full_tokens[:, -N_generate:]                     # [N_gen, N_generate]

    texts = [model.to_string(row) for row in full_tokens]
    continuation_texts = [model.to_string(row) for row in generated_tokens]

    # ---- 3. Estimate the final macroscopic state from the NEW tokens only ------------
    distributions = [
        context_estimator.estimate(generated_tokens[i], model, S_init)
        for i in tqdm(range(N_generations), desc="Estimating final pi", disable=not verbose)
    ]

    # ---- 4. Pairwise JSD distance matrix --------------------------------------------
    scale = 1e5 / model.cfg.d_vocab if normalize_jsd else 1.0
    jsd_matrix = torch.zeros((N_generations, N_generations), device=device)
    for i in tqdm(range(N_generations), desc="Pairwise JSD", disable=not verbose):
        vals_i, keys_i = distributions[i]
        for j in range(i + 1, N_generations):
            vals_j, keys_j = distributions[j]
            distance = scale * compute_aligned_jsd(
                vals_1=vals_i, keys_1=keys_i,
                vals_2=vals_j, keys_2=keys_j,
            )
            jsd_matrix[i, j] = distance
            jsd_matrix[j, i] = distance

    # ---- 5. DBSCAN on the precomputed distances -------------------------------------
    distance_np = jsd_matrix.detach().cpu().numpy().astype(np.float64)
    distance_np = np.clip(0.5 * (distance_np + distance_np.T), 0.0, None)  # kill float asymmetry
    np.fill_diagonal(distance_np, 0.0)

    clustering = DBSCAN(eps=jsd_threshold, min_samples=1, metric="precomputed").fit(distance_np)
    cluster_labels = clustering.labels_                                  # no -1: min_samples=1
    branching_factor = int(len(set(cluster_labels.tolist())))
    cluster_sizes = np.bincount(cluster_labels, minlength=branching_factor)

    if verbose:
        off_diagonal = distance_np[~np.eye(N_generations, dtype=bool)]
        print(
            f"Branching factor: {branching_factor} cluster(s) over {N_generations} generations "
            f"(sizes: {cluster_sizes.tolist()})"
        )
        print(
            f"Pairwise JSD: "
            f"mean {off_diagonal.mean():.4f}, min {off_diagonal.min():.4f}, "
            f"max {off_diagonal.max():.4f}, threshold {jsd_threshold}"
        )

    return {
        "texts": texts,
        "continuation_texts": continuation_texts,
        "context_tokens": context_tokens,
        "generated_tokens": generated_tokens,
        "distributions": distributions,
        "branching_factor": branching_factor,
        "jsd_matrix": jsd_matrix,
        "cluster_labels": cluster_labels,
        "cluster_sizes": cluster_sizes,
    }


def pi_to_P_one_layer(pi: torch.Tensor, heads: list, model: HookedTransformer, chunk_size=128, temperature: float = 1.0, p: float = 0.9):
    layer = 0
    assert pi.numel() == model.cfg.d_vocab
    # pi = pi.view(1, model.cfg.d_vocab)

    ln1 = model.blocks[layer].ln1
    ln_final = model.ln_final

    E = model.W_E # shape: [vocab, d_model]
    Q = model.W_Q[layer, heads] # shape: [num_heads, d_model, d_head]
    b_Q = model.b_Q[layer, heads]
    K = model.W_K[layer, heads] # shape: [num_heads, d_model, d_head]
    b_K = model.b_K[layer, heads]
    V = model.W_V[layer, heads] # shape: [num_heads, d_model, d_head]
    b_V = model.b_V[layer, heads]
    O = model.W_O[layer, heads] # shape: [num_heads, d_head, d_model]
    b_O = model.b_O[layer]
    U = model.W_U # shape [d_model, vocab]
    b_U = model.b_U # shape [vocab]

    # QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
    # OV_core = V @ O  # shape [num_heads, d_model, d_model]

    ## Attention computation
    attention_resid_pre = ln1(E) # shape: [vocab, d_model]

    # pre compute all keys and values
    k = einops.einsum(attention_resid_pre, K, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    k = k + b_K.unsqueeze(1) # Broadcast bias over vocab: [num_heads, vocab, d_head]
    v = einops.einsum(attention_resid_pre, V, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    v = v + b_V.unsqueeze(1) # shape: [num_heads, vocab, d_head]

    # define chunk processing function for checkpointing
    def compute_P_chunk(pi, q_chunk, E_chunk):
        QK_raw = (q_chunk @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, chunk_size, vocab]
        QK_weighted = pi.view(1, 1, -1) * torch.exp(QK_raw) # shape: [num_heads, chunk_size, vocab]
        attention_weights = QK_weighted / QK_weighted.sum(dim=-1, keepdim=True) # shape: [num_heads, chunk_size, vocab], sums to 1 along dim=-1

        weighted_v = einops.einsum(attention_weights, v, "num_heads chunk_size vocab, num_heads vocab d_head -> num_heads chunk_size d_head") # shape: [num_heads, chunk_size, d_head]
        head_outputs = einops.einsum(weighted_v, O, "num_heads chunk_size d_head, num_heads d_head d_model -> num_heads chunk_size d_model")
        attention_output = head_outputs.sum(dim=0) + b_O # shape: [chunk_size, d_model]

        # Unembedding
        final_resid = attention_output + E_chunk # shape: [chunk_size, d_model]
        final_resid_norm = ln_final(final_resid)
        chunk_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [chunk_size, vocab]

        # apply top-p nucleus to each row independently
        chunk_probs_dense = torch.softmax(chunk_logits_matrix / temperature, dim=-1)
        sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs < p
        # We shift the mask right by 1 to strictly include the token that crosses the threshold 'p'
        sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        # scatter to create the mask back on the original vocabulary
        mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))

        # re-normalize rows to sum to 1
        row_sums = filtered_probs.sum(dim=-1, keepdim=True)
        chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)

        ## STE (Straight-Through Estimator) - copy gradients for all tokens
        chunk_probs = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense

        # ## Apply Entmax instead of softmax + top-p
        # chunk_probs = entmax15(chunk_logits_matrix / temperature, dim=-1)

        # convert the chunk to sparse representation
        chunk_sparse = chunk_probs.to_sparse()
        return chunk_sparse.indices(), chunk_sparse.values()

    # Lists to accumulate sparse tensor components
    all_indices = []
    all_values = []
    # compute queries and outputs in chunks
    for i in tqdm(range(0, model.cfg.d_vocab, chunk_size), desc="Chunking Sparse P:"):
        q = einops.einsum(attention_resid_pre[i:i+chunk_size], Q, "chunk_size d_model, num_heads d_model d_head -> num_heads chunk_size d_head")
        q = q + b_Q.unsqueeze(1) # Broadcast bias over vocab: [num_heads, chunk_size, d_head]
        E_chunk = E[i:i+chunk_size]
        
        # Apply checkpointing over chunks
        chunk_indices, chunk_values = checkpoint(
            compute_P_chunk,
            pi, q, E_chunk,
            use_reentrant=False
        )
        
        # Clone indices to avoid modifying the original view, then apply row offset
        indices = chunk_indices.clone()
        indices[0] += i # 'i' is the current row offset in the d_vocab loop

        all_indices.append(indices)
        all_values.append(chunk_values)

        # logit_matrix_chunks.append(chunk_logits_matrix)

    ## Build final sparse matrix
    # Concatenate all indices and values
    full_indices = torch.cat(all_indices, dim=1)
    full_values = torch.cat(all_values, dim=0)

    # Construct the final full d_vocab x d_vocab sparse transition matrix
    full_sparse_P = torch.sparse_coo_tensor(
        full_indices, 
        full_values, 
        size=(model.cfg.d_vocab, model.cfg.d_vocab)
    )

    # full_logits_matrix = torch.cat(logit_matrix_chunks)
    return full_sparse_P

def pi_to_pi_P_one_layer(pi: torch.Tensor, heads: list, model: HookedTransformer, chunk_size=512, temperature: float = 1.0, p: float = 0.9):
    layer = 0
    assert pi.numel() == model.cfg.d_vocab
    # pi = pi.view(1, model.cfg.d_vocab)

    ln1 = model.blocks[layer].ln1
    ln_final = model.ln_final

    E = model.W_E # shape: [vocab, d_model]
    Q = model.W_Q[layer, heads] # shape: [num_heads, d_model, d_head]
    b_Q = model.b_Q[layer, heads]
    K = model.W_K[layer, heads] # shape: [num_heads, d_model, d_head]
    b_K = model.b_K[layer, heads]
    V = model.W_V[layer, heads] # shape: [num_heads, d_model, d_head]
    b_V = model.b_V[layer, heads]
    O = model.W_O[layer, heads] # shape: [num_heads, d_head, d_model]
    b_O = model.b_O[layer]
    U = model.W_U # shape [d_model, vocab]
    b_U = model.b_U # shape [vocab]

    # QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
    # OV_core = V @ O  # shape [num_heads, d_model, d_model]

    ## Attention computation
    attention_resid_pre = ln1(E) # shape: [vocab, d_model]

    # pre compute all keys and values
    k = einops.einsum(attention_resid_pre, K, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    k = k + b_K.unsqueeze(1) # Broadcast bias over vocab: [num_heads, vocab, d_head]
    v = einops.einsum(attention_resid_pre, V, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    v = v + b_V.unsqueeze(1) # shape: [num_heads, vocab, d_head]

    # define chunk processing function for checkpointing
    def compute_P_chunk(pi, q_chunk, E_chunk):
        QK_raw = (q_chunk @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, chunk_size, vocab]
        QK_weighted = pi.view(1, 1, -1) * torch.exp(QK_raw) # shape: [num_heads, chunk_size, vocab]
        attention_weights = QK_weighted / QK_weighted.sum(dim=-1, keepdim=True) # shape: [num_heads, chunk_size, vocab], sums to 1 along dim=-1

        weighted_v = einops.einsum(attention_weights, v, "num_heads chunk_size vocab, num_heads vocab d_head -> num_heads chunk_size d_head") # shape: [num_heads, chunk_size, d_head]
        head_outputs = einops.einsum(weighted_v, O, "num_heads chunk_size d_head, num_heads d_head d_model -> num_heads chunk_size d_model")
        attention_output = head_outputs.sum(dim=0) + b_O # shape: [chunk_size, d_model]

        # Unembedding
        final_resid = attention_output + E_chunk # shape: [chunk_size, d_model]
        final_resid_norm = ln_final(final_resid)
        chunk_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [chunk_size, vocab]

        # apply top-p nucleus to each row independently
        chunk_probs_dense = torch.softmax(chunk_logits_matrix / temperature, dim=-1)
        sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs < p
        # We shift the mask right by 1 to strictly include the token that crosses the threshold 'p'
        sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        # scatter to create the mask back on the original vocabulary
        mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))

        # re-normalize rows to sum to 1
        row_sums = filtered_probs.sum(dim=-1, keepdim=True)
        chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)

        ## STE (Straight-Through Estimator) - copy gradients for all tokens
        chunk_probs = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense

        # ## Apply Entmax instead of softmax + top-p
        # chunk_probs = entmax15(chunk_logits_matrix / temperature, dim=-1)

        return chunk_probs # shape: [chunk_size, vocab]
    
    def compute_pi_P_chunk(pi, q_chunk, E_chunk, pi_chunk):
        # compute the dense chunk of the P matrix
        chunk_probs = compute_P_chunk(pi, q_chunk, E_chunk)
        return pi_chunk.view(1, -1) @ chunk_probs

    # Lists to accumulate the vector-matrix product
    pi_P_accum = torch.zeros_like(pi)

    # compute queries and outputs in chunks
    for i in tqdm(range(0, model.cfg.d_vocab, chunk_size), desc="Chunking pi_P"):
        q = einops.einsum(attention_resid_pre[i:i+chunk_size], Q, "chunk_size d_model, num_heads d_model d_head -> num_heads chunk_size d_head")
        q = q + b_Q.unsqueeze(1) # Broadcast bias over vocab: [num_heads, chunk_size, d_head]
        E_chunk = E[i:i+chunk_size]
        pi_chunk = pi[i:i+chunk_size]
        
        # Apply checkpointing over chunks
        pi_P_chunk = checkpoint(
            compute_pi_P_chunk,
            pi, q, E_chunk, pi_chunk,
            use_reentrant=False
        )

        pi_P_accum += pi_P_chunk.squeeze(0)

    return pi_P_accum

def pi_to_pi_P_one_layer_topk(pi: torch.Tensor, heads: list, model: HookedTransformer, top_tokens, temperature: float = 1.0, p: float = 0.9):
    layer = 0
    assert pi.numel() == model.cfg.d_vocab
    # pi = pi.view(1, model.cfg.d_vocab)

    ln1 = model.blocks[layer].ln1
    ln_final = model.ln_final

    E = model.W_E # shape: [vocab, d_model]
    Q = model.W_Q[layer, heads] # shape: [num_heads, d_model, d_head]
    b_Q = model.b_Q[layer, heads]
    K = model.W_K[layer, heads] # shape: [num_heads, d_model, d_head]
    b_K = model.b_K[layer, heads]
    V = model.W_V[layer, heads] # shape: [num_heads, d_model, d_head]
    b_V = model.b_V[layer, heads]
    O = model.W_O[layer, heads] # shape: [num_heads, d_head, d_model]
    b_O = model.b_O[layer]
    U = model.W_U # shape [d_model, vocab]
    b_U = model.b_U # shape [vocab]

    # QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
    # OV_core = V @ O  # shape [num_heads, d_model, d_model]

    ## Attention computation
    attention_resid_pre = ln1(E) # shape: [vocab, d_model]

    # pre compute all keys and values
    k = einops.einsum(attention_resid_pre, K, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    k = k + b_K.unsqueeze(1) # Broadcast bias over vocab: [num_heads, vocab, d_head]
    v = einops.einsum(attention_resid_pre, V, "vocab d_model, num_heads d_model d_head -> num_heads vocab d_head")
    v = v + b_V.unsqueeze(1) # shape: [num_heads, vocab, d_head]

    # define chunk processing function for checkpointing
    def compute_P_chunk(pi, q_chunk, E_chunk):
        QK_raw = (q_chunk @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, chunk_size, vocab]
        QK_weighted = pi.view(1, 1, -1) * torch.exp(QK_raw) # shape: [num_heads, chunk_size, vocab]
        attention_weights = QK_weighted / QK_weighted.sum(dim=-1, keepdim=True) # shape: [num_heads, chunk_size, vocab], sums to 1 along dim=-1
        # print(f"attention weights:\n{attention_weights.topk(5)}")

        weighted_v = einops.einsum(attention_weights, v, "num_heads chunk_size vocab, num_heads vocab d_head -> num_heads chunk_size d_head") # shape: [num_heads, chunk_size, d_head]
        head_outputs = einops.einsum(weighted_v, O, "num_heads chunk_size d_head, num_heads d_head d_model -> num_heads chunk_size d_model")
        attention_output = head_outputs.sum(dim=0) + b_O # shape: [chunk_size, d_model]

        # Unembedding
        final_resid = attention_output + E_chunk # shape: [chunk_size, d_model]
        final_resid_norm = ln_final(final_resid)
        chunk_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [chunk_size, vocab]

        # apply top-p nucleus to each row independently
        chunk_probs_dense = torch.softmax(chunk_logits_matrix / temperature, dim=-1)
        sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs < p
        # We shift the mask right by 1 to strictly include the token that crosses the threshold 'p'
        sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        # scatter to create the mask back on the original vocabulary
        mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))

        # re-normalize rows to sum to 1
        row_sums = filtered_probs.sum(dim=-1, keepdim=True)
        chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)

        ## STE (Straight-Through Estimator) - copy gradients for all tokens
        chunk_probs = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense

        # ## Apply Entmax instead of softmax + top-p
        # chunk_probs = entmax15(chunk_logits_matrix / temperature, dim=-1)

        return chunk_probs # shape: [chunk_size, vocab]
    
    def compute_pi_P_chunk(pi, q_chunk, E_chunk, pi_chunk):
        # compute the dense chunk of the P matrix
        chunk_probs = compute_P_chunk(pi, q_chunk, E_chunk)
        # print(f"top tokens:\n{top_tokens}")
        # print(f"P_probs:\n{chunk_probs.topk(5)}")
        return pi_chunk.view(1, -1) @ chunk_probs

    # Lists to accumulate the vector-matrix product
    # pi_P_accum = torch.zeros_like(pi)

    # compute queries and outputs in chunks
    q = einops.einsum(attention_resid_pre[top_tokens], Q, "chunk_size d_model, num_heads d_model d_head -> num_heads chunk_size d_head")
    q = q + b_Q.unsqueeze(1) # Broadcast bias over vocab: [num_heads, chunk_size, d_head]
    E_chunk = E[top_tokens]
    pi_chunk = pi[top_tokens]
    
    # Apply checkpointing over chunks
    # pi_P_chunk = checkpoint(
    #     compute_pi_P_chunk,
    #     pi, q, E_chunk, pi_chunk,
    #     use_reentrant=False
    # )

    pi_P = compute_pi_P_chunk(pi, q, E_chunk, pi_chunk).squeeze(0)

    # pi_P_accum += pi_P_chunk.squeeze(0)

    return pi_P

def pi_from_context(tokens: torch.Tensor, vocab_size=48262):
    pi = torch.zeros(vocab_size, device=tokens.device)
    n_context = tokens.numel()

    values, counts = torch.unique(tokens, return_counts=True)
    pi[values] = counts.float()
    pi = pi / n_context
    return pi

def context_from_pi(pi: torch.Tensor, N: int):
    # pi[1] = 0 # zero out BOS probability
    threshold = 1/(2*N)
    pi_local = pi.clone()
    pi_local[pi_local < threshold] = 0
    tokens = torch.multinomial(pi_local, num_samples=N, replacement=True)
    # tokens_BOS_prepended = torch.cat((torch.tensor([1]), tokens), dim=0) 
    return tokens

def pi_t_from_context(tokens: torch.Tensor, vocab_size=48262):
    n_context = tokens.numel()

    pi_t = [pi_from_context(tokens[:t], vocab_size=vocab_size).view(1, -1) for t in range(1,n_context+1)]
    return torch.cat(pi_t, dim=0)

def loss_from_pi_t(pi_t: torch.Tensor, model, step=10, scale=1e5, chunk_size=1024, heads=[0,1,2,3,4,5,6,7], p=0.9, temperature=1):
    T, vocab_size = pi_t.shape

    loss = []
    for t in tqdm(range(0, T, step), desc="Computing loss"):
        top_tokens = pi_t[t].topk(chunk_size).indices
        pi_P_t = pi_to_pi_P_one_layer_topk(pi_t[t], heads, model, top_tokens, temperature, p).squeeze().clamp(min=1e-15)
        loss.append(scale * JSD(pi_t[t].clamp(min=1e-15), pi_P_t).item())
        
    return loss

def get_kgram_distribution_from_tokens(tokens, K: int, offset: int = 1, pad_id: int = 2, device='cpu'):
    """
    Computes the K-gram distribution from a sequence of tokens in a sparse format.
    
    Args:
        tokens: A 1D torch.Tensor of token IDs, or a list of integers.
        K: The size of the context window (K-gram length).
        offset: the index of the final token of the first K-gram
        pad_id: the pad_id of the model
        device: The PyTorch device to place the output tensors on.
        
    Returns:
        values: 1D tensor of shape [N_active] containing the probability mass (pi_active).
        indices: 2D tensor of shape [N_active, K] containing the K-gram token IDs.
    """
    if not isinstance(tokens, torch.Tensor):
        tokens = torch.tensor(tokens, dtype=torch.long, device=device)
    else:
        tokens = tokens.to(device)
        
    # Ensure it's a 1D sequence
    if tokens.dim() > 1:
        tokens = tokens.squeeze()

    if offset >= K - 1:
        tokens = tokens[offset - K + 1:]
    else:
        # pad with pad_id K - offset - 1 times
        tokens = torch.cat([torch.full((K - offset - 1,), pad_id), tokens])
        
    seq_len = tokens.shape[0]
    if seq_len < K:
        raise ValueError(f"Sequence length ({seq_len}) is smaller than K ({K}). No valid K-grams.")

    # 1. Extract all full/valid K-grams using a sliding window
    # .unfold(dimension, size, step) creates a view of all sequential K-grams
    all_kgrams = tokens.unfold(0, K, 1)  # Shape: [seq_len - K + 1, K]

    # 2. Count the occurrences of each unique K-gram
    # dim=0 groups by the entire K-gram row
    unique_kgrams, counts = torch.unique(all_kgrams, dim=0, return_counts=True)

    # 3. Convert counts into a proper probability distribution (simplex)
    values = counts.float() / counts.sum()

    # values represents `pi_active` (the macroscopic density measure)
    # unique_kgrams represents the active tracking keys
    return values, unique_kgrams

def get_sparse_kgram_dist_from_string(text: str, tokenizer, K: int, device='cpu'):
    """
    Wrapper to directly process a string into a sparse K-gram distribution.
    """
    # Note: add_special_tokens=False is usually preferred here so you don't 
    # accidentally inject BOS/EOS tokens into the middle of your sliding windows 
    # unless you are explicitly tracking the attention sink behavior.
    tokens = tokenizer.encode(text, add_special_tokens=False)
    
    return get_kgram_distribution_from_tokens(tokens, K, device=device)

In [ ]:
import torch
import einops
import numpy as np
from torch.utils.checkpoint import checkpoint

# =====================================================================
# 0. UTILITY AND HELPER FUNCTIONS
# =====================================================================

def extract_bos_sink(model: HookedTransformer) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Extracts the Key and Value representations of the <BOS> token at position 0.
    This acts as the static "attention sink" for the SDE framework.
    
    Args:
        model: The TransformerLens HookedTransformer instance.
        
    Returns:
        sink_k: Tensor of shape [n_layers, n_heads, d_head]
        sink_v: Tensor of shape [n_layers, n_heads, d_head]
    """
    model.eval()
    
    # 1. Isolate the <BOS> token
    bos_id = model.tokenizer.bos_token_id
    if bos_id is None:
        raise ValueError("Model tokenizer does not have a defined <BOS> token.")
        
    # Shape: [batch=1, pos=1]
    tokens = torch.tensor([[bos_id]], device=model.cfg.device)
    
    # 2. Forward pass with targeted caching
    # We use a names_filter to save memory, only catching Keys and Values.
    with torch.no_grad():
        _, cache = model.run_with_cache(
            tokens, 
            names_filter=lambda name: name.endswith("hook_k") or name.endswith("hook_v")
        )
        
    # 3. Extract and reshape
    # stack_activation naturally returns shape: [n_layers, batch, pos, n_heads, d_head]
    # We squeeze out the batch (dim 1) and pos (dim 1 again after first squeeze) dimensions.
    sink_k = cache.stack_activation("k").squeeze(1).squeeze(1)
    sink_v = cache.stack_activation("v").squeeze(1).squeeze(1)
    
    return sink_k, sink_v

def extract_frozen_sigma(model, dummy_tokens, layer_idx):
    with torch.no_grad():
        # Run a normal forward pass and cache the activations
        _, cache = model.run_with_cache(dummy_tokens)
    
        # Get the raw residual stream right before LayerNorm
        resid_pre = cache[f"blocks.{layer_idx}.hook_resid_pre"]
    
        # Calculate the exact standard deviation used by PyTorch LayerNorm
        eps = model.blocks[layer_idx].ln1.eps
        variance = resid_pre.var(dim=-1, keepdim=True, unbiased=False)
        sigma = torch.sqrt(variance + eps)
    
    # Average across the batch and sequence dimension to get our frozen constant
    return sigma.mean().item()

def estimate_c_far_for_head(
    model, 
    layer_idx: int, 
    head_idx: int, 
    K_i: int, 
    ln_avg_sigma: float, 
    seq_len: int = 800,
    pos_offset: int = 100 # Matches the offset from your abstract_forward_pass
) -> float:
    """
    Empirically estimates the baseline positional exponentiated mass (C_far) 
    for tokens outside the local sliding window, isolated from semantic noise.
    """
    device = model.cfg.device
    d_head = model.cfg.d_head
    
    # 1. Extract Absolute Positional Embeddings
    # We grab a sufficiently long sequence to get a stable statistical average
    pos_embeds = model.W_pos[pos_offset : pos_offset + seq_len, :] # [seq_len, d_model]
    
    # 2. LayerNorm Linearization (Strictly isolated to positional vectors)
    centered_pos = pos_embeds - pos_embeds.mean(dim=-1, keepdim=True)
    pos_pre = centered_pos / ln_avg_sigma
    
    ln_module = model.blocks[layer_idx].ln1
    if hasattr(ln_module, 'w') and ln_module.w is not None:
        pos_pre = pos_pre * ln_module.w
        
    # 3. Decoupled QK Projections for this specific head
    # Note: We intentionally omit b_Q and b_K here, as they are absorbed into the semantic stream!
    W_Q = model.W_Q[layer_idx, head_idx] # [d_model, d_head]
    W_K = model.W_K[layer_idx, head_idx] # [d_model, d_head]
    
    q_p = pos_pre @ W_Q # [seq_len, d_head]
    k_p = pos_pre @ W_K # [seq_len, d_head]
    
    # 4. Compute the pure Positional Attention Score Matrix
    score_matrix = (q_p @ k_p.T) / np.sqrt(d_head) # [seq_len, seq_len]
    
    # 5. Masking
    q_idx = torch.arange(seq_len, device=device).view(-1, 1) # Column vector
    k_idx = torch.arange(seq_len, device=device).view(1, -1) # Row vector
    distance = q_idx - k_idx
    
    # Valid global background tokens must be:
    # a) In the past (distance > 0)
    # b) Outside the local sliding window (distance >= K_i)
    # c) NOT the BOS token / Sink (k_idx != 0) 
    #    *Note: We assume k_idx=0 is the sink relative to our extracted block
    
    valid_background_mask = (distance >= K_i) & (k_idx != 0)
    
    # Extract only the valid far-field raw scores
    far_scores = score_matrix[valid_background_mask]
    
    if len(far_scores) == 0:
        # Edge case: If K_i is larger than seq_len, there is no "far" field.
        return 0.0
    
    # 6. Calculate the Expected Exponentiated Mass
    # We take the mean of the EXPONENTIATED scores, because C_far acts 
    # directly on the unnormalized probability mass M_global.
    c_far = torch.exp(far_scores).mean()
    
    return c_far.item()

def estimate_L_ctx_for_head(
    model, 
    layer_idx: int, 
    head_idx: int, 
    ln_avg_sigma: float, 
    K_i: int,
    seq_len: int = 800,
    pos_offset: int = 100
) -> int:
    """
    Empirically estimates the Integration Horizon (L_ctx) for a given head 
    by calculating where the pure positional attention signal decays into background noise.
    """
    device = model.cfg.device
    d_head = model.cfg.d_head
    
    # 1. Extract and Linearize Absolute Positional Embeddings
    pos_embeds = model.W_pos[pos_offset : pos_offset + seq_len, :]
    centered_pos = pos_embeds - pos_embeds.mean(dim=-1, keepdim=True)
    pos_pre = centered_pos / ln_avg_sigma
    
    ln_module = model.blocks[layer_idx].ln1
    if hasattr(ln_module, 'w') and ln_module.w is not None:
        pos_pre = pos_pre * ln_module.w
        
    # 2. Decoupled QK Projections (No bias)
    W_Q = model.W_Q[layer_idx, head_idx]
    W_K = model.W_K[layer_idx, head_idx]
    
    q_p = pos_pre @ W_Q 
    k_p = pos_pre @ W_K 
    
    # 3. Compute the Full Pure Positional Score Matrix
    score_matrix = (q_p @ k_p.T) / np.sqrt(d_head)
    
    # CRITICAL: Banish the Attention Sink (Position 0)
    # The sink is an anomaly that ruins distance-based calculations.
    score_matrix[:, 0] = -1e4
    
    # 4. Extract the Expected Exponentiated Mass Profile E(d)
    # d represents the relative distance: d = q_idx - k_idx
    E_d = torch.zeros(seq_len - 1, device=device)
    
    for d in range(1, seq_len):
        # Extract the sub-diagonal representing distance 'd'
        diag_raw_scores = torch.diagonal(score_matrix, offset=-d)
        
        # We average the EXPONENTIATED mass, just like in the SDE equations
        E_d[d-1] = torch.exp(diag_raw_scores).mean()
        
    # 5. Establish the Thermodynamic Noise Floor
    # We use the back-half of the sequence as our guaranteed "Far-Field"
    far_field_start = seq_len // 2
    far_field_E = E_d[far_field_start:]
    
    mu_far = far_field_E.mean()
    sigma_far = far_field_E.std()
    
    # 6. Detect the Horizon (Signal > 3 Standard Deviations above Noise)
    # We add 1e-6 to sigma to prevent division-by-zero errors in perfectly flat heads
    threshold = mu_far + 3 * (sigma_far + 1e-6)
    
    # Find all distances where the signal breaches the noise floor
    active_distances = torch.where(E_d > threshold)[0]
    
    if len(active_distances) == 0:
        # A purely semantic head has no significant positional geometry. 
        # It integrates over the entire context.
        L_ctx = seq_len
    else:
        # The true horizon is the maximum distance + 1 (since tensor is 0-indexed)
        # We add a small safety buffer (e.g., 2 tokens) to capture the exact tail
        # TODO: make the last item that doesn't pass the threshold?
        L_ctx = active_distances.max().item() + 1 + 2
        
    # Ensure L_ctx is logically bounded
    # It cannot be smaller than your explicit K-gram local window size (K_i)
    # It cannot be larger than the total sequence length
    L_ctx = max(min(L_ctx, seq_len), K_i + 1)
    
    return L_ctx

def profile_thermodynamic_heads(
    model,
    K_list: list[int],
    num_prompts: int = 600,
    seq_len: int = 1024,
    batch_size: int = 4
):
    """
    Computes the Integration Horizon (L_ctx) and Background Density (C_far) 
    for every head in a TransformerLens model using the Random Ensemble Method.
    
    Args:
        model: TransformerLens HookedTransformer instance.
        K_list: List of ints, the local explicit K-gram window sizes per layer.
        num_prompts: Number of random prompts to average over (cancels semantic noise).
        seq_len: The sequence length for the random prompts.
        batch_size: Batch size for forward passes to prevent CUDA OOM.
        
    Returns:
        L_ctx_list: list of Tensors [num_heads], one per layer.
        C_far_list: list of Tensors [num_heads], one per layer.
    """
    device = model.cfg.device
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    vocab_size = model.cfg.d_vocab
    bos_id = model.tokenizer.bos_token_id if model.tokenizer is not None else 1
    
    print(f"Generating Random Ensemble: {num_prompts} prompts of length {seq_len}...")
    
    # 1. Generate Random Prompts (Isotropic Semantic Noise)
    # We sample randomly from the vocabulary to ensure E[Semantic] ~ 0.
    tokens = torch.randint(1, vocab_size, (num_prompts, seq_len), device=device)
    # tokens = torch.ones((num_prompts, seq_len), device=device).long() * 1000
    # Enforce the BOS token at position 0 for all prompts to create the strict attention sink
    tokens[:, 0] = bos_id
    
    # Dictionary to hold the summed attention patterns per layer
    summed_patterns = {
        l: torch.zeros((n_heads, seq_len, seq_len), device=device) 
        for l in range(n_layers)
    }
    
    print("Running forward passes and extracting attention patterns...")
    # 2. Batched Forward Passes to avoid OOM
    with torch.no_grad():
        for b in tqdm(range(0, num_prompts, batch_size)):
            batch_tokens = tokens[b : b + batch_size]
            
            # We explicitly only cache the attention patterns to save memory
            _, cache = model.run_with_cache(
                batch_tokens, 
                names_filter=lambda name: "attn.hook_pattern" in name
            )
            
            for l in range(n_layers):
                hook_name = f"blocks.{l}.attn.hook_pattern"
                # hook_pattern shape: [batch, n_heads, seq_q, seq_k]
                # We sum across the batch dimension immediately
                summed_patterns[l] += cache[hook_name].sum(dim=0)
                
            del cache # Free computational graph memory
            torch.cuda.empty_cache()

    # 3. Analyze the Averaged Thermodynamics
    L_ctx_list = []
    C_far_list = []
    
    print("Computing SNR horizons and thermodynamic baselines...")
    
    for l in range(n_layers):
        K_i = K_list[l]
        
        # Calculate the mean post-softmax probability pattern over the ensemble
        mean_pattern = summed_patterns[l] / num_prompts # [n_heads, seq_len, seq_len]
        
        L_ctx_layer = torch.zeros(n_heads, dtype=torch.long, device=device)
        C_far_layer = torch.zeros(n_heads, dtype=torch.float32, device=device)
        
        for h in range(n_heads):
            A = mean_pattern[h] # [seq_len, seq_len]
            
            # We track E(d) for distances d from 1 to seq_len - 2
            # (We stop at seq_len - 2 because at d = seq_len - 1, the only key is k=0, 
            # which is the BOS sink, and we MUST exclude the sink).
            max_d = seq_len - 1
            E_d = torch.zeros(max_d - K_i, device=device)
            
            for d in range(1+K_i, max_d):
                # Extract the sub-diagonal (q - k = d)
                diag = torch.diagonal(A, offset=-d)
                
                # CRITICAL: The first element of every offset=-d diagonal is (q=d, k=0).
                # This is the interaction with the BOS token. 
                # We MUST slice it out (diag[1:]) so the sink doesn't ruin the baseline density!
                if len(diag) > 1:
                    E_d[d-K_i-1] = diag[1:].mean()
            
            # Define the "Far Field" as the second half of the sequence
            far_field_start = seq_len // 2
            far_field_E = E_d[far_field_start:]
            
            # Calculate the Thermodynamic Noise Floor
            mu_far = far_field_E.mean()
            sigma_far = far_field_E.std()
            
            # Detect the Horizon (Signal must be 3 standard deviations above noise)
            # Add tiny epsilon to prevent division/thresholding errors on perfectly flat heads
            threshold = mu_far + 10 * (sigma_far + 1e-7)
            
            active_distances = torch.where(E_d > threshold)[0]
            
            if len(active_distances) == 0:
                # Pure Semantic Head: Positional geometry is entirely uniform noise
                L_ctx = seq_len
            else:
                # True horizon is max active index + 1 (0-indexed) + a 2-token buffer
                L_ctx = active_distances.max().item() + 1 + 2 + K_i
                
            # Clamp the values based on fundamental boundaries
            L_ctx = max(min(L_ctx, seq_len), K_i + 1)
            
            L_ctx_layer[h] = L_ctx
            C_far_layer[h] = mu_far
            
        L_ctx_list.append(L_ctx_layer)
        C_far_list.append(C_far_layer)
        
    print("Profiling Complete.")
    return L_ctx_list, C_far_list

@torch.no_grad()
def profile_deep_heads_empirical(
    model, 
    corpus_tokens: torch.Tensor, 
    ensemble_size: int, 
    seq_len: int, 
    batch_size: int = 8
):
    """
    Empirically profiles all attention heads in a Transformer to extract 
    their thermodynamic SDE parameters (mu and sigma) across relative distances.
    
    Args:
        model: TransformerLens HookedTransformer.
        corpus_tokens: A massive 1D tensor of token IDs (e.g., flattened Wikitext).
        ensemble_size: Total number of sequences to process for the statistical ensemble.
        seq_len: The context window length (L) to track.
        batch_size: Number of sequences to process simultaneously.
        
    Returns:
        mu_tensor: (n_layers, n_heads, seq_len) - The Deep Positional Baseline
        sigma_tensor: (n_layers, n_heads, seq_len) - The Contextual Semantic Spread
        mu_bos_tensor: (n_layers, n_heads, seq_len) - The BOS Positional Baseline
        sigma_bos_tensor: (n_layers, n_heads, seq_len) - The BOS Semantic Spread
    """
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    device = model.cfg.device

    # =====================================================================
    # 1. INITIALIZE RUNNING STATISTIC TENSORS
    # We track the sum and sum of squares to compute exact variance over 
    # hundreds of thousands of contextualized token interactions.
    # =====================================================================
    sum_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)
    sum_sq_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)
    
    sum_bos_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)
    sum_sq_bos_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)

    # We must explicitly count valid interactions per distance, because 
    # longer distances appear less frequently in a fixed seq_len window.
    counts = torch.zeros(seq_len, dtype=torch.float64, device=device)
    counts_bos = torch.zeros(seq_len, dtype=torch.float64, device=device)

    # Extract the true BOS token ID from the model's tokenizer
    bos_token_id = model.tokenizer.bos_token_id
    if bos_token_id is None:
        raise ValueError("Model tokenizer does not have a defined BOS token.")

    # We want final sequences of length `seq_len`.
    # Therefore, we chunk the raw corpus into pieces of length `seq_len - 1`
    chunk_len = seq_len - 1
    total_tokens_needed = ensemble_size * chunk_len
    
    if corpus_tokens.shape[0] < total_tokens_needed:
        raise ValueError("Corpus is too small for the requested ensemble size and seq_len.")
    
    # Shape: [ensemble_size, seq_len - 1]
    raw_chunks = corpus_tokens[:total_tokens_needed].view(ensemble_size, chunk_len)

    # Create a column vector of pure BOS tokens
    # Shape: [ensemble_size, 1]
    bos_column = torch.full(
        (ensemble_size, 1), 
        bos_token_id, 
        dtype=raw_chunks.dtype, 
        device=raw_chunks.device
    )

    # Concatenate the BOS token to the front of every chunk
    # Shape: [ensemble_size, seq_len]
    prompts = torch.cat([bos_column, raw_chunks], dim=1)

    # Filter to explicitly capture ONLY the pre-softmax attention scores
    attn_hook_filter = lambda name: name.endswith("attn_scores")

    print(f"Running Empirical Profiling: {ensemble_size} sequences of length {seq_len}...")
    
    # =====================================================================
    # 2. THE STATISTICAL ENSEMBLE LOOP
    # =====================================================================
    for i in tqdm(range(0, ensemble_size, batch_size)):
        batch = prompts[i : i + batch_size].to(device)
        actual_batch_size = batch.shape[0]

        # Run forward pass, caching only the pre-softmax logits
        _, cache = model.run_with_cache(batch, names_filter=attn_hook_filter)

        for layer in range(n_layers):
            hook_name = f"blocks.{layer}.attn.hook_attn_scores"
            
            # Shape: [batch, n_heads, seq_len, seq_len]
            attn_scores = cache[hook_name] 

            # =================================================================
            # 3. RELATIVE DISTANCE EXTRACTION & BOS GUILLOTINE
            # =================================================================
            for d in range(seq_len):
                # Extract the d-th sub-diagonal (lower triangle: i - j = d)
                # Shape: [batch, n_heads, seq_len - d]
                diag = torch.diagonal(attn_scores, offset=-d, dim1=-2, dim2=-1)
                
                # BOS EXTRACTION
                # The first element of the diagonal (j=0) is the query-to-BOS score.
                # Relative distance d corresponds exactly to absolute query position T.
                bos_logits = diag[..., 0]
                sum_bos_logits[layer, :, d] += bos_logits.sum(dim=0)
                sum_sq_bos_logits[layer, :, d] += (bos_logits ** 2).sum(dim=0)
                
                if layer == 0:
                    counts_bos[d] += actual_batch_size

                # THE BOS GUILLOTINE:
                # The very first element of any sub-diagonal corresponds to j=0.
                # Because the BOS token is a massive semantic sink, including it 
                # artificially explodes the positional variance. We mathematically 
                # sever it by slicing [..., 1:].
                if diag.shape[-1] > 1:
                    diag_no_bos = diag[..., 1:] 
                    
                    # Aggregate running statistics
                    # Sum across batch and the sequence positions
                    sum_logits[layer, :, d] += diag_no_bos.sum(dim=(0, 2))
                    sum_sq_logits[layer, :, d] += (diag_no_bos ** 2).sum(dim=(0, 2))

                    # Update total count of evaluated logits for distance d
                    # (actual_batch_size) * (elements_in_diagonal - 1_for_BOS)
                    if layer == 0: 
                        counts[d] += actual_batch_size * (diag.shape[-1] - 1)

    # =====================================================================
    # 4. COMPUTE MACROSCOPIC PARAMETERS
    # =====================================================================
    # Prevent division by zero for distances that had 0 valid interactions 
    # (e.g., d = seq_len - 1, which only has 1 element that got sliced out by the Guillotine)
    safe_counts = counts.clamp(min=1).view(1, 1, -1)
    safe_counts_bos = counts_bos.clamp(min=1).view(1, 1, -1)

    # Content Parameters
    mu = sum_logits / safe_counts
    expected_sq = sum_sq_logits / safe_counts
    variance = torch.relu(expected_sq - (mu ** 2))
    sigma = torch.sqrt(variance)

    # BOS Parameters
    mu_bos = sum_bos_logits / safe_counts_bos
    expected_sq_bos = sum_sq_bos_logits / safe_counts_bos
    variance_bos = torch.relu(expected_sq_bos - (mu_bos ** 2))
    sigma_bos = torch.sqrt(variance_bos)

    # Cast back to standard precision for further SDE computations
    return mu.float(), sigma.float(), mu_bos.float(), sigma_bos.float()

@torch.no_grad()
def profile_deep_heads_empirical_instruct(
    model: HookedTransformer, 
    prompts: torch.Tensor,  # Now accepts the pre-formatted [ensemble_size, seq_len] tensor directly
    seq_len: int, 
    batch_size: int = 8
):
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    device = model.cfg.device
    ensemble_size = prompts.shape[0]

    # Verify input dimensions match the requested sequence length
    if prompts.shape[1] != seq_len:
        raise ValueError(f"Input prompts have length {prompts.shape[1]}, expected {seq_len}.")

    # 1. INITIALIZE RUNNING STATISTIC TENSORS
    sum_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)
    sum_sq_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)
    
    sum_bos_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)
    sum_sq_bos_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device=device)

    counts = torch.zeros(seq_len, dtype=torch.float64, device=device)
    counts_bos = torch.zeros(seq_len, dtype=torch.float64, device=device)

    # Filter to explicitly capture ONLY the pre-softmax attention scores
    attn_hook_filter = lambda name: name.endswith("attn_scores")

    print(f"Running Empirical Profiling: {ensemble_size} intact chat sequences of length {seq_len}...")
    
    # 2. STATISTICAL ENSEMBLE LOOP
    for i in tqdm(range(0, ensemble_size, batch_size)):
        batch = prompts[i : i + batch_size].to(device)
        actual_batch_size = batch.shape[0]

        # Run forward pass, caching only pre-softmax logits to conserve VRAM
        _, cache = model.run_with_cache(batch, names_filter=attn_hook_filter, return_type=None)

        for layer in range(n_layers):
            hook_name = f"blocks.{layer}.attn.hook_attn_scores"
            # Shape: [batch, n_heads, seq_len, seq_len]
            attn_scores = cache[hook_name] 

            # 3. RELATIVE DISTANCE EXTRACTION & BOS GUILLOTINE
            for d in range(seq_len):
                # Extract d-th sub-diagonal (lower triangle: i - j = d)
                # Shape: [batch, n_heads, seq_len - d]
                diag = torch.diagonal(attn_scores, offset=-d, dim1=-2, dim2=-1)
                
                # BOS EXTRACTION (j=0 is always the first element of the diagonal)
                bos_logits = diag[..., 0]
                sum_bos_logits[layer, :, d] += bos_logits.sum(dim=0, dtype=torch.float64)
                sum_sq_bos_logits[layer, :, d] += (bos_logits.double() ** 2).sum(dim=0)
                
                if layer == 0:
                    counts_bos[d] += actual_batch_size

                # THE BOS GUILLOTINE: Sever j=0 to prevent massive attention sinks 
                # from artificially exploding the positional variance of content tokens.
                if diag.shape[-1] > 1:
                    diag_no_bos = diag[..., 1:] 
                    
                    sum_logits[layer, :, d] += diag_no_bos.sum(dim=(0, 2), dtype=torch.float64)
                    sum_sq_logits[layer, :, d] += (diag_no_bos.double() ** 2).sum(dim=(0, 2))

                    if layer == 0: 
                        counts[d] += actual_batch_size * (diag.shape[-1] - 1)

            del attn_scores # Clear cache explicitly to prevent memory fragmentation on long runs

        # Clear cache explicitly to prevent memory fragmentation on long runs
        del cache

        # De-fragment VRAM every 50 steps
        if i > 0 and i % 50 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    # 4. COMPUTE MACROSCOPIC PARAMETERS
    safe_counts = counts.clamp(min=1).view(1, 1, -1)
    safe_counts_bos = counts_bos.clamp(min=1).view(1, 1, -1)

    # Content Parameters
    mu = sum_logits / safe_counts
    expected_sq = sum_sq_logits / safe_counts
    variance = torch.relu(expected_sq - (mu ** 2))
    sigma = torch.sqrt(variance)

    # BOS Parameters
    mu_bos = sum_bos_logits / safe_counts_bos
    expected_sq_bos = sum_sq_bos_logits / safe_counts_bos
    variance_bos = torch.relu(expected_sq_bos - (mu_bos ** 2))
    sigma_bos = torch.sqrt(variance_bos)

    return mu.float(), sigma.float(), mu_bos.float(), sigma_bos.float()

### MULTI-GPU PROFILING VERSION
# =====================================================================
# 1. THE WORKER FUNCTION (Executes on an assigned GPU)
# =====================================================================
def _profiling_worker(
    gpu_id: int,
    model_name: str,
    prompts_chunk: torch.Tensor,
    seq_len: int,
    batch_size: int
):
    """
    Independent worker that loads the model on a specific GPU, profiles 
    its assigned chunk of data, and returns raw FP64 sums and counts.
    """
    device = f"cuda:{gpu_id}"
    print(f"[GPU {gpu_id}] Initializing {model_name} in bfloat16 on {device}...")
    
    # Load model independently on this worker's GPU
    # bfloat16 halves static VRAM without altering interpretability metrics
    model = HookedTransformer.from_pretrained(
        model_name,
        device=device,
        dtype=torch.bfloat16,
        fold_ln=False,
        center_writing_weights=False
    )
    
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    ensemble_size = prompts_chunk.shape[0]
    
    # Initialize local running statistics strictly on the CPU
    # Offloading accumulators to CPU prevents VRAM creep over deep ensemble runs
    sum_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device="cpu")
    sum_sq_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device="cpu")
    sum_bos_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device="cpu")
    sum_sq_bos_logits = torch.zeros((n_layers, n_heads, seq_len), dtype=torch.float64, device="cpu")
    
    counts = torch.zeros(seq_len, dtype=torch.float64, device="cpu")
    counts_bos = torch.zeros(seq_len, dtype=torch.float64, device="cpu")
    
    attn_hook_filter = lambda name: name.endswith("attn_scores")
    
    print(f"[GPU {gpu_id}] Profiling {ensemble_size} sequences (Batch size: {batch_size})...")
    
    with torch.no_grad():
        for i in range(0, ensemble_size, batch_size):
            batch = prompts_chunk[i : i + batch_size].to(device)
            actual_batch_size = batch.shape[0]
            
            # return_type=None bypasses the massive 128k Llama vocabulary projection,
            # saving over 1GB of temporary VRAM per forward pass
            _, cache = model.run_with_cache(
                batch, 
                names_filter=attn_hook_filter, 
                return_type=None
            )
            
            for layer in range(n_layers):
                hook_name = f"blocks.{layer}.attn.hook_attn_scores"
                attn_scores = cache[hook_name]
                
                for d in range(seq_len):
                    # Extract d-th sub-diagonal (lower triangle: i - j = d)
                    diag = torch.diagonal(attn_scores, offset=-d, dim1=-2, dim2=-1)
                    
                    # 1. BOS EXTRACTION (j=0 is always index 0 of the diagonal)
                    bos_logits = diag[..., 0]
                    sum_bos_logits[layer, :, d] += bos_logits.sum(dim=0, dtype=torch.float64).cpu()
                    sum_sq_bos_logits[layer, :, d] += (bos_logits.double() ** 2).sum(dim=0).cpu()
                    
                    if layer == 0:
                        counts_bos[d] += actual_batch_size
                        
                    # 2. CONTENT EXTRACTION (BOS Guillotine: slice [..., 1:])
                    # Severs the BOS attention sink so it doesn't inflate positional variance
                    if diag.shape[-1] > 1:
                        diag_no_bos = diag[..., 1:]
                        sum_logits[layer, :, d] += diag_no_bos.sum(dim=(0, 2), dtype=torch.float64).cpu()
                        sum_sq_logits[layer, :, d] += (diag_no_bos.double() ** 2).sum(dim=(0, 2)).cpu()
                        
                        if layer == 0:
                            counts[d] += actual_batch_size * (diag.shape[-1] - 1)
                            
                del attn_scores
            del cache
            
            # Periodic VRAM defragmentation to maintain clean cache blocks
            if i > 0 and i % 50 == 0:
                gc.collect()
                torch.cuda.empty_cache()
                
    # Clean up GPU memory before terminating worker process
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[GPU {gpu_id}] Completed profiling assigned chunk.")
    
    # Return raw statistical accumulators (NOT averages) for global reduction
    return sum_logits, sum_sq_logits, sum_bos_logits, sum_sq_bos_logits, counts, counts_bos


# =====================================================================
# 2. THE MULTI-GPU CONDUCTOR (Splits data & aggregates results)
# =====================================================================
def profile_deep_heads_multigpu(
    model_name: str,
    prompts: torch.Tensor,
    seq_len: int,
    gpu_ids: list[int],
    batch_size_per_gpu: int = 8
):
    """
    Distributes an ensemble of intact instruction prompts across multiple GPUs
    using joblib/loky for interactive notebook compatibility.
    
    Args:
        model_name: HuggingFace model identifier (e.g., 'meta-llama/Llama-3.2-1B-Instruct').
        prompts: Intact tokenized sequences of shape [ensemble_size, seq_len].
        seq_len: The context window length to profile.
        gpu_ids: List of integer CUDA device IDs to utilize (e.g., [0, 1, 2]).
        batch_size_per_gpu: Number of sequences evaluated simultaneously per GPU.
        
    Returns:
        mu, sigma, mu_bos, sigma_bos: FP32 tensors of shape [n_layers, n_heads, seq_len].
    """
    num_gpus = len(gpu_ids)
    total_sequences = prompts.shape[0]
    
    if prompts.shape[1] != seq_len:
        raise ValueError(f"Prompts sequence length ({prompts.shape[1]}) does not match requested seq_len ({seq_len}).")
    
    print(f"Splitting {total_sequences} sequences across {num_gpus} GPUs ({gpu_ids})...")
    chunks = torch.tensor_split(prompts, num_gpus, dim=0)
    
    # backend="loky" leverages cloudpickle, preventing serialization failures in Jupyter
    results = Parallel(n_jobs=num_gpus, backend="loky")(
        delayed(_profiling_worker)(
            gpu_id=gpu_ids[i],
            model_name=model_name,
            prompts_chunk=chunks[i],
            seq_len=seq_len,
            batch_size=batch_size_per_gpu
        )
        for i in range(num_gpus) if chunks[i].shape[0] > 0
    )
    
    print("All GPU workers completed! Performing global map-reduce reduction...")
    
    # 1. REDUCE: Sum all FP64 statistical accumulators globally across workers
    total_sum = sum(r[0] for r in results)
    total_sum_sq = sum(r[1] for r in results)
    total_bos_sum = sum(r[2] for r in results)
    total_bos_sum_sq = sum(r[3] for r in results)
    
    total_counts = sum(r[4] for r in results)
    total_counts_bos = sum(r[5] for r in results)
    
    # 2. COMPUTE MACROSCOPIC PARAMETERS
    safe_counts = total_counts.clamp(min=1).view(1, 1, -1)
    safe_counts_bos = total_counts_bos.clamp(min=1).view(1, 1, -1)

    mu = total_sum / safe_counts
    expected_sq = total_sum_sq / safe_counts
    variance = torch.relu(expected_sq - (mu ** 2))
    sigma = torch.sqrt(variance)

    mu_bos = total_bos_sum / safe_counts_bos
    expected_sq_bos = total_bos_sum_sq / safe_counts_bos
    variance_bos = torch.relu(expected_sq_bos - (mu_bos ** 2))
    sigma_bos = torch.sqrt(variance_bos)

    return mu.float(), sigma.float(), mu_bos.float(), sigma_bos.float()

def compute_marginalized_expected_attention(
    mu: torch.Tensor, 
    sigma: torch.Tensor, 
    mu_bos: torch.Tensor, 
    sigma_bos: torch.Tensor
) -> torch.Tensor:
    """
    Computes the expected normalized attention pattern over relative distances
    by marginalizing the log-normal expected masses over all valid absolute query positions.

    Args:
        mu: (n_layers, n_heads, seq_len) - Content positional baseline
        sigma: (n_layers, n_heads, seq_len) - Content semantic spread
        mu_bos: (n_layers, n_heads, seq_len) - BOS positional baseline
        sigma_bos: (n_layers, n_heads, seq_len) - BOS semantic spread
        
    Returns:
        a_bar: (n_layers, n_heads, seq_len) - The expected attention mass for each relative distance.
    """
    seq_len = mu.shape[-1]

    # Step 1: Unnormalized Expected Masses via log-normal MGF
    # M_content[..., d] is the mass at relative distance d
    M_content = torch.exp(mu + 0.5 * (sigma ** 2))
    
    # M_bos[..., T] is the BOS mass for a query at absolute position T
    M_bos = torch.exp(mu_bos + 0.5 * (sigma_bos ** 2))

    # Step 2: Compute the Partition Function Z_T for each query position T
    # A query at position T attends to content tokens at relative distances d in [0, T-1].
    # We precompute the cumulative sum of M_content to quickly get this total mass.
    M_content_cumsum = torch.cumsum(M_content, dim=-1)
    
    Z = torch.zeros_like(M_content)
    
    # Base case: Query at T=0 attends only to itself (the BOS token)
    Z[..., 0] = M_bos[..., 0]
    
    # For T > 0, denominator is BOS mass + all valid content mass up to distance T-1
    for T in range(1, seq_len):
        Z[..., T] = M_bos[..., T] + M_content_cumsum[..., T-1]

    # Step 3: Compute Conditional Probabilities and Marginalize over T
    a_bar_d = torch.zeros_like(M_content)
    
    # We iterate over relative distances d.
    # A token is a valid content token at distance d ONLY IF the query position T > d.
    for d in range(seq_len - 1): # Maximum possible content distance is seq_len - 2
        valid_T_count = seq_len - 1 - d
        
        if valid_T_count > 0:
            # Broadcast the mass at distance d over all valid query positions T
            mass_d = M_content[..., d].unsqueeze(-1)
            
            # Extract the partition functions for all valid T (which are T > d)
            Z_valid = Z[..., d+1:]
            
            # Compute probabilities for distance d conditioned on T, then sum and average
            p_d_given_T = mass_d / Z_valid
            a_bar_d[..., d] = p_d_given_T.sum(dim=-1) / valid_T_count

    return a_bar_d

def compute_unmarginalized_expected_attention(
    mu: torch.Tensor, 
    sigma: torch.Tensor, 
    mu_bos: torch.Tensor, 
    sigma_bos: torch.Tensor
) -> torch.Tensor:
    """
    Computes the un-marginalized expected attention tensor A_{T, d} per head.
    
    Args:
        mu: (n_layers, n_heads, seq_len) - Content positional baseline
        sigma: (n_layers, n_heads, seq_len) - Content semantic spread
        mu_bos: (n_layers, n_heads, seq_len) - BOS positional baseline
        sigma_bos: (n_layers, n_heads, seq_len) - BOS semantic spread
        
    Returns:
        a_bar_unmarginalized: (n_layers, n_heads, seq_len, seq_len) 
                              indexed precisely as [layer, head, T, relative_distance].
    """
    n_layers, n_heads, seq_len = mu.shape

    # =========================================================================
    # 1. COMPUTE UNNORMALIZED EXPECTED MASSES (Log-Normal MGF)
    # Shape of both: [n_layers, n_heads, seq_len]
    # =========================================================================
    M_content = torch.exp(mu + 0.5 * (sigma ** 2))
    M_bos = torch.exp(mu_bos + 0.5 * (sigma_bos ** 2))

    # =========================================================================
    # 2. VECTORIZED PARTITION FUNCTION (Z_T)
    # A query at absolute pos T can only attend to content distances d in [0, T-1].
    # Therefore, the sum of content mass available to query T is cumsum(T-1).
    # =========================================================================
    M_content_cumsum = torch.cumsum(M_content, dim=-1)
    
    # We shift the cumsum right by 1, inserting a 0 at T=0
    content_mass_sum = torch.zeros_like(M_content)
    content_mass_sum[..., 1:] = M_content_cumsum[..., :-1]

    # Z_T = Mass_BOS(T) + sum_{d=0}^{T-1} Mass_Content(d)
    # Shape: [n_layers, n_heads, seq_len]  (indexed by T)
    Z = M_bos + content_mass_sum

    # =========================================================================
    # 3. OUTER BROADCASTING: CONTENT PROBABILITIES
    # We need M_content[d] / Z[T] for all pairs of (T, d). 
    # Un-squeezing creates an implicit outer division across the last two dims.
    # =========================================================================
    # M_content shape: [n_layers, n_heads, 1, seq_len]  <- broadcasts over T
    # Z shape:         [n_layers, n_heads, seq_len, 1]  <- broadcasts over d
    # Result shape:    [n_layers, n_heads, seq_len, seq_len] (indexed as [..., T, d])
    raw_content_probs = M_content.unsqueeze(-2) / Z.unsqueeze(-1)

    # Apply strict causal masking: content is only valid where distance d < T.
    # tril(..., diagonal=-1) keeps everything strictly below the main diagonal.
    causal_mask = torch.tril(torch.ones((seq_len, seq_len), device=mu.device), diagonal=-1).bool()
    
    content_probs = torch.where(
        causal_mask, 
        raw_content_probs, 
        torch.tensor(0.0, dtype=raw_content_probs.dtype, device=raw_content_probs.device)
    )

    # =========================================================================
    # 4. DIAGONAL EMBEDDING: BOS PROBABILITIES
    # For any query T, its attention to BOS sits precisely at relative distance d = T.
    # =========================================================================
    # Shape: [n_layers, n_heads, seq_len]
    bos_probs = M_bos / Z 

    # diag_embed takes our 1D BOS vectors and projects them onto the main diagonal 
    # of a new [seq_len, seq_len] matrix, filling the rest with zeros.
    bos_diag = torch.diag_embed(bos_probs, dim1=-2, dim2=-1)

    # =========================================================================
    # 5. SYNTHESIS
    # Content populates the lower triangle; BOS populates the main diagonal; 
    # the upper triangle remains 0.0.
    # =========================================================================
    a_bar_unmarginalized = content_probs + bos_diag

    return a_bar_unmarginalized

def load_and_tokenize_continuous_corpus(
    model: HookedTransformer,
    dataset_name: str = "wikitext",
    dataset_config: str = "wikitext-2-raw-v1",
    split: str = "train"
) -> torch.Tensor:
    """
    Loads a dataset and flattens it into a massive 1D continuous tensor of token IDs.
    Crucially, it strips all inherent special tokens to preserve the continuous 
    thermodynamic equilibrium for SDE profiling.
    
    Args:
        model: The HookedTransformer containing the appropriate tokenizer.
        dataset_name: HuggingFace dataset name (e.g., 'wikitext').
        dataset_config: HuggingFace dataset config.
        split: The dataset split to use.
        
    Returns:
        corpus_tokens: 1D torch.Tensor of token IDs on the model's device.
    """
    print(f"Loading {dataset_name} ({dataset_config}) split: '{split}'...")
    dataset = load_dataset(dataset_name, dataset_config, split=split)
    
    # Filter out empty strings/newlines to avoid artificially inflating structural gaps
    dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)
    
    print("Tokenizing corpus (safeguarding continuous flow by omitting special tokens)...")
    
    # We use HF's map function for batched, fast tokenization
    def tokenize_function(examples):
        return model.tokenizer(
            examples["text"],
            add_special_tokens=False, # don't add special tokens, we add them later
            truncation=False          # We want the full continuous text
        )
        
    tokenized_dataset = dataset.map(
        tokenize_function, 
        batched=True, 
        remove_columns=dataset.column_names # Drop the text, keep only input_ids
    )
    
    print("Flattening into a single macroscopic 1D phase-space tensor...")
    # tokenized_dataset['input_ids'] is a massive list of lists.
    # itertools.chain is the most computationally efficient way to flatten this in Python.
    flat_token_list = list(itertools.chain.from_iterable(tokenized_dataset['input_ids']))
    
    # Convert to PyTorch tensor and move to the exact device your model is on
    corpus_tokens = torch.tensor(flat_token_list, dtype=torch.long, device=model.cfg.device)
    
    print(f"Successfully loaded and flattened {corpus_tokens.shape[0]:,} continuous tokens.")
    
    return corpus_tokens

def prepare_intact_instruction_corpus(
    model: HookedTransformer, 
    ensemble_size: int,
    seq_len: int,
    dataset_name: str = "HuggingFaceH4/ultrachat_200k", 
    split: str = "train_sft"
) -> torch.Tensor:
    """
    Extracts intact conversation beginnings from an instruction dataset.
    Filters for chats >= seq_len and truncates from the right to preserve
    BOS, system prompts, and opening conversational headers.
    
    Returns:
        torch.Tensor of shape [ensemble_size, seq_len]
    """
    print(f"Loading dataset: {dataset_name}...")
    dataset = load_dataset(dataset_name, split=split, streaming=True)
    tokenizer = model.tokenizer
    
    valid_sequences = []
    
    print(f"Extracting {ensemble_size} intact sequences of length {seq_len}...")
    for example in dataset:
        messages = example.get("messages") or example.get("conversations")
        if not messages:
            continue
            
        # Apply native chat template (includes BOS and structure tokens naturally)
        formatted_chat = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        # Tokenize without injecting extra special tokens
        tokens = tokenizer.encode(formatted_chat, add_special_tokens=False)
        
        # Only keep sequences that fully cover our target sequence length
        if len(tokens) >= seq_len:
            # Slice from index 0 to keep the conversational opening intact
            valid_sequences.append(tokens[:seq_len])
            
        if len(valid_sequences) >= ensemble_size:
            break
            
    if len(valid_sequences) < ensemble_size:
        raise ValueError(
            f"Only found {len(valid_sequences)} sequences >= length {seq_len}. "
            "Try a larger dataset or a shorter seq_len."
        )
        
    print(f"Successfully extracted {len(valid_sequences)} intact conversation chunks.")
    return torch.tensor(valid_sequences, dtype=torch.long)

def compute_attention_mass_horizons(
    post_softmax_tensor: torch.Tensor, 
    thresholds: tuple = (0.85, 0.90, 0.95), 
    dim: int = -1
) -> dict[float, torch.LongTensor]:
    """
    Finds the exact positional indices along dimension `dim` where the cumulative 
    attention probability mass first breaches given percentage thresholds.

    Args:
        post_softmax_tensor: Tensor of attention weights (must sum to <= 1.0 along `dim`).
        thresholds: Tuple of target probability mass thresholds to cross.
        dim: Dimension along which to accumulate mass (default is last dimension).

    Returns:
        dict: Mapping from each threshold float to a LongTensor of crossing indices.
              The returned tensors have the same shape as the input minus the `dim` axis.
    """
    # 1. Accumulate mass in float64 to prevent numerical underflow over long sequences
    cumsum_mass = post_softmax_tensor.to(torch.float64).cumsum(dim=dim)
    
    max_idx = post_softmax_tensor.shape[dim] - 1
    results = {}
    
    for target in thresholds:
        # Create a boolean mask representing where cumulative mass exceeds or equals target
        breached_mask = cumsum_mass >= target
        
        # In PyTorch, argmax on a boolean/uint8 tensor returns the index of the FIRST max element.
        # Since True is 1 and False is 0, this identifies the exact first crossing index.
        first_crossing_idx = breached_mask.to(torch.uint8).argmax(dim=dim)
        
        # Safeguard: If the threshold was never reached (e.g., due to masking or truncation),
        # argmax on all-False defaults to index 0. We force unbreached cases to snap to max_idx.
        any_breached = breached_mask.any(dim=dim)
        first_crossing_idx = torch.where(
            any_breached, 
            first_crossing_idx, 
            torch.tensor(max_idx, device=first_crossing_idx.device)
        )
        
        results[target] = first_crossing_idx
        
    return results

def compute_adversarial_influence_horizon(
    mu_tensor: torch.Tensor,
    sigma_tensor: torch.Tensor,
    multiplier: float = 3.0,
    threshold: float = 0.05,
    dim: int = -1
) -> torch.LongTensor:
    """
    Computes the exact distance index per attention head where the maximum possible 
    adversarial Softmax weight permanently drops below a target threshold.

    Args:
        mu_tensor: Positional baseline logits, shape [n_layers, n_heads, seq_len].
        sigma_tensor: Contextual semantic spread (std dev), shape [n_layers, n_heads, seq_len].
        multiplier: Standard deviation scalar for semantic reach (e.g., 3.0 for 99.7% bounds).
        threshold: The maximum allowable attention probability ceiling (e.g., 0.05).
        dim: The relative distance axis (default is last dimension).

    Returns:
        cutoff_indices: LongTensor of shape [n_layers, n_heads] containing the first 
                        distance index where the adversarial weight drops below threshold.
    """
    seq_len = mu_tensor.shape[dim]

    # 1. Calculate Best-Case (B) and Worst-Case (W) logit boundaries
    B = mu_tensor + (multiplier * sigma_tensor)
    W = mu_tensor - (multiplier * sigma_tensor)

    # 2. Numerical Stability Shift (Log-Sum-Exp Trick)
    # Factor out the absolute peak logit along the sequence. Since B >= W always, 
    # the maximum logit across the entire playing field is guaranteed to be max(B).
    M, _ = B.max(dim=dim, keepdim=True)
    
    B_shifted = B - M
    W_shifted = W - M

    # Exponentiate shifted values (strictly bounded in (0, 1], eliminating overflow)
    exp_B = torch.exp(B_shifted)
    exp_W = torch.exp(W_shifted)

    # 3. Compute global competing worst-case sum strictly once
    # Accumulated in float64 to maintain extreme tail precision
    sum_W = exp_W.to(torch.float64).sum(dim=dim, keepdim=True)

    # 4. Vectorized calculation of adversarial Softmax ceiling per position
    # P_adv(d) = exp(B_d) / (exp(B_d) + sum_W - exp(W_d))
    denominator = exp_B.to(torch.float64) + sum_W - exp_W.to(torch.float64)
    
    # Cast back to native model precision
    P_adv = (exp_B / denominator).to(mu_tensor.dtype)

    # 5. Boolean mask identifying where the weight drops below the threshold
    sub_threshold_mask = P_adv < threshold

    # In PyTorch, argmax on a boolean/uint8 tensor locates the FIRST True element
    cutoff_idx = sub_threshold_mask.to(torch.uint8).argmax(dim=dim)

    # 6. Fallback Protection
    # If a head NEVER drops below the threshold (e.g., a pure global semantic head 
    # where all positions remain highly viable), argmax defaults to index 0. 
    # We forcefully snap unbreached heads to the absolute sequence boundary.
    any_met = sub_threshold_mask.any(dim=dim)
    
    cutoff_indices = torch.where(
        any_met,
        cutoff_idx,
        torch.tensor(seq_len, device=cutoff_idx.device)
    )

    return cutoff_indices

def compute_unembedding_svd(model: HookedTransformer) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Extracts W_U, applies mean-centering, and computes the reduced SVD once.
    Returns strictly the left singular vectors (U) and singular values (S).
    """
    W_U = model.W_U.detach().to(torch.float32)
    
    # Sanitize native FP16 overflow NaNs if present
    W_U = torch.nan_to_num(W_U, nan=0.0, posinf=0.0, neginf=0.0)

    # Bake Final LayerNorm centering into the projection operator
    if hasattr(model, 'ln_final') and model.ln_final is not None:
        W_U = W_U - W_U.mean(dim=0, keepdim=True)

    # full_matrices=False is critical: keeps U as [d_model, d_model] instead of [d_model, vocab]
    U, S, _ = torch.linalg.svd(W_U, full_matrices=False)
    
    return U, S

def svd_alignment_score(
    resid_vectors: torch.Tensor, 
    U: torch.Tensor, 
    S: torch.Tensor, 
    top_k_modes: int = 10
) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
    """
    High-speed SVD alignment profiler. 
    
    Args:
        resid_vectors: Raw residual stream delta tensor [..., d_model]
        U: Precomputed left singular vectors [d_model, d_model]
        S: Precomputed singular values [d_model]
    """
    orig_shape = resid_vectors.shape
    d_model = orig_shape[-1]
    
    # 1. Flatten arbitrary batch/seq topologies to [N, d_model]
    flat_vecs = resid_vectors.detach().reshape(-1, d_model).to(dtype=U.dtype, device=U.device)
    flat_vecs = torch.nan_to_num(flat_vecs, nan=0.0, posinf=0.0, neginf=0.0)

    # 2. Compute magnitudes and safe unit vectors
    vec_norms = torch.linalg.norm(flat_vecs, dim=-1, keepdim=True)
    unit_vecs = torch.where(vec_norms > 1e-12, flat_vecs / vec_norms, torch.zeros_like(flat_vecs))

    # 3. Project onto orthonormal basis U (Dot products = Cosine Similarities)
    # [N, d_model] @ [d_model, d_model] -> [N, d_model]
    svd_cosine_scores = torch.matmul(unit_vecs, U)
    directional_projections = torch.matmul(flat_vecs, U)

    # 4. Macroscopic Risk Metrics
    scaled_projections = directional_projections * S.unsqueeze(0)
    effective_logit_error = torch.linalg.norm(scaled_projections, dim=-1)

    rms_singular_value = torch.sqrt(torch.mean(S ** 2))
    expected_iso_error = vec_norms.squeeze(-1) * rms_singular_value
    
    gain_exposure = torch.where(
        expected_iso_error > 1e-12,
        effective_logit_error / expected_iso_error,
        torch.zeros_like(effective_logit_error)
    )

    top_k_energy = torch.sum(directional_projections[:, :top_k_modes] ** 2, dim=-1)
    safe_total_energy = torch.where(vec_norms.squeeze(-1)**2 > 1e-12, vec_norms.squeeze(-1)**2, torch.ones_like(vec_norms.squeeze(-1)))
    top_k_fraction = top_k_energy / safe_total_energy

    max_cosine, max_mode_idx = torch.max(torch.abs(svd_cosine_scores), dim=-1)

    # Restore user's original tensor dimensions
    out_shape = orig_shape[:-1]
    
    metrics = {
        "effective_logit_norm_error": effective_logit_error.reshape(out_shape),
        "normalized_gain_exposure": gain_exposure.reshape(out_shape),
        "top_k_energy_fraction": top_k_fraction.reshape(out_shape),
        "peak_mode_cosine": max_cosine.reshape(out_shape),
        "peak_mode_index": max_mode_idx.reshape(out_shape)
    }

    return svd_cosine_scores.reshape(*out_shape, d_model), metrics

# =====================================================================
# 1. THE CORE MECHANICS: SINGLE LAYER (SHRINKING PYRAMID)
# =====================================================================
# def single_layer_forward(
#     context_vals: torch.Tensor,   # [N_c] -> Macroscopic probability measure
#     context_resid: torch.Tensor,  # [N_c, S_in, d_model] -> Frozen background KV
#     query_resid: torch.Tensor,    # [N_q, S_in, d_model] -> Active tracking states
#     layer_idx: int,
#     K_i: int,                     # Local window size for this specific layer
#     model,                        # HookedTransformer instance
#     sem_heads: list,              # Semantic heads in this layer
#     pos_heads: list               # Positional heads in this layer
# ):
#     """
#     Computes a single transformer layer update using the Shrinking Pyramid technique.
#     It strictly computes only the required S_out positions to prevent wasted FLOPs.
#     """
#     N_c = context_resid.size(0)
#     N_q = query_resid.size(0)
#     S_in = context_resid.size(1)
#     S_out = S_in - K_i + 1        # The shrinking sequence dimension

#     # prepend BOS to all S_in k-grams
#     bos_resid = model.W_E[1] + model.W_pos[0]
#     extended_context_resid = torch.cat([bos_resid.repeat(N_c,1,1), context_resid], dim=1) # [N_c, S_in+1, d_model]
#     extended_query_resid = torch.cat([bos_resid.repeat(N_q,1,1), query_resid], dim=1) # [N_q, S_in+1, d_model]
    
#     # 1. Macro-Micro Batching for Efficiency
#     # Shape: [N_c + N_q, S_in, d_model]
#     combined_resid = torch.cat([extended_context_resid, extended_query_resid], dim=0)
#     combined_resid_pre = model.blocks[layer_idx].ln1(combined_resid)
    
#     # We only need queries for the last S_out positions
#     q_resid_pre = combined_resid_pre[:, -S_out:, :] # [N_c + N_q, S_out, d_model]
#     final_attn_out = torch.zeros_like(q_resid_pre)  # [N_c + N_q, S_out, d_model]

#     # ---------------------------------------------------------
#     # A. Semantic Heads (Context-Driven Active Set)
#     # ---------------------------------------------------------
#     if len(sem_heads) > 0:
#         Q_sem = model.W_Q[layer_idx, sem_heads] # [num_sem, d_model, d_head]
#         K_sem = model.W_K[layer_idx, sem_heads]
#         V_sem = model.W_V[layer_idx, sem_heads]
#         O_sem = model.W_O[layer_idx, sem_heads]
        
#         # Queries: from the combined batch (both context and active queries update)
#         q_s = einops.einsum(q_resid_pre, Q_sem, "batch S_out d_model, num_sem d_model d_head -> num_sem batch S_out d_head")
#         q_s = q_s + model.b_Q[layer_idx, sem_heads].unsqueeze(1).unsqueeze(1)
        
#         # Keys & Values: strictly from the final token of the frozen context
#         context_final_token = combined_resid_pre[:N_c, -1, :] # [N_c, d_model]
#         k_s = einops.einsum(context_final_token, K_sem, "N_c d_model, num_sem d_model d_head -> num_sem N_c d_head")
#         k_s = k_s + model.b_K[layer_idx, sem_heads].unsqueeze(1)
#         v_s = einops.einsum(context_final_token, V_sem, "N_c d_model, num_sem d_model d_head -> num_sem N_c d_head")
#         v_s = v_s + model.b_V[layer_idx, sem_heads].unsqueeze(1)
        
#         # Attention with Integration Measure (context_vals)
#         QK_raw = einops.einsum(q_s, k_s, "num_sem batch S_out d_head, num_sem N_c d_head -> num_sem batch S_out N_c")
#         QK_raw = QK_raw / np.sqrt(model.cfg.d_head)
        
#         QK_weighted = context_vals.view(1, 1, 1, -1) * torch.exp(QK_raw)
#         attn_weights_s = QK_weighted / QK_weighted.sum(dim=-1, keepdim=True).clamp(min=1e-9)
#         # print(f"attn_weights_s:\n{attn_weights_s.topk(5)}")
        
#         weighted_v_s = einops.einsum(attn_weights_s, v_s, "num_sem batch S_out N_c, num_sem N_c d_head -> num_sem batch S_out d_head")
#         out_s = einops.einsum(weighted_v_s, O_sem, "num_sem batch S_out d_head, num_sem d_head d_model -> batch S_out d_model")
#         final_attn_out += out_s

#     # ---------------------------------------------------------
#     # B. Positional Heads (Local Sliding Window)
#     # ---------------------------------------------------------
#     if len(pos_heads) > 0:
#         Q_pos = model.W_Q[layer_idx, pos_heads] # [num_pos, d_model, d_head]
#         K_pos = model.W_K[layer_idx, pos_heads]
#         V_pos = model.W_V[layer_idx, pos_heads]
#         O_pos = model.W_O[layer_idx, pos_heads]
        
#         q_p = einops.einsum(q_resid_pre, Q_pos, "batch S_out d_model, num_pos d_model d_head -> num_pos batch S_out d_head")
#         q_p = q_p + model.b_Q[layer_idx, pos_heads].unsqueeze(1).unsqueeze(1)
        
#         # Keys & Values: from the full S_in sequence of the current batch
#         k_p = einops.einsum(combined_resid_pre, K_pos, "batch S_in d_model, num_pos d_model d_head -> num_pos batch S_in d_head")
#         k_p = k_p + model.b_K[layer_idx, pos_heads].unsqueeze(1).unsqueeze(1)
#         v_p = einops.einsum(combined_resid_pre, V_pos, "batch S_in d_model, num_pos d_model d_head -> num_pos batch S_in d_head")
#         v_p = v_p + model.b_V[layer_idx, pos_heads].unsqueeze(1).unsqueeze(1)
        
#         QK_raw_p = einops.einsum(q_p, k_p, "num_pos batch S_out d_head, num_pos batch S_in d_head -> num_pos batch S_out S_in")
#         QK_raw_p = QK_raw_p / np.sqrt(model.cfg.d_head)
        
#         # Construct the K_i local sliding window mask
#         q_indices = torch.arange(S_out, device=QK_raw_p.device).view(1, 1, S_out, 1)
#         k_indices = torch.arange(S_in+1, device=QK_raw_p.device).view(1, 1, 1, S_in+1)
#         mask = (k_indices >= q_indices) & (k_indices <= q_indices + K_i - 1)
        
#         QK_raw_p = torch.where(mask, QK_raw_p, torch.tensor(-1e4, device=QK_raw_p.device))
#         attn_weights_p = torch.softmax(QK_raw_p, dim=-1)
        
#         weighted_v_p = einops.einsum(attn_weights_p, v_p, "num_pos batch S_out S_in, num_pos batch S_in d_head -> num_pos batch S_out d_head")
#         out_p = einops.einsum(weighted_v_p, O_pos, "num_pos batch S_out d_head, num_pos d_head d_model -> batch S_out d_model")
#         final_attn_out += out_p

#     # ---------------------------------------------------------
#     # C. Residual & MLP Application
#     # ---------------------------------------------------------
#     final_attn_out += model.b_O[layer_idx] # add the final attention layer output bias
#     shrunken_resid = combined_resid[:, -S_out:, :] + final_attn_out
    
#     # mlp_in = model.blocks[layer_idx].ln2(shrunken_resid)
#     # mlp_out = model.blocks[layer_idx].mlp(mlp_in)
#     # next_resid = shrunken_resid + mlp_out

#     next_resid = shrunken_resid
    
#     # Split back into context and query streams
#     next_context_resid = next_resid[:N_c]
#     next_query_resid = next_resid[N_c:]
    
#     return next_context_resid, next_query_resid

### New version - 3-part, semantic + positional + sink computation 
def single_layer_forward(
    context_vals: torch.Tensor,       # [N_c] -> Macroscopic probability measure (\pi)
    context_sem_resid: torch.Tensor,  # [N_c, S_in, d_model] -> Frozen pure semantic background
    query_sem_resid: torch.Tensor,    # [N_q, S_in, d_model] -> Active pure semantic tracking states
    context_keys: torch.Tensor,       # [N_c, S_init]        -> [UPDATE] Raw context token IDs for auto pad masking
    query_keys: torch.Tensor,         # [N_q, S_init]        -> [UPDATE] Raw query token IDs for auto pad masking
    pos_embeds_local: torch.Tensor,   # [S_in, d_model]      -> Absolute pos embeddings for the current window
    pos_embeds_q: torch.Tensor,       # [S_out, d_model]     -> Absolute pos embeddings for the query
    layer_idx: int,
    K_i: int,                         # Local window size for this specific layer
    model,                            # HookedTransformer instance
    active_heads: list,               # Unified list of active heads (no sem/pos distinction)
    ln_avg_sigma,                     # scalar -> frozen LN sigma, OR [d_vocab] -> exact per-token table
    L_ctx: torch.Tensor,              # [num_heads] -> Receptive field / integration measure
    C_far: torch.Tensor,              # [num_heads] -> Baseline pos scalar for background tokens
    sink_k: torch.Tensor,             # [num_heads, d_head] -> Pre-computed BOS Key
    sink_v: torch.Tensor              # [num_heads, d_head] -> Pre-computed BOS Value
):
    """
    Computes a single transformer layer update using the 3-part thermodynamic partition function
    (Global Density + Local K-Gram + Attention Sink), completely decoupled from positional noise.
    """
    N_c = context_sem_resid.size(0)
    N_q = query_sem_resid.size(0)
    S_in = context_sem_resid.size(1)
    S_out = S_in - K_i + 1            # The shrinking sequence dimension
    d_head = model.cfg.d_head
    scale = np.sqrt(d_head)
    
    # =========================================================================
    # 1. LAYER-NORM LINEARIZATION (The Freeze Hack)
    # =========================================================================
    combined_sem_resid = torch.cat([context_sem_resid, query_sem_resid], dim=0) # [N_c + N_q, S_in, d_model]
    
    # [UPDATE]: Concatenate raw input keys to match the combined batch dimension of the residuals
    combined_keys = torch.cat([context_keys, query_keys], dim=0) # [N_c + N_q, S_init]

    # A. Linear Mean Subtraction (Centering)
    centered_sem = combined_sem_resid - combined_sem_resid.mean(dim=-1, keepdim=True)
    centered_pos_local = pos_embeds_local - pos_embeds_local.mean(dim=-1, keepdim=True)
    centered_pos_q = pos_embeds_q - pos_embeds_q.mean(dim=-1, keepdim=True)
    
    # B. Apply the LN denominator.
    #    Two modes, selected by the TYPE of `ln_avg_sigma`:
    #      * scalar (float or 0-d/1-element tensor) -> the original "freeze hack": one
    #        frozen sigma for the whole layer. Costs ~4.9e-3 JSD on attn-only-1l.
    #      * [d_vocab] tensor -> the EXACT per-token table. Only legal at layer 0 of a
    #        position-free model, where resid_pre[j] == W_E[token_j] exactly, so
    #        sigma_j = sqrt(mean((W_E[v]-mean)^2) + eps) is a pure token lookup.
    #        This makes LN exact for EVERY row of the batch simultaneously (unlike the
    #        W_E-rescaling trick, which can only be exact for one query token per call),
    #        and it leaves `combined_sem_resid` un-rescaled so the residual-stream read
    #        in section 5 stays exact too.
    use_sigma_table = (
        torch.is_tensor(ln_avg_sigma)
        and ln_avg_sigma.dim() == 1
        and ln_avg_sigma.numel() == model.cfg.d_vocab
    )
    if use_sigma_table:
        if layer_idx != 0 or S_in != combined_keys.size(1):
            raise ValueError(
                "A per-token LN sigma table is only valid at layer 0 of a position-free "
                "model (resid_pre[j] == W_E[token_j]). Deeper layers, or a shrunken "
                f"window, must use a scalar sigma. Got layer_idx={layer_idx}, "
                f"S_in={S_in}, S_init={combined_keys.size(1)}."
            )
        sem_sigma = ln_avg_sigma[combined_keys[:, -S_in:]].unsqueeze(-1)  # [N_c+N_q, S_in, 1]
        # The positional stream is identically zero whenever the table mode is legal
        # (W_pos == 0), so its divisor is arbitrary; use the table mean to stay finite.
        pos_sigma = ln_avg_sigma.mean()
    else:
        sem_sigma = ln_avg_sigma
        pos_sigma = ln_avg_sigma

    sem_resid_pre = centered_sem / sem_sigma
    pos_local_pre = centered_pos_local / pos_sigma
    pos_q_pre = centered_pos_q / pos_sigma

    # C. Apply Learned Gamma (If it exists in this specific architecture)
    ln_module = model.blocks[layer_idx].ln1
    if hasattr(ln_module, 'w') and ln_module.w is not None:
        # Model has affine parameters (e.g., GPT-2, Llama)
        sem_resid_pre = sem_resid_pre * ln_module.w
        pos_local_pre = pos_local_pre * ln_module.w
        pos_q_pre = pos_q_pre * ln_module.w

    # Extract query residuals
    q_s_resid = sem_resid_pre[:, -S_out:, :] # [N_c + N_q, S_out, d_model]

    # =========================================================================
    # 2. PROJECTIONS (Decoupled Semantic and Positional Streams)
    # =========================================================================
    Q = model.W_Q[layer_idx, active_heads] # [num_heads, d_model, d_head]
    K = model.W_K[layer_idx, active_heads]
    V = model.W_V[layer_idx, active_heads]
    O = model.W_O[layer_idx, active_heads]
    
    # -- Queries --
    q_s = einops.einsum(q_s_resid, Q, "batch S_out d_model, heads d_model d_head -> heads batch S_out d_head")
    q_s = q_s + model.b_Q[layer_idx, active_heads].view(-1, 1, 1, d_head)
    
    q_p = einops.einsum(pos_q_pre, Q, "S_out d_model, heads d_model d_head -> heads S_out d_head")
    
    # -- Keys and Values --
    # Global Background (uses final token of the context residual)
    context_final_token = sem_resid_pre[:N_c, -1, :] # [N_c, d_model]
    k_s_global = einops.einsum(context_final_token, K, "N_c d_model, heads d_model d_head -> heads N_c d_head")
    k_s_global = k_s_global + model.b_K[layer_idx, active_heads].view(-1, 1, d_head)
    v_s_global = einops.einsum(context_final_token, V, "N_c d_model, heads d_model d_head -> heads N_c d_head")
    v_s_global = v_s_global + model.b_V[layer_idx, active_heads].view(-1, 1, d_head)
    
    # Local Window (uses full sequence)
    k_s_local = einops.einsum(sem_resid_pre, K, "batch S_in d_model, heads d_model d_head -> heads batch S_in d_head")
    k_s_local = k_s_local + model.b_K[layer_idx, active_heads].view(-1, 1, 1, d_head)
    v_local = einops.einsum(sem_resid_pre, V, "batch S_in d_model, heads d_model d_head -> heads batch S_in d_head")
    v_local = v_local + model.b_V[layer_idx, active_heads].view(-1, 1, 1, d_head)
    
    k_p_local = einops.einsum(pos_local_pre, K, "S_in d_model, heads d_model d_head -> heads S_in d_head")

    # =========================================================================
    # 3. EXPONENTIATED PROBABILITY MASS (The Partition Function)
    # =========================================================================
    
    # -- Term 1: The Global Semantic Background --
    sem_score_global = einops.einsum(q_s, k_s_global, "heads batch S_out d_head, heads N_c d_head -> heads batch S_out N_c") / scale
    E_sem_global = torch.exp(sem_score_global)
    
    L_global = (L_ctx - K_i - 1).view(-1, 1, 1, 1)
    C_far_view = C_far.view(-1, 1, 1, 1)
    pi_view = context_vals.view(1, 1, 1, N_c)
    
    M_global = L_global * pi_view * E_sem_global * C_far_view # [heads, batch, S_out, N_c]
    
    # -- Term 2: The Explicit Local K-gram --
    sem_score_local = einops.einsum(q_s, k_s_local, "heads batch S_out d_head, heads batch S_in d_head -> heads batch S_out S_in") / scale
    E_sem_local = torch.exp(sem_score_local)
    
    pos_score_local = einops.einsum(q_p, k_p_local, "heads S_out d_head, heads S_in d_head -> heads S_out S_in") / scale
    E_pos_local = torch.exp(pos_score_local)
    
    # Construct sliding window mask (1.0 for allowed, 0.0 for outside window)
    q_indices = torch.arange(S_out, device=E_pos_local.device).view(1, S_out, 1)
    k_indices = torch.arange(S_in, device=E_pos_local.device).view(1, 1, S_in)
    mask = (k_indices >= q_indices) & (k_indices <= q_indices + K_i - 1)
    E_pos_local = E_pos_local * mask.float()
    
    M_local = E_sem_local * E_pos_local.unsqueeze(1) # [heads, batch, S_out, S_in]

    # [UPDATE START]: Unified Dead-Token Masking (PAD + BOS)
    pad_token_id = getattr(model.tokenizer, "pad_token_id", None)
    if pad_token_id is None and hasattr(model.tokenizer, "to_dict"):
        pad_token_id = model.tokenizer.to_dict().get("pad_token_id", None)

    bos_token_id = getattr(model.tokenizer, "bos_token_id", None)
    if bos_token_id is None and hasattr(model.tokenizer, "to_dict"):
        bos_token_id = model.tokenizer.to_dict().get("bos_token_id", None)

    active_keys = combined_keys[:, -S_in:] # Slice to active window length
    
    # Build a unified boolean mask of positions to kill in the local K-gram
    ignore_mask = torch.zeros_like(active_keys, dtype=torch.bool)

    if pad_token_id is not None and pad_token_id != -1:
        ignore_mask |= (active_keys == pad_token_id)

    if bos_token_id is not None and bos_token_id != -1:
        # [NEW]: Conceptually replace BOS with PAD for the local window.
        # Forces the model to resolve BOS 100% through the dedicated M_sink term.
        ignore_mask |= (active_keys == bos_token_id)

    # Invert to get valid positions (1.0 for keep, 0.0 for wipe)
    valid_mask = (~ignore_mask).view(1, -1, 1, S_in).to(device=M_local.device, dtype=M_local.dtype)
    M_local = M_local * valid_mask
    # [UPDATE END]

    # -- Term 3: The Attention Sink (BOS) --
    sink_score = einops.einsum(q_s, sink_k[active_heads], "heads batch S_out d_head, heads d_head -> heads batch S_out") / scale
    M_sink = torch.exp(sink_score) # [heads, batch, S_out]

    # =========================================================================
    # 4. NORMALIZATION & V-PROJECTION
    # =========================================================================
    
    # Weight the values directly with the unnormalized masses
    out_global = einops.einsum(M_global, v_s_global, "heads batch S_out N_c, heads N_c d_head -> heads batch S_out d_head")
    out_local = einops.einsum(M_local, v_local, "heads batch S_out S_in, heads batch S_in d_head -> heads batch S_out d_head")
    out_sink = einops.einsum(M_sink, sink_v[active_heads], "heads batch S_out, heads d_head -> heads batch S_out d_head")
    
    # Compute partition function (Z)
    Z_global = M_global.sum(dim=-1)
    Z_local = M_local.sum(dim=-1)
    Z_total = Z_global + Z_local + M_sink # [heads, batch, S_out]
    
    # Normalize
    weighted_v = (out_global + out_local + out_sink) / Z_total.unsqueeze(-1).clamp(min=1e-9)
    
    # Final Output Projection
    attn_out = einops.einsum(weighted_v, O, "heads batch S_out d_head, heads d_head d_model -> batch S_out d_model")
    attn_out = attn_out + model.b_O[layer_idx]

    # =========================================================================
    # 5. RESIDUAL UPDATE (Semantic Subspace Only)
    # =========================================================================
    shrunken_resid = combined_sem_resid[:, -S_out:, :] + attn_out
    
    if hasattr(model.blocks[layer_idx], "mlp"):
        mlp_in = model.blocks[layer_idx].ln2(shrunken_resid + pos_embeds_q.unsqueeze(0)) ### ADDED DUMMY POSITION EMBEDDING
        mlp_out = model.blocks[layer_idx].mlp(mlp_in)
        next_resid = shrunken_resid + mlp_out
    else:
        # Bypass MLP for pure attention dynamics (or insert MLP here acting on pure semantics)
        next_resid = shrunken_resid
    
    next_context_sem_resid = next_resid[:N_c]
    next_query_sem_resid = next_resid[N_c:]
    
    return next_context_sem_resid, next_query_sem_resid


# =====================================================================
# 2. THE MULTI-LAYER FORWARD PASS (LOGITS & TRANSITION PROBS)
# =====================================================================
# def abstract_forward_pass(
#     context_vals: torch.Tensor,   # [N_context]
#     context_keys: torch.Tensor,   # [N_context, S_init]
#     query_keys: torch.Tensor,     # [N_queries, S_init]
#     model, 
#     K_list: list,                 # List of K_i window sizes per layer
#     sem_heads_list: list,         # List of semantic heads per layer
#     pos_heads_list: list,         # List of positional heads per layer
#     temperature: float,
#     top_p: float
# ) -> torch.Tensor:
#     """
#     Loops through the Transformer layers and returns the transition matrix P_active
#     for the active queries, conditioned strictly on the frozen context.
#     """

#     # 1. Embeddings: [N, S_init, d_model]
#     context_resid = model.W_E[context_keys] 
#     query_resid = model.W_E[query_keys]     
    
#     # 2. Absolute Positional Embeddings (Layer 0 only)
#     position_offset = 100
#     pos_embeds = model.W_pos[position_offset:position_offset+context_keys.size(1), :] 
#     # context_resid = context_resid + pos_embeds
#     # query_resid = query_resid + pos_embeds
    
#     # 3. Apply Layers
#     for layer_idx in range(model.cfg.n_layers):
#         context_resid, query_resid = single_layer_forward(
#             context_vals, 
#             context_resid, 
#             query_resid, 
#             layer_idx,
#             K_list[layer_idx],
#             model, 
#             sem_heads_list[layer_idx], 
#             pos_heads_list[layer_idx]
#         )
    
#     # 4. Unembedding (Query residual sequence length is now strictly 1)
#     final_query_resid = query_resid.squeeze(1) # [N_queries, d_model]
#     final_query_resid = model.ln_final(final_query_resid)
    
#     logits = final_query_resid @ model.W_U + model.b_U # [N_queries, |V|]
    
#     # 5. Temperature and Top-P Masking (w/ Straight-Through Estimator)
#     if temperature != 1.0:
#         logits = logits / temperature
        
#     chunk_probs_dense = torch.softmax(logits, dim=-1)
    
#     if top_p < 1.0:
#         sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
#         cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#         sorted_mask = cumulative_probs < top_p
#         sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        
#         mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
#         mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
#         filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))
        
#         row_sums = filtered_probs.sum(dim=-1, keepdim=True)
#         chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)
        
#         # STE for gradient preservation
#         P_active = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense
#     else:
#         P_active = chunk_probs_dense
        
#     return P_active # Shape: [N_queries, |V|]

## New version - combined, 3-part partition function
def abstract_forward_pass(
    context_vals: torch.Tensor,   # [N_context] -> The probability density (\pi)
    context_keys: torch.Tensor,   # [N_context, S_init]
    query_keys: torch.Tensor,     # [N_queries, S_init]
    model, 
    temperature: float,
    top_p: float,
    K_list: list,
    active_heads_list: list,
    ln_avg_sigma_list: list,
    L_ctx_list: list,
    C_far_list: list,
    sink_k_list: list,
    sink_v_list: list
) -> torch.Tensor:
    """
    Loops through the Transformer layers and returns the transition matrix P_active
    using completely decoupled semantic and positional representations.
    """

    # 1. PURE SEMANTIC Embeddings: [N, S_init, d_model]
    # Positional embeddings are explicitly NOT added to the residual stream here.
    context_sem_resid = model.W_E[context_keys] 
    query_sem_resid = model.W_E[query_keys]     
    
    # 2. Extract Absolute Positional Embeddings
    S_init = context_keys.size(1)
    position_offset = 100
    # Pre-extract all positional embeddings needed for the full initial context length
    pos_embeds_full = model.W_pos[position_offset : position_offset + S_init, :] # [S_init, d_model]
    
    # 3. Apply Layers
    for layer_idx in range(model.cfg.n_layers):
        K_i = K_list[layer_idx]
        current_S_in = context_sem_resid.size(1)
        S_out = current_S_in - K_i + 1
        
        # Slicing the correct absolute positions for the shrinking window
        # Because the window shrinks from the "past", the most recent tokens are always at the end.
        pos_embeds_local = pos_embeds_full[-current_S_in:] # [current_S_in, d_model]
        pos_embeds_q = pos_embeds_full[-S_out:]            # [S_out, d_model]

        context_sem_resid, query_sem_resid = single_layer_forward(
            context_vals=context_vals, 
            context_sem_resid=context_sem_resid, 
            query_sem_resid=query_sem_resid, 
            context_keys=context_keys,                       # [UPDATE]: Passed down for pad masking
            query_keys=query_keys,                           # [UPDATE]: Passed down for pad masking
            pos_embeds_local=pos_embeds_local,
            pos_embeds_q=pos_embeds_q,
            layer_idx=layer_idx,
            K_i=K_i,
            model=model, 
            active_heads=active_heads_list[layer_idx], 
            ln_avg_sigma=ln_avg_sigma_list[layer_idx],
            L_ctx=L_ctx_list[layer_idx],
            C_far=C_far_list[layer_idx],
            sink_k=sink_k_list[layer_idx],
            sink_v=sink_v_list[layer_idx]
        )
    
    # 4. Unembedding (Query residual sequence length is now strictly 1)
    # We apply final LN directly to the pure semantic residual, avoiding positional bias
    # TODO: add a dummy local positional embeddings before final LN and unembedding
    final_query_resid = query_sem_resid.squeeze(1) # [N_queries, d_model]
    final_query_resid = model.ln_final(final_query_resid + pos_embeds_q) ## ADDED A DUMMY POSITIONAL EMBEDDING
    
    logits = final_query_resid @ model.W_U + model.b_U # [N_queries, |V|]
    
    # 5. Temperature and Top-P Masking (w/ Straight-Through Estimator)
    if temperature != 1.0:
        logits = logits / temperature
        
    chunk_probs_dense = torch.softmax(logits, dim=-1)
    
    if top_p < 1.0:
        sorted_probs, sorted_indices = torch.sort(chunk_probs_dense, descending=True, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        sorted_mask = cumulative_probs < top_p
        sorted_mask = torch.cat([torch.ones_like(sorted_mask[:, :1]), sorted_mask[:, :-1]], dim=-1)
        
        mask = torch.zeros_like(chunk_probs_dense, dtype=torch.bool)
        mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
        filtered_probs = torch.where(mask, chunk_probs_dense, torch.zeros_like(chunk_probs_dense))
        
        row_sums = filtered_probs.sum(dim=-1, keepdim=True)
        chunk_probs_sparse = filtered_probs / row_sums.clamp(min=1e-9)
        
        # STE for gradient preservation
        P_active = chunk_probs_sparse.detach() - chunk_probs_dense.detach() + chunk_probs_dense
    else:
        P_active = chunk_probs_dense
        
    return P_active # Shape: [N_queries, |V|]

# =====================================================================
# 3. THE SINGLE MARKOV TRANSITION (POWER ITERATION)
# =====================================================================
# def single_power_iteration_step(
#     curr_vals: torch.Tensor,      # [N_queries]
#     curr_keys: torch.Tensor,      # [N_queries, S_init]
#     context_vals: torch.Tensor,   # [N_context] -> FROZEN
#     context_keys: torch.Tensor,   # [N_context, S_init] -> FROZEN
#     N: int,                       # Pruning limit (Max active states)
#     pruning_K: int,               # Top-K next states to keep per query
#     model,
#     K_list: list,
#     sem_heads_list: list,
#     pos_heads_list: list,
#     temperature: float,
#     top_p: float
# ):
#     """
#     Executes one step of pi * P_pi. Designed to be wrapped in torch.utils.checkpoint.
#     """
#     S_init = curr_keys.size(1)
    
#     # 1. Forward Pass to get Transition Matrix
#     P_active = abstract_forward_pass(
#         context_vals, context_keys, curr_keys, 
#         model, K_list, sem_heads_list, pos_heads_list, 
#         temperature, top_p
#     ) # [N_queries, V]
    
#     N_queries, V = P_active.shape
#     device = curr_vals.device
    
#     # 2. Multiplication: Broadcast to unaggregated next probabilities
#     next_vals_unaggregated = curr_vals.unsqueeze(1) * P_active # [N_queries, V]
    
#     # Select the Top-K next states per query BEFORE creating the keys
#     actual_K = min(pruning_K, V)
#     top_next_vals, top_next_indices = torch.topk(next_vals_unaggregated, k=actual_K, dim=-1) # [N_queries, actual_K]
    
#     # 3. Deterministic Key Shifting
#     shifted_keys = curr_keys[:, 1:] # [N_queries, S_init-1]
#     shifted_keys_expanded = shifted_keys.unsqueeze(1).expand(-1, actual_K, -1)
    
#     top_next_indices_expanded = top_next_indices.unsqueeze(-1) # [N_queries, actual_K, 1]
#     new_keys = torch.cat([shifted_keys_expanded, top_next_indices_expanded], dim=-1) # [N_queries, actual_K, S_init]
    
#     # 4. Flattening
#     new_keys_flat = new_keys.view(-1, S_init)            # [N_queries * actual_K, S_init]
#     next_vals_flat = top_next_vals.view(-1)              # [N_queries * actual_K]
    
#     # 5. Differentiable Aggregation
#     unique_keys, inverse_indices = torch.unique(new_keys_flat, dim=0, return_inverse=True)
#     agg_vals = torch.zeros(unique_keys.size(0), dtype=next_vals_flat.dtype, device=device)
#     agg_vals.scatter_add_(0, inverse_indices, next_vals_flat)
    
#     # 6. Differentiable Pruning (Top-N overall)
#     k_prune = min(N, agg_vals.size(0))
#     top_vals, top_indices = torch.topk(agg_vals, k=k_prune)
#     next_keys = unique_keys[top_indices]
    
#     # 7. Re-normalization
#     next_vals = top_vals / top_vals.sum()
    
#     return next_vals, next_keys

### New Version
def single_power_iteration_step(
    curr_vals: torch.Tensor,      # [N_queries]
    curr_keys: torch.Tensor,      # [N_queries, S_init]
    context_vals: torch.Tensor,   # [N_context] -> FROZEN
    context_keys: torch.Tensor,   # [N_context, S_init] -> FROZEN
    N: int,                       # Pruning limit (Max active states)
    pruning_K: int,               # Top-K next states to keep per query
    model,
    temperature: float,
    top_p: float,
    K_list: list,
    active_heads_list: list,
    ln_avg_sigma_list: list,
    L_ctx_list: list,
    C_far_list: list,
    sink_k_list: list,
    sink_v_list: list
):
    """
    Executes one step of pi * P_pi. Designed to be wrapped in torch.utils.checkpoint.
    """
    S_init = curr_keys.size(1)
    
    # 1. Forward Pass to get Transition Matrix
    P_active = abstract_forward_pass(
        context_vals, context_keys, curr_keys, 
        model, temperature, top_p,
        K_list, active_heads_list, ln_avg_sigma_list,
        L_ctx_list, C_far_list, sink_k_list, sink_v_list
    ) # [N_queries, V]
    
    N_queries, V = P_active.shape
    device = curr_vals.device
    
    # 2. Multiplication: Broadcast to unaggregated next probabilities
    next_vals_unaggregated = curr_vals.unsqueeze(1) * P_active # [N_queries, V]
    
    # Select the Top-K next states per query BEFORE creating the keys
    actual_K = min(pruning_K, V)
    top_next_vals, top_next_indices = torch.topk(next_vals_unaggregated, k=actual_K, dim=-1) # [N_queries, actual_K]
    
    # 3. Deterministic Key Shifting
    shifted_keys = curr_keys[:, 1:] # [N_queries, S_init-1]
    shifted_keys_expanded = shifted_keys.unsqueeze(1).expand(-1, actual_K, -1)
    
    top_next_indices_expanded = top_next_indices.unsqueeze(-1) # [N_queries, actual_K, 1]
    new_keys = torch.cat([shifted_keys_expanded, top_next_indices_expanded], dim=-1) # [N_queries, actual_K, S_init]
    
    # 4. Flattening
    new_keys_flat = new_keys.view(-1, S_init)            # [N_queries * actual_K, S_init]
    next_vals_flat = top_next_vals.view(-1)              # [N_queries * actual_K]
    
    # 5. Differentiable Aggregation
    unique_keys, inverse_indices = torch.unique(new_keys_flat, dim=0, return_inverse=True)
    agg_vals = torch.zeros(unique_keys.size(0), dtype=next_vals_flat.dtype, device=device)
    agg_vals.scatter_add_(0, inverse_indices, next_vals_flat)
    
    # 6. Differentiable Pruning (Top-N overall)
    k_prune = min(N, agg_vals.size(0))
    top_vals, top_indices = torch.topk(agg_vals, k=k_prune)
    next_keys = unique_keys[top_indices]
    
    # 7. Re-normalization
    next_vals = top_vals / top_vals.sum()
    
    return next_vals, next_keys


# =====================================================================
# 4. THE MAIN MACROSCOPIC LOOP (STATIONARY DISTRIBUTION)
# =====================================================================
# def compute_stationary_distribution(
#     pi_vals: torch.Tensor,        # [N] Initial probabilities
#     pi_keys: torch.Tensor,        # [N, S_init] Initial K-grams
#     N: int,                       # Fixed number of non-zero elements to track
#     pruning_K: int,               # Top-K next states to keep per query
#     model,
#     K_list: list,                 # Receptive fields per layer
#     sem_heads_list: list,         
#     pos_heads_list: list,         
#     M: int,                       # Number of power iterations
#     temperature: float = 1.0,
#     top_p: float = 1.0
# ):
#     """
#     Unrolls M power iterations to approximate the stationary distribution 
#     of the specific Markov chain defined by the frozen input pi.
#     """
#     # The background context is permanently frozen for the duration of the loop
#     context_vals = pi_vals
#     context_keys = pi_keys
    
#     # The active tracking states initialize at pi
#     curr_vals = pi_vals
#     curr_keys = pi_keys
    
#     for _ in range(M):
#         # We use use_reentrant=False (modern standard for PyTorch checkpointing),
#         # allowing it to properly route gradients through curr_vals and context_vals
#         # while safely ignoring the non-differentiable keys.
#         curr_vals, curr_keys = checkpoint(
#             single_power_iteration_step,
#             curr_vals,
#             curr_keys,
#             context_vals,  # Frozen
#             context_keys,  # Frozen
#             N,
#             pruning_K,
#             model,
#             K_list,
#             sem_heads_list,
#             pos_heads_list,
#             temperature,
#             top_p,
#             use_reentrant=False
#         )
        
#     return curr_vals, curr_keys

### New Version
def compute_stationary_distribution(
    pi_vals: torch.Tensor,        # [N] Initial probabilities
    pi_keys: torch.Tensor,        # [N, S_init] Initial K-grams
    N: int,                       # Fixed number of non-zero elements to track
    pruning_K: int,               # Top-K next states to keep per query
    model,
    M: int,                       # Number of power iterations
    K_list: list,                 # Receptive fields per layer       
    active_heads_list: list,
    ln_avg_sigma_list: list,
    L_ctx_list: list,
    C_far_list: list,
    sink_k_list: list,
    sink_v_list: list,
    temperature: float = 1.0,
    top_p: float = 1.0,
):
    """
    Unrolls M power iterations to approximate the stationary distribution 
    of the specific Markov chain defined by the frozen input pi.
    """
    # The background context is permanently frozen for the duration of the loop
    context_vals = pi_vals
    context_keys = pi_keys
    
    # The active tracking states initialize at pi
    curr_vals = pi_vals
    curr_keys = pi_keys
    
    for _ in range(M):
        # We use use_reentrant=False (modern standard for PyTorch checkpointing),
        # allowing it to properly route gradients through curr_vals and context_vals
        # while safely ignoring the non-differentiable keys.
        curr_vals, curr_keys = checkpoint(
            single_power_iteration_step,
            curr_vals,
            curr_keys,
            context_vals,  # Frozen
            context_keys,  # Frozen
            N,
            pruning_K,
            model,
            temperature,
            top_p,
            K_list,
            active_heads_list,
            ln_avg_sigma_list,
            L_ctx_list,
            C_far_list,
            sink_k_list,
            sink_v_list,
            use_reentrant=False
        )
        
    return curr_vals, curr_keys

In [ ]:
### K-gram exploration class

class MetastableKGramEngine:
    """
    Dynamically isolates the minimal approximately closed (metastable) set of K-grams
    for a Transformer slaved to a frozen macroscopic context state (\pi).
    
    Uses Pruned Power Iteration with 2D Joint Flux Thresholding, Native Coalescing,
    and a rigorous 3-part thermodynamic flux partition.
    """
    def __init__(
        self,
        model: Any,                        # HookedTransformer instance
        k_value: int,                      # K-gram sequence length (K)
        frozen_context_vals: torch.Tensor, # [N_c] -> Macroscopic density (\pi)
        frozen_context_keys: torch.Tensor, # [N_c, S_init] -> Frozen background tokens
        forward_pass_kwargs: Dict[str, Any],# Hooks & params (K_list, active_heads, LN_sigma, etc.)
        epsilon_power_method: float = 1e-6,# Minimum joint flux threshold to survive guillotine
        epsilon_dijkstra: float = 1e-7,    # Minimum joint flux threshold to survive guillotine
        query_batch_size: int = 1024,      # Safe minibatch limit for the residual stream
        suffix_chunk_size: int = 2048,     # Bounding limit for dense VRAM flux accumulator (~411MB)
        device: str = "cuda"
    ):
        self.model = model
        self.k_value = k_value
        self.frozen_context_vals = frozen_context_vals.to(device)
        self.frozen_context_keys = frozen_context_keys.to(device)
        self.forward_pass_kwargs = forward_pass_kwargs
        self.epsilon_power_method = epsilon_power_method
        self.epsilon_dijkstra = epsilon_dijkstra
        self.query_batch_size = query_batch_size
        self.suffix_chunk_size = suffix_chunk_size
        self.device = device

    @torch.no_grad()
    def run_pruned_power_iteration(
        self,
        initial_kgrams: torch.Tensor,      # [N_init, K] Seed states
        max_iterations: int = 100,
        frontier_threshold: float = 1e-4,  # Halt when new frontier discovery drops below this
        leakage_threshold: float = 1e-2,   # Audit warning if dissipation exceeds this
        min_stable_steps: int = 2,         # Consecutive steps required to confirm halt
        initial_measure: Optional[torch.Tensor] = None,  # [N_init] seed mass; None -> uniform
    ) -> Dict[str, Any]:
        """
        Executes exact post-aggregation pruned power iteration.
        """
        # 1. Initialize Global Support & Stationary Measure
        global_support_keys, seed_inverse = torch.unique(
            initial_kgrams.to(self.device), dim=0, return_inverse=True
        )
        N_init = global_support_keys.size(0)

        if initial_measure is None:
            # Start with uniform measure over seed states (measure-independent, breadth-first)
            global_stat_probs = torch.ones(N_init, dtype=torch.float32, device=self.device) / N_init
        else:
            # Warm start from a supplied measure (e.g. the current pi of an optimization
            # loop), aggregated through the de-duplication of the seed keys.
            seed_measure = initial_measure.detach().to(
                device=self.device, dtype=torch.float32
            ).flatten()
            if seed_measure.numel() != initial_kgrams.size(0):
                raise ValueError(
                    f"initial_measure has {seed_measure.numel()} entries but "
                    f"initial_kgrams has {initial_kgrams.size(0)} rows."
                )
            global_stat_probs = torch.zeros(N_init, dtype=torch.float32, device=self.device)
            global_stat_probs.scatter_add_(0, seed_inverse, seed_measure)
            seed_total = global_stat_probs.sum()
            if float(seed_total) <= 0.0:
                raise ValueError("initial_measure sums to zero; nothing to propagate.")
            global_stat_probs = global_stat_probs / seed_total
        
        # O(1) Lookup: map tuple(k_gram_tokens) -> row index in global_support_keys
        state_to_idx = {tuple(row.tolist()): i for i, row in enumerate(global_support_keys)}

        # Dynamically determine vocabulary size robustly via a dummy 1-state query
        dummy_P = abstract_forward_pass(
            context_vals=self.frozen_context_vals,
            context_keys=self.frozen_context_keys,
            query_keys=global_support_keys[:1],
            model=self.model,
            **self.forward_pass_kwargs
        )
        vocab_size = dummy_P.size(-1)

        # Audit History
        history_internal: List[float] = []
        history_frontier: List[float] = []
        history_leakage: List[float] = []
        
        stable_steps_count = 0
        converged = False
        final_iter = 0

        print(f"[*] Starting Post-Aggregation Power Iteration (Seeds: {N_init}, K={self.k_value}, Vocab: {vocab_size})")

        for iteration in range(1, max_iterations + 1):
            final_iter = iteration
            N_current = global_support_keys.size(0)
            
            # Filter active queries (ignore practically zero mass interior nodes to save compute)
            active_mask = global_stat_probs >= (self.epsilon_power_method / 100.0)
            active_indices = torch.where(active_mask)[0]
            
            if active_indices.numel() == 0:
                print("[!] Catastrophic mass collapse: All active queries dropped to zero.")
                break

            query_keys_active = global_support_keys[active_indices] # [N_act, K]
            p_q_active = global_stat_probs[active_indices]          # [N_act]

            # -----------------------------------------------------------------
            # STEP A: Topological Suffix Indexing & Parent Sorting
            # -----------------------------------------------------------------
            if query_keys_active.size(1) > 1:
                suffixes = query_keys_active[:, 1:] # [N_act, K-1]
                unique_suffixes, suffix_inverse_indices = torch.unique(suffixes, dim=0, return_inverse=True)
            else:
                # K == 1 (unigrams): the K-1 suffix is the empty sequence, so every parent
                # shares one and the same suffix and all flux converges onto a single row.
                # torch.unique(dim=0) cannot process a zero-width tensor, so build the
                # degenerate indexing explicitly: one unique suffix of width 0.
                unique_suffixes = query_keys_active.new_empty((1, 0))      # [1, 0]
                suffix_inverse_indices = torch.zeros(
                    query_keys_active.size(0), dtype=torch.long, device=query_keys_active.device
                )
            num_unique_suffixes = unique_suffixes.size(0)

            # Sort active queries by their suffix ID to group converging parents adjacently
            sorted_suffix_ids, sort_order = torch.sort(suffix_inverse_indices)
            q_keys_sorted = query_keys_active[sort_order]
            p_q_sorted = p_q_active[sort_order]

            all_surviving_successors: List[torch.Tensor] = []
            all_surviving_fluxes: List[torch.Tensor] = []
            total_dissipated_flux = 0.0

            # -----------------------------------------------------------------
            # STEP B: Chunked Exact Aggregation & Post-Pruning
            # -----------------------------------------------------------------
            # Iterate over unique suffixes in VRAM-bounded blocks (~411MB per block)
            for chunk_start_suffix in range(0, num_unique_suffixes, self.suffix_chunk_size):
                chunk_end_suffix = min(chunk_start_suffix + self.suffix_chunk_size, num_unique_suffixes)
                current_chunk_size = chunk_end_suffix - chunk_start_suffix

                # Use binary search to find exact array slice of parents belonging to these suffix IDs
                query_start_idx = torch.searchsorted(sorted_suffix_ids, chunk_start_suffix)
                query_end_idx = torch.searchsorted(sorted_suffix_ids, chunk_end_suffix)

                if query_start_idx == query_end_idx:
                    continue

                sub_q_keys = q_keys_sorted[query_start_idx : query_end_idx] # [num_parents_in_chunk, K]
                sub_p_q = p_q_sorted[query_start_idx : query_end_idx]       # [num_parents_in_chunk]
                
                # Relative row index in our accumulator tensor [0, current_chunk_size - 1]
                sub_row_indices = sorted_suffix_ids[query_start_idx : query_end_idx] - chunk_start_suffix

                # Allocate pristine VRAM flux accumulator strictly for this suffix block
                chunk_accumulated_flux = torch.zeros(
                    (current_chunk_size, vocab_size), 
                    dtype=torch.float32, 
                    device=self.device
                )

                # Process parent queries through Transformer in residual-stream-safe minibatches
                num_parents = sub_q_keys.size(0)
                for batch_start in range(0, num_parents, self.query_batch_size):
                    batch_end = min(batch_start + self.query_batch_size, num_parents)
                    
                    batch_q = sub_q_keys[batch_start : batch_end]
                    batch_p = sub_p_q[batch_start : batch_end]
                    batch_rows = sub_row_indices[batch_start : batch_end]

                    # Query Transformer Forward Pass -> Transition Probs [batch_B, |V|]
                    batch_P = abstract_forward_pass(
                        context_vals=self.frozen_context_vals,
                        context_keys=self.frozen_context_keys,
                        query_keys=batch_q,
                        model=self.model,
                        **self.forward_pass_kwargs
                    )

                    # Compute exact raw flux contribution
                    batch_flux = batch_p.unsqueeze(1) * batch_P # [batch_B, |V|]

                    # Natively aggregate exact multi-path flux across duplicate suffix rows
                    chunk_accumulated_flux.scatter_add_(
                        0, batch_rows.unsqueeze(1).expand(-1, vocab_size), batch_flux
                    )

                # --- EXECUTE POST-AGGREGATION GUILLOTINE ---
                # The accumulator now holds 100% exact converged probability flux
                survivor_mask = chunk_accumulated_flux >= self.epsilon_power_method
                dissipated_mask = ~survivor_mask
                
                total_dissipated_flux += chunk_accumulated_flux[dissipated_mask].sum().item()

                rows, cols = torch.where(survivor_mask)
                if rows.numel() > 0:
                    surviving_fluxes = chunk_accumulated_flux[rows, cols]
                    
                    # Assemble successor K-grams (Guaranteed 100% Unique)
                    # Prefix is unique_suffixes[chunk_start_suffix + rows]
                    surviving_prefixes = unique_suffixes[chunk_start_suffix + rows]
                    appended_tokens = cols.unsqueeze(1)
                    successors = torch.cat([surviving_prefixes, appended_tokens], dim=1) # [N_chunk_survivors, K]

                    all_surviving_successors.append(successors)
                    all_surviving_fluxes.append(surviving_fluxes)

            if not all_surviving_successors:
                print("[!] Catastrophic Dissipation: All fully aggregated fluxes dropped below epsilon_prune.")
                break

            # Assemble surviving states across suffix chunks (Zero duplicate coalescing needed!)
            surviving_successors_cat = torch.cat(all_surviving_successors, dim=0) # [M_surv, K]
            surviving_fluxes_cat = torch.cat(all_surviving_fluxes, dim=0)         # [M_surv]

            # -----------------------------------------------------------------
            # STEP C: Global Absorption & The 3-Part Thermodynamic Audit
            # -----------------------------------------------------------------
            next_stat_probs = torch.zeros(N_current, dtype=torch.float32, device=self.device)
            
            succ_tuples = [tuple(r) for r in surviving_successors_cat.tolist()]
            frontier_keys_list: List[torch.Tensor] = []
            frontier_flux_list: List[torch.Tensor] = []
            
            internal_flux_sum = 0.0
            frontier_flux_sum = 0.0

            for idx_succ, key_tuple in enumerate(succ_tuples):
                flux_val = surviving_fluxes_cat[idx_succ]
                if key_tuple in state_to_idx:
                    idx_global = state_to_idx[key_tuple]
                    next_stat_probs[idx_global] += flux_val
                    internal_flux_sum += flux_val.item()
                else:
                    frontier_keys_list.append(surviving_successors_cat[idx_succ])
                    frontier_flux_list.append(flux_val)
                    frontier_flux_sum += flux_val.item()

            # Absorb newly discovered Frontier States into persistent global tensors
            if frontier_keys_list:
                new_keys_tensor = torch.stack(frontier_keys_list, dim=0)
                new_fluxes_tensor = torch.stack(frontier_flux_list, dim=0)
                
                start_idx = N_current
                for i, key_tuple in enumerate([tuple(r) for r in new_keys_tensor.tolist()]):
                    state_to_idx[key_tuple] = start_idx + i
                    
                global_support_keys = torch.cat([global_support_keys, new_keys_tensor], dim=0)
                next_stat_probs = torch.cat([next_stat_probs, new_fluxes_tensor], dim=0)

            # -----------------------------------------------------------------
            # STEP D: Renormalization & Convergence Check
            # -----------------------------------------------------------------
            total_flux = internal_flux_sum + frontier_flux_sum + total_dissipated_flux
            
            internal_rate = internal_flux_sum / total_flux
            frontier_rate = frontier_flux_sum / total_flux
            leakage_rate = total_dissipated_flux / total_flux

            history_internal.append(internal_rate)
            history_frontier.append(frontier_rate)
            history_leakage.append(leakage_rate)

            # Renormalize state measure for step t+1
            global_stat_probs = next_stat_probs / next_stat_probs.sum()

            print(
                f"Iter {iteration:02d} | Set Size: {global_support_keys.size(0):<6} | "
                f"Internal Rate: {internal_rate * 100:>6.2f}% | "
                f"Frontier Rate: {frontier_rate * 100:>6.2f}% | "
                f"Leakage Rate: {leakage_rate * 100:>6.2f}%"
            )

            if frontier_rate < frontier_threshold and leakage_rate < leakage_threshold:
                stable_steps_count += 1
                if stable_steps_count >= min_stable_steps:
                    print(f"\n[+] Metastable Set Successfully Isolated at Iteration {iteration}!")
                    converged = True
                    break
            else:
                stable_steps_count = 0

        return {
            "closed_support_keys": global_support_keys,
            "stationary_measure": global_stat_probs,
            "history_internal_rates": history_internal,
            "history_frontier_rates": history_frontier,
            "history_leakage_rates": history_leakage,
            "is_converged": converged,
            "total_iterations": final_iter
        }
    
    @torch.no_grad()
    def run_dijkstra_support_discovery(
        self,
        initial_kgrams: torch.Tensor,      # [N_init, K] Seed states
        audit_interval: int = 2000         # Print progress every N expanded nodes
    ) -> Dict[str, Any]:
        """
        Explores the K-gram surprisal manifold using Batched Dijkstra.
        Halts exactly when all paths with cumulative surprisal <= S_max are exhausted.
        """
        # 1. Deduplicate initial seed states
        seed_keys, _ = torch.unique(initial_kgrams.to(self.device), dim=0, return_inverse=True)
        N_init = seed_keys.size(0)

        # 2. Compute exact mathematical Surprisal Barrier (S_max)
        # Accounting for initial uniform distribution mass: p_0 = 1 / N_init
        effective_epsilon = self.epsilon_dijkstra * N_init
        if effective_epsilon >= 1.0:
            raise ValueError(f"[!] Degenerate pruning threshold: epsilon * N_init ({effective_epsilon}) >= 1.0")
            
        S_max = -math.log(effective_epsilon)

        print(f"[*] Starting Dijkstra Support Discovery (Seeds: {N_init}, K={self.k_value})")
        print(f"[*] Absolute Action Barrier S_max: {S_max:.4f} nats (Flux limit: {self.epsilon_dijkstra:.2e})")

        # 3. Priority Queue (Min-Heap) & Visited Sets
        # Heap stores tuples: (cumulative_action_float, kgram_token_tuple)
        heap: List[Tuple[float, Tuple[int, ...]]] = []
        best_action: Dict[Tuple[int, ...], float] = {}
        expanded_set: Set[Tuple[int, ...]] = set()

        # Initialize heap with seed states at action S = 0.0
        for row in seed_keys:
            state_tuple = tuple(row.tolist())
            best_action[state_tuple] = 0.0
            heapq.heappush(heap, (0.0, state_tuple))

        total_forward_passes = 0
        last_audit = 0

        # 4. Batched Dijkstra Traversal Loop
        while heap:
            batch_tuples: List[Tuple[int, ...]] = []
            batch_actions: List[float] = []

            # Pop up to query_batch_size unvisited states from the min-heap
            while heap and len(batch_tuples) < self.query_batch_size:
                current_action, state_tuple = heapq.heappop(heap)

                # Standard Dijkstra Closed-List Check
                if state_tuple in expanded_set:
                    continue

                # If we popped a state whose best known action already exceeded S_max, we are done!
                if current_action > S_max:
                    break

                # Mark state as permanently expanded (shortest path finalized)
                expanded_set.add(state_tuple)
                batch_tuples.append(state_tuple)
                batch_actions.append(current_action)

            if not batch_tuples:
                break

            total_forward_passes += 1

            # Audit logging
            if len(expanded_set) - last_audit >= audit_interval:
                last_audit = len(expanded_set)
                curr_max_act = max(batch_actions)
                print(f" -> Expanded Support: {len(expanded_set):<7} | Current Frontier Action: {curr_max_act:<6.2f} / {S_max:.2f} nats")

            # Convert popped batch to tensors
            q_keys_chunk = torch.tensor(batch_tuples, dtype=torch.long, device=self.device)   # [B, K]
            actions_chunk = torch.tensor(batch_actions, dtype=torch.float32, device=self.device) # [B]

            # Query Transformer Forward Pass -> Transition Probabilities [B, |V|]
            P_chunk = abstract_forward_pass(
                context_vals=self.frozen_context_vals,
                context_keys=self.frozen_context_keys,
                query_keys=q_keys_chunk,
                model=self.model,
                **self.forward_pass_kwargs
            )

            # Compute Step Surprisal S(u -> v) = -log P(v|u)
            # Clamped at 1e-30 to prevent log(0) -> -inf
            step_surprisal = -torch.log(P_chunk.clamp(min=1e-30)) # [B, |V|]

            # Additive Action Accumulation: S_next = S_u + S_(u->v)
            next_actions = actions_chunk.unsqueeze(1) + step_surprisal # [B, |V|]

            # Guillotine Pruning on the Action Manifold
            valid_mask = next_actions <= S_max
            rows, cols = torch.where(valid_mask)

            if rows.numel() == 0:
                continue

            # Construct Shifted Successor K-grams purely for surviving edges
            prefixes = q_keys_chunk[rows, 1:] # [N_survivors, K-1]
            appended = cols.unsqueeze(1)      # [N_survivors, 1]
            successors_chunk = torch.cat([prefixes, appended], dim=1) # [N_survivors, K]
            surviving_next_actions = next_actions[rows, cols]         # [N_survivors]

            # Convert surviving successors to Python structures for fast hash-routing
            succ_tuples_list = [tuple(r) for r in successors_chunk.tolist()]
            surv_acts_list = surviving_next_actions.tolist()

            # Push surviving successors into the priority queue
            for succ_tuple, cand_action in zip(succ_tuples_list, surv_acts_list):
                # If successor is already in closed list, its shortest path is already finalized
                if succ_tuple in expanded_set:
                    continue

                # Open list relaxation check
                if cand_action < best_action.get(succ_tuple, float('inf')):
                    best_action[succ_tuple] = cand_action
                    heapq.heappush(heap, (cand_action, succ_tuple))

        # 5. Finalize Closed Support Tensor
        sorted_support_tuples = sorted(list(expanded_set))
        closed_support_tensor = torch.tensor(sorted_support_tuples, dtype=torch.long, device=self.device)
        final_actions_tensor = torch.tensor([best_action[s] for s in sorted_support_tuples], dtype=torch.float32, device=self.device)

        print(f"\n[+] Dijkstra Manifold Exhausted! Final Support Size: {closed_support_tensor.size(0)} K-grams.")
        print(f"[*] Total GPU Forward Pass Batches Executed: {total_forward_passes}")

        return {
            "closed_support_keys": closed_support_tensor, # [N_final, K]
            "support_actions_nats": final_actions_tensor, # [N_final] -> Minimum surprisal from seed
            "total_support_size": closed_support_tensor.size(0),
            "total_forward_passes": total_forward_passes
        }

@torch.no_grad()
def extract_substochastic_matrix(
    closed_support_keys: torch.Tensor, # [N, K] canonical integer K-grams
    frozen_context_vals: torch.Tensor, # [\pi] macroscopic slow context density
    frozen_context_keys: torch.Tensor, # Frozen background KV tokens
    model: Any,                        # HookedTransformer instance
    forward_pass_kwargs: Dict[str, Any],# Hooks & params (active_heads, LN_sigma, etc.)
    epsilon_edge: float = 1e-9,        # Noise floor to prune minuscule softmax tails
    query_batch_size: int = 1024,      # Safe minibatch limit for residual stream forward pass
    device: str = "cuda"
) -> Dict[str, Any]:
    """
    Constructs the augmented (N+1) x (N+1) sub-stochastic Markov transition matrix
    over the closed K-gram support set, routing all escaping probability flux into a Sink state.
    
    Returns a dictionary containing a memory-safe PyTorch Sparse COO tensor.
    """
    # 1. Canonicalize Support Set
    states, _ = torch.unique(closed_support_keys.to(device), dim=0, return_inverse=True)
    N = states.size(0)
    sink_idx = N

    # Dynamically extract vocabulary size via a single dummy query
    dummy_P = abstract_forward_pass(
        context_vals=frozen_context_vals,
        context_keys=frozen_context_keys,
        query_keys=states[:1],
        model=model,
        **forward_pass_kwargs
    )
    vocab_size = dummy_P.size(-1)

    print(f"[*] Extracting Sub-Stochastic Matrix | Support: {N} states | Vocab: {vocab_size}")

    # 2. Build 64-bit Suffix-Prefix Hash Table
    if states.size(1) > 1:
        query_suffixes = states[:, 1:] # [N, K-1]
        target_prefixes = states[:, :-1] # [N, K-1]

        # Unified space of all (K-1)-grams
        combined_km1, km1_inverse = torch.unique(
            torch.cat([query_suffixes, target_prefixes], dim=0), dim=0, return_inverse=True
        )
        query_suffix_ids = km1_inverse[:N].to(torch.int64)
        target_prefix_ids = km1_inverse[N:].to(torch.int64)
    else:
        # K == 1 (unigrams): both the parent suffix and the target prefix are the empty
        # sequence, so the entire (K-1)-gram space collapses to a single class with id 0.
        # torch.unique(dim=0) cannot process the zero-width tensor, so assign it directly.
        # The 1D coordinate below then degenerates to the plain token id, i.e. the edge
        # u -> v exists iff the successor token v is itself a state of the support.
        query_suffix_ids = torch.zeros(N, dtype=torch.int64, device=device)
        target_prefix_ids = torch.zeros(N, dtype=torch.int64, device=device)

    target_last_tokens = states[:, -1].to(torch.int64)

    # Map every target state in C to a unique 1D 64-bit integer coordinate
    target_coords = target_prefix_ids * vocab_size + target_last_tokens
    sorted_target_coords, sort_order = torch.sort(target_coords)
    sorted_target_indices = torch.arange(N, dtype=torch.long, device=device)[sort_order]

    # Edge Accumulators
    all_rows: List[torch.Tensor] = []
    all_cols: List[torch.Tensor] = []
    all_vals: List[torch.Tensor] = []
    
    internal_prob_sums = torch.zeros(N, dtype=torch.float32, device=device)

    # 3. Minibatch Query Processing over the Support Set
    for batch_start in range(0, N, query_batch_size):
        batch_end = min(batch_start + query_batch_size, N)
        batch_q_keys = states[batch_start:batch_end]
        batch_global_indices = torch.arange(batch_start, batch_end, dtype=torch.long, device=device)

        # Query Transformer Forward Pass -> Transition Probs [B, |V|]
        P_batch = abstract_forward_pass(
            context_vals=frozen_context_vals,
            context_keys=frozen_context_keys,
            query_keys=batch_q_keys,
            model=model,
            **forward_pass_kwargs
        )

        # Prune Softmax tail noise
        batch_r, batch_c = torch.where(P_batch >= epsilon_edge)
        if batch_r.numel() == 0:
            continue

        probs = P_batch[batch_r, batch_c]
        parent_global_idx = batch_global_indices[batch_r]
        parent_suffix_id = query_suffix_ids[parent_global_idx]

        # Candidate transition 1D coordinates
        trans_coords = parent_suffix_id * vocab_size + batch_c.to(torch.int64)

        # Lightning binary search against established closed targets
        insert_positions = torch.searchsorted(sorted_target_coords, trans_coords)
        
        valid_bounds = insert_positions < N
        matched_mask = valid_bounds & (
            sorted_target_coords[insert_positions.clamp(max=N-1)] == trans_coords
        )

        # Extract internal surviving transitions
        if matched_mask.any():
            internal_rows = parent_global_idx[matched_mask]
            internal_cols = sorted_target_indices[insert_positions[matched_mask]]
            internal_vals = probs[matched_mask]

            all_rows.append(internal_rows)
            all_cols.append(internal_cols)
            all_vals.append(internal_vals)

            # Accumulate surviving internal flux per query inside this batch
            internal_prob_sums.scatter_add_(0, internal_rows, internal_vals)

    # 4. Compute Conservation Leakage to Sink State
    # P(i -> Sink) = 1.0 - sum(P_internal)
    leakage_probs = (1.0 - internal_prob_sums).clamp(min=0.0)
    leaking_nodes = torch.where(leakage_probs >= epsilon_edge)[0]

    if leaking_nodes.numel() > 0:
        all_rows.append(leaking_nodes)
        all_cols.append(torch.full_like(leaking_nodes, sink_idx))
        all_vals.append(leakage_probs[leaking_nodes])

    # 5. Add Sink Self-Loop (Absorbing Boundary)
    all_rows.append(torch.tensor([sink_idx], dtype=torch.long, device=device))
    all_cols.append(torch.tensor([sink_idx], dtype=torch.long, device=device))
    all_vals.append(torch.tensor([1.0], dtype=torch.float32, device=device))

    # Construct Final PyTorch Sparse COO Tensor
    final_rows = torch.cat(all_rows, dim=0)
    final_cols = torch.cat(all_cols, dim=0)
    final_vals = torch.cat(all_vals, dim=0)

    sparse_transition_matrix = torch.sparse_coo_tensor(
        indices=torch.stack([final_rows, final_cols], dim=0),
        values=final_vals,
        size=(N + 1, N + 1)
    ).coalesce()

    mean_leakage = leakage_probs.mean().item()
    print(f"[+] Matrix Extraction Complete | Sparse Edges: {final_vals.numel()} | Mean Leakage: {mean_leakage:.4e}")

    return {
        "sparse_transition_matrix": sparse_transition_matrix, # Shape [N+1, N+1]
        "canonical_support_keys": states,                     # Shape [N, K]
        "sink_state_index": sink_idx,                         # Integer N
        "leakage_per_state": leakage_probs,                   # Shape [N]
        "mean_leakage_rate": mean_leakage
    }

class MetastableMSMAnalyzer:
    """
    Executes SCC-pruned non-reversible spectral analysis and Doob-conditioned PCCA+ clustering.
    
    Natively extracts the Maximal Strongly Connected Component (the recurrent core)
    at initialization, eliminating dangling boundary leaves to guarantee exact
    stochasticity during Doob's h-transformation.
    """
    def __init__(self, sparse_transition_dict: Dict[str, Any]):
        raw_keys = sparse_transition_dict["canonical_support_keys"] # Shape [N_raw, K]
        self.sink_idx = sparse_transition_dict["sink_state_index"]  # Integer N_raw
        
        # 1. Convert PyTorch Sparse COO to SciPy CSR Matrix natively
        torch_P = sparse_transition_dict["sparse_transition_matrix"].coalesce()
        indices = torch_P.indices().cpu().numpy()
        values = torch_P.values().cpu().numpy().astype(np.float64)
        shape = torch_P.shape
        
        P_full_csr = sp.csr_matrix((values, (indices[0], indices[1])), shape=shape)
        
        # Extract purely internal N x N sub-stochastic block P_C (excluding Sink)
        P_C_csr = P_full_csr[:self.sink_idx, :self.sink_idx]
        N_raw = self.sink_idx

        print(f"[*] Initializing MSM Analyzer | Raw Internal Manifold: {N_raw} states.")

        # ---------------------------------------------------------------------
        # TOPOLOGICAL FILTER: Tarjan's Strongly Connected Component Extraction
        # ---------------------------------------------------------------------
        # Extract all SCCs (~30 ms for 100,000 states)
        num_scc, scc_labels = csgraph.connected_components(P_C_csr, connection='strong')

        # Identify the single Giant Recurrent Component (The core metastable basin)
        component_sizes = np.bincount(scc_labels)
        giant_scc_id = np.argmax(component_sizes)
        self.core_mask = (scc_labels == giant_scc_id)
        self.core_size = int(self.core_mask.sum())

        num_pruned = N_raw - self.core_size
        print(f"[+] Tarjan Topological Shear: Sheared off {num_pruned} dangling transient leaves.")
        print(f"[+] Pristine Recurrent Core Size: {self.core_size} states (Maximal SCC).")

        if self.core_size == 0:
            raise ValueError("[!] Degenerate Topology: The strongly connected core collapsed to 0 states.")

        # 2. Slice CSR matrix and Canonical Keys strictly to the Recurrent Core
        # Modern SciPy robust slicing: mat[mask][:, mask]
        self.P_core_csr = P_C_csr[self.core_mask][:, self.core_mask]
        self.core_keys = raw_keys[torch.from_numpy(self.core_mask)] # [N_core, K]

    def perform_spectral_decomposition(self, k_eigenvalues: int = 10) -> Dict[str, Any]:
        """
        Solves for the top K eigenvalues and right eigenvectors of the recurrent core,
        along with the Quasi-Stationary Distribution (First Left Eigenvector).
        Includes an automatic fallback to dense solvers if the core collapsed to a tiny loop.
        """
        print(f"[*] Solving non-reversible spectrum for Top {k_eigenvalues} states (Core N={self.core_size})...")
        
        evals: np.ndarray
        evecs: np.ndarray
        qsd_raw: np.ndarray

        # Fallback safeguard: ARPACK fails if k >= N - 1
        if self.core_size < max(k_eigenvalues + 2, 20):
            print(f"[*] Core size ({self.core_size}) small. Bypassing ARPACK -> using dense LAPACK solver.")
            P_dense = self.P_core_csr.toarray()
            
            # Extract both left (vl) and right (vr) eigenvectors natively
            evals, vl, vr = la.eig(P_dense, left=True, right=True)
            evecs = vr
            
            # Sort strictly by complex modulus magnitude |lambda| descending
            sort_order = np.argsort(-np.abs(evals))
            evals = evals[sort_order]
            evecs = evecs[:, sort_order]
            vl = vl[:, sort_order]
            
            # Isolate the leading left eigenvector
            qsd_raw = np.real(vl[:, 0])
            
        else:
            # Safe ARPACK extraction asking for k+1 to audit spectral gaps properly
            target_k = min(k_eigenvalues + 1, self.core_size - 2)
            evals, evecs = spla.eigs(self.P_core_csr, k=target_k, which='LM', tol=1e-12)
            
            # Sort strictly by complex modulus magnitude |lambda| descending
            sort_order = np.argsort(-np.abs(evals))
            evals = evals[sort_order]
            evecs = evecs[:, sort_order]

            # Solve for the leading left eigenvector using the transpose matrix P^T
            _, vl = spla.eigs(self.P_core_csr.T, k=1, which='LM', tol=1e-12)
            qsd_raw = np.real(vl[:, 0])

        # Isolate Perron-Frobenius leading eigenstate
        lambda_quasi = np.real(evals[0])
        r_quasi = np.real(evecs[:, 0])

        # Enforce positive sign convention for physical mass representation
        if np.mean(r_quasi) < 0:
            r_quasi = -r_quasi

        # --- QUASI-STATIONARY DISTRIBUTION NORMALIZATION ---
        # 1. Enforce positive sign convention
        if np.mean(qsd_raw) < 0:
            qsd_raw = -qsd_raw
            
        # 2. Clean microscopic numerical negative noise (Perron-Frobenius guarantees >= 0)
        qsd_raw = np.maximum(qsd_raw, 0.0)
        
        # 3. L1-Normalize to form a valid discrete probability distribution
        qsd_distribution = qsd_raw / np.sum(qsd_raw)

        # Container Escape Rate: gamma = 1.0 - |lambda_1|
        basin_escape_rate = 1.0 - np.abs(evals[0])

        # Calculate Implied Characteristic Mixing Timescales (in tokens)
        moduli = np.abs(evals)
        with np.errstate(divide='ignore'):
            implied_timescales = -1.0 / np.log(moduli.clip(min=1e-15, max=0.9999999999))

        # Calculate Spectral Gaps: Gap_i = |lambda_i| - |lambda_{i+1}|
        spectral_gaps = moduli[:-1] - moduli[1:]

        print(f"[+] Core Spectrum Resolved! Quasi-Stationary Mass Retention: {lambda_quasi * 100:.4f}% per step.")
        print(f"[+] Recurrent Basin Thermodynamic Escape Rate: {basin_escape_rate:.4e} per step.")

        return {
            "evals_complex": evals[:k_eigenvalues],
            "evecs_right_complex": evecs[:, :k_eigenvalues],
            "lambda_quasi": lambda_quasi,
            "r_quasi_positive": r_quasi,
            "qsd_distribution": qsd_distribution,       # Added QSD to output dictionary
            "basin_escape_rate": basin_escape_rate,
            "implied_timescales_tokens": implied_timescales[:k_eigenvalues],
            "spectral_gaps": spectral_gaps[:k_eigenvalues]
        }

    def run_conditioned_pcca(self, spectral_results: Dict[str, Any], num_clusters: int = 3) -> Dict[str, Any]:
        """
        Executes Doob's h-transform over the SCC core, extracts its exact invariant measure \pi,
        constructs the Time-Reversed Adjoint chain P*, and fits PCCA+ on the 
        symmetrized reversible manifold P_sym = (P + P*) / 2.
        """
        if not DEEPTIME_AVAILABLE:
            raise RuntimeError("[!] Cannot run PCCA+: deeptime package is not installed.")

        if num_clusters > self.core_size:
            raise ValueError(f"[!] Requested {num_clusters} clusters, but core only contains {self.core_size} states.")

        print(f"[*] Executing Doob's h-Transformation (64-bit precision)...")
        lambda_quasi = np.float64(spectral_results["lambda_quasi"])
        r_quasi = spectral_results["r_quasi_positive"].astype(np.float64)

        # Clamp right eigenstate safely above zero to prevent division errors
        r_clamped = np.maximum(r_quasi, 1e-30)
        R = sp.diags(r_clamped)
        R_inv = sp.diags(1.0 / r_clamped)

        # 1. Doob Conditioned Forward Chain: P_cond = (1 / lambda) * R^-1 @ P_core @ R
        P_cond = (1.0 / lambda_quasi) * R_inv.dot(self.P_core_csr).dot(R)

        # Scrub machine epsilon floating-point drift
        row_sums = P_cond.sum(axis=1).A.squeeze()
        Row_norm = sp.diags(1.0 / np.maximum(row_sums, 1e-15))
        P_cond_stoch = Row_norm.dot(P_cond) # Strictly stochastic forward chain

        print(f"[*] Extracting Core Invariant Measure (\pi) for Adjoint Reversibilization...")
        # Fit a temporary non-reversible MSM strictly to solve for the exact stationary vector \pi
        temp_msm = msm.MarkovStateModel(P_cond_stoch, reversible=False, transition_matrix_tolerance=1e-12)
        pi_stat = temp_msm.stationary_distribution.astype(np.float64)

        # 2. Construct the Time-Reversed Adjoint Chain: P* = \Pi^-1 @ P_cond^T @ \Pi
        pi_clamped = np.maximum(pi_stat, 1e-30)
        Pi_mat = sp.diags(pi_clamped)
        Pi_inv = sp.diags(1.0 / pi_clamped)

        P_adjoint = Pi_inv.dot(P_cond_stoch.T).dot(Pi_mat)

        print(f"[*] Symmetrizing Dynamics: P_sym = 0.5 * (P_cond + P*)...")
        # 3. Fill's Additive Symmetrization
        P_sym = 0.5 * (P_cond_stoch + P_adjoint)

        # Final double-precision scrubbing to guarantee absolute numerical reversibility
        sym_row_sums = P_sym.sum(axis=1).A.squeeze()
        P_sym_stoch = sp.diags(1.0 / np.maximum(sym_row_sums, 1e-15)).dot(P_sym)

        print(f"[*] Fitting Reversibilized MSM & PCCA+ ({num_clusters} macrostates)...")
        # We can now explicitly activate reversible=True! Spectrum is guaranteed 100% real.
        final_msm = msm.MarkovStateModel(P_sym_stoch, reversible=True, transition_matrix_tolerance=1e-12)
        pcca_obj = final_msm.pcca(n_metastable_sets=num_clusters)
        
        memberships = pcca_obj.memberships # Shape [N_core, num_clusters]
        crisp_assignments = np.argmax(memberships, axis=1)

        cluster_masses = [np.sum(pi_stat[crisp_assignments == c]) for c in range(num_clusters)]

        print(f"[+] PCCA+ Simplex Converged! Macrostate Invariant Mass Split: {[round(m*100, 2) for m in cluster_masses]}%")

        return {
            "pcca_memberships": memberships,               # [N_core, M] Float
            "crisp_clusters": crisp_assignments,           # [N_core] Integer
            "stationary_measure_pi": pi_stat,              # [N_core] Float (Exact unperturbed Doob mass)
            "macrostate_mass_proportions": cluster_masses, # List[float] summing to 1.0
            "core_support_keys": self.core_keys            # [N_core, K] PyTorch Integer Tensor
        }

#### Plotting Functions and Profiling

In [ ]:
### Plotting Functions ###

def plot_head_thermodynamic_profile(
    mu_slice: torch.Tensor, 
    sigma_slice: torch.Tensor, 
    layer_idx: int = None, 
    head_idx: int = None,
    sigma_multiplier: float = 3.0
):
    """
    Plots the positional baseline and semantic spread of a specific attention head.
    
    Args:
        mu_slice: 1D Tensor of positional means (shape: [seq_len])
        sigma_slice: 1D Tensor of contextual semantic variances (shape: [seq_len])
        layer_idx: (Optional) Layer index for the title.
        head_idx: (Optional) Head index for the title.
        sigma_multiplier: Sets the bounds. 3.0 captures 99.7% of semantic features.
    """
    # Move to CPU and convert to numpy for matplotlib
    mu = mu_slice.detach().cpu().numpy()
    sigma = sigma_slice.detach().cpu().numpy()
    
    seq_len = len(mu)
    distances = np.arange(seq_len)
    
    # Calculate the competitive bounds (Best-case vs Worst-case semantic draws)
    upper_bound = mu + (sigma_multiplier * sigma)
    lower_bound = mu - (sigma_multiplier * sigma)
    
    # Create the figure (Wide aspect ratio to stretch out the 1024 points cleanly)
    fig, ax = plt.subplots(figsize=(14, 5), dpi=150)
    
    # 1. Plot the Semantic Spread (The "Tube")
    ax.fill_between(
        distances, 
        lower_bound, 
        upper_bound, 
        color='#4C72B0', 
        alpha=0.3, 
        label=f'Semantic Spread ($\pm {sigma_multiplier}\sigma$)'
    )
    
    # 2. Plot the Positional Baseline (The exact structural routing)
    ax.plot(
        distances, 
        mu, 
        color='#003366', 
        linewidth=2, 
        label=r'Positional Baseline ($\mu_d$)',
        marker="o",
        markersize=3
    )
    
    # Add a horizontal line at the far-field noise floor (estimated from the last 10% of context)
    noise_floor_mu = np.mean(mu[int(seq_len * 0.9):])
    ax.axhline(noise_floor_mu, color='red', linestyle='--', alpha=0.7, label='Far-Field Thermodynamic Noise Floor')

    # Formatting and labels
    title = "Thermodynamic Head Profile"
    if layer_idx is not None and head_idx is not None:
        title += f" (Layer {layer_idx}, Head {head_idx})"
        
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Relative Distance ($d = i - j$)', fontsize=12)
    ax.set_ylabel('Pre-Softmax Logit Score', fontsize=12)
    
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper right', framealpha=0.9)
    
    # Because deep heads often have intense local action and flat far-fields, 
    # it is sometimes helpful to set the x-axis limits to start exactly at 1 (skipping d=0 if it's an artifact).
    ax.set_xlim(0, seq_len - 1)
    
    plt.tight_layout()
    plt.show()

### Load Model and Generate

In [ ]:
# Load model
# device = utils.get_device()
device = 'cuda:0'
model_name = "attn-only-1l"
# model_name = "attn-only-2l"
# model_name = "gelu-2l"
# model_name = "gpt2-small"
# model_name = "roneneldan/TinyStories-1M"
# model_name = "tiny-stories-1L-21M"
# model_name = "pythia-14m"
# model_name = "pythia-31m"
# model_name = "pythia-70m"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
model = HookedTransformer.from_pretrained(model_name, device=device)
# model = HookedTransformer.from_pretrained(model_name, device=device, fold_ln=False, dtype=torch.bfloat16)

model.cfg.use_attn_result = True # make the model use the inefficient OV computation
# torch.set_grad_enabled(False) # turn off automatic differentiation

# U_basis, S_spectrum = compute_unembedding_svd(model)

In [ ]:
layer = 0
head = 0
U, S, Vh = torch.linalg.svd(model.W_V[layer,head])
plt.figure(figsize=(10,5))
plt.plot(S.cpu(), '-o')
plt.xlabel("#Order")
plt.ylabel("Singular Value")
plt.grid()
plt.ylim(bottom=0.5, top=1.6)
plt.title(f"SVD Spectrum of Value Projection, head {layer}.{head}")
plt.show()

# print(f"Mean singular value: {S.max() / S.mean()}")
print(f"PAPR: {S.max() / S.mean()}")

In [ ]:
# generate
prompt = "Once upon a time there was a beautiful princess. She was in love with the proud prince who was going to rule all of the kingdom. She was"
# prompt = "The music was splendid. The clarinet played beautifully, and the saxophones roared with jazz delight. The orchestra was playing greate music with amazing groove and"
# prompt = "Question: what is the best movie of all time? Answer:"
# generated = "The players ran on the grass and passed the ball between them. The goalkeeper was standing between the posts, and the football was going great. The players kicked the ball was the ball and the ball was the ball and the ball was the ball. The ball was the ball was the ball was the ball and the ball was the ball. The ball was the ball was the ball was the ball of the ball. The ball was ball was the ball was the ball, the ball was the ball. The ball was the ball was the ball was the ball. The ball was the ball was the ball was the ball and the ball was the ball. The ball was the ball was the ball was the ball. The ball was the ball was the ball was the ball. The ball was the ball was the ball. The ball was the ball was the ball was the ball. The ball was the ball was the ball and the ball was the ball. The ball was the ball was the ball. The ball was the ball was the ball. The ball was the ball was the ball was the ball. The ball was the ball was the ball. The ball was the ball was the ball. The ball was the ball was the ball. The ball was the ball. The ball was the ball was the ball. The ball was the ball was the ball. The ball was the ball was ball. The ball was the ball. The ball was the ball was the ball. The ball was the ball. The ball was the ball. The ball was the ball was the ball. The ball was the ball was the ball. The ball was ball. The ball was the ball. The ball was ball was ball. The ball was ball. The ball was ball. The ball was the ball. The ball was ball. The ball was ball. The ball was the ball was the ball. The ball was the ball. The ball was ball. The ball. The ball was ball. The ball was ball. The ball was ball. The ball was ball. The ball was ball. The ball was ball. The ball was ball. The ball was ball. The ball. The ball was the ball. The ball was ball. The ball was ball. The ball was ball. The ball. The ball was ball. The ball was ball. The ball was ball. The ball was ball. The ball was ball. The ball. The ball was ball. The ball. The ball. The ball was ball. The ball. The ball. The ball was ball. The ball. The ball. The ball was ball. The ball. The ball was ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball. The ball"
# generated = 'The music was splendid. The clarinet played beautifully, and the saxophones roared with jazz delight. The orchestra was playing greate music with amazing groove and the melodic/ole-s-Amplification, producing further feel extremely sharp, nice, crisp, with relaxed atmosphere and care. Perfect for a lively atmosphere, melodic piano accompaniment.\nThe St. Patrick’s-formed drum ensemble.The band played on how to not-ize the singer’s influence on tempo. While any musician I chose a music conductor or stage that if the audience was not properly laid in jazz. The ensemble was, brilliant as an instrumental \'3 of Symphony No.2, it sounded classic and sound had very hard to develop than Europe. Also, the sound into the world’s second chamber bassin. Up each course will blend the classical sounds of the playing of jazz. Orchestrator Jazz show, and jazz and minor.\nThe band features are in the same chamber melodetis, groove set is a powerful material to the and production chamber. This was the most expensive: chords of improv action. The fusion band in full and formal musicians to create a piece that heaps of jazz and on them and blends more freely into the music. The pop you live in a vinyl show-punk chamber (found in such an interactive Tilez is a band that all of the music comes in a bold 75 x 10 a night to the studio version of the CD and Tentowers. The bass flute. Our recordings are of solid rock music on the bass have been jam for use in compressing the first dance clubs, the bass and melodies. Up the song and play in other lines have helped make him play a key one of the key pieces.\nThe nine - The underval is set is tethered. With 3s 601 is set, the record instruments you will continue playing with a role that plays an instrument or give a far-reaching blues. This set could be a place to play the piano on a great jazz piano! He is an ennio’s. You could hear from the beautiful harms in the grooves on the track, playing and piano. The piano is a highly vocal; recording is also the first electric piano, instrument to a stage piano, a piano version of piano and a piano. After all, first recorded the piano is made up of 30 years, the piano, the piano, piano and it would keep your performance.\nIf you have any sound and feels like more deeply moving, which a lot of space in your age. If it is any one or more pianist, the pianist, and accompanist, and piano and piano is an accompanist!\nFor pianist is the piano but does the Pinet is more harmonious with the piano accompanist for your piano and played by bass or bass clarinet piano. It is the feeling of.\nThe acoustic track is one of the piano’s wide-open door, Keygrass piano, and the highest say that piano fits well. We are a musician, the piano to listen to the composition of the piano and the contemporary’s melodies into piano, although, that I rarely hear, see the melodies, a whole range of piano melodies, rhythmic, and piano, orchestral works and perform with pianos, just like Roland Allen, and piano.\nThis can be more expressive and he works with keyboard.\nThalpit Strings - pianos, he, with fugitives, and in one, while Chlalman and his Ellman met those he found his most while playing piano, his piano keys piano is at home.\nQ. Does this piano sound Piano : piano, violin, piano?\nWhen I buy pianos are pianos, and two piano keys. Music."\nThe general modern I\'d recommend the Piano and piano for piano as with, pianos are pianos and horn. If one sounds on piano, 4 or so, the original organ or two pianos'
# generated = "Once upon a time there was a beautiful princess. She was in love with the proud prince who was going to rule all of the kingdom. She was very happy.\n\nOne day, the princess decided to go for a walk in the forest. She wandered around and looked for the trees and the birds. She looked for a long time, but she couldn't find the prince.\n\nThe prince noticed a big, bad witch. He was always mean to her and asked her to forgive him. She was very kind and told him he should never wander off. The prince was very sad and he knew he was in trouble.\n\nThe prince and princess decided to talk to the old witch and tell her stories. He told her about his adventures and how he would always forgive her. The princess was so happy that she gave him a big hug.\n\nThe old witch was so happy that she had made a new friend. She told the prince to never wander off again and to always be kind and forgive. The princess was glad that she was there to help her friend.\n"
# prompt = bla + " very" * 11
# prompt = "So many millions of tokens of Arbitrary Noise that interrupt the prompt repeatedly that seem unrelated.\nWill this matter to the model at all?\nI believe that it should, because the doctor said so."
input_tokens = model.to_tokens(prompt)

generated = model.generate(
    prompt,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.9,
    top_p=0.8,
    prepend_bos=True,
    stop_at_eos=True,
    # return_type="tokens"
)

print(generated)
# print(model.to_string(generated[0]))

In [ ]:
## Generation with Knockout
# prompt = " downstream upstream downstream upstream downstream upstream downstream upstream downstream upstream downstream upstream downstream upstream account account account account account account account account account account account account account account account account account account upstream account upstream downstream" * 1 + " tiny" * 0
# prompt = " account" * 46*3 + " downstream" * 30*3 + " upstream" * 18*3
# prompt = " music jazz playing" * 33
prompt = " Matra" * 200
# prompt = "Once upon a time there was a beautiful princess. She was in love with the proud prince who was going to rule all of the kingdom. She was"
# possible_tokens = torch.tensor([15023, 17423, 2320])
# prompt = model.to_string(possible_tokens[torch.multinomial(torch.tensor([0.2, 0.2, 0.6]), num_samples=100, replacement=True)])
# prompt = model.to_string(context_from_pi(bla, N=200))
# prompt = torch.tensor([[26924] * 100]).to(device)
# prompt = 'An astronaut (from the Ancient Greek  (), meaning \'star\', and  (), meaning \'sailor\') is a person trained, equipped, and deployed by a human spaceflight program to serve as a commander or crew member aboard a spacecraft. Although generally reserved for professional space travelers, the term is sometimes applied to anyone who travels into space, including scientists, politicians, journalists, and tourists.\n\n"Astronaut" technically applies to all human space travelers regardless of nationality. However, astronauts fielded by Russia or the Soviet Union are typically known instead as cosmonauts (from the Russian "kosmos" (космос), meaning "space", also borrowed from Greek ). Comparatively recent developments in crewed spaceflight made by China have led to the rise of the term taikonaut (from the Mandarin "tàikōng" (), meaning "space"), although its use is somewhat informal and its origin is unclear. In China, the People\'s Liberation Army Astronaut Corps astronauts and their foreign counterparts are all officially called hángtiānyuán (, meaning "heaven navigator" or literally "heaven-sailing staff").\n\nSince 1961, 600 astronauts have flown in space.  Until 2002, astronauts were sponsored and trained exclusively by governments, either by the military or by civilian space agencies. With the suborbital flight of the privately funded SpaceShipOne in 2004, a new category of astronaut was created: the commercial astronaut.\n\nDefinition \n\nThe criteria for what constitutes human spaceflight vary, with some focus on the point where the atmosphere becomes so thin that centrifugal force, rather than aerodynamic force, carries a significant portion of the weight of the flight object. The Fédération Aéronautique Internationale (FAI) Sporting Code for astronautics recognizes only flights that exceed the Kármán line'
max_new_tokens = 500

do_sample = True
temperature = 1.0
top_p = 1.0
prepend_bos = True

print("Normal Generation:")
clean_output = model.generate(prompt, max_new_tokens=max_new_tokens, do_sample=do_sample, top_p=top_p, temperature=temperature, prepend_bos=prepend_bos)
print(f"Result: {clean_output}\n")

# 2. Generate with temporary knockout
target_layer = 0
target_head = [0,1,2,3,5,6,7]
knockout_positional = True

patched_output = generate_with_knockout(
    model, 
    prompt, 
    layer=target_layer, 
    head_idx=target_head, 
    max_tokens=max_new_tokens,
    knockout_positional=knockout_positional,
    do_sample=do_sample,
    top_p=top_p,
    prepend_bos=prepend_bos
)

print(f"Result: {patched_output}\n")


In [ ]:
pi = pi_t_from_context(model.to_tokens(patched_output)[0])
diffs = torch.diff(pi.detach(), dim=0)

plt.figure(figsize=(10,8))
plt.plot(np.arange(diffs.shape[0]) * diffs.abs().sum(dim=-1).cpu().numpy())
plt.grid()
plt.show()

In [ ]:
plt.plot(pi[:, 16722].cpu().numpy())
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
plt.hist(pi[-1].topk(200).values.cpu().numpy(), bins=100)
plt.grid()
plt.show()

In [ ]:
# Run with cache
# prompt = "Once upon a time there was a beautiful princess. She was in love with the proud prince who was going to rule all of the kingdom. She was"
# prompt = " Jack" * 1000
# prompt = ' yes' * 1000
# input_tokens = model.to_tokens(prompt, prepend_bos=True)
# input_tokens = torch.tensor([[1] + [18462] * 1000])
input_tokens = model.to_tokens(generated)
# input_tokens = model.to_tokens(patched_output)
# input_tokens = model.to_tokens(clean_output)
# print(input_tokens.device)
layer = 0
head_idx = []
min_position = 0
knockout_positional = False

hook_name = f"blocks.{layer}.attn.hook_result"
hook_fnc = partial(zero_head_hook, head_idx=head_idx, min_position=min_position)
hooks = [(hook_name, hook_fnc)]

# temporary layer 1 knockout
# hook_name_layer_1 = f"blocks.1.attn.hook_result"
# hook_fnc_layer_1 = partial(zero_head_hook, head_idx=[0,1,2,3,4,5,6,7], min_position=min_position)
# hooks += [(hook_name_layer_1, hook_fnc_layer_1)]

if knockout_positional:
    positional_hook_name = "hook_pos_embed"
    positional_hook_fn = remove_pos_embed_hook
    hooks += [(positional_hook_name, positional_hook_fn)]

with torch.no_grad():
    with model.hooks(fwd_hooks=hooks):
        logits, cache = model.run_with_cache(input_tokens, remove_batch_dim=True, pos_slice=[6,7,8], 
                                             names_filter=lambda name: name.endswith("hook_result")
                                             )
    
probs = F.softmax(logits / 1.0, dim=-1)

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(probs[0,:, 3894].cpu().numpy(), '--.')
plt.grid()
plt.show()

In [ ]:
### DLA computation ###

# Compute DLA by projecting the head results through the unembedding matrix (W_U)
# W_U shape: [d_model, vocab_size]
# Output shape: [seq_len, num_heads, vocab_size]
dla_0 = einops.einsum(
    cache["blocks.0.attn.hook_result"] - cache["blocks.0.attn.hook_result"].mean(dim=-1, keepdim=True), 
    model.W_U, 
    "seq_len num_heads d_model, d_model vocab_size -> seq_len num_heads vocab_size"
)
if model.cfg.n_layers > 1:
    dla_1 = einops.einsum(
        cache["blocks.1.attn.hook_result"] - cache["blocks.1.attn.hook_result"].mean(dim=-1, keepdim=True), 
        model.W_U, 
        "seq_len num_heads d_model, d_model vocab_size -> seq_len num_heads vocab_size"
    )
else:
    dla_1 = torch.zeros_like(dla_0)
dla_x0 = einops.einsum(
    cache["blocks.0.hook_resid_pre"] - cache["blocks.0.hook_resid_pre"].mean(dim=-1, keepdim=True), 
    model.W_U, 
    "seq_len d_model, d_model vocab_size -> seq_len vocab_size"
)
# ln_bias_dla = einops.einsum(
#     model.ln_final.b,
#     model.W_U,
#     "d_model, d_model vocab_size -> vocab_size"
# )
b_O_0_dla = (model.blocks[0].attn.b_O - model.blocks[0].attn.b_O.mean(dim=-1, keepdim=True)) @ model.W_U
unembed_bias_dla = model.b_U 
# total_bias_logits = ln_bias_dla + unembed_bias_dla

# ---Apply Final LayerNorm Scaling ---
# True DLA often divides by the scaling factor of the final LayerNorm to reflect 
# the exact contribution to the final logits. Uncomment if you need strict exactness.
ln_scale = cache["ln_final.hook_scale"] # Shape: [seq_len, 1]
# We unsqueeze to broadcast across the num_heads dimension: [seq_len, 1, 1]
dla_0 = dla_0 / ln_scale.unsqueeze(1)
dla_1 = dla_1 / ln_scale.unsqueeze(1)
dla_x0 = dla_x0 / ln_scale
b_O_0_dla = b_O_0_dla / ln_scale

dla = torch.concat([dla_x0.unsqueeze(1), unembed_bias_dla.unsqueeze(0).repeat(dla_0.shape[0], 1).unsqueeze(1), dla_0, b_O_0_dla.unsqueeze(1), dla_1], dim=1)

print(f"DLA tensor shape: {dla.shape}")

In [ ]:
prompt_dla = torch.zeros((dla.shape[0]-1, dla.shape[1]))
chosen_tokens_inds = input_tokens[0, 1:]


prompt_dla = dla[range(prompt_dla.shape[0]), :, chosen_tokens_inds]
# prompt_dla = prompt_dla - prompt_dla[:, 1:2]
max_dla = torch.max(torch.abs(prompt_dla)).item()

# Plot as heatmap
# plt.figure(figsize=(20, 8))
# plt.imshow((prompt_dla).cpu().numpy().T, cmap='coolwarm', aspect='auto', vmin=-max_dla, vmax=max_dla)
# plt.colorbar(label='DLA')
# plt.xlabel('Position')
# plt.ylabel('Head')
# plt.yticks(range(18), ["$x_0$", "$b_U$"] + [f"L{layer}H{head}" for layer in range(2) for head in range(8)])
# plt.title(f'DLA of all heads')
# plt.tight_layout()

plt.figure(figsize=(15,8))
plt.plot(prompt_dla[:100,:model.cfg.n_heads+2+1].cpu().numpy(), "--.", label=["$x_0$", "$b_U$"] + [f"L{layer}H{head}" for layer in range(1) for head in range(model.cfg.n_heads)] + [f"$b_(O0)$"])
# plt.plot(prompt_dla[:,model.cfg.n_heads+2+1:].cpu().numpy(), "--x", label=[f"L{layer}H{head}" for layer in range(1,2) for head in range(model.cfg.n_heads)])
plt.xlabel("Position")
plt.ylabel("DLA")
plt.grid()
plt.legend()

plt.show()

In [ ]:
### Plot prompt logits, max logits and 2-nd max logits

prompt_dla = torch.zeros((dla.shape[0]-1, dla.shape[1]))
chosen_tokens_inds = input_tokens[0, 1:]

prompt_dla = dla[range(prompt_dla.shape[0]), :, chosen_tokens_inds]
# prompt_dla = prompt_dla - prompt_dla[:, 1:2]
max_dla = torch.max(torch.abs(prompt_dla)).item()

prompt_logits = logits[0, range(prompt_dla.shape[0]), chosen_tokens_inds]
prompt_max_logits = torch.max(logits[0, :-1], dim=-1).values
prompt_max_2_logits = torch.topk(logits[0, :-1], k=2, dim=-1).values[:, 1]

plt.figure(figsize=(10,8))
plt.plot(prompt_logits.cpu().numpy())
# plt.plot(prompt_dla.sum(dim=-1).cpu().numpy())
plt.plot(prompt_max_logits.cpu().numpy(), '--r')
plt.plot(prompt_max_2_logits.cpu().numpy(), '--b')
plt.grid()
plt.title(f"Chosen and Maximal Logits")
plt.xlabel("Position")
plt.ylabel("Logit")
plt.figure(figsize=(10,8))
# plt.plot(prompt_logits[350:470].cpu().numpy())
plt.plot((prompt_max_logits-prompt_max_2_logits).cpu().numpy(), '.')
# plt.stem(.cpu().numpy(), 'b.')
plt.grid()
plt.title(f"Top-1 to Top-2 Logit Diff")
plt.xlabel("Position")
plt.ylabel("Logit Diff")

plt.show()

In [ ]:
### DLA of Repetitive Pattern
jump = 1
prompt_dla = torch.zeros((dla.shape[0]-1, dla.shape[1]))
chosen_tokens_inds = input_tokens[0, 1:]


# pick out the chosen tokens
prompt_dla = dla[range(prompt_dla.shape[0]), :, chosen_tokens_inds]
prompt_logits = logits[0, range(prompt_dla.shape[0]), chosen_tokens_inds]
prompt_max_logits = torch.max(logits[0, :-1], dim=-1).values

prompt_dla_matrix = prompt_dla.reshape(-1, jump, 18)
prompt_logits_matrix = prompt_logits.reshape(-1, jump)
prompt_max_logits_matrix = prompt_max_logits.reshape(-1, jump)

# get centered logits
logits_centered = logits - logits.mean(dim=-1, keepdim=True)
prompt_logits_centered = logits_centered[0, range(prompt_dla.shape[0]), chosen_tokens_inds]
prompt_max_logits_centered = torch.max(logits_centered[0, :-1], dim=-1).values

prompt_logits_centered_matrix = prompt_logits_centered.reshape(-1, jump)
prompt_max_logits_centered_matrix = prompt_max_logits_centered.reshape(-1, jump)

# get probs matrix
probs = F.softmax(logits[0], dim=-1)  # shape [position, vocab]
prompt_probs = probs[range(prompt_dla.shape[0]), chosen_tokens_inds]
probs_matrix = prompt_probs.reshape(-1, jump)

for offset in range(jump):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,10))
    axs[0].plot(prompt_logits_matrix[:, offset].cpu().numpy())
    axs[0].plot(prompt_max_logits_matrix[:, offset].cpu().numpy(), 'r.')
    axs[0].grid()
    axs[0].set_title(f"Chosen and Maximal Logits")
    axs[0].set_xlabel("#Repetition")
    axs[0].set_ylabel("Logit")

    axs[1].plot(prompt_dla_matrix[:, offset].cpu().numpy(), label=["$x_0$", "$b_U$"] + [f"L{layer}H{head}" for layer in range(2) for head in range(8)])
    axs[1].grid()
    axs[1].set_title(f"DLA for chosen token")
    axs[1].legend()
    axs[1].set_xlabel("#Repetition")
    axs[1].set_ylabel("DLA")
    fig.suptitle(f"Offset {offset}, token: {model.to_string(chosen_tokens_inds[offset])}")

    # plt.figure(figsize=(10,8))
    # plt.plot((prompt_max_logits_matrix[:, offset] - prompt_logits_matrix[:, offset]).cpu().numpy())
    # plt.grid()
    # plt.title(f"Maximal to Chosen Logit Diff")
    # plt.xlabel("#Repetition")
    # plt.ylabel("Logit Diff")

    plt.figure(figsize=(10,8))
    plt.plot((probs_matrix[1:, offset] - probs_matrix[:-1, offset]).cumsum(dim=-1).cpu().numpy())
    plt.grid()
    plt.title(f"Probability Derivative Vs. #Repetition")
    plt.xlabel("#Repetition")
    plt.ylabel("Probability Diff")
    
plt.show()

In [ ]:
plt.figure(figsize=(15,8))
# plt.plot(prompt_dla[600:800,2:10].cpu().numpy(), "o", label=[f"L{layer}H{head}" for layer in range(1) for head in range(8)])
# plt.plot(prompt_dla[:,2:18].max(dim=-1).indices.cpu(), '--.')
plt.hist(prompt_dla[:,2:18].max(dim=-1).indices.cpu(), bins=np.arange(-0.5,16.5,1))
plt.xlabel("Position")
plt.ylabel("DLA")
plt.grid()
plt.legend()


plt.show()

In [ ]:
plt.figure(figsize=(15,8))
plt.plot((slow_dla[80:100].sum(dim=-1) - fast_dla[80:100].sum(dim=-1)).cpu(), '--.')
plt.xlabel("Position")
plt.ylabel("DLA")
plt.grid()
# plt.legend()


plt.show()

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(probs_matrix.cpu().numpy(), label=[f"{model.to_string(token)}" for token in input_tokens[0,1:jump+1]])
plt.plot(0.5 * np.ones(shape=(probs_matrix.shape[0], 1)), 'k', label="$y=1/2$")
plt.grid()
plt.ylabel("Probability")
plt.xlabel("#Repetition")
plt.legend()
plt.show()

In [ ]:
logits_centered_matrix = logits_centered[0,:-1].reshape(-1, jump, logits_centered.shape[-1])
# final_resid = cache['blocks.1.hook_resid_post']
# logits_centered_matrix = final_resid[:-1].reshape(-1, jump, final_resid.shape[-1])

offset = 2
plt.figure(figsize=(10,8))
plt.plot(F.cosine_similarity(logits_centered_matrix[:-1,offset], logits_centered_matrix[1:,offset], dim=-1).cpu().numpy())
plt.grid()
plt.xlabel("#Repetition")
plt.ylabel("Cosine Similarity")
plt.title(f"Cosine Similarity Vs. Repetition, Offset {offset}")
plt.ylim(bottom=0.99, top=1.01)
plt.show()

In [ ]:
bla = dla[:,9,:]
probs_bla = F.softmax(bla, dim=-1)
plt.plot(probs_bla[0::2,1706].cpu().numpy())
plt.plot(probs_bla[0::2].max(dim=-1).values.cpu().numpy())
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
token_id = model.to_single_token(" met")
max_prob_ids = np.argmax(probs[0], axis=-1)
plt.plot(probs[0, :, token_id], "--.")
plt.plot(np.pad(max_prob_ids[1:], (1, 0)) == token_id, ".", label="Win?")
plt.grid()
plt.legend()

plt.show()

In [ ]:
# Get attention patterns and visualize them
layer = 1
print(type(cache))
attention_pattern = cache["attn", layer]
print(attention_pattern.shape) # [head_idx, destination, source]
str_tokens = model.to_str_tokens(clean_output)

In [ ]:
print(f"Layer {layer} head Attention Patterns:")
cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern)

In [ ]:
layer = 0
head = 5
seq_len = 100

attention_pattern = cache["attn", layer]
plt.figure(figsize=(10,8))
plt.imshow(attention_pattern[head][:seq_len,   :seq_len].cpu().numpy(), interpolation="none", cmap="Reds", norm="linear")
plt.colorbar()
plt.title(f"Head {layer}.{head} (Linear Scale)")

plt.figure(figsize=(10,8))
plt.imshow(attention_pattern[head][:seq_len,   :seq_len].cpu().numpy(), interpolation="none", cmap="Reds", norm="log")
plt.colorbar()
plt.title(f"Head {layer}.{head} (Log Scale)")

plt.show()

In [ ]:
row = 800
plt.figure(figsize=(10,8))
plt.plot(attention_pattern[head][row][1:row+1].cpu())
plt.grid()
plt.title(f"Attention Pattern for Repeating Token, Head {layer}.{head}, Position {row}")
plt.show()

In [ ]:
## Plot Attention Coeffs
head = 3
jump = 9
offset = -2
plt.plot(attention_pattern[head, offset].cpu().numpy(), '--o')
# plt.plot(attention_pattern[head, offset].cpu().numpy()[offset-jump+1::-jump][::-1], '--o')
plt.ylim(bottom=0)
plt.xlabel("Position")
plt.ylabel("Attention Weight")
plt.grid()
# plt.show()

# print(attention_pattern[head, -1].cpu().numpy().sum())

## Plot Token Relative Attention Coeff Vs. Repeats
relative_coeffs = [attention_pattern[head, pos].cpu().numpy()[pos-jump+1::-jump].sum() / attention_pattern[head, pos].cpu().numpy().sum() for pos in np.arange(jump+np.mod(offset, jump), len(attention_pattern[head, offset].cpu().numpy()), jump)]
# print(attention_pattern[head, offset].cpu().numpy()[offset-jump+1::-jump].sum() / attention_pattern[head, offset].cpu().numpy()[0])
plt.figure()
plt.plot(relative_coeffs)
plt.xlabel("#Repeats")
plt.ylabel("Relative Coeff")
plt.ylim(bottom=0, top=1)
plt.grid()
plt.show()

In [ ]:
layer = 1

# initial residual stream (embedding + positional)
x0 = cache["blocks.0.hook_resid_pre"] # shape: [batch, sequence_length, d_model]

# 1. The Residual Stream
# shape: [batch, sequence_length, d_model]
resid_pre = cache[f"blocks.{layer}.hook_resid_pre"] # Before attention
resid_post = cache[f"blocks.{layer}.hook_resid_post"] # After attention & addition

# 2. The Q, K, and V Projections
# shape: [batch, sequence_length, num_heads, d_head]
q_vectors = cache[f"blocks.{layer}.attn.hook_q"] # shape [position, head_idx, d_head]
k_vectors = cache[f"blocks.{layer}.attn.hook_k"]
v_vectors = cache[f"blocks.{layer}.attn.hook_v"]


# 3. Attention Scores
# hook_attn_scores: Unnormalized dot products (Q*K^T)
# hook_pattern: Normalized probabilities (after Softmax)
# shape: [batch, num_heads, query_pos, key_pos]
unnormalized_scores = cache[f"blocks.{layer}.attn.hook_attn_scores"]
attention_pattern = cache[f"blocks.{layer}.attn.hook_pattern"]

# 4. Computed Outputs (Before being added to residual stream)
# hook_z: The output of the OV circuit before the final W_O projection
# hook_attn_out: The final output of each head after W_O projection (the exact H_t we modeled)
# shape (hook_attn_out): [batch, sequence_length, num_heads, d_model]
z_vectors = cache[f"blocks.{layer}.attn.hook_z"]
head_outputs = cache[f"blocks.{layer}.attn.hook_result"]

# 5. Final Logits
# Already returned from run_with_cache. shape: [batch, sequence_length, vocab_size]
print(f"Logits shape: {logits.shape}")

In [ ]:
### Compute QK and OV matrices
layer = 11
head = 8
E = model.W_E.cpu() # shape: [vocab, d_model]
E_pos = model.W_pos.cpu() # shape: [n_context, d_model]
Q = model.W_Q[layer, head].cpu() # shape [d_model, d_head]
b_Q = model.b_Q[layer, head].cpu() # shape: [d_head]
K = model.W_K[layer, head].cpu() # shape [d_model, d_head]
b_K = model.b_K[layer, head].cpu() # shape: [d_head]
V = model.W_V[layer, head].cpu() # shape [d_model, d_head]
b_V = model.b_V[layer, head].cpu() # shape: [d_head]
O = model.W_O[layer, head].cpu() # shape [d_head, d_model]
b_O = model.b_O[layer].cpu() # shape: [d_model]
U = model.W_U.cpu() # shape [d_model, vocab]
b_U = model.b_U.cpu() # shape [vocab]

dummy_tokens = torch.tensor(np.arange(1000,1000+model.cfg.n_ctx)).unsqueeze(0)
ln_sigma = extract_frozen_sigma(model, dummy_tokens, layer_idx=layer)

E_ln = model.blocks[layer].ln1(E)
# E_pos_ln = model.blocks[layer].ln1(E_pos)
E_pos_ln = (E_pos - E_pos.mean(dim=-1, keepdim=True)) / ln_sigma

## Core matrices
OV_core = V @ O # shape [d_model, d_model]
QK_core = Q @ K.T # shaoe [d_model, d_model]

E_0 = E + E_pos[7]
E_1 = E + E_pos[8]

# Full matrices
QK_full = (E_ln @ Q + b_Q) @ (E_ln @ K + b_K).T # shape [vocab, vocab]
# QK_full_pos = E_pos @ QK_core @ E_pos.T # shape [n_context, n_context]
QK_full_pos = (E_pos_ln @ Q + b_Q) @ (E_pos_ln @ K + b_K).T # shape [n_context, n_context]
OV_full = E @ OV_core @ U
OV_full_pos = E_pos @ OV_core @ U


In [ ]:
plt.figure(figsize=(10,8))

matrix_to_plot = QK_full_pos.detach().tril()

tops_per_token = matrix_to_plot.topk(3, dim=-1)
strongest_tokens = tops_per_token.values[:,0].sort(descending=True)

y_ind = 0
y_offset = 1000
x_ind = 0
x_offset = 1000
v_max = matrix_to_plot[y_ind:y_ind+y_offset, x_ind:x_ind+x_offset].abs().max().item()
plt.imshow(matrix_to_plot[y_ind:y_ind+y_offset, x_ind:x_ind+x_offset], cmap="coolwarm", vmin=-v_max, vmax=v_max)
# plt.xticks(range(x_offset), range(x_ind, x_ind+x_offset))
# plt.yticks(range(y_offset), range(y_ind, y_ind+y_offset))
plt.colorbar()
plt.show()

In [ ]:
# plot positional attention at a specific row
i_row = 800
row = QK_full_pos[i_row].detach().clone()
row[i_row+1:] = -1e15
row_softmax = F.softmax(row / np.sqrt(model.cfg.d_head), dim=-1)
plt.figure(figsize=(10,8))
plt.plot(row_softmax[1:i_row+1], '--o')

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
# plt.hist(matrix_to_plot[:,1], bins=100)
plt.plot(matrix_to_plot.mean(dim=0)[:1000], '--o')
plt.grid()
plt.show()

In [ ]:
# Softmax over the output dimension (dim=-1) to get the normalized distribution
# P_OV[i, j] = Probability of head predicting token j given source token i
temperature = 0.01
P_OV = F.softmax(OV_full / temperature, dim=-1) # Shape: [d_vocab, d_vocab]

E_virtual = P_OV @ E
K_virtual = E_virtual @ K

Q_standard = E @ Q

QK_virtual_full = Q_standard @ K_virtual.T # i (row): destination token - regular token embedding. j (column): source. OV weighted embeddings by token j. attention from token i to output of value of token j.

OV_standard = E @ V @ O
OV_virtual = E_virtual @ V @ O
# OV_virtual_diag = (OV_standard @ OV_virtual.T).diag()
OV_virtual_diag = F.cosine_similarity(OV_standard, OV_virtual, dim=-1)

feedback_scores = QK_virtual_full * QK_full * OV_virtual_diag.view(1, -1) # multiply each column by the F-OV score

In [ ]:
## Find Stable Tokens
with torch.no_grad():
    d_vocab = model.cfg.d_vocab
    layer = 0
    # heads = [0,1,2,3,4,5,6,7]
    heads = [1,3]
    num_heads = len(heads)

    ln1 = model.blocks[layer].ln1
    ln_final = model.ln_final

    E = model.W_E.cpu() # shape: [vocab, d_model]
    Q = model.W_Q[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
    b_Q = model.b_Q[layer, heads].cpu()
    K = model.W_K[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
    b_K = model.b_K[layer, heads].cpu()
    V = model.W_V[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
    b_V = model.b_V[layer, heads].cpu()
    O = model.W_O[layer, heads].cpu() # shape: [num_heads, d_head, d_model]
    b_O = model.b_O[layer].cpu()
    U = model.W_U.cpu() # shape [d_model, vocab]
    b_U = model.b_U.cpu() # shape [vocab]

    # QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
    # OV_core = V @ O  # shape [num_heads, d_model, d_model]

    ## Attention computation
    attention_resid_pre = ln1(E) # shape: [L, d_model]

    attention_weights = torch.eye(d_vocab).unsqueeze(0).repeat(num_heads, 1, 1) # shape: [num_heads, L, L], sums to 1 along dim=1

    v = einops.einsum(attention_resid_pre, V, "L d_model, num_heads d_model d_head -> num_heads L d_head")
    v = v + b_V.unsqueeze(1) # shape: [num_heads, L, d_head]

    weighted_v = attention_weights.mT @ v # shape: [num_heads, L, d_head]
    head_outputs = einops.einsum(weighted_v, O, "num_heads L d_head, num_heads d_head d_model -> num_heads L d_model")
    attention_output = head_outputs.sum(dim=0) + b_O # shape: [L, d_model]

    # value_bias = einops.einsum(model.b_V, model.W_O, "layer num_heads d_head, seq num_heads d_head d_model -> layer num_heads d_model")[:, heads].sum(dim=1).cpu() # shape: [1, d_model]
    # attention_output = (attention_weights.mT @ attention_resid_pre @ OV_core).sum(dim=0) + b_O + value_bias # shape: [L, d_model]

    # Unembedding
    final_resid = attention_output + E # shape: [L, d_model]
    final_resid_norm = ln_final(final_resid)
    full_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [L, vocab]
    max_OV_token_ind = full_logits_matrix.max(dim=-1).indices
    token_is_stable = max_OV_token_ind == np.arange(model.cfg.d_vocab)
    stable_tokens = token_is_stable.nonzero()
    unstable_tokens = (~token_is_stable).nonzero()

    top_logits = full_logits_matrix.topk(2, dim=-1)
    top_stable_logits_sorted = (top_logits.values[stable_tokens, 0] - top_logits.values[stable_tokens, 1]).squeeze().sort()

In [ ]:
new_stable_tokens = stable_tokens[~torch.isin(stable_tokens, sem_stable_tokens)]

In [ ]:
sorted_index = 1000
token = stable_tokens[top_stable_logits_sorted.indices[sorted_index]]
value = top_stable_logits_sorted.values[sorted_index]
print(f"Token {token.item()}:{model.to_string(token)}, Top-2 Logit Diff: {value}")

plt.hist(top_stable_logits_sorted.values, bins=100)
plt.xlabel("Top-2 Logit Difference")
plt.ylabel("Counts")

plt.figure()
plt.plot(top_stable_logits_sorted.values)
plt.xlabel("Sorted Index")
plt.ylabel("Top-2 Logit Difference")
plt.show()


In [ ]:
layer = 0
heads = [3]
token1 = 10193
token2 = 2820
pi = torch.zeros(model.cfg.d_vocab, 1) # shape: [vocab, 1]
pi[token1] = pi[token2] = 0.5
# pi[token1] = 1

ln1 = model.blocks[layer].ln1
ln_final = model.ln_final

E = model.W_E.cpu() # shape: [vocab, d_model]
Q = model.W_Q[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_Q = model.b_Q[layer, heads].cpu()
K = model.W_K[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_K = model.b_K[layer, heads].cpu()
V = model.W_V[layer, heads].cpu() # shape: [num_heads, d_model, d_head]
b_V = model.b_V[layer, heads].cpu()
O = model.W_O[layer, heads].cpu() # shape: [num_heads, d_head, d_model]
b_O = model.b_O[layer].cpu()
U = model.W_U.cpu() # shape [d_model, vocab]
b_U = model.b_U.cpu() # shape [vocab]

# QK_core = Q @ K.mT # shape [num_heads, d_model, d_model]
# OV_core = V @ O  # shape [num_heads, d_model, d_model]

## Attention computation
attention_resid_pre = ln1(E[[token1, token2]]) # shape: [L, d_model]

q = einops.einsum(attention_resid_pre, Q, "L d_model, num_heads d_model d_head -> num_heads L d_head")
q = q + b_Q.unsqueeze(1) # Broadcast bias over L: [num_heads, L, d_head]
k = einops.einsum(attention_resid_pre, K, "L d_model, num_heads d_model d_head -> num_heads L d_head")
k = k + b_K.unsqueeze(1) # Broadcast bias over L: [num_heads, L, d_head]

QK_raw = (q @ k.mT) / np.sqrt(model.cfg.d_head) # shape: [num_heads, L, L]
QK_weighted = pi[[token1, token2]] * torch.exp(QK_raw)
attention_weights = QK_weighted / QK_weighted.sum(dim=1, keepdim=True) # shape: [num_heads, L, L], sums to 1 along dim=1

v = einops.einsum(attention_resid_pre, V, "L d_model, num_heads d_model d_head -> num_heads L d_head")
v = v + b_V.unsqueeze(1) # shape: [num_heads, L, d_head]

weighted_v = attention_weights.mT @ v # shape: [num_heads, L, d_head]
head_outputs = einops.einsum(weighted_v, O, "num_heads L d_head, num_heads d_head d_model -> num_heads L d_model")
attention_output = head_outputs.sum(dim=0) + b_O # shape: [L, d_model]

# value_bias = einops.einsum(model.b_V, model.W_O, "layer num_heads d_head, seq num_heads d_head d_model -> layer num_heads d_model")[:, heads].sum(dim=1).cpu() # shape: [1, d_model]
# attention_output = (attention_weights.mT @ attention_resid_pre @ OV_core).sum(dim=0) + b_O + value_bias # shape: [L, d_model]

# Unembedding
final_resid = attention_output + E[[token1, token2]] # shape: [L, d_model]
final_resid_norm = ln_final(final_resid)
full_logits_matrix = final_resid_norm @ U + b_U.view(1, -1) # shape: [L, vocab]

pattern_stable = (full_logits_matrix[0].topk(1).indices == token2) and (full_logits_matrix[1].topk(1).indices == token1)
print("Pattern Stable!" if pattern_stable else "Pattern Unbstable")

In [ ]:
sorted_diag = full_logits_matrix.diag().sort()
stable_indices_sorted = [index for index in range(model.cfg.d_vocab) if sorted_diag.indices[index] in stable_tokens]
unstable_indices_sorted = [index for index in range(model.cfg.d_vocab) if sorted_diag.indices[index] not in stable_tokens]

plt.figure(figsize=(10,8))
plt.plot(stable_indices_sorted, sorted_diag.values[stable_indices_sorted], 'b.', markersize=0.1)
plt.plot(unstable_indices_sorted, sorted_diag.values[unstable_indices_sorted], 'r.', markersize=0.1)
# plt.plot([index in stable_tokens for index in full_logits_matrix.diag().sort().indices], 'r.')
plt.grid()
plt.show()

## Fixed Point Finder

### Original 1-Layer Optimization

In [ ]:
### Black Box Optimization
vocab_size = model.cfg.d_vocab
eps = 1e-10

# Turn off model requires grad
for p in model.parameters():
    p.requires_grad = False

# move to cpu
# model_cpu = model.cpu()

# Initialize optimization parameter
pi_init = torch.ones(vocab_size, device=device) * eps
# pi_init[432] = (0.9/(1-0.9)) * pi_init.sum()
pi_init[12263] = 0.6
pi_init[16354] = 0.4
# pi_init[17423] = 0.25
# pi_init = pi_t_from_context(model.to_tokens(patched_output)[0])[-100].clamp(min=1e-10)
pi_init = pi_init / pi_init.sum()
pi_logits = nn.Parameter(torch.log(pi_init)) # for Natural GD
# pi = nn.Parameter(pi_init) # for Exponentiated GD

# Define loss function
criterion = nn.MSELoss(reduction='sum')
# criterion = nn.L1Loss(reduction='sum')

# heads = [0,1,2,3,4,5,6,7]
heads = [0,3]

# Optimization parameters
lr = 1e0
n_iterations = 1000
val_iterations = 500
print_iterations = 20
temperature = 1.0
# chunk_size = 2**10
chunk_size = 64
p = 0.9 # nucleus sampling parameter
# optimizer = torch.optim.Adam([pi_logits], lr=lr, eps=1e-15)
optimizer = torch.optim.SGD([pi_logits], lr=lr, momentum=0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max = n_iterations, eta_min=1e-2)

losses = []
full_losses = []
pi_list = []
pi_P_list = []
for i_iteration in tqdm(range(n_iterations)):
    # zero grad
    optimizer.zero_grad()

    pi = F.softmax(pi_logits, dim=-1)

    top_tokens = pi.topk(chunk_size).indices

    # forward pass
    # P_logits = pi_to_P_one_layer(pi, heads, model, chunk_size=512, temperature=temperature, p=p)
    # P = pi_to_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p) # Sparse matrix!

    # P = F.softmax(P_logits / temperature, dim=-1)

    # pi_col = pi.unsqueeze(1) # change to column vector [vocab, 1]
    # pi_P_col = torch.sparse.mm(P.t(), pi_col) # Perform Sparse @ Dense multiplication: P^T @ pi^T, shape: [vocab, 1]
    # pi_P = pi_P_col.squeeze(1)
    # pi_P = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
    pi_P = pi_to_pi_P_one_layer_topk(pi, heads, model, top_tokens=top_tokens, temperature=temperature, p=p)
    pi_P = pi_P.squeeze()
    pi_P = pi_P.clamp(min=1e-15)

    # loss = criterion(pi, pi_P) * 1e3
    # loss = F.kl_div(pi.log(), pi_P.log(), reduction='batchmean', log_target=True)

    ## Forward KL
    # loss = 1e5 * F.kl_div(  # computes KL(p || pi_P)
    #     input=pi_P.log(), 
    #     target=pi.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    ## Reverse KL
    # loss = 1e5 * F.kl_div(  # computes KL(pi_P || p)
    #     input=pi.log(), 
    #     target=pi_P.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    ## JSD
    loss = 1e5 * JSD(pi.unsqueeze(0), pi_P.unsqueeze(0)) / vocab_size
    # M_pi = 0.5 * (pi + pi_P)
    # loss = 0.5 * 1e5 * (F.kl_div(  
    #     input=M_pi.log(), 
    #     target=pi_P.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # ) + F.kl_div(  
    #     input=M_pi.log(), 
    #     target=pi.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )
    # )

    if i_iteration % val_iterations == 0:
        with torch.no_grad():
            pi_P_full = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
            pi_P_full = pi_P_full.squeeze().clamp(min=1e-15)
            M_pi_full = 0.5 * (pi + pi_P_full)
            loss_full = 0.5 * 1e5 * (F.kl_div(  
                input=M_pi_full.log(), 
                target=pi_P_full.log(), 
                reduction='batchmean', 
                log_target=True
            ) + F.kl_div(  
                input=M_pi_full.log(), 
                target=pi.log(), 
                reduction='batchmean', 
                log_target=True
            )
            )

    losses.append(loss.item())
    full_losses.append(loss_full.item())
    pi_list.append(pi.detach().cpu())
    pi_P_list.append(pi_P.detach().cpu())
    loss.backward()

    # Natural Gradient scaling
    with torch.no_grad():
        g = pi_logits.grad
        # print(f"Original Gradient: {g.mean()}, {g.max()}")
        
        # Damping factor to prevent division by zero for dead tokens
        # (Because your distribution is sparse, this lambda is your lifeline)
        lmbda = 1e-10
        
        # 1. A_inv is just 1 / (p + lambda)
        A_inv = 1.0 / (pi + lmbda)
        
        # 2. Compute the three dot products/element-wise ops
        A_inv_g = A_inv * g                      # Vector: A^{-1} g
        A_inv_p = A_inv * pi                     # Vector: A^{-1} p
        
        p_A_inv_g = torch.sum(pi * A_inv_g)      # Scalar: p^T A^{-1} g
        p_A_inv_p = torch.sum(pi * A_inv_p)      # Scalar: p^T A^{-1} p
        
        # 3. Apply the Sherman-Morrison formula to get the exact preconditioned gradient
        numerator = A_inv_p * p_A_inv_g
        denominator = 1.0 - p_A_inv_p
        
        exact_natural_grad = A_inv_g + (numerator / denominator.clamp(min=1e-15))
        
        # 4. Overwrite the gradient
        pi_logits.grad = exact_natural_grad

        # print(f"A_inv: {A_inv.mean()}, {A_inv.max()}")
        # print(f"A_inv_p: {A_inv_p.mean()}, {A_inv_p.max()}")
        # print(f"p_A_inv_p: {p_A_inv_p}")
        # print(f"numerator: {numerator.mean()}, {numerator.max()}")
        # print(f"denominator: {denominator.mean()}, {denominator.max()}")
        # print(f"exact_natural_grad: {exact_natural_grad.mean()}, {exact_natural_grad.max()}")
        
        # 5. Clip to prevent wild swings from the lambda division
        torch.nn.utils.clip_grad_norm_([pi_logits], max_norm=1.0)

    # --- EXPONENTIATED GRADIENT UPDATE ---
    # with torch.no_grad():
    #     g = pi.grad
        
    #     # The Log-Space Trick for Bulletproof Numerical Stability
    #     # We add 'eps' to avoid log(0) for completely dead tokens
    #     z = torch.log(pi + 1e-12) - (lr * g)
        
    #     # Softmax naturally normalizes the vector so it sums to 1 again
    #     pi_new = F.softmax(z, dim=-1)
        
    #     # In-place copy to maintain the nn.Parameter reference for the next loop
    #     pi.copy_(pi_new)
        
    #     # Manually zero the gradient for the next iteration
    #     pi.grad.zero_()

    if i_iteration % print_iterations == 0:
        grad_mean = pi_logits.grad.abs().mean().item()
        grad_max = pi_logits.grad.abs().max().item()
        # grad_mean = pi.grad.abs().mean().item()
        # grad_max = pi.grad.abs().max().item()
        print(f"Gradient Mean: {grad_mean:.2e}, Max: {grad_max:.2e}")
        # print(f"Loss: {loss.item():.2e}")
        print(f"Loss: {loss.item():.2e} || Full Loss: {loss_full.item():.2e}")
        print(f"pi:\t{pi.topk(5).values.tolist()},\t{pi.topk(5).indices.tolist()}")
        print(f"pi_P:\t{pi_P.topk(5).values.tolist()},\t{pi_P.topk(5).indices.tolist()}")

    optimizer.step()
    scheduler.step()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), losses)
plt.plot(np.arange(1, len(losses)+1), full_losses)
# plt.plot(np.arange(1, len(losses)+1), np.log10(losses))
# plt.plot(np.arange(1, len(losses)+1), np.log10(full_losses))
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss Vs. Iteration")

plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), [torch.log(pi_t[6345]) for pi_t in pi_list])
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Pi[token]")
plt.title("Pi[token] Vs. Iteration")

plt.show()

In [ ]:
### Black Box Optimization - Exponentiated Gradient Descent
vocab_size = model.cfg.d_vocab
eps = 1e-10

# Turn off model requires grad
for p in model.parameters():
    p.requires_grad = False

# move to cpu
# model_cpu = model.cpu()

# Initialize optimization parameter
pi_init = torch.ones(vocab_size, device=device) * eps
# pi_init[432] = (0.9/(1-0.9)) * pi_init.sum()
# pi_init[17593] = 0.4
# pi_init[16367] = 0.6
# pi_init[1] = 0.01

pi_init = pi_from_context(model.to_tokens(patched_output)[0]).clamp(min=1e-10)
pi_init = pi_init / pi_init.sum()
# pi_logits = nn.Parameter(torch.log(pi_init)) # for Natural GD
pi = nn.Parameter(pi_init) # for Exponentiated GD

# Define loss function
criterion = nn.MSELoss(reduction='sum')
# criterion = nn.L1Loss(reduction='sum')

# heads = [0,1,2,3,4,5,6,7]
heads = [0,3]

# Optimization parameters
lr = 1e-2
n_iterations = 5000
val_iterations = 500
print_iterations = 20
temperature = 1.0
chunk_size = 2**9
p = 0.9 # nucleus sampling parameter
# optimizer = torch.optim.Adam([pi_logits], lr=lr, eps=1e-15)
# optimizer = torch.optim.SGD([pi_logits], lr=lr, momentum=0)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max = n_iterations, eta_min=1e-1)

losses = []
losses_full = []
pi_list = []
pi_P_list = []
for i_iteration in tqdm(range(n_iterations)):
    # zero grad
    # optimizer.zero_grad()

    # pi = F.softmax(pi_logits, dim=-1)

    top_tokens = pi.topk(chunk_size).indices

    # forward pass
    # P_logits = pi_to_P_one_layer(pi, heads, model, chunk_size=512, temperature=temperature, p=p)
    # P = pi_to_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p) # Sparse matrix!

    # P = F.softmax(P_logits / temperature, dim=-1)

    # pi_col = pi.unsqueeze(1) # change to column vector [vocab, 1]
    # pi_P_col = torch.sparse.mm(P.t(), pi_col) # Perform Sparse @ Dense multiplication: P^T @ pi^T, shape: [vocab, 1]
    # pi_P = pi_P_col.squeeze(1)
    # pi_P = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
    pi_P = pi_to_pi_P_one_layer_topk(pi, heads, model, top_tokens=top_tokens, temperature=temperature, p=p)
    pi_P = pi_P.squeeze()
    pi_P = pi_P.clamp(min=1e-15)
    

    # loss = criterion(pi, pi_P) * 1
    # loss = F.kl_div(pi.log(), pi_P.log(), reduction='batchmean', log_target=True)

    ## Forward KL
    # loss = 1e5 * F.kl_div(  # computes KL(p || pi_P)
    #     input=pi_P.log(), 
    #     target=pi.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    ## Reverse KL
    # loss = 1e5 * F.kl_div(  # computes KL(pi_P || p)
    #     input=pi.log(), 
    #     target=pi_P.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    # JSD
    M_pi = 0.5 * (pi + pi_P)
    loss = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi.log(), 
        target=pi_P.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi.log(), 
        target=pi.clamp(min=1e-15).log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

    if i_iteration % val_iterations == 0:
        with torch.no_grad():
            pi_P_full = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
            pi_P_full = pi_P_full.squeeze().clamp(min=1e-15)
            M_pi_full = 0.5 * (pi + pi_P_full)
            loss_full = 0.5 * 1e5 * (F.kl_div(  
                input=M_pi_full.log(), 
                target=pi_P_full.log(), 
                reduction='batchmean', 
                log_target=True
            ) + F.kl_div(  
                input=M_pi_full.log(), 
                target=pi.log(), 
                reduction='batchmean', 
                log_target=True
            )
            )

    losses.append(loss.item())
    losses_full.append(loss_full.item())
    pi_list.append(pi.detach().cpu())
    pi_P_list.append(pi_P.detach().cpu())
    loss.backward()

    # Natural Gradient scaling
    # with torch.no_grad():
    #     g = pi_logits.grad
        
    #     # Damping factor to prevent division by zero for dead tokens
    #     # (Because your distribution is sparse, this lambda is your lifeline)
    #     lmbda = 1e-11
        
    #     # 1. A_inv is just 1 / (p + lambda)
    #     A_inv = 1.0 / (pi + lmbda)
        
    #     # 2. Compute the three dot products/element-wise ops
    #     A_inv_g = A_inv * g                      # Vector: A^{-1} g
    #     A_inv_p = A_inv * pi                     # Vector: A^{-1} p
        
    #     p_A_inv_g = torch.sum(pi * A_inv_g)      # Scalar: p^T A^{-1} g
    #     p_A_inv_p = torch.sum(pi * A_inv_p)      # Scalar: p^T A^{-1} p
        
    #     # 3. Apply the Sherman-Morrison formula to get the exact preconditioned gradient
    #     numerator = A_inv_p * p_A_inv_g
    #     denominator = 1.0 - p_A_inv_p
        
    #     exact_natural_grad = A_inv_g + (numerator / denominator)
        
    #     # 4. Overwrite the gradient
    #     pi_logits.grad = exact_natural_grad
        
    #     # 5. Clip to prevent wild swings from the lambda division
    #     torch.nn.utils.clip_grad_norm_([pi_logits], max_norm=1.0)
    if i_iteration % print_iterations == 0:
        grad_mean = pi.grad.abs().mean().item()
        grad_max = pi.grad.abs().max().item()
        print(f"Gradient Mean: {grad_mean:.2e}, Max: {grad_max:.2e}")
        print(f"Loss: {loss.item():.2e}")
        print(f"pi top-5: {pi.topk(5)}")
        print(f"pi_P top-5: {pi_P.topk(5)}")

    # --- EXPONENTIATED GRADIENT UPDATE ---
    with torch.no_grad():
        g = pi.grad
        
        # The Log-Space Trick for Bulletproof Numerical Stability
        # We add 'eps' to avoid log(0) for completely dead tokens
        z = torch.log(pi + 1e-12) - (lr * g)
        
        # Softmax naturally normalizes the vector so it sums to 1 again
        pi_new = F.softmax(z, dim=-1)
        
        # In-place copy to maintain the nn.Parameter reference for the next loop
        pi.copy_(pi_new)
        
        # Manually zero the gradient for the next iteration
        pi.grad.zero_()

    # grad_mean = pi_logits.grad.abs().mean().item()
    # grad_max = pi_logits.grad.abs().max().item()

    # optimizer.step()
    # scheduler.step()

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), np.log10(losses))
plt.plot(np.arange(1, len(losses)+1), np.log10(losses_full))
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss Vs. Iteration")

plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), [pi_t[17593] for pi_t in pi_list])
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Pi[token]")
plt.title("Pi[token] Vs. Iteration")

plt.show()

In [ ]:
# plt.hist(pi_list[-1].topk(100).values.log(), bins=50)
# plt.hist(pi_P_list[-1].topk(100).values.log(), bins=50)
# plt.show()

In [ ]:
## pi iteration
vocab_size = model.cfg.d_vocab
eps = 1e-10

# Turn off model requires grad
for p in model.parameters():
    p.requires_grad = False

# move to cpu
# model_cpu = model.cpu()

# Initialize optimization parameter
pi_init = torch.ones(vocab_size, device=device) * eps
# pi_init[432] = (0.9/(1-0.9)) * pi_init.sum()
# pi_init[2501] = 0.99
# pi_init[10193] = 0.47
# pi_init[1] = 0.01

# pi_init = pi_from_context(model.to_tokens(patched_output)).clamp(min=1e-10)
pi_init = pi_from_context(model.to_tokens(prompt)).clamp(min=1e-10)
pi_init = pi_init / pi_init.sum()
pi = pi_init

heads = [0,1,2,3,4,5,6,7]
# heads = [3]

# Optimization parameters
lr = 3e-3
alpha = 0.995
n_iterations = 3000
val_iterations = 500
print_iterations = 20
temperature = 1.0
chunk_size = 2**10
p = 0.9 # nucleus sampling parameter


losses = []
losses_full = []
pi_list = []
pi_P_list = []
for i_iteration in tqdm(range(n_iterations)):

    top_tokens = pi.topk(chunk_size).indices
    with torch.no_grad():
        pi_P = pi_to_pi_P_one_layer_topk(pi, heads, model, top_tokens=top_tokens, temperature=temperature, p=p)
    pi_P = pi_P.squeeze()
    pi_P = pi_P.clamp(min=1e-15)
    pi = pi.clamp(min=1e-15)

    # loss = criterion(pi, pi_P) * 1
    # loss = F.kl_div(pi.log(), pi_P.log(), reduction='batchmean', log_target=True)

    ## Forward KL
    # loss = 1e5 * F.kl_div(  # computes KL(p || pi_P)
    #     input=pi_P.log(), 
    #     target=pi.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    ## Reverse KL
    # loss = 1e5 * F.kl_div(  # computes KL(pi_P || p)
    #     input=pi.log(), 
    #     target=pi_P.log(), 
    #     reduction='batchmean', 
    #     log_target=True
    # )

    # JSD
    M_pi = 0.5 * (pi + pi_P)
    loss = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi.log(), 
        target=pi_P.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi.log(), 
        target=pi.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

    if i_iteration % val_iterations == 0:
        with torch.no_grad():
            pi_P_full = pi_to_pi_P_one_layer(pi, heads, model, chunk_size=chunk_size, temperature=temperature, p=p)
            pi_P_full = pi_P_full.squeeze().clamp(min=1e-15)
            M_pi_full = 0.5 * (pi + pi_P_full)
            loss_full = 0.5 * 1e5 * (F.kl_div(  
                input=M_pi_full.log(), 
                target=pi_P_full.log(), 
                reduction='batchmean', 
                log_target=True
            ) + F.kl_div(  
                input=M_pi_full.log(), 
                target=pi.log(), 
                reduction='batchmean', 
                log_target=True
            )
            )

    losses.append(loss.item())
    losses_full.append(loss_full.item())
    pi_list.append(pi.detach().cpu())
    pi_P_list.append(pi_P.detach().cpu())

    if i_iteration % print_iterations == 0:
        # grad_mean = pi.grad.abs().mean().item()
        # grad_max = pi.grad.abs().max().item()
        # print(f"Gradient Mean: {grad_mean:.2e}, Max: {grad_max:.2e}")
        print(f"Loss: {loss.item():.2e}")
        print(f"pi top-5: {pi.topk(5)}")
        print(f"pi_P top-5: {pi_P.topk(5)}")

    pi = alpha * pi + (1 - alpha) * pi_P

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), losses)
plt.plot(np.arange(1, len(losses)+1), losses_full)
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss Vs. Iteration")

plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), [pi_t[25860] for pi_t in pi_list])
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Pi[token]")
plt.title("Pi[token] Vs. Iteration")

plt.show()

### Multi-Layer + Positional Black Box Optimization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import math

N_union_list = []
criterion = nn.L1Loss(reduction="sum")

# =====================================================================
# 1. JSD SUPPORT ALIGNMENT HELPER
# =====================================================================
def compute_aligned_jsd(vals_1: torch.Tensor, keys_1: torch.Tensor, 
                        vals_2: torch.Tensor, keys_2: torch.Tensor) -> torch.Tensor:
    """
    Computes Jensen-Shannon Divergence between two sparse distributions 
    with potentially mismatched K-gram supports by dynamically uniting them.
    """
    # Find the Union Support
    cat_keys = torch.cat([keys_1, keys_2], dim=0)
    unique_keys, inverse_indices = torch.unique(cat_keys, dim=0, return_inverse=True)
    N_unique = unique_keys.size(0)
    N_union_list.append(N_unique)
    
    # Split inverse indices back to original distributions
    idx_1 = inverse_indices[:keys_1.size(0)]
    idx_2 = inverse_indices[keys_1.size(0):]
    
    # Scatter values onto the unified support (Missing keys get eps)
    aligned_1 = torch.ones(N_unique, dtype=vals_1.dtype, device=vals_1.device) * 1e-10
    aligned_2 = torch.ones(N_unique, dtype=vals_2.dtype, device=vals_2.device) * 1e-10
    
    aligned_1.scatter_(0, idx_1, vals_1)
    aligned_2.scatter_(0, idx_2, vals_2)

    aligned_1 = aligned_1 / aligned_1.sum()
    aligned_2 = aligned_2 / aligned_2.sum()
    
    # Compute standard JSD
    # M = 0.5 * (aligned_1 + aligned_2)
    # kl_1 = torch.sum(aligned_1 * torch.log((aligned_1 + 1e-11) / (M + 1e-11)))
    # kl_2 = torch.sum(aligned_2 * torch.log((aligned_2 + 1e-11) / (M + 1e-11)))

    # loss = 0.5 * kl_1 + 0.5 * kl_2

    # print(f"keys 1 shape: {keys_1.shape}\nkeys_2 shape: {keys_2.shape}")
    # print(f"N_unique: {N_unique}")
    # print(f"unique_keys: {unique_keys}")
    # print(f"aligned_1 shape: {aligned_1.shape}")
    # print(f"idx_1 shape: {idx_1.shape}")

    loss = JSD(aligned_1.unsqueeze(0), aligned_2.unsqueeze(0)) # unsqueeze to create batch dimension
    # loss = criterion(aligned_1, aligned_2)
    
    return loss


# =====================================================================
# 2. BLACK BOX OPTIMIZATION SETUP
# =====================================================================
vocab_size = model.cfg.d_vocab
device = next(model.parameters()).device
eps = 1e-10

for p in model.parameters():
    p.requires_grad = False

# --- Multi-Layer Positional/Semantic Definitions ---
## one-layer attention-only model
# K_list = [3] 
# sem_heads_list = [[0,3,4,6]] 
# pos_heads_list = [[1,2,5,7]] 
## two-layers attention-only model
# K_list = [5, 1]
# sem_heads_list = [[1,2,4,6,7], [0,1,2,3,4,5,6,7]]
# pos_heads_list = [[0,3,5], []] 
# K_list = [5, 3]
# sem_heads_list = [[4,6,7], [3,5,6,7]]
# pos_heads_list = [[0,1,2,3,5], [0,1,2,4]] 
K_list = [7, 1]
sem_heads_list = [[4,6,7], []]
pos_heads_list = [[0,1,2,3,5], []] 

# The exact initial sequence length required to compute 1 output token through L layers
S_init = sum(K_list) - len(K_list) + 1  #
M_iterations = 1
N_active = 256
N_tracking = int(vocab_size / 4)
K_pruning = N_active * 2  # the number of transition to save for each query

# --- Initialization of the Active K-Gram Subspace ---
initial_k_grams = [
    # [27864],
    [330,330,330,330,330,330,330],
    # [1094,24714,1094,24714,1094],
    # [2,2,1],
    # [2, 1] 
]
# Pad with random or base-rate K-grams to reach N_active capacity
while len(initial_k_grams) < N_tracking:
    initial_k_grams.append(torch.randint(0, vocab_size, (S_init,), ).tolist()) # note - k-grams are not guaranteed to be unique

# The Coordinate Tensor: [N_tracking, S_init]
pi_keys = torch.tensor(initial_k_grams, device=device, dtype=torch.long)
pi_keys = torch.unique(pi_keys, dim=0)

# The Probability Tensor: [N_tracking]
pi_vals = torch.ones(pi_keys.shape[0], device=device) * eps
pi_vals[torch.all(pi_keys == torch.tensor(initial_k_grams[0], device=device), dim=1)] = 1
# pi_vals[torch.all(pi_keys == torch.tensor(initial_k_grams[1], device=device), dim=1)] = 0.5
# pi_vals[torch.all(pi_keys == torch.tensor(initial_k_grams[2], device=device), dim=1)] = 0.01
pi_vals = pi_vals / pi_vals.sum()

# --- Optimization Parameters ---
lr_max = 1e0
lr_min = 1e-1
n_iterations = 1
val_iterations = 20000
print_iterations = 1
temperature = 1.0
top_p = 1.0

losses = []
full_losses = []

# =====================================================================
# 3. THE EXPLORE-THEN-OPTIMIZE LOOP
# =====================================================================
for i_iteration in tqdm(range(n_iterations)):
    
    # -----------------------------------------------------------------
    # PHASE 1: EXPLORATION PASS (NO GRADIENTS) 
    # Fast rollout to discover emergent K-grams organically produced 
    # by the transition matrix P_pi.
    # -----------------------------------------------------------------
    # only use the top N k-grams as queries
    # TODO: increase to larger number for pass with no gradient?
    top_pi_vals, top_pi_inds = pi_vals.topk(N_active)
    top_pi_keys = pi_keys[top_pi_inds]

    with torch.no_grad():
        pi_P_vals_explore, pi_P_keys_explore = compute_stationary_distribution(
            top_pi_vals, top_pi_keys, N_tracking, K_pruning, model, 
            K_list, sem_heads_list, pos_heads_list, 
            M_iterations, temperature, top_p
        )
        
        # Determine the expanded Union Support [cite: 1158]
        cat_keys = torch.cat([pi_keys, pi_P_keys_explore], dim=0)
        unique_keys, inverse_indices = torch.unique(cat_keys, dim=0, return_inverse=True)
        N_union = unique_keys.size(0)
        
        # Expand current pi onto the Union Support [cite: 1159]
        # Existing keys maintain their current mass. Newly discovered keys get 2 * epsilon.
        pi_vals_union = torch.ones(N_union, device=device) * 2 * eps
        pi_vals_union.scatter_(0, inverse_indices[:pi_keys.size(0)], pi_vals)
        pi_vals_union = pi_vals_union / pi_vals_union.sum() 
        # print(f"N_union:\n{N_union}")
        # N_union_list.append(N_union)
        # print(f"pi_vals_union:\n{pi_vals_union.topk(5)}")
        
    # -----------------------------------------------------------------
    # PHASE 2: OPTIMIZATION PASS (WITH GRADIENTS) [cite: 1161, 1162]
    # Re-initialize the active parameters over the expanded support.
    # The new keys are now valid leaf parameters inside the autograd graph.
    # -----------------------------------------------------------------
    pi_logits_union = nn.Parameter(torch.log(pi_vals_union))

    current_lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * i_iteration / n_iterations))
    
    # We instantiate a fresh optimizer since SGD w/ momentum=0 has no internal state buffer
    optimizer = torch.optim.SGD([pi_logits_union], lr=current_lr, momentum=0)
    optimizer.zero_grad()
    
    pi_vals_active = F.softmax(pi_logits_union, dim=-1)

    # only use the top N k-grams as queries
    top_pi_vals_active, top_pi_active_inds = pi_vals_active.topk(N_active)
    top_pi_active_keys = unique_keys[top_pi_active_inds]
    
    # Compute the rigorously tracked forward pass mapping for the exact Jacobian/Gradients
    pi_P_vals, pi_P_keys = compute_stationary_distribution(
        top_pi_vals_active, top_pi_active_keys, N_active, K_pruning, model, 
        K_list, sem_heads_list, pos_heads_list, 
        M_iterations, temperature, top_p
    )
    
    # Calculate JSD Loss. Because pi_vals_active contains epsilon mass for the new keys,
    # the JSD acts as a continuous gradient pull toward highly favorable new states.
    ## TODO: change to only top_pi_vals_active and not the full tracking buffer pi_vals_active?
    loss = 1e5 * compute_aligned_jsd(pi_vals_active, unique_keys, pi_P_vals.clamp(1e-10), pi_P_keys) / vocab_size
    # loss = 1e5 * compute_aligned_jsd(top_pi_vals_active, top_pi_active_keys, pi_P_vals.clamp(1e-15), pi_P_keys)
    losses.append(loss.item())

    loss.backward()

    # -----------------------------------------------------------------
    # PHASE 3: NATURAL GRADIENT ON THE SIMPLEX [cite: 1102, 1106, 1107]
    # -----------------------------------------------------------------
    with torch.no_grad():
        g = pi_logits_union.grad
        
        # if i_iteration % print_iterations == 0:
            # print(f"pi_logits_union: {pi_logits_union.topk(10)}")
            # print(f"grad: {pi_logits_union.grad.abs().topk(10)}")
        lmbda = 1e-10
        
        # Sherman-Morrison exact preconditioned gradient for probability simplex
        A_inv = 1.0 / (pi_vals_active + lmbda)
        A_inv_g = A_inv * g                      
        A_inv_p = A_inv * pi_vals_active                
        
        p_A_inv_g = torch.sum(pi_vals_active * A_inv_g) 
        p_A_inv_p = torch.sum(pi_vals_active * A_inv_p) 
        
        numerator = A_inv_p * p_A_inv_g
        denominator = 1.0 - p_A_inv_p
        
        exact_natural_grad = A_inv_g + (numerator / denominator.clamp(min=1e-10))
        if i_iteration % print_iterations == 0:
            print(f"exact_natural_grad norm: {torch.norm(exact_natural_grad)}")
        pi_logits_union.grad = exact_natural_grad
        # if i_iteration % print_iterations == 0:
            # print(f"scaled grad: {pi_logits_union.grad.abs().topk(10)}")
        
        torch.nn.utils.clip_grad_norm_([pi_logits_union], max_norm=1.0)
        # if i_iteration % print_iterations == 0:
            # print(f"scaled grad: {pi_logits_union.grad.abs().topk(10)}")


    # with torch.no_grad():
    #     g = pi_logits_union.grad
        
    #     # We only apply the preconditioner to the parameters that actually 
    #     # participated in the forward pass. The waiting room keys just coast.
        
    #     # Create a mask for the active indices
    #     active_mask = torch.zeros_like(g, dtype=torch.bool)
    #     active_mask[top_pi_active_inds] = True
        
    #     g_active = g[active_mask]
    #     p_active = pi_vals_active[active_mask]
        
    #     lmbda = 1e-10 
    #     A_inv = 1.0 / (p_active + lmbda)
        
    #     A_inv_g = A_inv * g_active                      
    #     A_inv_p = A_inv * p_active                
        
    #     p_A_inv_g = torch.sum(p_active * A_inv_g) 
    #     p_A_inv_p = torch.sum(p_active * A_inv_p) 
        
    #     numerator = A_inv_p * p_A_inv_g
    #     denominator = 1.0 - p_A_inv_p
        
    #     exact_natural_grad_active = A_inv_g + (numerator / denominator.clamp(min=1e-15))
        
    #     # Reassemble: Active keys get Natural Gradients, Waiting keys keep raw gradients (which are 0)
    #     final_grad = g.clone()
    #     final_grad[active_mask] = exact_natural_grad_active
        
    #     pi_logits_union.grad = final_grad
    #     torch.nn.utils.clip_grad_norm_([pi_logits_union], max_norm=1.0)
        
    
    optimizer.step()

    # -----------------------------------------------------------------
    # PHASE 4: PRUNING & MIGRATION
    # Enforce O(N) sparsity by severing the weakest unpromising transitions
    # -----------------------------------------------------------------
    with torch.no_grad():
        updated_pi_vals = F.softmax(pi_logits_union, dim=-1)
        
        # Take the top N_active states to become the definitive baseline for Iteration t+1
        top_vals, top_idx = torch.topk(updated_pi_vals, k=min(N_tracking, N_union))
        
        pi_vals = top_vals / top_vals.sum()
        pi_keys = unique_keys[top_idx]


    # -----------------------------------------------------------------
    # PHASE 5: LOGGING & VALIDATION 
    # -----------------------------------------------------------------
    # # Validation (Simulate intractable "Full" loss by heavily expanding the capacity)
    # if i_iteration % val_iterations == 0:
    #     with torch.no_grad():
    #         pi_P_vals_val, pi_P_keys_val = compute_stationary_distribution(
    #             pi_vals, pi_keys, N_active * 10, model, 
    #             K_list, sem_heads_list, pos_heads_list, 
    #             M_iterations, temperature, top_p
    #         )
    #         loss_full = 1e5 * compute_aligned_jsd(pi_vals, pi_keys, pi_P_vals_val, pi_P_keys_val)
    #         full_losses.append(loss_full.item())

    if i_iteration % print_iterations == 0:
        grad_mean = pi_logits_union.grad.abs().mean().item()
        grad_max = pi_logits_union.grad.abs().max().item()
        print(f"\nIteration {i_iteration} | Grad Mean: {grad_mean:.2e}, Max: {grad_max:.2e}")
        
        val_loss_str = f"{full_losses[-1]:.2e}" if full_losses else "N/A"
        print(f"Loss: {loss.item():.2e} || Full Loss: {val_loss_str}")
        
        # Display the highest tracked K-grams and their probabilities
        top_k_display = min(5, N_active)
        top_pi_vals, top_pi_idx = pi_vals.topk(top_k_display)
        top_P_vals, top_P_idx = pi_P_vals.topk(top_k_display)
        
        print(f"pi:\n{top_pi_vals.tolist()}\nKeys:\n{pi_keys[top_pi_idx].tolist()}")
        print(f"pi_P:\n{top_P_vals.tolist()}\nKeys:\n{pi_P_keys[top_P_idx].tolist()}")

### Profiling Timescales

In [ ]:
### Profile model heads' timescales
# continuous_corpus = load_and_tokenize_continuous_corpus(model)

# TinyStories
continuous_corpus = load_and_tokenize_continuous_corpus(
    model=model,
    dataset_name="roneneldan/TinyStories",
    dataset_config="default", 
    split="train[:10000]"
)

continuous_corpus = continuous_corpus[:1_000_000]

In [ ]:
mu_tensor, sigma_tensor, mu_bos_tensor, sigma_bos_tensor = profile_deep_heads_empirical(
    model=model, 
    corpus_tokens=continuous_corpus, 
    ensemble_size=800, 
    seq_len=model.cfg.n_ctx,
    batch_size=2
)

a_bar = compute_marginalized_expected_attention(mu_tensor, sigma_tensor, mu_bos_tensor, sigma_bos_tensor)
a_bar_T = compute_unmarginalized_expected_attention(mu_tensor, sigma_tensor, mu_bos_tensor, sigma_bos_tensor)

In [ ]:
receptive_fields = compute_attention_mass_horizons(F.softmax(mu_tensor, dim=-1), thresholds=(0.85, 0.9, 0.95))
receptive_fields_a_bar = compute_attention_mass_horizons(a_bar_T[:,:,400], thresholds=(0.85, 0.9, 0.95))
adversarial_receptive_windows = compute_adversarial_influence_horizon(mu_tensor, sigma_tensor, multiplier=0.9, threshold=0.1)

In [ ]:
layer = 0
head = 15
seq_len = 70
plot_head_thermodynamic_profile(
    mu_tensor[layer, head,:seq_len], 
    sigma_tensor[layer, head,:seq_len], 
    layer, 
    head,
    sigma_multiplier=1.0
)

print(f"Sigma: {sigma_tensor[layer, head, :100].mean()}")

T = 70
plt.figure(figsize=(10,8))
plt.plot(a_bar_T[layer,head,T][:T].cpu(), "-o")
plt.grid()
plt.xlabel("Relative Position")
plt.ylabel("Marginalized Expected Attention")
plt.title(f"a_bar for head {layer}.{head}")
plt.show()



In [ ]:
plt.figure()
# plt.plot([0]*16, receptive_fields[0.85][0].cpu().numpy(), '.')
plt.plot(np.arange(16), receptive_fields[0.9][0].cpu().numpy(), '.')
plt.grid()
plt.xlabel("Head Index")
plt.ylabel("Receptive Field")
plt.show()

### NEW VERSION - Partition-Function Based Optimization

In [ ]:
### NEW VERSION - MIXED 3-PART PARTITION COMPUTATION ###
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import math

N_union_list = []
criterion = nn.L1Loss(reduction="sum")

# =====================================================================
# 2. BLACK BOX OPTIMIZATION SETUP
# =====================================================================
vocab_size = model.cfg.d_vocab
device = next(model.parameters()).device
eps = 1e-10

for p in model.parameters():
    p.requires_grad = False

# --- Pre-Compute required values --- 
knockout_positional = True
if knockout_positional:
    positional_hook_name = "hook_pos_embed"
    positional_hook_fn = remove_pos_embed_hook
    hooks = [(positional_hook_name, positional_hook_fn)]
else:
    hooks = []

with torch.no_grad():
    with model.hooks(fwd_hooks=hooks):
        sink_k, sink_v = extract_bos_sink(model)
sink_k_list = [sink_k[layer] for layer in range(model.cfg.n_layers)]
sink_v_list = [sink_v[layer] for layer in range(model.cfg.n_layers)]

dummy_tokens = torch.tensor(np.arange(1000,1000+model.cfg.n_ctx)).unsqueeze(0)
ln_avg_sigma_list = [extract_frozen_sigma(model, dummy_tokens, layer) for layer in range(model.cfg.n_layers)]

# --- Multi-Layer Positional/Semantic Definitions ---
### attn-only-1l
# K_list = [10]
# active_heads_list = [[0,1,2,3,4,5,6,7]]
K_list = [1]
active_heads_list = [[3]]
# L_ctx_list = [torch.tensor([200, 11, 20, 350, 550, 11, 220, 11]).to(model.cfg.device)]
# C_far_list = [torch.tensor([1e-1]).to(model.cfg.device)]
# C_far_list = [torch.tensor([estimate_c_far_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list = [torch.tensor([estimate_L_ctx_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list, C_far_list = profile_thermodynamic_heads(model, K_list, num_prompts=100, seq_len=1024, batch_size=1)
L_ctx_all, C_far_all = profile_thermodynamic_heads(model, K_list, num_prompts=100, seq_len=1024, batch_size=1)   # [n_heads] each
L_ctx_list = [L_ctx_all[0][active_heads_list[0]]]   # tensor([...]) — 1 entry
C_far_list = [C_far_all[0][active_heads_list[0]]]   # tensor([...]) — 1 entry

### attn-only-2l
# K_list = [10, 10]
# active_heads_list = [[0,1,2,3,4,5,6,7], [0,1,2,3,4,5,6,7]]
# L_ctx_list = [torch.tensor([200, 11, 20, 350, 550, 11, 220, 11]).to(model.cfg.device)]
# C_far_list = [torch.tensor([1e-1]).to(model.cfg.device)]
# C_far_list = [torch.tensor([estimate_c_far_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list = [torch.tensor([estimate_L_ctx_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list, C_far_list = profile_thermodynamic_heads(model, K_list, num_prompts=100, seq_len=1024, batch_size=4)

# The exact initial sequence length required to compute 1 output token through L layers
S_init = sum(K_list) - len(K_list) + 1  #
M_iterations = 1
M_iterations_explore = 1
N_active = 256
N_tracking = int(vocab_size / 4)
K_pruning = N_active * 2  # the number of transition to save for each query

# --- Initialization of the Active K-Gram Subspace ---
# initial_k_grams = [
    # [15023, 17423]*5,
    # [17423, 15023]*5,
    # [5049]*10,
    # [16206],
    # [31557],
    # [1094,24714,1094,24714,1094],
    # [2,2,1],
    # [2, 1] 
    # [2, 1] 
# ]
pi_K_vals, pi_K_indices = get_kgram_distribution_from_tokens(model.to_tokens(corpus[0][:5000]), K=K_list[0])
initial_k_grams = pi_K_indices.tolist()
# Pad with random or base-rate K-grams to reach N_active capacity
while len(initial_k_grams) < N_tracking:
    initial_k_grams.append(torch.randint(0, vocab_size, (S_init,), ).tolist()) # note - k-grams are not guaranteed to be unique

# The Coordinate Tensor: [N_tracking, S_init]
pi_keys = torch.tensor(initial_k_grams, device=device, dtype=torch.long)
pi_keys = torch.unique(pi_keys, dim=0)

# The Probability Tensor: [N_tracking]
pi_vals = torch.ones(pi_keys.shape[0], device=device) * eps
# pi_vals[torch.all(pi_keys == torch.tensor(initial_k_grams[0], device=device), dim=1)] = 0.9
# pi_vals[torch.all(pi_keys == torch.tensor(initial_k_grams[1], device=device), dim=1)] = 0.5
# pi_vals[torch.all(pi_keys == torch.tensor(initial_k_grams[2], device=device), dim=1)] = 0.3
for i in range(pi_K_vals.size(0)):
    pi_vals[torch.all(pi_keys == pi_K_indices[i].to(device), dim=1)] = pi_K_vals[i]
pi_vals = pi_vals / pi_vals.sum()

# --- Optimization Parameters ---
lr_max = 1e0
lr_min = 1e-1
n_iterations = 3000
val_iterations = 20000
print_iterations = 20
temperature = 1.0
top_p = 1.0

losses = []
full_losses = []

# =====================================================================
# 3. THE EXPLORE-THEN-OPTIMIZE LOOP
# =====================================================================
for i_iteration in tqdm(range(n_iterations)):
    
    # -----------------------------------------------------------------
    # PHASE 1: EXPLORATION PASS (NO GRADIENTS) [cite: 1157, 1158]
    # Fast rollout to discover emergent K-grams organically produced 
    # by the transition matrix P_pi.
    # -----------------------------------------------------------------
    # only use the top N k-grams as queries
    # TODO: increase to larger number for pass with no gradient?
    top_pi_vals, top_pi_inds = pi_vals.topk(N_active)
    top_pi_keys = pi_keys[top_pi_inds]

    with torch.no_grad():
        pi_P_vals_explore, pi_P_keys_explore = compute_stationary_distribution(
            top_pi_vals, top_pi_keys, N_tracking, K_pruning, model,
            M_iterations_explore, K_list, active_heads_list, ln_avg_sigma_list,
            L_ctx_list, C_far_list, sink_k_list, sink_v_list
        )
        
        # Determine the expanded Union Support [cite: 1158]
        cat_keys = torch.cat([pi_keys, pi_P_keys_explore], dim=0)
        unique_keys, inverse_indices = torch.unique(cat_keys, dim=0, return_inverse=True)
        N_union = unique_keys.size(0)
        
        # Expand current pi onto the Union Support [cite: 1159]
        # Existing keys maintain their current mass. Newly discovered keys get 2 * epsilon.
        pi_vals_union = torch.ones(N_union, device=device) * 2 * eps
        pi_vals_union.scatter_(0, inverse_indices[:pi_keys.size(0)], pi_vals)
        pi_vals_union = pi_vals_union / pi_vals_union.sum() 
        # print(f"N_union:\n{N_union}")
        # N_union_list.append(N_union)
        # print(f"pi_vals_union:\n{pi_vals_union.topk(5)}")
        
    # -----------------------------------------------------------------
    # PHASE 2: OPTIMIZATION PASS (WITH GRADIENTS) [cite: 1161, 1162]
    # Re-initialize the active parameters over the expanded support.
    # The new keys are now valid leaf parameters inside the autograd graph.
    # -----------------------------------------------------------------
    pi_logits_union = nn.Parameter(torch.log(pi_vals_union))

    current_lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * i_iteration / n_iterations))
    
    # We instantiate a fresh optimizer since SGD w/ momentum=0 has no internal state buffer
    optimizer = torch.optim.SGD([pi_logits_union], lr=current_lr, momentum=0)
    optimizer.zero_grad()
    
    pi_vals_active = F.softmax(pi_logits_union, dim=-1)

    # only use the top N k-grams as queries
    top_pi_vals_active, top_pi_active_inds = pi_vals_active.topk(N_active)
    top_pi_active_keys = unique_keys[top_pi_active_inds]
    
    # Compute the rigorously tracked forward pass mapping for the exact Jacobian/Gradients
    pi_P_vals, pi_P_keys = compute_stationary_distribution(
        top_pi_vals_active, top_pi_active_keys, N_active, K_pruning, model, 
        M_iterations, K_list, active_heads_list, ln_avg_sigma_list,
        L_ctx_list, C_far_list, sink_k_list, sink_v_list
    )
    
    # Calculate JSD Loss. Because pi_vals_active contains epsilon mass for the new keys,
    # the JSD acts as a continuous gradient pull toward highly favorable new states.
    loss = compute_aligned_jsd(pi_vals_active, unique_keys, pi_P_vals.clamp(1e-10), pi_P_keys) * 1e5 / vocab_size
    # loss = 1e5 * compute_aligned_jsd(top_pi_vals_active, top_pi_active_keys, pi_P_vals.clamp(1e-15), pi_P_keys)
    losses.append(loss.item())

    loss.backward()

    # -----------------------------------------------------------------
    # PHASE 3: NATURAL GRADIENT ON THE SIMPLEX
    # -----------------------------------------------------------------
    with torch.no_grad():
        g = pi_logits_union.grad
        
        # if i_iteration % print_iterations == 0:
            # print(f"pi_logits_union: {pi_logits_union.topk(10)}")
            # print(f"grad: {pi_logits_union.grad.abs().topk(10)}")
        lmbda = 1e-10
        
        # Sherman-Morrison exact preconditioned gradient for probability simplex
        A_inv = 1.0 / (pi_vals_active + lmbda)
        A_inv_g = A_inv * g                      
        A_inv_p = A_inv * pi_vals_active                
        
        p_A_inv_g = torch.sum(pi_vals_active * A_inv_g) 
        p_A_inv_p = torch.sum(pi_vals_active * A_inv_p) 
        
        numerator = A_inv_p * p_A_inv_g
        denominator = 1.0 - p_A_inv_p
        
        exact_natural_grad = A_inv_g + (numerator / denominator.clamp(min=1e-10))
        if i_iteration % print_iterations == 0:
            print(f"exact_natural_grad norm: {torch.norm(exact_natural_grad)}")
        pi_logits_union.grad = exact_natural_grad
        # if i_iteration % print_iterations == 0:
            # print(f"scaled grad: {pi_logits_union.grad.abs().topk(10)}")
        
        torch.nn.utils.clip_grad_norm_([pi_logits_union], max_norm=1.0)
        # if i_iteration % print_iterations == 0:
            # print(f"scaled grad: {pi_logits_union.grad.abs().topk(10)}")
        
    
    optimizer.step()

    # -----------------------------------------------------------------
    # PHASE 4: PRUNING & MIGRATION 
    # Enforce O(N) sparsity by severing the weakest unpromising transitions
    # -----------------------------------------------------------------
    with torch.no_grad():
        updated_pi_vals = F.softmax(pi_logits_union, dim=-1)
        
        # Take the top N_active states to become the definitive baseline for Iteration t+1
        top_vals, top_idx = torch.topk(updated_pi_vals, k=min(N_tracking, N_union))
        
        pi_vals = top_vals / top_vals.sum()
        pi_keys = unique_keys[top_idx]


    # -----------------------------------------------------------------
    # PHASE 5: LOGGING & VALIDATION 
    # -----------------------------------------------------------------
    # # Validation (Simulate intractable "Full" loss by heavily expanding the capacity)
    # if i_iteration % val_iterations == 0:
    #     with torch.no_grad():
    #         pi_P_vals_val, pi_P_keys_val = compute_stationary_distribution(
    #             pi_vals, pi_keys, N_active * 10, model, 
    #             K_list, sem_heads_list, pos_heads_list, 
    #             M_iterations, temperature, top_p
    #         )
    #         loss_full = 1e5 * compute_aligned_jsd(pi_vals, pi_keys, pi_P_vals_val, pi_P_keys_val)
    #         full_losses.append(loss_full.item())

    if i_iteration % print_iterations == 0:
        grad_mean = pi_logits_union.grad.abs().mean().item()
        grad_max = pi_logits_union.grad.abs().max().item()
        print(f"\nIteration {i_iteration} | Grad Mean: {grad_mean:.2e}, Max: {grad_max:.2e}")
        
        val_loss_str = f"{full_losses[-1]:.2e}" if full_losses else "N/A"
        print(f"Loss: {loss.item():.2e} || Full Loss: {val_loss_str}")
        
        # Display the highest tracked K-grams and their probabilities
        top_k_display = min(5, N_active)
        top_pi_vals, top_pi_idx = pi_vals.topk(top_k_display)
        top_P_vals, top_P_idx = pi_P_vals.topk(top_k_display)
        
        print(f"pi:\n{top_pi_vals.tolist()}\nKeys:\n{pi_keys[top_pi_idx].tolist()}")
        print(f"pi_P:\n{top_P_vals.tolist()}\nKeys:\n{pi_P_keys[top_P_idx].tolist()}")

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(np.arange(1, len(losses)+1), (losses))
# plt.plot(np.arange(1, len(losses)+1), np.log10(losses))
# plt.plot(np.arange(1, len(losses)+1), np.log10(losses_full))
plt.grid()
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss Vs. Iteration")

# plt.figure(figsize=(10, 8))
# plt.plot(N_union_list)
# plt.grid()
# plt.xlabel("Iteration")
# plt.ylabel("N_unique")
# plt.title("N_unique Vs. Iteration")

# plt.figure(figsize=(10, 8))
# plt.plot(np.arange(1, len(losses)+1), [pi_t[17593] for pi_t in pi_list])
# plt.grid()
# plt.xlabel("Iteration")
# plt.ylabel("Pi[token]")
# plt.title("Pi[token] Vs. Iteration")

top_to_display = 50
top_values = pi_vals[:top_to_display]

token_labels = [
    repr(model.to_string(token_id.item()))
    for token_id in pi_keys[:top_to_display]
]

plt.figure(figsize=(16, 10))
plt.barh(token_labels[::-1], top_values.cpu().numpy()[::-1])
plt.xlabel("Probability")
plt.ylabel("Token")
plt.title(f"Top-{top_to_display} Tokens in $\\pi_{{final}}$")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
print(top_pi_vals[pi_keys[top_pi_idx][:,-1] == 17423].sum())

### K-gram Exploration

In [ ]:
### K-Gram Exploration

vocab_size = model.cfg.d_vocab

## Exploration Parameters
epsilon_power_method = 1e-6
epsilon_dijkstra = 5e-7
epsilon_leakage = 1e-3
epsilon_frontier = 1e-3
max_iterations = 20
query_batch_size = 1024
suffix_chunk_size = 2048
temperature = 1.0
top_p = 0.9
min_stable_steps = 1

for p in model.parameters():
    p.requires_grad = False

# --- Pre-Compute required values --- 
knockout_positional = True
if knockout_positional:
    positional_hook_name = "hook_pos_embed"
    positional_hook_fn = remove_pos_embed_hook
    hooks = [(positional_hook_name, positional_hook_fn)]
else:
    hooks = []
with torch.no_grad():
    with model.hooks(fwd_hooks=hooks):
        sink_k, sink_v = extract_bos_sink(model)
sink_k_list = [sink_k[layer] for layer in range(model.cfg.n_layers)]
sink_v_list = [sink_v[layer] for layer in range(model.cfg.n_layers)]

dummy_tokens = torch.tensor(np.arange(1000,1000+model.cfg.n_ctx)).unsqueeze(0)
ln_avg_sigma_list = [extract_frozen_sigma(model, dummy_tokens, layer) for layer in range(model.cfg.n_layers)]

# --- Multi-Layer Positional/Semantic Definitions ---
### attn-only-1l
# K_list = [10]
# active_heads_list = [[0,1,2,3,4,5,6,7]]
K_list = [1]
active_heads_list = [[3]]
# L_ctx_list = [torch.tensor([150, 10, 10, 200, 300, 10, 150, 10]).to(model.cfg.device)]
# C_far_list = [torch.tensor([1e-1]).to(model.cfg.device)]
# C_far_list = [torch.tensor([estimate_c_far_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=500, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list = [torch.tensor([estimate_L_ctx_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
L_ctx_list, C_far_list = profile_thermodynamic_heads(model, K_list, num_prompts=100, seq_len=1024, batch_size=1)

### attn-only-2l
# K_list = [10, 10]
# active_heads_list = [[0,1,2,3,4,5,6,7], [0,1,2,3,4,5,6,7]]
# # L_ctx_list = [torch.tensor([200, 11, 20, 350, 550, 11, 220, 11]).to(model.cfg.device)]
# # C_far_list = [torch.tensor([1e-1]).to(model.cfg.device)]
# # C_far_list = [torch.tensor([estimate_c_far_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# # L_ctx_list = [torch.tensor([estimate_L_ctx_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list, C_far_list = profile_thermodynamic_heads(model, K_list, num_prompts=100, seq_len=1024, batch_size=4)

# tinystories-1L
# K_list = [40]
# active_heads_list = [[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]]
# L_ctx_list = [torch.tensor([40, 100, 150, 100, 40, 40, 40, 40, 40, 40, 40, 180, 40, 40, 180, 40]).to(model.cfg.device)]
# C_far_list = [torch.tensor([1e-1]).to(model.cfg.device)]
# C_far_list = [torch.tensor([estimate_c_far_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=400, pos_offset=50) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list = [torch.tensor([estimate_L_ctx_for_head(model, layer, head, K_list[layer], ln_avg_sigma_list[layer], seq_len=800, pos_offset=100) for head in active_heads_list[layer]]).to(model.cfg.device) for layer in range(model.cfg.n_layers)]
# L_ctx_list, C_far_list = profile_thermodynamic_heads(model, K_list, num_prompts=100, seq_len=1024, batch_size=1)

# The exact initial sequence length required to compute 1 output token through L layers
S_init = sum(K_list) - len(K_list) + 1  #

forward_pass_kwargs = {
        "K_list": K_list,
        "ln_avg_sigma_list": ln_avg_sigma_list,
        "C_far_list": C_far_list,
        "L_ctx_list": L_ctx_list,
        "sink_k_list": sink_k_list,
        "sink_v_list": sink_v_list,
        "temperature": temperature,
        "top_p": top_p,
        "active_heads_list": active_heads_list
    }

# --- Initialization of the Active K-Gram Subspace ---
# context_k_grams = [
    # [15023, 17423]*5,
    # [17423, 15023]*5,
    # [5049]*10,
    # [16206],
    # [31557],
    # [1094,24714,1094,24714,1094],
    # [2,2,1],
    # [2, 1] 
    # [2, 1] 
# ]
curr_position = 400
effective_context_window_size = 1000
first_position = max(0, curr_position-effective_context_window_size)
# pi_K_vals, pi_K_indices = get_kgram_distribution_from_tokens(model.to_tokens(generated)[0,first_position:curr_position+1], K=10)
pi_K_vals, pi_K_indices = get_kgram_distribution_from_tokens(model.to_tokens(result['text'])[0,first_position:curr_position+1], K=S_init, pad_id=model.tokenizer.pad_token_id, offset=S_init)
# pi_K_vals[-2] = 0.3 # patch a specific K-gram distribution
context_k_grams = pi_K_indices.tolist()

# The Coordinate Tensor: [N_tracking, S_init]
pi_keys = torch.tensor(context_k_grams, device=device, dtype=torch.long)
pi_keys = torch.unique(pi_keys, dim=0)

# The Probability Tensor: [N_tracking]
pi_vals = torch.zeros(pi_keys.shape[0], device=device)
# pi_vals[torch.all(pi_keys == torch.tensor(context_k_grams[0], device=device), dim=1)] = 0.9
# pi_vals[torch.all(pi_keys == torch.tensor(context_k_grams[1], device=device), dim=1)] = 0.5
# pi_vals[torch.all(pi_keys == torch.tensor(context_k_grams[2], device=device), dim=1)] = 0.3
for i in range(pi_K_vals.size(0)):
    pi_vals[torch.all(pi_keys == pi_K_indices[i].to(device), dim=1)] = pi_K_vals[i]
pi_vals = pi_vals / pi_vals.sum()

exploration_engine = MetastableKGramEngine(
    model,
    k_value=S_init,
    frozen_context_keys=pi_keys,
    frozen_context_vals=pi_vals,
    forward_pass_kwargs=forward_pass_kwargs,
    epsilon_power_method=epsilon_power_method,
    epsilon_dijkstra=epsilon_dijkstra,
    query_batch_size=query_batch_size,
    suffix_chunk_size=suffix_chunk_size,
    device=model.cfg.device
)

# initial_k_grams_list = [[1062]*S_init]
# initial_k_grams = torch.tensor(initial_k_grams_list)
# initial_k_grams = model.to_tokens(generated)[:,:curr_position][:,-S_init:]
initial_k_grams = model.to_tokens(result['text'])[:,curr_position-S_init:curr_position]
# initial_k_grams = model.to_tokens(bla)[:,-S_init:]
# initial_k_grams = results_dijk["closed_support_keys"]

results_power = exploration_engine.run_pruned_power_iteration(
    initial_kgrams=initial_k_grams,
    max_iterations=max_iterations,
    frontier_threshold=epsilon_frontier,
    leakage_threshold=epsilon_leakage,
    min_stable_steps=min_stable_steps
)

# results_dijk = exploration_engine.run_dijkstra_support_discovery(
#     initial_kgrams=initial_k_grams,
#     audit_interval=2000
# )

# set_power = set(tuple(r.tolist()) for r in results_power["closed_support_keys"])
# set_dijk = set(tuple(r.tolist()) for r in results_dijk["closed_support_keys"])

# overlap = len(set_power.intersection(set_dijk))


In [ ]:
substochastic_matrix_dict = extract_substochastic_matrix(
    closed_support_keys=results_power["closed_support_keys"],
    # closed_support_keys=results_dijk["closed_support_keys"],
    frozen_context_vals=pi_vals,
    frozen_context_keys=pi_keys,
    model=model,
    forward_pass_kwargs=forward_pass_kwargs,
    epsilon_edge=1e-9,
    query_batch_size=query_batch_size,
    device=device
)

# 1. Initialize Upgraded Analyzer (Shears dangling leaves instantly)
analyzer = MetastableMSMAnalyzer(substochastic_matrix_dict)
initial_k_grams_in_core = model.to_string(initial_k_grams)[0] in model.to_string(analyzer.core_keys)

# 2. Extract Spectrum
spec = analyzer.perform_spectral_decomposition(k_eigenvalues=40)
survival_vector = spec["r_quasi_positive"] / np.max(spec["r_quasi_positive"])
qsd = torch.tensor(spec['qsd_distribution'])

print("\n" + "="*55)
print("     TOPOLOGICAL & SPECTRAL AUDIT")
print("="*55)
print(f"Original Basin Size : {analyzer.sink_idx} K-grams")
print(f"Recurrent Core Size : {analyzer.core_size} K-grams")
print(f"Basin Retention Rate: {spec['lambda_quasi'] * 100:.4f}% probability mass / step")

print("\nCharacteristic Timescales:")
for i, (t_tok, gap, val) in enumerate(zip(spec["implied_timescales_tokens"], spec["spectral_gaps"], spec["evals_complex"])):
    print(f" -> Mode {i+1:02d} | Modulus: {np.abs(val):.4f} | Implied Decay: {t_tok:6.1f} tokens | Gap: {gap:.4f}")


In [ ]:
# Print the strongest K-gram states associated with the first two spectral modes.
n_modes = min(10, len(spec["evals_complex"]))
n_states = 20

eigenvalues = spec["evals_complex"]
right_vectors = spec["evecs_right_complex"]
core_keys = analyzer.core_keys

for mode_idx in range(n_modes):
    eigenvalue = eigenvalues[mode_idx]
    mode_vector = right_vectors[:, mode_idx]

    strongest_indices = np.argsort(np.abs(mode_vector))[::-1][:n_states]

    print("\n" + "=" * 90)
    print(
        f"MODE {mode_idx + 1} | "
        f"eigenvalue = {eigenvalue.real:.6f} "
        f"{eigenvalue.imag:+.6f}i | "
        f"|lambda| = {abs(eigenvalue):.6f}"
    )
    print("=" * 90)

    for rank, state_idx in enumerate(strongest_indices, start=1):
        kgram = core_keys[state_idx]
        vector_value = mode_vector[state_idx]

        print(
            f"{rank:2d}. state={state_idx:6d} | "
            f"magnitude={abs(vector_value):.6e} | "
            f"phase={np.angle(vector_value):+.4f} rad | "
            f"K-gram={repr(model.to_string(kgram))}"
        )

In [ ]:
# 3. Fit PCCA+ Macrostates
num_macro_clusters = 3
pcca_res = analyzer.run_conditioned_pcca(spec, num_clusters=num_macro_clusters)

core_keys = pcca_res["core_support_keys"] # PyTorch Tensor [N_core, K]
memberships = pcca_res["pcca_memberships"] # NumPy Array [N_core, num_clusters]
pi_mass = pcca_res["stationary_measure_pi"] # NumPy Array [N_core]

# 2. Global Macrostate Summary
print(f"Total Recurrent Core K-grams: {core_keys.size(0)}")

for cluster_idx in range(memberships.shape[1]):
    macro_mass = pcca_res["macrostate_mass_proportions"][cluster_idx]
    print(f"\n" + "="*60)
    print(f" MACROSTATE {cluster_idx + 1} (Captures {macro_mass*100:.2f}% of equilibrium meaning)")
    print("="*60)

    # Extract all K-grams assigned to this cluster
    cluster_probs = memberships[:, cluster_idx]
    assigned_mask = pcca_res["crisp_clusters"] == cluster_idx
    
    num_assigned = assigned_mask.sum()
    print(f"Assigned K-grams: {num_assigned} states")

    # Sort the assigned K-grams strictly by their stationary importance (\pi) inside this topic!
    # This filters out the diffuse connective grammar and bubbles the true "Topic Anchors" to the top.
    assigned_indices = np.where(assigned_mask)[0]
    sorted_anchors = assigned_indices[np.argsort(-pi_mass[assigned_indices])]

    print("\nTop 5 Most Thermodynamic K-Gram Anchors:")
    for rank, s_idx in enumerate(sorted_anchors[:5]):
        # Map the core integer row index directly to its vocabulary tensor row
        kgram_tensor = core_keys[s_idx]
        
        # Ask your HookedTransformer to decode the integer IDs to text
        kgram_str = model.to_string(kgram_tensor)
        
        state_pi = pi_mass[s_idx] * 100
        pcca_conf = cluster_probs[s_idx] * 100
        print(f" {rank+1}. [Mass: {state_pi:5.2f}% | PCCA Conf: {pcca_conf:5.1f}%] -> {repr(kgram_str)}")

In [ ]:
# 2. Isolate the eigenvector for Mode 02 (index 1)
# Make sure your K-gram labels list perfectly matches the row indices of P_ocean
mode = 0
cycle_vector = spec['evecs_right_complex'][:, mode]
kgram_labels = analyzer.core_keys # Your list of 85,085 string labels

# 3. Calculate Magnitude and Phase
magnitudes = np.abs(cycle_vector)
phases = np.angle(cycle_vector)

# 4. Filter out the background noise
# A good heuristic is grabbing the top 1% or using a strict cutoff.
# Let's grab the indices of the states with the top 50 highest magnitudes
top_indices = np.argsort(magnitudes)[-100:]

# 5. Extract the surviving states
cycle_states = []
for idx in top_indices:
    cycle_states.append({
        "kgram": model.to_string(kgram_labels[idx]),
        "magnitude": magnitudes[idx],
        "phase": phases[idx]
    })

# 6. Sort chronologically by Phase Angle to reconstruct the sequence
cycle_states_sorted = sorted(cycle_states, key=lambda x: x["phase"])

# 7. Print the reconstructed cycle
print(f"--- MODE {mode} CYCLE RECONSTRUCTION ---")
for state in cycle_states_sorted:
    print(f"Phase: {state['phase']:>6.2f} rad | Mag: {state['magnitude']:.4f} | K-gram: {state['kgram']}")

In [ ]:
u, s, vT = spla.svds(analyzer.P_core_csr, k=10)

In [ ]:
plt.figure(figsize=(10,8))
plt.imshow(np.log10(analyzer.P_core_csr.todense()[:,:]+1e-12))
plt.colorbar()
# plt.show()

plt.figure(figsize=(10,8))
plt.plot(np.log10(np.sort(spec['qsd_distribution'])))
# plt.colorbar()
plt.show()

#### Analyze a Trajectory

In [ ]:
# Analyze a trajectory
## Exploration Parameters
epsilon_power_method = 1e-6
epsilon_dijkstra = 5e-7
epsilon_leakage = 1e-3
epsilon_frontier = 1e-3
max_iterations = 30
query_batch_size = 1024
suffix_chunk_size = 2048
temperature = 0.7
top_p = 1.0
min_stable_steps = 1

forward_pass_kwargs = {
        "K_list": K_list,
        "ln_avg_sigma_list": ln_avg_sigma_list,
        "C_far_list": C_far_list,
        "L_ctx_list": L_ctx_list,
        "sink_k_list": sink_k_list,
        "sink_v_list": sink_v_list,
        "temperature": temperature,
        "top_p": top_p,
        "active_heads_list": active_heads_list
    }

positions = range(20, 830, 20)
effective_context_window_size = 200

lambda_0_list = []
timescale_0_list = []
lambda_1_list = []
timescale_1_list = []
q_func_list = []
SCC_size_list = []
qsd_probs = {"The concert will perform the concert, which will perform": [], " The concert was performed by the concert was performed at": []}
context_probs = {"The concert will perform the concert, which will perform": [], " The concert was performed by the concert was performed at": []}
qsd_size = []
qsd_kgrams = []
qsd_values = []
# context_probs = {}
# for key, val in qsd_probs.items():
#     context_probs[key] = val

for i, curr_position in tqdm(enumerate(positions)):
    first_position = max(0, curr_position-effective_context_window_size)
    pi_K_vals, pi_K_indices = get_kgram_distribution_from_tokens(model.to_tokens(generated)[0,first_position:curr_position], K=10)
    # pi_K_vals, pi_K_indices = get_kgram_distribution_from_tokens(model.to_tokens(prompt)[0,:], K=10)
    # pi_K_vals[-2] = 0.3 # patch a specific K-gram distribution
    context_k_grams = pi_K_indices.tolist()

    # The Coordinate Tensor: [N_tracking, S_init]
    pi_keys = torch.tensor(context_k_grams, device=device, dtype=torch.long)
    pi_keys = torch.unique(pi_keys, dim=0)

    # The Probability Tensor: [N_tracking]
    pi_vals = torch.zeros(pi_keys.shape[0], device=device)
    # pi_vals[torch.all(pi_keys == torch.tensor(context_k_grams[0], device=device), dim=1)] = 0.9
    # pi_vals[torch.all(pi_keys == torch.tensor(context_k_grams[1], device=device), dim=1)] = 0.5
    # pi_vals[torch.all(pi_keys == torch.tensor(context_k_grams[2], device=device), dim=1)] = 0.3
    for i in range(pi_K_vals.size(0)):
        pi_vals[torch.all(pi_keys == pi_K_indices[i].to(device), dim=1)] = pi_K_vals[i]
    pi_vals = pi_vals / pi_vals.sum()

    exploration_engine = MetastableKGramEngine(
        model,
        k_value=S_init,
        frozen_context_keys=pi_keys,
        frozen_context_vals=pi_vals,
        forward_pass_kwargs=forward_pass_kwargs,
        epsilon_power_method=epsilon_power_method,
        epsilon_dijkstra=epsilon_dijkstra,
        query_batch_size=query_batch_size,
        suffix_chunk_size=suffix_chunk_size,
        device=model.cfg.device
    )

    # initial_k_grams_list = [[1062]*S_init]
    # initial_k_grams = torch.tensor(initial_k_grams_list)
    initial_k_grams = model.to_tokens(generated)[:,:curr_position][:,-S_init:]
    # initial_k_grams = model.to_tokens(prompt)[:,-S_init:]
    # initial_k_grams = model.to_tokens(bla)[:,-S_init:]
    # initial_k_grams = results_dijk["closed_support_keys"]

    results_power = exploration_engine.run_pruned_power_iteration(
        initial_kgrams=initial_k_grams,
        max_iterations=max_iterations,
        frontier_threshold=epsilon_frontier,
        leakage_threshold=epsilon_leakage,
        min_stable_steps=min_stable_steps
    )

    results_dijk = exploration_engine.run_dijkstra_support_discovery(
        initial_kgrams=initial_k_grams,
        audit_interval=2000
    )

    substochastic_matrix_dict = extract_substochastic_matrix(
        closed_support_keys=results_power["closed_support_keys"],
        # closed_support_keys=results_dijk["closed_support_keys"],
        frozen_context_vals=pi_vals,
        frozen_context_keys=pi_keys,
        model=model,
        forward_pass_kwargs=forward_pass_kwargs,
        epsilon_edge=1e-9,
        query_batch_size=query_batch_size,
        device=device
    )

    # 1. Initialize Upgraded Analyzer (Shears dangling leaves instantly)
    analyzer = MetastableMSMAnalyzer(substochastic_matrix_dict)
    initial_k_grams_in_core = model.to_string(initial_k_grams)[0] in model.to_string(analyzer.core_keys)

    # 2. Extract Spectrum
    spec = analyzer.perform_spectral_decomposition(k_eigenvalues=10)
    survival_vector = spec["r_quasi_positive"] / np.max(spec["r_quasi_positive"])
    qsd = torch.tensor(spec['qsd_distribution'], device=device)

    print("\n" + "="*55)
    print("     TOPOLOGICAL & SPECTRAL AUDIT")
    print("="*55)
    print(f"Original Basin Size : {analyzer.sink_idx} K-grams")
    print(f"Recurrent Core Size : {analyzer.core_size} K-grams")
    print(f"Basin Retention Rate: {spec['lambda_quasi'] * 100:.4f}% probability mass / step")

    print("\nCharacteristic Timescales:")
    for i, (t_tok, gap, val) in enumerate(zip(spec["implied_timescales_tokens"], spec["spectral_gaps"], spec["evals_complex"])):
        print(f" -> Mode {i+1:02d} | Modulus: {np.abs(val):.4f} | Implied Decay: {t_tok:6.1f} tokens | Gap: {gap:.4f}")

    lambda_0_list.append(np.abs(spec['evals_complex'][0]))
    timescale_0_list.append(spec['implied_timescales_tokens'][0])
    lambda_1_list.append(np.abs(spec['evals_complex'][1]))
    timescale_1_list.append(spec['implied_timescales_tokens'][1])
    q_func_list.append(compute_aligned_jsd(qsd, analyzer.core_keys, pi_vals, pi_keys))
    SCC_size_list.append(analyzer.core_size)
    for kgram in qsd_probs.keys():
        kgram_ind = torch.all(analyzer.core_keys == model.to_tokens(kgram, prepend_bos=False), dim=1).nonzero().flatten()
        if len(kgram_ind):
            qsd_probs[kgram].append(spec["qsd_distribution"][kgram_ind.item()])
        else:
            qsd_probs[kgram].append(0)

        kgram_ind = torch.all(pi_keys == model.to_tokens(kgram, prepend_bos=False), dim=1).nonzero().flatten()
        if len(kgram_ind):
            context_probs[kgram].append(pi_vals[kgram_ind.item()].item())
        else:
            context_probs[kgram].append(0)
    
    qsd_size.append(1 + compute_attention_mass_horizons(torch.sort(qsd, descending=True).values, thresholds={0.9})[0.9].item())
    qsd_top_vals, qsd_top_inds = qsd.topk(qsd_size[-1])
    qsd_values.append(qsd_top_vals)
    qsd_kgrams.append(analyzer.core_keys[qsd_top_inds])


q_func_tensor = torch.tensor(q_func_list).cpu()

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(positions, lambda_0_list, '--o', label="$\lambda_0$")
plt.plot(positions, lambda_1_list, '--o', label="$\lambda_1$")
plt.grid()
plt.xlabel("Position")
plt.ylabel("$\lambda$")
plt.title("Substochastic Matrix Eigenvalues Vs. Position")
plt.legend()

plt.figure(figsize=(10,8))
plt.plot(positions, timescale_0_list, '--o', label="$\t_0$")
plt.plot(positions, timescale_1_list, '--o', label="$\t_1$")
plt.grid()
plt.xlabel("Position")
plt.ylabel("Timescale [tokens]")
plt.title("Relaxation Timescales Vs. Position")
plt.legend()

plt.figure(figsize=(10,8))
plt.plot(positions, SCC_size_list, '--o')
plt.grid()
plt.xlabel("Position")
plt.ylabel("SCC Core Size")
plt.title("SCC Size Vs. Position")

plt.figure(figsize=(10,8))
plt.plot(positions, qsd_size, '--o')
plt.grid()
plt.xlabel("Position")
plt.ylabel(r"QSD 90% size Size")
plt.title("QSD Size Vs. Position")

plt.figure(figsize=(10,8))
plt.plot(positions, q_func_tensor, '--o')
plt.grid()
plt.xlabel("Position")
plt.ylabel("JSD")
plt.title("JSD Vs. Position")

plt.figure(figsize=(10,8))
for kgram, probs in qsd_probs.items():
    plt.plot(positions, probs, '--o', label=kgram)
for kgram, probs in context_probs.items():
    plt.plot(positions, probs, '-o', label=kgram)
plt.grid()
plt.xlabel("Position")
plt.ylabel("Probability")
plt.title("Stationary and Context Probability Vs. Position")
plt.legend()

plt.show()

In [ ]:
# Compute pairwise JSD distance matrix for the QSD support distributions
N_qsd = len(qsd_kgrams)
device = qsd_values[0].device if isinstance(qsd_values[0], torch.Tensor) else torch.device("cpu")

distance_matrix = torch.zeros((N_qsd, N_qsd), device=device)

for i in range(N_qsd):
    for j in range(i, N_qsd):
        dist = compute_aligned_jsd(
            vals_1=qsd_values[i], keys_1=qsd_kgrams[i],
            vals_2=qsd_values[j], keys_2=qsd_kgrams[j]
        )
        distance_matrix[i, j] = dist
        distance_matrix[j, i] = dist

plt.figure(figsize=(10, 8))
plt.imshow(1-distance_matrix.cpu().numpy(), cmap="Reds", aspect="auto")
plt.colorbar(label="JSD")
plt.title("Pairwise QSD JSD Affinity Matrix")
plt.xlabel("QSD Index")
plt.ylabel("QSD Index")
plt.tight_layout()
plt.show()

### Examining Trajectories

In [ ]:
## compute loss at positional and non-positional fixed points
heads = [0,1,2,3,4,5,6,7]
with torch.no_grad():

    pi_positional = pi_from_context(model.to_tokens(clean_output)).clamp(min=1e-15)
    # pi_positional = torch.zeros(model.cfg.d_vocab, device=device).clamp(min=1e-15)
    # pi_positional[2320] = 1
    pi_P_positional = pi_to_pi_P_one_layer(pi_positional, heads, model, chunk_size=2048, temperature=1, p=0.9).squeeze().clamp(min=1e-15)
    M_pi_pos = 0.5 * (pi_positional + pi_P_positional)
    loss_pos = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi_pos.log(), 
        target=pi_P_positional.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi_pos.log(), 
        target=pi_positional.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

    pi_nonpositional = pi_from_context(model.to_tokens(patched_output)).clamp(min=1e-15)
    pi_P_nonpositional = pi_to_pi_P_one_layer(pi_nonpositional, heads, model, chunk_size=2048, temperature=1, p=0.9).squeeze().clamp(min=1e-15)
    M_pi_nonpos = 0.5 * (pi_nonpositional + pi_P_nonpositional)
    loss_nonpos = 0.5 * 1e5 * (F.kl_div(  
        input=M_pi_nonpos.log(), 
        target=pi_P_nonpositional.log(), 
        reduction='batchmean', 
        log_target=True
    ) + F.kl_div(  
        input=M_pi_nonpos.log(), 
        target=pi_nonpositional.log(), 
        reduction='batchmean', 
        log_target=True
    )
    )

print(f"Positional Pi Loss:\t{loss_pos}")
print(f"Non-Positional Pi Loss:\t{loss_nonpos}")

In [ ]:
with torch.no_grad():
    pi_t_clean = pi_t_from_context(model.to_tokens(clean_output)[0])
    pi_t_patched = pi_t_from_context(model.to_tokens(patched_output)[0])
plt.figure(figsize=(10,8))
plt.plot(pi_t_clean[:, 5210].cpu().numpy())
plt.plot(pi_t_patched[:, 5210].cpu().numpy())
plt.grid()
plt.show()

In [ ]:
with torch.no_grad():
    pi_t_pos = pi_t_from_context(model.to_tokens(clean_output)[0])
    pi_t_semantic = pi_t_from_context(model.to_tokens(patched_output)[0])
    loss_t = loss_from_pi_t(pi_t_semantic[:], model, step=5, scale=1e5, heads=[0,3])
    # loss_t_semantic = loss_from_pi_t(pi_t_semantic[:], model, step=1, scale=1e5)

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(loss_t)
# plt.plot([42, 42], [0,1])
# plt.plot([46+30, 46+30], [0,1])
# plt.plot([46+30+18, 46+30+18], [0,1.4], '--')
plt.plot([200/5, 200/5], [0,1], '--')
plt.xlabel("Position")
plt.ylabel("JSD Loss")
plt.grid()
plt.title("JSD Loss Vs. Position")

plt.show()

## Wide Semantic Head Optimization

In [ ]:
### Wide Semantic Head Optimization - Multi-GPU Fixed Point Search over a Real Corpus ###
# One active head per GPU, one independent optimization per (text, position) pair.

def sample_optimization_positions(n_tokens: int, n_positions: int, min_position: int, n_separation: int, rng: random.Random):
    """
    Randomly choose up to `n_positions` context lengths in [min_position, n_tokens),
    pairwise separated by at least `n_separation` tokens.

    Returns: sorted list of ints (each is a context length, i.e. tokens[:position]).
    """
    candidates = list(range(min_position, n_tokens))  # all admissible cut points
    rng.shuffle(candidates)

    chosen = []
    for candidate in candidates:
        if len(chosen) >= n_positions:
            break
        if all(abs(candidate - other) >= n_separation for other in chosen):
            chosen.append(candidate)
    return sorted(chosen)


def pi_to_sparse_cpu(pi: torch.Tensor, top_k: int = 4096):
    """
    Compress a dense distribution [vocab] to a coalesced CPU sparse COO tensor holding
    only its top-k entries - results are shipped back across processes, so we never
    move full dense 128k-vocab vectors around.
    """
    values, indices = pi.detach().topk(min(top_k, pi.numel()))
    keep = values > 0
    return torch.sparse_coo_tensor(
        indices[keep].view(1, -1).cpu(),          # shape: [1, n_kept]
        values[keep].float().cpu(),               # shape: [n_kept]
        size=(pi.numel(),)
    ).coalesce()


def optimize_pi_one_layer(pi_init: torch.Tensor, heads: list, model: HookedTransformer,
                          lr: float = 1e0, max_iterations: int = 1000, chunk_size: int = 64,
                          temperature: float = 1.0, top_p: float = 0.9,
                          loss_threshold: float = 1e-5, loss_scale: float = 1e5,
                          lmbda: float = 1e-10, max_grad_norm: float = 1.0,
                          eta_min: float = 1e-2, print_iterations: int = 0):
    """
    Natural Gradient Descent fixed point search for the 1-layer, token-level operator:
    minimize  loss_scale * JSD(pi, pi P_pi) / vocab_size  over the simplex.

    Follows cell 52 (Sherman-Morrison natural gradient on `pi_logits`), with no
    exploration phase, no validation pass and an early stop on `loss_threshold`.

    Args:
        pi_init: [vocab] initial distribution (the empirical unigram measure of the context).
        heads:   list of layer-0 head indices that are active.

    Returns:
        pi_best:     [vocab] the distribution with the lowest loss seen (detached).
        best_loss:   float, the corresponding loss.
        n_performed: int, number of gradient iterations actually run.
        losses:      list[float], the full loss trace.
    """
    vocab_size = model.cfg.d_vocab

    # --- Parameterization: pi = softmax(pi_logits), so pi stays on the simplex by construction
    pi_init = (pi_init / pi_init.sum()).clamp(min=1e-10)
    pi_logits = nn.Parameter(torch.log(pi_init).float())   # shape: [vocab]

    optimizer = torch.optim.SGD([pi_logits], lr=lr, momentum=0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=max_iterations, eta_min=eta_min)

    losses = []
    pi_best = F.softmax(pi_logits.detach(), dim=-1)
    best_loss = float('inf')
    n_performed = 0

    for i_iteration in range(max_iterations):
        optimizer.zero_grad()

        pi = F.softmax(pi_logits, dim=-1)                          # shape: [vocab]
        top_tokens = pi.detach().topk(chunk_size).indices          # shape: [chunk_size]

        # --- Forward pass: the state-dependent transition operator, restricted to the top-k queries
        pi_P = pi_to_pi_P_one_layer_topk(pi, heads, model, top_tokens=top_tokens,
                                         temperature=temperature, p=top_p)
        pi_P = pi_P.squeeze().clamp(min=1e-15)                     # shape: [vocab]

        # --- Scaled JSD loss
        loss = loss_scale * JSD(pi.clamp(min=1e-15).unsqueeze(0), pi_P.unsqueeze(0)) / vocab_size

        # --- Robustness: a non-finite loss means the top-k support collapsed; stop and keep the best iterate
        if not torch.isfinite(loss):
            print(f"  [warning] non-finite loss at iteration {i_iteration}, stopping early.")
            break

        loss_value = loss.item()
        losses.append(loss_value)
        n_performed = i_iteration + 1

        if loss_value < best_loss:
            best_loss = loss_value
            pi_best = pi.detach().clone()

        if loss_value < loss_threshold:
            break

        loss.backward()

        # --- Exact natural gradient on the simplex (Fisher metric + Sherman-Morrison rank-1 correction)
        with torch.no_grad():
            g = pi_logits.grad
            if g is None or not torch.isfinite(g).all():
                print(f"  [warning] non-finite gradient at iteration {i_iteration}, stopping early.")
                break

            pi_detached = pi.detach()
            A_inv = 1.0 / (pi_detached + lmbda)                    # shape: [vocab]
            A_inv_g = A_inv * g
            A_inv_p = A_inv * pi_detached

            p_A_inv_g = torch.sum(pi_detached * A_inv_g)           # scalar
            p_A_inv_p = torch.sum(pi_detached * A_inv_p)           # scalar

            numerator = A_inv_p * p_A_inv_g
            denominator = (1.0 - p_A_inv_p).clamp(min=1e-15)

            natural_grad = A_inv_g + numerator / denominator
            natural_grad = torch.nan_to_num(natural_grad, nan=0.0, posinf=0.0, neginf=0.0)
            pi_logits.grad = natural_grad

            # Clip to prevent wild swings from the 1/pi division on dead tokens
            torch.nn.utils.clip_grad_norm_([pi_logits], max_norm=max_grad_norm)

        if print_iterations and (i_iteration % print_iterations == 0):
            print(f"  iter {i_iteration:5d} | loss {loss_value:.3e} | grad max {pi_logits.grad.abs().max().item():.2e}")

        optimizer.step()
        scheduler.step()

    return pi_best, best_loss, n_performed, losses


def _wide_optimization_worker(gpu_id: int, head: int, model_name: str, corpus: list,
                              max_tokens: int = 1100, min_position: int = 100, n_positions: int = 1,
                              n_separation: int = 50, loss_threshold: float = 1e-5,
                              lr: float = 1e0, max_iterations: int = 1000, chunk_size: int = 64,
                              temperature: float = 1.0, top_p: float = 0.9, loss_scale: float = 1e5,
                              store_top_k: int = 4096, seed: int = 0, model_dtype=torch.float32):
    """
    Runs on a single GPU with a single active head. Loads its own copy of the model,
    then for every text in the corpus optimizes `n_positions` independent initial conditions.

    Returns: list of result dicts, one per optimization.
    """
    device = f"cuda:{gpu_id}"
    print(f"[GPU {gpu_id} | head {head}] Loading {model_name} on {device}...")

    model = HookedTransformer.from_pretrained(model_name, device=device, fold_ln=False, dtype=model_dtype)
    for param in model.parameters():
        param.requires_grad = False
    torch.set_grad_enabled(True)  # the notebook disables grad globally at load time

    vocab_size = model.cfg.d_vocab
    rng = random.Random(seed + head)   # per-head reproducible position sampling
    results = []

    for i_text, text in enumerate(tqdm(corpus, desc=f"[GPU {gpu_id} | head {head}] texts", position=gpu_id)):
        # --- Tokenize and chop to at most max_tokens tokens
        tokens = model.to_tokens(text)[0][:max_tokens]      # shape: [n_tokens]
        n_tokens = tokens.numel()
        if n_tokens <= min_position:
            continue

        # --- Pick the independent cut points for this text
        positions = sample_optimization_positions(n_tokens, n_positions, min_position, n_separation, rng)

        for position in positions:
            # --- Initial condition: the empirical unigram distribution of the context up to `position`
            context_tokens = tokens[:position]                              # shape: [position]
            pi_init = pi_from_context(context_tokens, vocab_size=vocab_size)  # shape: [vocab]

            pi_final, final_loss, n_iterations, losses = optimize_pi_one_layer(
                pi_init, [head], model,
                lr=lr, max_iterations=max_iterations, chunk_size=chunk_size,
                temperature=temperature, top_p=top_p,
                loss_threshold=loss_threshold, loss_scale=loss_scale,
            )

            # --- Short summary of this single optimization
            if losses:
                print(f"[GPU {gpu_id} | head {head}] text {i_text}, position {position}: "
                      f"initial loss {losses[0]:.3e} -> final loss {final_loss:.3e} "
                      f"after {n_iterations} iterations")
            else:
                print(f"[GPU {gpu_id} | head {head}] text {i_text}, position {position}: "
                      f"no valid iterations (diverged immediately)")

            results.append({
                "head": head,
                "text": text,
                "position": position,
                "n_tokens": n_tokens,
                "pi_init": pi_to_sparse_cpu(pi_init, top_k=store_top_k),     # sparse [vocab]
                "pi_final": pi_to_sparse_cpu(pi_final, top_k=store_top_k),   # sparse [vocab]
                "final_loss": final_loss,
                "n_iterations": n_iterations,
                "losses": losses,
            })

        # Keep VRAM clean across long corpora
        if i_text % 10 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[GPU {gpu_id} | head {head}] Done: {len(results)} optimizations.")
    return results


def run_wide_head_optimization(corpus: list, heads_to_run: list, gpu_ids: list, model_name: str,
                               max_tokens: int = 1100, min_position: int = 100, n_positions: int = 1,
                               n_separation: int = 50, loss_threshold: float = 1e-5,
                               lr: float = 1e0, max_iterations: int = 1000, chunk_size: int = 64,
                               temperature: float = 1.0, top_p: float = 0.9, loss_scale: float = 1e5,
                               store_top_k: int = 4096, seed: int = 0, model_dtype=torch.float32):
    """
    Multi-GPU conductor: runs the same corpus-wide optimization for every head in
    `heads_to_run`, splitting the heads across `gpu_ids` (one head per GPU at a time).

    Args:
        corpus:       list of raw strings.
        heads_to_run: list of layer-0 head indices; each one gets its own experiment.
        gpu_ids:      list of CUDA device ids to fan out over.

    Returns:
        dict {head_index: list of per-optimization result dicts}.
    """
    print(f"Running {len(heads_to_run)} heads {heads_to_run} over {len(corpus)} texts on GPUs {gpu_ids}...")

    # backend="loky" uses cloudpickle, so notebook-defined functions survive serialization
    all_results = Parallel(n_jobs=len(gpu_ids), backend="loky")(
        delayed(_wide_optimization_worker)(
            gpu_id=gpu_ids[i_head % len(gpu_ids)],   # round-robin: heads are split across the GPUs
            head=head,
            model_name=model_name,
            corpus=corpus,
            max_tokens=max_tokens,
            min_position=min_position,
            n_positions=n_positions,
            n_separation=n_separation,
            loss_threshold=loss_threshold,
            lr=lr,
            max_iterations=max_iterations,
            chunk_size=chunk_size,
            temperature=temperature,
            top_p=top_p,
            loss_scale=loss_scale,
            store_top_k=store_top_k,
            seed=seed,
            model_dtype=model_dtype,
        )
        for i_head, head in enumerate(heads_to_run)
    )

    results_by_head = {head: head_results for head, head_results in zip(heads_to_run, all_results)}

    print("\n=== Wide optimization summary ===")
    for head, head_results in results_by_head.items():
        if len(head_results) == 0:
            print(f"head {head}: no optimizations completed")
            continue
        final_losses = torch.tensor([r["final_loss"] for r in head_results])
        print(f"head {head}: {len(head_results)} optimizations | "
              f"median final loss {final_losses.median().item():.3e} | "
              f"min {final_losses.min().item():.3e} | max {final_losses.max().item():.3e}")

    return results_by_head


### Example Run - WikiText Corpus

In [ ]:
### Example: run the wide optimization over WikiText ###

# --- Corpus: real text from WikiText-103, keeping only articles long enough to reach `min_position`
# wikitext_dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

iterable_dataset = load_dataset(
    "wikimedia/wikipedia", 
    "20231101.en", 
    split="train", 
    streaming=True
)
# 2. Grab only the first 500 examples
small_iterable = iterable_dataset.take(500)
# 3. Convert the iterable back into a standard, in-memory Hugging Face Dataset
small_dataset = Dataset.from_list(list(small_iterable))

# corpus = [text for text in wikitext_dataset["text"][:20000] if len(text) > 3000][:1]  # list of strings
# corpus = [text for text in small_dataset["text"][:20000] if len(text) > 5000][:90]  # list of strings
# corpus = [" rap" * 400, " yoga Matra" * 300, " account" * 46*3 + " downstream" * 30*3 + " upstream" * 18*3]
corpus = [model.to_string(context_from_pi(pi_film_cluster, N=1000))]
print(f"Corpus: {len(corpus)} texts")

# --- Sampling of initial conditions
max_tokens = 1500      # chop every text to at most this many tokens
min_position = 700     # earliest context length to optimize from
n_positions = 1        # independent initial conditions per text
n_separation = 50      # minimum token separation between chosen positions

# --- Optimization parameters (shared by every head / GPU)
lr = 1e0
max_iterations = 3500
chunk_size = 512        # number of top-k query tokens in pi_to_pi_P_one_layer_topk
temperature = 1.0
top_p = 0.9            # nucleus parameter
loss_scale = 1e5
loss_threshold = 5e-5  # early stop once the scaled JSD drops below this

# --- Experiment fan-out: one active head per GPU
heads_to_run = [0, 3, 4, 6]
gpu_ids = [1, 2, 3, 4]

wide_results = run_wide_head_optimization(
    corpus=corpus,
    heads_to_run=heads_to_run,
    gpu_ids=gpu_ids,
    model_name=model_name,
    max_tokens=max_tokens,
    min_position=min_position,
    n_positions=n_positions,
    n_separation=n_separation,
    loss_threshold=loss_threshold,
    lr=lr,
    max_iterations=max_iterations,
    chunk_size=chunk_size,
    temperature=temperature,
    top_p=top_p,
    loss_scale=loss_scale,
)

# wide_results: {head: [ {text, position, pi_init, pi_final, final_loss, n_iterations, losses}, ... ]}
example_result = wide_results[heads_to_run[0]][0]
print(f"\nHead {example_result['head']}, position {example_result['position']}, "
      f"final loss {example_result['final_loss']:.3e} after {example_result['n_iterations']} iterations")
print(f"pi_final top-5: {example_result['pi_final'].to_dense().topk(5)}")


In [ ]:
results_dir = "/home/galk/LanguageDynamics/results/wide_optimizations"
os.makedirs(results_dir, exist_ok=True)

wide_run_config = {
    "model_name": model_name,
    "heads_to_run": heads_to_run,
    "gpu_ids": gpu_ids,
    "max_tokens": max_tokens,
    "min_position": min_position,
    "n_positions": n_positions,
    "n_separation": n_separation,
    "lr": lr,
    "max_iterations": max_iterations,
    "chunk_size": chunk_size,
    "temperature": temperature,
    "top_p": top_p,
    "loss_scale": loss_scale,
    "loss_threshold": loss_threshold,
    "corpus_size": len(corpus),
}

save_path = os.path.join(
    results_dir,
    f"wide_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pt",
)

torch.save(
    {
        "wide_results": wide_results,
        "config": wide_run_config,
    },
    save_path,
)

print(f"Saved wide optimization results to: {save_path}")

In [ ]:
head = heads_to_run[3]
ind = 19

losses = wide_results[head][ind]["losses"]
print(f"pi_final top-20: {model.to_string(wide_results[head][ind]['pi_final'].to_dense().topk(20).indices)}")

if wide_results[head][ind]["branching_results_final"] is not None:
    print(f"branching_results_final: {wide_results[head][ind]['branching_results_final']["branching_factor"]}")

pi_final = wide_results[head][ind]["pi_final"].to_dense()
top_values, top_indices = torch.topk(pi_final, k=50)

token_labels = [
    repr(model.to_string(token_id.item()))
    for token_id in top_indices
]

plt.figure(figsize=(16, 10))
plt.barh(token_labels[::-1], top_values.cpu().numpy()[::-1])
plt.xlabel("Probability")
plt.ylabel("Token")
plt.title(f"Top-20 Tokens in $\\pi_{{final}}$ — Head {head}, Result {ind}")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title(f"Optimization Losses — Head {head}, Result {ind}")
plt.grid(True)
plt.show()

In [ ]:
results_dir = "/home/galk/LanguageDynamics/results/wide_optimizations"

pt_files = [
    os.path.join(results_dir, filename)
    for filename in os.listdir(results_dir)
    if filename.endswith(".pt")
]

if not pt_files:
    raise FileNotFoundError(f"No .pt files found in {results_dir}")

save_path = max(pt_files, key=os.path.getmtime)
saved_data = torch.load(save_path, map_location="cpu", weights_only=False)

wide_results = saved_data["wide_results"]
all_results = [
    result
    for head_results in wide_results.values()
    for result in head_results
]

best_results = sorted(
    all_results,
    key=lambda result: float(result["final_loss"])
)[:10]

print(f"Loaded: {save_path}")
print(f"Total optimization results: {len(all_results)}\n")

for rank, result in enumerate(best_results, start=1):
    pi_final = result["pi_final"].coalesce()
    values = pi_final.values()
    token_ids = pi_final.indices()[0]

    k = min(20, values.numel())
    top_values, top_indices = torch.topk(values, k=k)
    top_token_ids = token_ids[top_indices]

    top_tokens = [
        repr(model.to_string(int(token_id)))
        for token_id in top_token_ids
    ]

    print(f"{'=' * 80}")
    print(f"Rank {rank} | Head {result['head']} | Position {result['position']}")
    print(f"Best loss: {float(result['final_loss']):.6e}")
    print(f"Top 20 final tokens: {top_tokens}")
    print(f"Original text prompt:\n{result['text'][:3000]}\n")

In [ ]:
result = best_results[0]
top_for_display = 50

losses = result["losses"]

pi_final = result["pi_final"].to_dense()
# pi_final = pi_film_cluster
top_values, top_indices = torch.topk(pi_final, k=top_for_display)

token_labels = [
    repr(model.to_string(token_id.item()))
    for token_id in top_indices
]

plt.figure(figsize=(16, 10))
plt.barh(token_labels[::-1], top_values.cpu().numpy()[::-1])
plt.xlabel("Probability")
plt.ylabel("Token")
plt.title(f"Top-{top_for_display} Tokens in $\\pi_{{final}}$ — Head {result['head']}")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(losses) + 1), losses)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title(f"Optimization Losses")
plt.grid(True)
plt.show()

In [ ]:
# Select an optimization result and its active layer-0 head
selected_result = result
active_head = int(selected_result["head"])
inactive_heads = [h for h in range(model.cfg.n_heads) if h != active_head]

# Recover the final distribution and sample a 500-token initial prompt
final_pi = selected_result["pi_final"].to_dense().to(device).clone()
# final_pi = selected_result["pi_init"].to_dense().to(device).clone()
sampled_tokens = context_from_pi(final_pi, N=500).to(device)
# sampled_tokens_2 = context_from_pi(final_pi, N=500).to(device)

# Decode without adding BOS here; model.generate will prepend it during tokenization
sampled_prompt = model.to_string(sampled_tokens)

# Ablate every head except the active head and remove positional embeddings
hooks = [
    (
        "blocks.0.attn.hook_z",
        partial(zero_head_hook, head_idx=inactive_heads, min_position=0),
    ),
    ("hook_pos_embed", remove_pos_embed_hook),
]

with torch.no_grad():
    with model.hooks(fwd_hooks=hooks):
        generated_tokens = model.generate(
            sampled_prompt,
            max_new_tokens=500,
            do_sample=True,
            temperature=1.0,
            top_p=0.9,
            prepend_bos=True,
            stop_at_eos=True,
            return_type="tokens"
        )

print(model.to_string(generated_tokens[0, :]))

In [ ]:
# Select an optimization result and its active layer-0 head
num_runs = 2
selected_result = result
active_head = int(selected_result["head"])
inactive_heads = [h for h in range(model.cfg.n_heads) if h != active_head]

# Recover the final distribution and sample a 500-token initial prompt
final_pi = selected_result["pi_final"].to_dense().to(device).clone()
# final_pi = selected_result["pi_init"].to_dense().to(device).clone()
sampled_tokens = context_from_pi(final_pi, N=500).to(device)
# sampled_tokens_2 = context_from_pi(final_pi, N=500).to(device)

# Decode without adding BOS here; model.generate will prepend it during tokenization
sampled_prompt = model.to_string(sampled_tokens)

# Ablate every head except the active head and remove positional embeddings
hooks = [
    (
        "blocks.0.attn.hook_z",
        partial(zero_head_hook, head_idx=inactive_heads, min_position=0),
    ),
    ("hook_pos_embed", remove_pos_embed_hook),
]

old_start = sampled_prompt
for run_idx in range(num_runs):
    with torch.no_grad():
        with model.hooks(fwd_hooks=hooks):
            generated_tokens = model.generate(
                old_start,
                max_new_tokens=500,
                do_sample=True,
                temperature=1.0,
                top_p=0.9,
                prepend_bos=True,
                stop_at_eos=True,
                return_type="tokens"
            )
    old_start = model.to_string(generated_tokens[0, 500:])

print(model.to_string(generated_tokens[0, -100:]))

In [ ]:
tokens = generated_tokens[0].flatten()

pi_first_500 = pi_from_context(tokens[:500], vocab_size=model.cfg.d_vocab).clamp(min=1e-15)
pi_full = pi_from_context(tokens[500:], vocab_size=model.cfg.d_vocab).clamp(min=1e-15)

jsd_first_vs_full = JSD(
    # final_pi.clamp(min=1e-15).unsqueeze(0),
    pi_first_500.unsqueeze(0),
    pi_full.unsqueeze(0),
).item() * 1e5 / model.cfg.d_vocab

print(f"JSD difference (first 500 tokens vs. full text): {jsd_first_vs_full:.6e}")

In [ ]:
# Finite sample size error comparison: compute JSD between two independent samples of 500 tokens each from the same distribution

JSD_list = []
for n_reps in range(100):
    # sample tokens
    tokens_1 = context_from_pi(pi_film_optim_2, N=500).to(device)
    tokens_2 = context_from_pi(pi_film_cluster, N=500).to(device)

    pi_1 = pi_from_context(tokens_1, vocab_size=model.cfg.d_vocab).clamp(min=1e-15)
    pi_2 = pi_from_context(tokens_2, vocab_size=model.cfg.d_vocab).clamp(min=1e-15)

    jsd = JSD(
        pi_1.unsqueeze(0),
        pi_2.unsqueeze(0),
    ).item() * 1e5 / model.cfg.d_vocab

    # print(f"JSD difference: {jsd:.6e}")
    JSD_list.append(jsd)

print(f"JSD mean: {np.mean(JSD_list):.6e} | std: {np.std(JSD_list):.6e}")

In [ ]:
pi_1 = best_results[0]["pi_final"].to_dense().to(device).clone()
pi_2 = best_results[2]["pi_final"].to_dense().to(device).clone()

jsd = JSD(
        pi_1.clamp(min=1e-15).unsqueeze(0),
        pi_2.clamp(min=1e-15).unsqueeze(0),
    ).item() * 1e5 / model.cfg.d_vocab

print(f"JSD difference: {jsd:.6e}")

In [ ]:
### Test branching factors

# --- 1. The macroscopic state to restart from -------------------------------------
result = best_results[2]

# (b) or from a wide-optimization result (cells 92-95):
pi = result["pi_final"].to_dense().to(device).clone()

# (c) or from a real context, as the control:
# pi = pi_from_context(model.to_tokens(some_text)[0], vocab_size=model.cfg.d_vocab)

# --- 2. The same hooked model the fixed point was found under ----------------------
active_head = result['head']
inactive_heads = [h for h in range(model.cfg.n_heads) if h != active_head]

fwd_hooks = [
    (
        f"blocks.{layer_idx}.attn.hook_z",
        partial(zero_head_hook, head_idx=inactive_heads, min_position=0),
    )
    for layer_idx in range(model.cfg.n_layers)
]
fwd_hooks.append(("hook_pos_embed", remove_pos_embed_hook))
# fwd_hooks = None   # <- the unmodified model, for comparison

# --- 3. The state estimator applied to the generated tokens ------------------------
K_list = [1]
S_init = sum(K_list) - len(K_list) + 1        # == 1 for a 1-layer model

context_estimator = UnigramContextEstimator(
    exclude_special=False,    
    top_n=None,               # VRAM knob; also denoises the tail of the histogram
)
# For n_layers > 1 use the K-gram estimator instead:
# context_estimator = KGramContextEstimator(top_n=2048)

# --- 4. Measure -------------------------------------------------------------------
branching = measure_branching_factor(
    model=model,
    pi=pi,
    N_generations=10,
    N_context=500,
    N_generate=500,
    temperature=1.0,
    top_p=1.0,
    fwd_hooks=fwd_hooks,
    context_estimator=context_estimator,
    jsd_threshold=0.5,               
    S_init=S_init,
    generation_batch_size=10,          # 32 x 1000 tokens at once will OOM on most cards
    seed=0,
    normalize_jsd=True
)

print(f"branching factor = {branching['branching_factor']}")
print(f"cluster sizes    = {branching['cluster_sizes'].tolist()}")


In [ ]:
print(model.to_string(branching['generated_tokens'][0][-50:]))

In [ ]:
for head in [0, 3, 4, 6]:

    fwd_hooks = [
        (
            f"blocks.{layer_idx}.attn.hook_z",
            partial(zero_head_hook, head_idx=[h for h in range(model.cfg.n_heads) if h != head], min_position=0),
        )
        for layer_idx in range(model.cfg.n_layers)
    ]
    fwd_hooks.append(("hook_pos_embed", remove_pos_embed_hook))

    context_estimator = UnigramContextEstimator(
        exclude_special=False,
        top_n=None,
    )

    for index, result in tqdm(enumerate(wide_results[head]), desc=f"Measuring branching factors for head {head}", total=len(wide_results[head])):
        pi_final = result["pi_final"].to_dense().to(device).clone()
        pi_init = result["pi_init"].to_dense().to(device).clone()

        branching_final = measure_branching_factor(
            model=model,
            pi=pi_final,
            N_generations=10,
            N_context=500,
            N_generate=500,
            temperature=1.0,
            top_p=1.0,
            fwd_hooks=fwd_hooks,
            context_estimator=context_estimator,
            jsd_threshold=0.5,
            S_init=1,
            generation_batch_size=10,
            seed=0,
            normalize_jsd=True,
            verbose=False
        )

        branching_init = measure_branching_factor(
                model=model,
                pi=pi_init,
                N_generations=10,
                N_context=500,
                N_generate=500,
                temperature=1.0,
                top_p=1.0,
                fwd_hooks=fwd_hooks,
                context_estimator=context_estimator,
                jsd_threshold=0.5,
                S_init=1,
                generation_batch_size=10,
                seed=0,
                normalize_jsd=True,
                verbose=False
            )

        print(f"Head {head} | Index {index} | Branching factor final: {branching_final['branching_factor']}")

        wide_results[head][index]['branching_results_final'] = branching_final
        wide_results[head][index]['branching_results_init'] = branching_init


In [ ]:
head = 4
branching_factors_final = [results['branching_results_final']['branching_factor'] for results in wide_results[head]]
branching_factors_init = [results['branching_results_init']['branching_factor'] for results in wide_results[head]]
final_losses = [results['final_loss'] for results in wide_results[head]]

# plot histogram of branching factors
plt.figure(figsize=(10, 6))
plt.hist(branching_factors_final, bins=np.arange(0.5, 11, 1), color='skyblue', edgecolor='black')
plt.title(f"Histogram of Branching Factors for Head {head}")
plt.xlabel("Branching Factor")
plt.xticks(np.arange(1, 11, 1))
plt.ylabel("Frequency")
plt.grid(axis='y', alpha=0.75)
plt.show()

# plot final loss vs branching factor
plt.figure(figsize=(10, 6))
plt.scatter(branching_factors_final, final_losses, color='orange', edgecolor='black')
plt.title(f"Final Loss vs Branching Factor for Head {head}")
plt.xlabel("Branching Factor")
plt.ylabel("Final Loss")
plt.xticks(np.arange(1, 11, 1))
plt.grid(True)
plt.show()

# plot histogram of branching factors
plt.figure(figsize=(10, 6))
plt.hist(branching_factors_init, bins=np.arange(0.5, 11, 1), color='skyblue', edgecolor='black')
plt.title(f"Histogram of Branching Factors (Initial Conditions) for Head {head}")
plt.xlabel("Branching Factor")
plt.xticks(np.arange(1, 11, 1))
plt.ylabel("Frequency")
plt.grid(axis='y', alpha=0.75)
plt.show()


## Approximation Quality Benchmark

How close is the *approximated* forward pass to the real one? Run both on a corpus of
real text at a randomly chosen late position and measure the JSD between the two
next-token distributions.

The context estimator (`ContextDistributionEstimator`) and the abstraction itself
(`ApproximateForwardPass`) are independent plug-in points, so new hypotheses -- no
positional embeddings, a finite sliding context window, a different context estimator --
are new subclasses and nothing else changes.

In [ ]:
### Approximation Quality Benchmark ###
# Measures how well an *approximated* forward pass reproduces the true next-token
# distribution of the concrete model, on a corpus of real text.
#
# Pipeline for one text:
#   text -> tokens [n_tokens]
#        -> sample a position t in [min_position, n_tokens)
#        -> REAL:   model(tokens[:t+1])            -> p_real  [d_vocab]
#        -> APPROX: estimate pi from tokens[:t+1]  -> (pi_vals [N_c], pi_keys [N_c, S_init])
#                   abstract_forward_pass(query = tokens[t-S_init+1 : t+1])
#                                                  -> p_approx [d_vocab]
#        -> error = JSD(p_real, p_approx)
#
# Both halves are pluggable:
#   * ContextDistributionEstimator -> how pi is estimated (unigram, K-gram, windowed, ...)
#   * ApproximateForwardPass       -> how the forward pass itself is abstracted
#     (partition-function version here; "no positional embeddings", "finite sliding
#      context window", ... are new subclasses, nothing else changes).

from abc import ABC, abstractmethod


# =====================================================================
# 0. SHARED NUMERICS
# =====================================================================

def apply_sampling_transform(
    logits: torch.Tensor,          # [..., d_vocab]
    temperature: float = 1.0,
    top_p: float = 1.0,
) -> torch.Tensor:
    """
    Temperature + top-p nucleus renormalization, identical in semantics to the block
    at the end of `abstract_forward_pass` (minus the straight-through estimator, which
    is only needed when gradients flow through it).

    Returns: probabilities, same shape as `logits`, summing to 1 along the last dim.
    """
    if temperature != 1.0:
        logits = logits / temperature

    probs_dense = torch.softmax(logits, dim=-1)                                  # [..., d_vocab]
    if top_p >= 1.0:
        return probs_dense

    sorted_probs, sorted_indices = torch.sort(probs_dense, descending=True, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    sorted_mask = cumulative_probs < top_p
    # Shift right by one so the token that crosses the threshold is kept.
    sorted_mask = torch.cat([torch.ones_like(sorted_mask[..., :1]), sorted_mask[..., :-1]], dim=-1)

    mask = torch.zeros_like(probs_dense, dtype=torch.bool)
    mask.scatter_(dim=-1, index=sorted_indices, src=sorted_mask)
    filtered_probs = torch.where(mask, probs_dense, torch.zeros_like(probs_dense))

    return filtered_probs / filtered_probs.sum(dim=-1, keepdim=True).clamp(min=1e-9)


def jsd_dense(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-15) -> float:
    """
    Jensen-Shannon divergence (nats, so bounded by ln 2 ~= 0.693) between two dense
    probability vectors.

    Args:
        p, q: [d_vocab] (or [1, d_vocab]). Clamped and renormalized before the call to
              `JSD`, because top-p filtering produces exact zeros and `JSD` takes logs.
    Returns: python float.
    """
    p = p.reshape(1, -1).double().clamp(min=eps)                                 # [1, d_vocab]
    q = q.reshape(1, -1).double().clamp(min=eps)                                 # [1, d_vocab]
    p = p / p.sum(dim=-1, keepdim=True)
    q = q / q.sum(dim=-1, keepdim=True)
    return JSD(p, q).item()


class RunningMoments:
    """Welford accumulator: streaming mean / variance, so nothing has to be kept in memory."""

    def __init__(self):
        self.count = 0
        self.mean = 0.0
        self._m2 = 0.0
        self.min = float("inf")
        self.max = float("-inf")

    def update(self, x: float):
        if not math.isfinite(x):
            return
        self.count += 1
        delta = x - self.mean
        self.mean += delta / self.count
        self._m2 += delta * (x - self.mean)
        self.min = min(self.min, x)
        self.max = max(self.max, x)

    @property
    def var(self) -> float:
        return self._m2 / (self.count - 1) if self.count > 1 else 0.0

    @property
    def std(self) -> float:
        return math.sqrt(self.var)

    @property
    def sem(self) -> float:
        """Standard error of the mean -- the right bar to put on a corpus average."""
        return self.std / math.sqrt(self.count) if self.count > 0 else float("nan")

    def summary(self) -> dict:
        return {
            "count": self.count,
            "mean": self.mean if self.count else float("nan"),
            "std": self.std,
            "sem": self.sem,
            "min": self.min if self.count else float("nan"),
            "max": self.max if self.count else float("nan"),
        }


# =====================================================================
# 1. CONTEXT DISTRIBUTION ESTIMATORS  (pluggable "how do we get pi?")
# =====================================================================

class ContextDistributionEstimator(ABC):
    """
    Turns the realized context `tokens[:position+1]` into the sparse macroscopic state
    (pi_vals, pi_keys) that `abstract_forward_pass` consumes as its frozen background.
    """

    name = "abstract"

    @abstractmethod
    def estimate(
        self,
        tokens: torch.Tensor,   # [n_context] -- the context INCLUDING the query token
        model,
        S_init: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (vals [N_c], keys [N_c, S_init]); vals sums to 1, keys is torch.long."""
        raise NotImplementedError


class UnigramContextEstimator(ContextDistributionEstimator):
    """
    Empirical *unigram* (token-frequency) distribution of the context, lifted to the
    [N_c, S_init] K-gram format required by `abstract_forward_pass`.

    The lift is the cheap one: each surviving token id becomes one context row whose
    LAST column is that token and whose earlier columns are `fill_token_id`.
    `single_layer_forward` builds the global background term from
    `sem_resid_pre[:N_c, -1, :]` -- the final column only -- so for a **1-layer** model
    (S_init == K_list[0]) this representation is exact. For deeper models the earlier
    columns do get processed, and filling them with PAD/BOS (which
    `single_layer_forward` explicitly masks out of M_local) makes each context row an
    isolated token rather than a real K-gram. That is a deliberate mean-field
    simplification, not an oversight -- use a K-gram estimator instead if you need it.

    Args:
        context_window: if set, only the last `context_window` tokens before (and
            including) the query are counted. This is how you test a finite sliding
            window hypothesis for the state estimator.
        exclude_special: drop BOS / EOS / PAD from the histogram (they are handled by
            the dedicated M_sink term, so counting them again double-counts).
        min_mass: drop context tokens carrying less than this probability mass.
        top_n: keep at most this many highest-mass tokens (VRAM knob; the global term
            costs O(heads * N_q * S_out * N_c)).
        fill_token_id: value for the leading S_init-1 columns. Defaults to the
            tokenizer's pad id, else bos id, else 0.
    """

    def __init__(
        self,
        context_window: Optional[int] = None,
        exclude_special: bool = False,
        min_mass: float = 0.0,
        top_n: Optional[int] = None,
        fill_token_id: Optional[int] = None,
    ):
        self.context_window = context_window
        self.exclude_special = exclude_special
        self.min_mass = min_mass
        self.top_n = top_n
        self.fill_token_id = fill_token_id
        self.name = f"unigram(window={context_window}, top_n={top_n})"

    @staticmethod
    def _special_token_ids(model) -> List[int]:
        tokenizer = getattr(model, "tokenizer", None)
        ids = []
        for attribute in ("bos_token_id", "eos_token_id", "pad_token_id"):
            token_id = getattr(tokenizer, attribute, None)
            if token_id is not None and token_id >= 0:
                ids.append(int(token_id))
        return ids

    def _resolve_fill_token_id(self, model) -> int:
        if self.fill_token_id is not None:
            return int(self.fill_token_id)
        tokenizer = getattr(model, "tokenizer", None)
        for attribute in ("pad_token_id", "bos_token_id"):
            token_id = getattr(tokenizer, attribute, None)
            if token_id is not None and token_id >= 0:
                return int(token_id)
        return 0

    def estimate(self, tokens: torch.Tensor, model, S_init: int):
        device = model.cfg.device
        context_tokens = tokens.flatten().to(device)                             # [n_context]
        if self.context_window is not None:
            context_tokens = context_tokens[-self.context_window:]

        if self.exclude_special:
            special_ids = self._special_token_ids(model)
            if special_ids:
                special = torch.tensor(special_ids, device=device, dtype=torch.long)
                context_tokens = context_tokens[~torch.isin(context_tokens, special)]

        if context_tokens.numel() == 0:
            raise ValueError("UnigramContextEstimator: empty context after filtering.")

        # Dense unigram histogram. vocab_size MUST be passed
        pi_dense = pi_from_context(context_tokens, vocab_size=model.cfg.d_vocab)  # [d_vocab]

        vals, token_ids = self._sparsify(pi_dense)                               # [N_c], [N_c]

        fill_token_id = self._resolve_fill_token_id(model)
        keys = torch.full(
            (token_ids.numel(), S_init), fill_token_id, dtype=torch.long, device=device
        )                                                                        # [N_c, S_init]
        keys[:, -1] = token_ids
        return vals, keys

    def _sparsify(self, pi_dense: torch.Tensor):
        """[d_vocab] dense -> (vals [N_c] renormalized to 1, token_ids [N_c])."""
        mask = pi_dense > max(self.min_mass, 0.0)
        token_ids = mask.nonzero(as_tuple=True)[0]                               # [N_c]
        vals = pi_dense[token_ids]                                               # [N_c]

        if self.top_n is not None and vals.numel() > self.top_n:
            top = vals.topk(self.top_n)
            token_ids = token_ids[top.indices]
            vals = top.values

        return vals / vals.sum().clamp(min=1e-12), token_ids


class KGramContextEstimator(ContextDistributionEstimator):
    """
    Empirical S_init-gram distribution of the context, i.e. the estimator the K-gram
    pipeline (cells 72/80) actually uses. Provided as the natural comparison point for
    `UnigramContextEstimator` on multi-layer models.
    """

    def __init__(self, context_window: Optional[int] = None, top_n: Optional[int] = None):
        self.context_window = context_window
        self.top_n = top_n
        self.name = f"kgram(window={context_window}, top_n={top_n})"

    def estimate(self, tokens: torch.Tensor, model, S_init: int):
        device = model.cfg.device
        context_tokens = tokens.flatten()                                        # [n_context]
        if self.context_window is not None:
            context_tokens = context_tokens[-self.context_window:]

        pad_id = getattr(getattr(model, "tokenizer", None), "pad_token_id", None)
        pad_id = 0 if pad_id is None or pad_id < 0 else int(pad_id)

        # offset >= K-1 so `get_kgram_distribution_from_tokens` never takes its CPU-only
        # left-padding branch (see doc section 4.4).
        vals, keys = get_kgram_distribution_from_tokens(
            context_tokens.cpu(), K=S_init, offset=S_init - 1, pad_id=pad_id, device="cpu"
        )                                                                        # [N_c], [N_c, S_init]

        if self.top_n is not None and vals.numel() > self.top_n:
            top = vals.topk(self.top_n)
            vals, keys = top.values, keys[top.indices]

        vals = (vals / vals.sum().clamp(min=1e-12)).to(device)
        return vals, keys.to(device=device, dtype=torch.long)


# =====================================================================
# 2. APPROXIMATE FORWARD PASSES  (pluggable "how do we abstract the model?")
# =====================================================================

class ApproximateForwardPass(ABC):
    """
    Predicts p(next token | tokens[:position+1]) *without* running the concrete model.

    Subclass contract: implement `predict`, returning (probs [d_vocab], info dict).
    Everything downstream (the benchmark, the statistics) is agnostic to how you do it,
    so "no positional embeddings", "finite sliding context window", "top-k context
    only", ... are each just another subclass.
    """

    name = "abstract"

    @abstractmethod
    def predict(self, tokens: torch.Tensor, position: int) -> Tuple[torch.Tensor, dict]:
        """
        Args:
            tokens:   [n_tokens] full token sequence for the text (BOS-prepended).
            position: index of the query token; the prediction is for position+1.
        Returns:
            probs: [d_vocab] on the model device, sums to 1.
            info:  free-form dict merged into the per-text result (e.g. the estimated pi).
        """
        raise NotImplementedError


class PartitionFunctionApproximateForwardPass(ApproximateForwardPass):
    """
    The current (cell-68 / cell-72) abstraction: 3-part thermodynamic partition function,
    decoupled semantic / positional streams, frozen linearized LayerNorm, explicit BOS sink.

    Args:
        model:               HookedTransformer with *learned absolute* positional
                             embeddings and loaded with fold_ln=False (see doc 4.2).
        context_estimator:   supplies (pi_vals, pi_keys).
        forward_pass_kwargs: the nine-key dict from cell 72 -- K_list, active_heads_list,
                             ln_avg_sigma_list, L_ctx_list, C_far_list, sink_k_list,
                             sink_v_list, temperature, top_p. `temperature` / `top_p` may
                             be overridden by the explicit arguments below.
        temperature, top_p:  sampling transform applied inside the abstract pass. Keep
                             these equal to the ones the benchmark applies to the real
                             pass, otherwise the JSD measures the sampling transform
                             rather than the approximation.
    """

    def __init__(
        self,
        model,
        context_estimator: ContextDistributionEstimator,
        forward_pass_kwargs: dict,
        temperature: Optional[float] = None,
        top_p: Optional[float] = None,
    ):
        self.model = model
        self.context_estimator = context_estimator

        kwargs = dict(forward_pass_kwargs)
        self.temperature = float(kwargs.pop("temperature", 1.0) if temperature is None else temperature)
        self.top_p = float(kwargs.pop("top_p", 1.0) if top_p is None else top_p)
        kwargs.pop("temperature", None)
        kwargs.pop("top_p", None)

        required = {
            "K_list", "active_heads_list", "ln_avg_sigma_list",
            "L_ctx_list", "C_far_list", "sink_k_list", "sink_v_list",
        }
        missing = required - set(kwargs)
        if missing:
            raise KeyError(f"forward_pass_kwargs is missing {sorted(missing)}")
        unexpected = set(kwargs) - required
        if unexpected:
            raise KeyError(f"forward_pass_kwargs has unexpected keys {sorted(unexpected)}")

        # Fail loudly here rather than as an opaque broadcast error deep inside
        # `single_layer_forward`: L_ctx / C_far must have exactly one entry per ACTIVE
        # head, but the profilers return one entry per head in the model.
        for layer_idx, active_heads in enumerate(kwargs["active_heads_list"]):
            for list_name in ("L_ctx_list", "C_far_list"):
                n_entries = kwargs[list_name][layer_idx].numel()
                if n_entries not in (len(active_heads), 1):
                    raise ValueError(
                        f"{list_name}[{layer_idx}] has {n_entries} entries but layer {layer_idx} "
                        f"has {len(active_heads)} active heads {active_heads}. Slice the profiler "
                        f"output, e.g. L_ctx_list[{layer_idx}][active_heads_list[{layer_idx}]]."
                    )

        self.forward_pass_kwargs = kwargs
        # Exact input length that collapses to a single output token after every layer.
        self.S_init = sum(kwargs["K_list"]) - len(kwargs["K_list"]) + 1
        self.name = (
            f"partition_function(S_init={self.S_init}, T={self.temperature}, "
            f"top_p={self.top_p}, ctx={context_estimator.name})"
        )

    def _build_query_keys(self, tokens: torch.Tensor, position: int) -> torch.Tensor:
        """The last S_init tokens ending at `position`, left-padded if the text is shorter."""
        device = self.model.cfg.device
        start = position + 1 - self.S_init
        if start >= 0:
            query = tokens[start: position + 1].to(device)                        # [S_init]
        else:
            pad_id = getattr(getattr(self.model, "tokenizer", None), "pad_token_id", None)
            pad_id = 0 if pad_id is None or pad_id < 0 else int(pad_id)
            padding = torch.full((-start,), pad_id, dtype=torch.long, device=device)
            query = torch.cat([padding, tokens[: position + 1].to(device)], dim=0) # [S_init]
        return query.view(1, self.S_init).long()                                  # [1, S_init]

    def predict(self, tokens: torch.Tensor, position: int):
        # 1. Estimate the macroscopic state pi from the realized context up to `position`.
        context_vals, context_keys = self.context_estimator.estimate(
            tokens[: position + 1], self.model, self.S_init
        )                                                                         # [N_c], [N_c, S_init]

        # 2. The query is the single K-gram ending at `position`.
        query_keys = self._build_query_keys(tokens, position)                     # [1, S_init]

        # 3. Abstract forward pass -> next-token distribution for that one query.
        P_active = abstract_forward_pass(
            context_vals=context_vals,
            context_keys=context_keys,
            query_keys=query_keys,
            model=self.model,
            temperature=self.temperature,
            top_p=self.top_p,
            **self.forward_pass_kwargs,
        )                                                                         # [1, d_vocab]

        info = {
            "context_vals": context_vals.detach().cpu(),                          # [N_c]
            "context_keys": context_keys.detach().cpu(),                          # [N_c, S_init]
            "query_keys": query_keys.detach().cpu(),                              # [1, S_init]
            "n_context_states": int(context_vals.numel()),
        }
        return P_active[0], info                                                  # [d_vocab], dict


# =====================================================================
# 3. THE BENCHMARK
# =====================================================================

class ApproximationQualityBenchmark:
    """
    Compares an `ApproximateForwardPass` against the concrete model on a text corpus.

    Args:
        model:            HookedTransformer (the ground truth).
        approximate_pass: the abstraction under test.
        real_hooks:       TransformerLens fwd_hooks [(name, fn), ...] installed (via
                          `with model.hooks(...)`) around the REAL forward pass only.
                          Use this to make the concrete model match the assumptions of
                          the abstraction (e.g. ablate non-active heads, knock out the
                          positional embeddings). None / [] means an untouched model.
        approx_hooks:     same, but installed around `approximate_pass.predict`. Most
                          abstractions never touch the model, but any that do (or that
                          call the model internally) see these.
        max_tokens:       every text is chopped to at most this many tokens (also
                          clamped to model.cfg.n_ctx).
        min_position:     earliest query position sampled; texts shorter than
                          min_position + 1 tokens are skipped and reported.
        n_positions:      independent query positions sampled per text.
        temperature/top_p: sampling transform applied to the REAL logits. Defaults to
                          the approximate pass's own values so the two sides are
                          compared under identical post-processing.
        store_probs:      'full' keeps both [d_vocab] vectors per result on CPU,
                          'topk' keeps only the top `store_topk` of each, 'none' keeps
                          neither. 'full' costs ~2 * d_vocab * 4 bytes per position.
    """

    def __init__(
        self,
        model,
        approximate_pass: ApproximateForwardPass,
        real_hooks: Optional[List[Tuple[str, Any]]] = None,
        approx_hooks: Optional[List[Tuple[str, Any]]] = None,
        max_tokens: int = 1000,
        min_position: int = 700,
        n_positions: int = 1,
        temperature: Optional[float] = None,
        top_p: Optional[float] = None,
        seed: int = 42,
        store_probs: str = "full",
        store_topk: int = 100,
        verbose: bool = True,
    ):
        self.model = model
        self.approximate_pass = approximate_pass
        self.real_hooks = list(real_hooks) if real_hooks else []
        self.approx_hooks = list(approx_hooks) if approx_hooks else []
        self.max_tokens = min(int(max_tokens), int(model.cfg.n_ctx))
        self.min_position = int(min_position)
        self.n_positions = int(n_positions)
        self.temperature = float(getattr(approximate_pass, "temperature", 1.0) if temperature is None else temperature)
        self.top_p = float(getattr(approximate_pass, "top_p", 1.0) if top_p is None else top_p)
        self.seed = seed
        self.store_probs = store_probs
        self.store_topk = store_topk
        self.verbose = verbose

        if self.min_position >= self.max_tokens:
            raise ValueError(
                f"min_position ({self.min_position}) must be < max_tokens ({self.max_tokens}); "
                f"note max_tokens is clamped to model.cfg.n_ctx = {model.cfg.n_ctx}."
            )

    # ---------------- ground truth ----------------

    def real_probs(self, tokens: torch.Tensor, position: int) -> torch.Tensor:
        """
        Concrete model next-token distribution at `position`, under `self.real_hooks`.
        Returns [d_vocab].
        """
        prefix = tokens[: position + 1].view(1, -1).to(self.model.cfg.device)      # [1, position+1]
        with self.model.hooks(fwd_hooks=self.real_hooks):
            logits = self.model(prefix, return_type="logits")                     # [1, position+1, d_vocab]
        return apply_sampling_transform(logits[0, -1], self.temperature, self.top_p)  # [d_vocab]

    # ---------------- one text ----------------

    def tokenize(self, text: str) -> torch.Tensor:
        tokens = self.model.to_tokens(text, prepend_bos=True)[0]                  # [n_tokens]
        return tokens[: self.max_tokens]

    def sample_positions(self, n_tokens: int, rng: random.Random) -> List[int]:
        """Uniform positions in [min_position, n_tokens); the query token is at `position`."""
        if n_tokens <= self.min_position:
            return []
        return [rng.randrange(self.min_position, n_tokens) for _ in range(self.n_positions)]

    def evaluate_position(self, text: str, tokens: torch.Tensor, position: int) -> dict:
        p_real = self.real_probs(tokens, position)                                # [d_vocab]
        with self.model.hooks(fwd_hooks=self.approx_hooks):
            p_approx, info = self.approximate_pass.predict(tokens, position)      # [d_vocab], dict
        p_approx = p_approx.detach().reshape(-1)

        loss_jsd = jsd_dense(p_real, p_approx)

        # Control: JSD between the true distribution and the raw context unigram. An
        # approximation that does not beat this is not using the model at all.
        context_unigram = pi_from_context(
            tokens[: position + 1].to(self.model.cfg.device), vocab_size=self.model.cfg.d_vocab
        )                                                                         # [d_vocab]

        result = {
            "text": text,
            "position": int(position),
            "n_tokens": int(tokens.numel()),
            "query_token": int(tokens[position].item()),
            "loss_jsd": loss_jsd,
            "jsd_baseline_context_unigram": jsd_dense(p_real, context_unigram),
            "total_variation": 0.5 * (p_real - p_approx).abs().sum().item(),
            "top1_agreement": bool(p_real.argmax().item() == p_approx.argmax().item()),
            "real_top1_rank_in_approx": int(
                (p_approx > p_approx[p_real.argmax()]).sum().item()
            ),
            "real_entropy_nats": float(-(p_real.clamp(min=1e-15) * p_real.clamp(min=1e-15).log()).sum()),
            "approx_entropy_nats": float(-(p_approx.clamp(min=1e-15) * p_approx.clamp(min=1e-15).log()).sum()),
        }
        result.update(info)

        if self.store_probs == "full":
            result["real_probs"] = p_real.detach().cpu()                          # [d_vocab]
            result["approx_probs"] = p_approx.detach().cpu()                      # [d_vocab]
        elif self.store_probs == "topk":
            k = min(self.store_topk, p_real.numel())
            real_top, approx_top = p_real.topk(k), p_approx.topk(k)
            result["real_probs_topk"] = (real_top.indices.cpu(), real_top.values.cpu())
            result["approx_probs_topk"] = (approx_top.indices.cpu(), approx_top.values.cpu())

        return result

    # ---------------- the corpus ----------------

    def run(self, corpus: List[str]) -> dict:
        """
        Args:
            corpus: list of raw strings.
        Returns:
            {"per_text": [result, ...], "statistics": {...}, "skipped": [...], "config": {...}}
        """
        rng = random.Random(self.seed)
        per_text: List[dict] = []
        skipped: List[dict] = []

        accumulators = {
            "loss_jsd": RunningMoments(),
            "jsd_baseline_context_unigram": RunningMoments(),
            "total_variation": RunningMoments(),
            "top1_agreement": RunningMoments(),
            "real_entropy_nats": RunningMoments(),
            "approx_entropy_nats": RunningMoments(),
        }

        iterator = tqdm(corpus, desc="Approximation benchmark") if self.verbose else corpus
        for text_idx, text in enumerate(iterator):
            tokens = self.tokenize(text)                                          # [n_tokens]
            positions = self.sample_positions(int(tokens.numel()), rng)
            if not positions:
                skipped.append({
                    "text_index": text_idx,
                    "n_tokens": int(tokens.numel()),
                    "reason": f"shorter than min_position + 1 = {self.min_position + 1} tokens",
                })
                continue

            for position in positions:
                try:
                    with torch.no_grad():
                        result = self.evaluate_position(text, tokens, position)
                except Exception as error:  # a single bad text must not kill a long run
                    skipped.append({"text_index": text_idx, "position": int(position), "reason": repr(error)})
                    continue

                result["text_index"] = text_idx
                per_text.append(result)
                for key, accumulator in accumulators.items():
                    accumulator.update(float(result[key]))

            if self.verbose and isinstance(iterator, tqdm):
                iterator.set_postfix(mean_jsd=f"{accumulators['loss_jsd'].mean:.4e}")

        statistics = {key: accumulator.summary() for key, accumulator in accumulators.items()}
        statistics["n_evaluated"] = len(per_text)
        statistics["n_skipped"] = len(skipped)
        # Headline numbers, named as asked for.
        statistics["mean_loss"] = statistics["loss_jsd"]["mean"]
        statistics["std_loss"] = statistics["loss_jsd"]["std"]

        return {
            "per_text": per_text,
            "statistics": statistics,
            "skipped": skipped,
            "config": {
                "approximation": self.approximate_pass.name,
                "real_hooks": [name for name, _ in self.real_hooks],
                "approx_hooks": [name for name, _ in self.approx_hooks],
                "max_tokens": self.max_tokens,
                "min_position": self.min_position,
                "n_positions": self.n_positions,
                "temperature": self.temperature,
                "top_p": self.top_p,
                "seed": self.seed,
            },
        }


def summarize_benchmark(results: dict) -> None:
    """Pretty-print the corpus-level statistics of an `ApproximationQualityBenchmark.run`."""
    statistics = results["statistics"]
    print(f"Approximation : {results['config']['approximation']}")
    print(f"Evaluated     : {statistics['n_evaluated']} positions ({statistics['n_skipped']} skipped)")
    print(f"JSD (nats)    : {statistics['loss_jsd']['mean']:.5e} +/- {statistics['loss_jsd']['std']:.3e} "
          f"(sem {statistics['loss_jsd']['sem']:.2e}, "
          f"range [{statistics['loss_jsd']['min']:.3e}, {statistics['loss_jsd']['max']:.3e}])")
    print(f"  baseline    : {statistics['jsd_baseline_context_unigram']['mean']:.5e}  "
          f"<- JSD(real, raw context unigram); the approximation must beat this")
    print(f"Total var.    : {statistics['total_variation']['mean']:.5e} +/- {statistics['total_variation']['std']:.3e}")
    print(f"Top-1 agree   : {statistics['top1_agreement']['mean']:.3%}")
    print(f"Entropy       : real {statistics['real_entropy_nats']['mean']:.4f} nats | "
          f"approx {statistics['approx_entropy_nats']['mean']:.4f} nats")


### Example Run - Partition-Function Approximation

In [ ]:
### Example: benchmark the partition-function approximation on a real corpus ###

# --- 0. Preconditions (see doc sections 4.2 / 4.8) -------------------------------------
# * `model` must have LEARNED ABSOLUTE positional embeddings (model.W_pos) and have been
#   loaded with fold_ln=False. Llama / any rotary model will raise AttributeError.
# * This is a pure evaluation, so gradients stay off.
for parameter in model.parameters():
    parameter.requires_grad = False

# --- 1. Abstraction parameters --------------------------------------------------------
# Re-uses exactly the same pre-computed quantities as the K-gram pipeline (cell 72). If
# you have already run that cell, `forward_pass_kwargs` is live and you can skip to 2.
knockout_positional = True
if knockout_positional:
    positional_hook_name = "hook_pos_embed"
    positional_hook_fn = remove_pos_embed_hook
    hooks = [(positional_hook_name, positional_hook_fn)]
else:
    hooks = []
with torch.no_grad():
    with model.hooks(fwd_hooks=hooks):
        sink_k, sink_v = extract_bos_sink(model)  # [n_layers, n_heads, d_head] x2
sink_k_list = [sink_k[layer] for layer in range(model.cfg.n_layers)]
sink_v_list = [sink_v[layer] for layer in range(model.cfg.n_layers)]

dummy_tokens = torch.tensor(np.arange(1000, 1000 + model.cfg.n_ctx)).unsqueeze(0)  # [1, n_ctx]
with torch.no_grad():
    with model.hooks(fwd_hooks=hooks):
        ln_avg_sigma_list = [extract_frozen_sigma(model, dummy_tokens, layer) for layer in range(model.cfg.n_layers)]

K_list = [1]                        # per-layer local window; S_init = sum(K) - len(K) + 1
active_heads_list = [[3]]

# NOTE: `profile_thermodynamic_heads` returns one entry per head in the MODEL, but
# `single_layer_forward` indexes L_ctx / C_far by position in `active_heads`. Slicing is
# mandatory: with a full-width L_ctx and a narrow head list the broadcast inside
# `single_layer_forward` silently succeeds and sums the same head n_heads times with
# n_heads different horizons. `PartitionFunctionApproximateForwardPass` refuses that.
L_ctx_all, C_far_all = profile_thermodynamic_heads(model, K_list, num_prompts=100, seq_len=1024, batch_size=1)
# L_ctx_list = [L_ctx_all[layer][active_heads_list[layer]] for layer in range(model.cfg.n_layers)]  # [n_active] per layer
# C_far_list = [C_far_all[layer][active_heads_list[layer]] for layer in range(model.cfg.n_layers)]  # [n_active] per layer
L_ctx_list = [torch.tensor([3], device=model.cfg.device) for layer in range(model.cfg.n_layers)]  # [n_active] per layer
C_far_list = [torch.tensor([1], device=model.cfg.device) for layer in range(model.cfg.n_layers)]  # [n_active] per layer

temperature = 1.0
top_p = 1.0                         # keep 1.0 for a pure approximation-quality measurement;
                                    # any top_p < 1 truncates BOTH sides and flatters the JSD.

forward_pass_kwargs = {
    "K_list": K_list,
    "active_heads_list": active_heads_list,
    "ln_avg_sigma_list": ln_avg_sigma_list,
    "L_ctx_list": L_ctx_list,
    "C_far_list": C_far_list,
    "sink_k_list": sink_k_list,
    "sink_v_list": sink_v_list,
    "temperature": temperature,
    "top_p": top_p,
}

# --- 1b. Forward-pass hooks --------------------------------------------------------
# Installed around the REAL pass only, to bring the concrete model closer to the
# assumptions the abstraction makes. The two knockouts are independent switches.
ablate_inactive_heads = True        # zero every head the abstraction does NOT model
ablate_positional_embeddings = True # run the real model with no positional embeddings

real_hooks = []

if ablate_inactive_heads:
    for layer, active_heads in enumerate(active_heads_list):
        inactive_heads = [head for head in range(model.cfg.n_heads) if head not in active_heads]
        if inactive_heads:
            real_hooks.append((
                f"blocks.{layer}.attn.hook_z",
                partial(zero_head_hook, head_idx=inactive_heads, min_position=0),
            ))

if ablate_positional_embeddings:
    real_hooks.append(("hook_pos_embed", remove_pos_embed_hook))

# The abstraction never runs the concrete model, so it needs no hooks.
approx_hooks = []

# --- 2. Corpus ------------------------------------------------------------------------
# Any list of strings works. Reusing the Wikipedia stream from the wide-optimization cell.

iterable_dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
small_dataset = Dataset.from_list(list(iterable_dataset.take(500)))
corpus = [text for text in small_dataset["text"] if len(text) > 5000][:100]
print(f"Corpus: {len(corpus)} texts")

# --- 3. Assemble and run --------------------------------------------------------------
benchmark = ApproximationQualityBenchmark(
    model=model,
    approximate_pass=PartitionFunctionApproximateForwardPass(
        model=model,
        context_estimator=UnigramContextEstimator(top_n=None, exclude_special=True),
        forward_pass_kwargs=forward_pass_kwargs,
    ),
    real_hooks=real_hooks,
    approx_hooks=approx_hooks,
    max_tokens=1000,
    min_position=700,
    n_positions=1,
    seed=42,
    store_probs="full",
)

approximation_results = benchmark.run(corpus)
summarize_benchmark(approximation_results)

# approximation_results["per_text"][i] holds: text, position, context_vals/context_keys
# (the estimated pi), real_probs, approx_probs, loss_jsd, and the auxiliary metrics.


In [ ]:
### Inspect the benchmark results ###

per_text = approximation_results["per_text"]
losses = np.array([result["loss_jsd"] for result in per_text])                    # [n_evaluated]
baselines = np.array([result["jsd_baseline_context_unigram"] for result in per_text])
statistics = approximation_results["statistics"]

plt.figure(figsize=(10, 8))
plt.hist(losses, bins=40, alpha=0.7, label="approximation")
plt.hist(baselines, bins=40, alpha=0.5, label="context-unigram baseline")
plt.axvline(statistics["loss_jsd"]["mean"], color="k", linestyle="--",
            label=f"mean = {statistics['loss_jsd']['mean']:.3e}")
plt.xlabel("JSD (nats)")
plt.ylabel("#Texts")
plt.title("Approximation Error Across the Corpus")
plt.legend()
plt.grid()

plt.figure(figsize=(10, 8))
plt.plot([result["position"] for result in per_text], losses, '.')
plt.xlabel("Query Position")
plt.ylabel("JSD (nats)")
plt.title("Approximation Error Vs. Position")
plt.grid()
plt.show()

# --- Worst case: where does the abstraction actually break? ---
worst = max(per_text, key=lambda result: result["loss_jsd"])
print(f"Worst JSD {worst['loss_jsd']:.4e} at position {worst['position']} "
      f"(text {worst['text_index']}, {worst['n_context_states']} context states)")
print(f"query token: {model.to_string(torch.tensor([worst['query_token']]))!r}")
if "real_probs" in worst:
    real_top = worst["real_probs"].topk(10)                                       # [10]
    approx_top = worst["approx_probs"].topk(10)                                   # [10]
    print("real  top-10:", [(model.to_string(i.view(1)), round(v.item(), 4))
                            for i, v in zip(real_top.indices, real_top.values)])
    print("approx top-10:", [(model.to_string(i.view(1)), round(v.item(), 4))
                             for i, v in zip(approx_top.indices, approx_top.values)])

# --- Comparing several approximations on the same positions ---
# The benchmark is seeded, so different approximations sample identical positions:
#
# variants = {
#     "unigram, full context": UnigramContextEstimator(top_n=2048),
#     "unigram, 200-token window": UnigramContextEstimator(context_window=200, top_n=2048),
#     "empirical S_init-grams": KGramContextEstimator(top_n=2048),
# }
# comparison = {}
# for label, estimator in variants.items():
#     benchmark.approximate_pass = PartitionFunctionApproximateForwardPass(
#         model=model, context_estimator=estimator, forward_pass_kwargs=forward_pass_kwargs
#     )
#     comparison[label] = benchmark.run(corpus)
#     print(label); summarize_benchmark(comparison[label]); print()


### Exact Approximation - 1-Layer, Single Semantic Head, No Positions

With no positional embeddings the attention score depends only on the *token ids*, so the
whole forward pass is an exact function of the context unigram **counts**. The 3-part
partition function can reproduce it bit-for-bit provided `pi`, `L_ctx` and `C_far` are
tuned per forward pass so that the sink, the local window and the far field partition the
context **exactly once**. See the derivation at the top of the next cell.

In [ ]:
### EXACT forward pass for a 1-layer, single-head, position-free model ###
import contextlib

# Why an exact answer is possible at all
# -------------------------------------
# Strip the positional embeddings from a 1-layer attention-only model and the pre-softmax
# score between query position t and key position j collapses to
#
#     s(x_t, x_j) = (LN(W_E[x_t]) W_Q + b_Q) . (LN(W_E[x_j]) W_K + b_K) / sqrt(d_head)
#
# i.e. a function of the two *token ids* only -- position has dropped out. Hence
#
#     Z      = sum_j exp(s(x_t, x_j)) = sum_v n_v * exp(s(x_t, v))
#     attn_z = sum_j exp(s) * v_j / Z = sum_v n_v * exp(s(x_t, v)) * v(v) / Z
#
# where n_v is the *count* of token v in the context. The concrete forward pass is
# therefore an exact function of the context unigram COUNTS -- not just the normalized
# unigram distribution; the total context length T matters, and that is precisely what
# `L_ctx` has to carry.
#
# Mapping that onto the 3-part partition function of `single_layer_forward`
# -------------------------------------------------------------------------
#   Z = sum_c M_global[c]  +  sum_s M_local[s]  +  M_sink
#
#   * M_sink   = exp(q_s . sink_k / sqrt(d)) is *exactly* the BOS term, because with no
#                positional embeddings `sink_k` is just the key of the BOS token.
#                It contributes exactly ONE occurrence of BOS.
#   * M_local  = exp(sem_score) * exp(pos_score) * window_mask * valid_mask.
#                With W_pos == 0 the positional factor is exp(0) == 1, the window mask
#                (S_out == 1) admits every one of the S_init query positions, and
#                `valid_mask` deletes every PAD/BOS position. So M_local contributes
#                exactly the NON-DEAD tokens of the last S_init positions, each once,
#                with its exact score. This is the "positional mass": the current token
#                and its immediate predecessors, counted explicitly.
#   * M_global = (L_ctx - K_i - 1) * pi_c * exp(s) * C_far, summed over context rows.
#                This must supply everything the other two terms did NOT: the far field.
#
# The tuning, therefore:
#
#   C_far  = 1.0                      (C_far is the mean exponentiated POSITIONAL mass of
#                                      a far token; with no positional embeddings it is
#                                      exp(0) = 1 by construction.)
#   L_ctx  = N_far + K_i + 1          (so that L_global = L_ctx - K_i - 1 == N_far, the
#                                      exact number of far tokens). It is therefore
#                                      PER-FORWARD-PASS: it grows with the prompt.
#                                      For a 1-layer model K_i == S_init, so this reads
#                                      L_ctx = T - (#BOS handled by the sink) - (#alive
#                                      local positions) + S_init + 1, and in the clean
#                                      case (one BOS at position 0, no PAD) it is simply
#                                      L_ctx = T = position + 1.
#   pi     = the unigram distribution of the FAR tokens ONLY, i.e.
#                n_far[v] = total_count[v] - alive_local_count[v] - [v == bos_id]
#            renormalized. Using the whole-context unigram here would double-count the
#            local window and the sink.
#   K_i    = free; any K in [1, T] works, and K == 1 (the query token alone in the local
#            window) is the cheapest. It does NOT need tuning for exactness -- it only
#            moves mass between M_local and M_global. What must be tuned is the PAIR
#            (K_i, pi, L_ctx) so that the three terms partition the context exactly once.
#
# The remaining gap: LayerNorm
# ----------------------------
# `single_layer_forward` linearizes LN with a single frozen scalar `ln_avg_sigma`, while
# the true LN divides by a per-token sigma_v. With no positional embeddings the layer-0
# residual at position j is exactly W_E[x_j], so sigma_v = ||W_E[v] - mean||_rms is a
# pure function of the token and can be made exact WITHOUT touching
# `single_layer_forward`: feed the model a rescaled embedding matrix
#
#     W_E'[v] = W_E[v] * (s / sigma_v),        ln_avg_sigma = s
#
# Centering is linear and commutes with the per-row scalar, so
# (W_E'[v] - mean) / s == (W_E[v] - mean) / sigma_v == the true LN output, for every v.
# The one place the *un*-normalized residual is still used is the residual-stream update
# `shrunken_resid = combined_sem_resid[:, -1, :] + attn_out`, which for a 1-layer model
# only ever reads the QUERY token. Choosing s = sigma_{x_t} (the query token's own sigma)
# makes W_E'[x_t] == W_E[x_t], so that read is exact too. The context rows' residuals are
# rescaled but they are discarded after layer 0, which is why this trick is specific to
# n_layers == 1.
#
# Preconditions for a bit-for-bit match (all checked in __init__ / predict):
#   * model.cfg.n_layers == 1 and the model is attention-only (no MLP).
#   * Loaded with fold_ln=True (TransformerLens default), so ln1 is a LayerNormPre with
#     no learnable gamma/beta -- `single_layer_forward` applies gamma but never beta.
#   * The REAL side of the benchmark must run with `real_hooks` = [remove_pos_embed_hook]
#     plus `zero_head_hook` on every head except the single active one.
#   * temperature == 1.0, top_p == 1.0 on both sides.


class ExactSemanticHeadForwardPass(ApproximateForwardPass):
    """
    `abstract_forward_pass` driven so that it reproduces the concrete model EXACTLY, for
    a 1-layer, attention-only, position-free model with a single active head.

    Everything the partition function needs is recomputed per query position, because
    L_ctx and pi both depend on the realized prompt length. Plugs straight into
    `ApproximationQualityBenchmark`; the expected JSD is float32 round-off (~1e-12).

    Args:
        model:        HookedTransformer, n_layers == 1, attn_only, learned absolute
                      positional embeddings (they are zeroed internally).
        active_head:  the single head the abstraction models. The real pass must ablate
                      every other head.
        K:            local-window size K_list[0]; also S_init. Any value in [1, n_ctx]
                      gives the same (exact) answer -- 1 is cheapest, larger K just moves
                      mass from M_global into the explicit M_local window.
        exact_layernorm: apply the W_E-rescaling trick above. Set False to see how much
                      the frozen-sigma linearization alone costs.
        mass_in_L_ctx: True  -> pi is a normalized distribution and L_ctx carries N_far
                               (the conceptually correct split; costs one float32
                               divide-then-multiply round trip, ~1e-7 relative).
                      False -> pi carries the raw far-field COUNTS and L_ctx is pinned at
                               K + 2 so L_global == 1. Numerically tighter, but `pi` is
                               then not a probability vector.
        temperature, top_p: keep at 1.0 for an approximation-quality measurement.
    """

    def __init__(
        self,
        model,
        active_head: int,
        K: int = 1,
        exact_layernorm: bool = True,
        mass_in_L_ctx: bool = True,
        temperature: float = 1.0,
        top_p: float = 1.0,
    ):
        if model.cfg.n_layers != 1:
            raise ValueError(
                f"ExactSemanticHeadForwardPass is exact only for n_layers == 1 "
                f"(got {model.cfg.n_layers}): the W_E-rescaling trick corrupts the "
                f"context rows' residual stream, which deeper layers would read."
            )
        if hasattr(model.blocks[0], "mlp"):
            raise ValueError("Model has an MLP; this exact construction assumes attn_only.")

        ln1 = model.blocks[0].ln1
        if getattr(ln1, "b", None) is not None:
            raise ValueError(
                "blocks[0].ln1 has a learnable beta, which `single_layer_forward` never "
                "applies. Load the model with fold_ln=True (the TransformerLens default)."
            )
        if getattr(ln1, "w", None) is not None:
            # gamma IS applied by single_layer_forward, so it is fine -- but it must be
            # applied to the *rescaled* embedding too, which it is (it is a right-multiply
            # after the divide). Nothing to do; just don't silently assume fold_ln.
            pass

        expected_scale = float(np.sqrt(model.cfg.d_head))
        actual_scale = float(getattr(model.cfg, "attn_scale", expected_scale))
        if not np.isclose(expected_scale, actual_scale):
            raise ValueError(
                f"model.cfg.attn_scale={actual_scale} but `single_layer_forward` hardcodes "
                f"sqrt(d_head)={expected_scale}; the scores would not match."
            )

        self.model = model
        self.active_head = int(active_head)
        self.K = int(K)
        self.S_init = int(K)                    # 1 layer => S_init = sum(K) - len(K) + 1 = K
        self.exact_layernorm = bool(exact_layernorm)
        self.mass_in_L_ctx = bool(mass_in_L_ctx)
        self.temperature = float(temperature)
        self.top_p = float(top_p)

        device = model.cfg.device
        self.device = device

        # ---- dead-token ids, resolved EXACTLY as `single_layer_forward` resolves them ----
        self.pad_token_id = self._resolve_special(model, "pad_token_id")
        self.bos_token_id = self._resolve_special(model, "bos_token_id")
        if self.bos_token_id is None:
            raise ValueError("No BOS id: M_sink has no token to correspond to.")
        dead_ids = [i for i in (self.pad_token_id, self.bos_token_id) if i is not None]
        self.dead_ids = torch.tensor(sorted(set(dead_ids)), device=device, dtype=torch.long)
        # Leading columns of the context K-gram rows. Must be a token `valid_mask` kills,
        # otherwise (for K > 1) the context rows would be non-isolated -- harmless for a
        # 1-layer model since their output is discarded, but keep it honest.
        self.fill_token_id = int(self.bos_token_id)

        self._zero_W_pos = torch.zeros_like(model.W_pos.detach())                 # [n_ctx, d_model]

        # ---- the BOS attention sink, measured with the positional embeddings removed ----
        with torch.no_grad(), self._zero_pos_embeddings():
            sink_k, sink_v = extract_bos_sink(model)          # [n_layers, n_heads, d_head] x2
        self.sink_k_list = [sink_k[0].detach()]               # full head axis: indexed by active_heads
        self.sink_v_list = [sink_v[0].detach()]

        # ---- per-token LayerNorm scale, and the pre-divided embedding matrix ----
        with torch.no_grad():
            W_E = model.W_E.detach()                                              # [d_vocab, d_model]
            centered = W_E - W_E.mean(dim=-1, keepdim=True)
            eps = float(getattr(ln1, "eps", 1e-5))
            self.sigma_table = (centered.pow(2).mean(dim=-1) + eps).sqrt()        # [d_vocab]
            self.W_E_unit = (W_E / self.sigma_table.unsqueeze(-1)).contiguous()   # [d_vocab, d_model]
            self._W_E_buffer = torch.empty_like(self.W_E_unit)                    # [d_vocab, d_model]
            self._frozen_sigma = float(self.sigma_table.mean())                   # only for exact_layernorm=False

        if model.unembed.W_U.data_ptr() == model.embed.W_E.data_ptr():
            raise ValueError("W_U aliases W_E; rescaling the embedding would corrupt the unembedding.")

        self.name = (
            f"exact_semantic_head(head={self.active_head}, K={self.K}, "
            f"exact_ln={self.exact_layernorm}, mass_in_L_ctx={self.mass_in_L_ctx})"
        )

    # ------------------------------------------------------------ exported config

    def forward_pass_kwargs(self, L_prompt: float, exact_layernorm: bool = None, ln_sigma=None):
        """
        The nine-key kwargs dict that drives `abstract_forward_pass` (and therefore
        `single_power_iteration_step`, `compute_stationary_distribution` and
        `MetastableKGramEngine`) in the EXACT regime, for a context of `L_prompt` tokens.

        The context is partitioned exactly once as

            L_prompt  =  1 (BOS -> M_sink)  +  K (local window -> M_local)
                         +  (L_prompt - K - 1) far tokens (-> M_global)

        so `L_global = L_ctx - K_i - 1 = L_prompt - K - 1 = N_far`, and the measure
        `context_vals` that you pair with this dict MUST therefore be the normalized
        **far-field** unigram (BOS/PAD excluded, sums to 1). Using a whole-context
        unigram here double-counts the local window and the sink.

        Unlike `predict`, this dict carries no per-query state, so it can drive a
        batch of many different queries against one shared context.

        IMPORTANT: everything this dict describes assumes W_pos == 0. Wrap every call
        that consumes it in `with self.position_free():`.

        Args:
            L_prompt:        total context length in tokens. Must be >= K + 1.
                             With no positional embeddings there is no n_ctx ceiling;
                             keep it <= model.cfg.n_ctx only if you want the result to be
                             comparable with a concrete forward pass.
            exact_layernorm: None -> use the value the instance was built with.
                             True  -> hand down the [d_vocab] per-token sigma table, which
                                      `single_layer_forward` applies exactly, per row.
                             False -> hand down the single frozen scalar (the ~4.9e-3 JSD
                                      ablation of section 2.10.3).
            ln_sigma:        explicit override of the LN denominator; wins over
                             `exact_layernorm`.
        """
        L_prompt = float(L_prompt)
        if L_prompt < self.K + 1:
            raise ValueError(
                f"L_prompt={L_prompt} < K + 1 = {self.K + 1}: the context cannot even hold "
                f"the BOS sink plus the local window, so L_global would be negative."
            )
        if exact_layernorm is None:
            exact_layernorm = self.exact_layernorm
        if ln_sigma is None:
            ln_sigma = self.sigma_table if exact_layernorm else self._frozen_sigma

        dtype = self.model.W_E.dtype
        return dict(
            K_list=[self.K],
            active_heads_list=[[self.active_head]],
            ln_avg_sigma_list=[ln_sigma],
            # L_global = L_ctx - K_i - 1 = N_far
            L_ctx_list=[torch.tensor([L_prompt], device=self.device, dtype=dtype)],
            # C_far is the mean exponentiated POSITIONAL mass of a far token; with
            # W_pos == 0 that is exp(0) == 1 exactly.
            C_far_list=[torch.ones(1, device=self.device, dtype=dtype)],
            sink_k_list=self.sink_k_list,
            sink_v_list=self.sink_v_list,
            temperature=self.temperature,
            top_p=self.top_p,
        )

    # ------------------------------------------------------------------ helpers

    @staticmethod
    def _resolve_special(model, attribute: str):
        tokenizer = getattr(model, "tokenizer", None)
        token_id = getattr(tokenizer, attribute, None)
        if token_id is None and hasattr(tokenizer, "to_dict"):
            token_id = tokenizer.to_dict().get(attribute, None)
        if token_id is None or int(token_id) < 0:
            return None
        return int(token_id)

    @contextlib.contextmanager
    def _zero_pos_embeddings(self):
        """Temporarily blank W_pos. `abstract_forward_pass` reads model.W_pos directly."""
        pos_parameter = self.model.pos_embed.W_pos
        saved = pos_parameter.data
        pos_parameter.data = self._zero_W_pos
        try:
            yield
        finally:
            pos_parameter.data = saved

    @contextlib.contextmanager
    def _rescaled_embeddings(self, sigma_query: float):
        """W_E <- W_E_unit * sigma_query, so that (x - mean)/sigma_query is the true LN."""
        embed_parameter = self.model.embed.W_E
        saved = embed_parameter.data
        torch.mul(self.W_E_unit, sigma_query, out=self._W_E_buffer)
        embed_parameter.data = self._W_E_buffer
        try:
            yield
        finally:
            embed_parameter.data = saved

    # Public alias: everything driven by `forward_pass_kwargs` must run inside this.
    position_free = _zero_pos_embeddings

    def _build_query_keys(self, tokens: torch.Tensor, position: int) -> torch.Tensor:
        """The last S_init tokens ending at `position`; left-padded with a dead token."""
        start = position + 1 - self.S_init
        if start >= 0:
            query = tokens[start: position + 1].to(self.device)                   # [S_init]
        else:
            padding = torch.full((-start,), self.fill_token_id, dtype=torch.long, device=self.device)
            query = torch.cat([padding, tokens[: position + 1].to(self.device)], dim=0)
        return query.view(1, self.S_init).long()                                  # [1, S_init]

    def _far_field(self, context_tokens: torch.Tensor, query_keys: torch.Tensor):
        """
        Split the context into: 1 x BOS (the sink) + the alive local window + the far field.

        Returns (far_counts [d_vocab], n_far float).
        """
        d_vocab = self.model.cfg.d_vocab
        total_counts = torch.bincount(context_tokens, minlength=d_vocab).to(torch.float64)

        window = query_keys.flatten()                                             # [S_init]
        alive_window = window[~torch.isin(window, self.dead_ids)]                 # [<= S_init]
        local_counts = torch.bincount(alive_window, minlength=d_vocab).to(torch.float64)

        far_counts = total_counts - local_counts
        # M_sink always fires exactly once, and it is exactly one BOS key/value.
        far_counts[self.bos_token_id] -= 1.0

        if float(far_counts.min()) < -1e-9:
            raise ValueError(
                "Negative far-field count: the local window / sink accounting is off. "
                "Is the prompt BOS-prepended (tokens[0] == bos_token_id)?"
            )
        far_counts.clamp_(min=0.0)
        return far_counts, float(far_counts.sum())

    # ------------------------------------------------------------------ the pass

    def predict(self, tokens: torch.Tensor, position: int):
        model = self.model
        context_tokens = tokens[: position + 1].flatten().to(self.device)         # [T]
        if int(context_tokens[0]) != self.bos_token_id:
            raise ValueError("Context must start with BOS; M_sink assumes exactly one leading BOS.")

        query_keys = self._build_query_keys(tokens, position)                     # [1, S_init]
        far_counts, n_far = self._far_field(context_tokens, query_keys)

        # ---- the far-field unigram measure pi, and the horizon L_ctx that scales it ----
        token_ids = far_counts.nonzero(as_tuple=True)[0]                          # [N_c]
        if token_ids.numel() == 0:
            # Whole context is already accounted for by the sink + the local window.
            token_ids = torch.tensor([self.bos_token_id], device=self.device, dtype=torch.long)
            context_vals = torch.ones(1, device=self.device, dtype=model.W_E.dtype)
            L_ctx_value = float(self.K + 1)                                       # => L_global == 0
        elif self.mass_in_L_ctx:
            context_vals = (far_counts[token_ids] / n_far).to(model.W_E.dtype)    # [N_c], sums to 1
            L_ctx_value = n_far + self.K + 1.0                                    # => L_global == n_far
        else:
            context_vals = far_counts[token_ids].to(model.W_E.dtype)              # [N_c], raw counts
            L_ctx_value = float(self.K + 2)                                       # => L_global == 1

        context_keys = torch.full(
            (token_ids.numel(), self.S_init), self.fill_token_id,
            dtype=torch.long, device=self.device,
        )                                                                         # [N_c, S_init]
        context_keys[:, -1] = token_ids                                           # only this column is read

        L_ctx_list = [torch.tensor([L_ctx_value], device=self.device, dtype=model.W_E.dtype)]
        C_far_list = [torch.ones(1, device=self.device, dtype=model.W_E.dtype)]   # no positions => exp(0)

        # ---- LayerNorm: exact per-token sigma via the rescaled embedding matrix ----
        if self.exact_layernorm:
            sigma_query = float(self.sigma_table[int(context_tokens[-1])])
            ln_avg_sigma_list = [sigma_query]
            embedding_context = self._rescaled_embeddings(sigma_query)
        else:
            ln_avg_sigma_list = [self._frozen_sigma]
            embedding_context = contextlib.nullcontext()

        with self._zero_pos_embeddings(), embedding_context:
            P_active = abstract_forward_pass(
                context_vals=context_vals,
                context_keys=context_keys,
                query_keys=query_keys,
                model=model,
                temperature=self.temperature,
                top_p=self.top_p,
                K_list=[self.K],
                active_heads_list=[[self.active_head]],
                ln_avg_sigma_list=ln_avg_sigma_list,
                L_ctx_list=L_ctx_list,
                C_far_list=C_far_list,
                sink_k_list=self.sink_k_list,
                sink_v_list=self.sink_v_list,
            )                                                                     # [1, d_vocab]

        info = {
            "context_vals": context_vals.detach().cpu(),                          # [N_c]
            "context_keys": context_keys.detach().cpu(),                          # [N_c, S_init]
            "query_keys": query_keys.detach().cpu(),                              # [1, S_init]
            "n_context_states": int(context_vals.numel()),
            "n_far": n_far,
            "L_ctx": L_ctx_value,
        }
        return P_active[0], info                                                  # [d_vocab], dict


### Example Run - Exact 1-Layer Forward Pass

In [ ]:
### Example: verify that the exact construction really is exact ###
# Run this straight after the benchmark cells. It reuses `corpus` from cell 102; any
# list of strings works.

for parameter in model.parameters():
    parameter.requires_grad = False

ACTIVE_HEAD = 3          # the single "semantic" head
K_EXACT = 1              # any K in [1, n_ctx] is exact; 1 is cheapest

exact_pass = ExactSemanticHeadForwardPass(
    model=model,
    active_head=ACTIVE_HEAD,
    K=K_EXACT,
    exact_layernorm=True,
    mass_in_L_ctx=True,
    temperature=1.0,
    top_p=1.0,
)

# The real side MUST be brought into the same regime: no positional embeddings, and every
# head except `ACTIVE_HEAD` ablated. Anything less and the JSD measures the model, not the
# approximation.
inactive_heads = [head for head in range(model.cfg.n_heads) if head != ACTIVE_HEAD]
real_hooks = [
    ("blocks.0.attn.hook_z", partial(zero_head_hook, head_idx=inactive_heads, min_position=0)),
    ("hook_pos_embed", remove_pos_embed_hook),
]

exact_benchmark = ApproximationQualityBenchmark(
    model=model,
    approximate_pass=exact_pass,
    real_hooks=real_hooks,
    approx_hooks=[],                 # the abstraction patches W_E / W_pos itself
    max_tokens=1000,
    min_position=700,
    n_positions=1,
    seed=42,
    store_probs="full",
)

exact_results = exact_benchmark.run(corpus)
summarize_benchmark(exact_results)

# Expected: mean JSD at float32 round-off, ~1e-13 nats, and top1_agreement == 1.0.
# If it is not, the usual culprits, in order of likelihood:
#   * the real pass was not run with BOTH hooks above;
#   * the model was loaded with fold_ln=False, so ln1 still carries a beta that
#     `single_layer_forward` never applies (the constructor raises on this);
#   * temperature / top_p differ between the two sides.

# --- Ablations: which ingredient is load-bearing? -------------------------------------
# Each row below breaks exactly one of the three tuned quantities and re-measures.
ablations = {
    "exact": exact_pass,
    "frozen sigma LN (ln_avg_sigma scalar)": ExactSemanticHeadForwardPass(
        model=model, active_head=ACTIVE_HEAD, K=K_EXACT, exact_layernorm=False),
    "K = 8 local window (must still be exact)": ExactSemanticHeadForwardPass(
        model=model, active_head=ACTIVE_HEAD, K=8),
    "counts in pi, L_ctx pinned (must still be exact)": ExactSemanticHeadForwardPass(
        model=model, active_head=ACTIVE_HEAD, K=K_EXACT, mass_in_L_ctx=False),
}

for label, approximate_pass in ablations.items():
    result = ApproximationQualityBenchmark(
        model=model, approximate_pass=approximate_pass, real_hooks=real_hooks,
        max_tokens=1000, min_position=700, n_positions=1, seed=42,
        store_probs="none", verbose=False,
    ).run(corpus[:20])
    statistics = result["statistics"]
    print(f"{label:48s} JSD = {statistics['loss_jsd']['mean']:.3e}   "
          f"top1 = {statistics['top1_agreement']['mean']:.3f}")


## Exact Single-Semantic-Head Fixed-Point Search

The fixed-point search of the previous sections, driven by the **exact** parameterization of
`ExactSemanticHeadForwardPass` instead of profiled estimates.

For a 1-layer, attention-only model with the positional embeddings zeroed, the attention score
`s(x_t, x_j)` depends only on the two **token ids** -- position drops out -- so the forward pass is an
exact function of the context unigram **counts**. That makes every parameter of the 3-part partition
function analytically determined rather than measured:

| Parameter | Value | Why |
|---|---|---|
| `C_far` | `1.0` | mean exponentiated *positional* mass of a far token; with `W_pos = 0` that is `exp(0)` |
| `L_ctx` | `L_PROMPT` | so `L_global = L_ctx - K - 1 = N_far`, the exact far-token count |
| `K_list` | `[K]`, free | `K` only shuttles mass between `M_local` and `M_global`; `K = 1` is cheapest |
| `ln_avg_sigma` | `[d_vocab]` table | exact per-token LayerNorm, applied per row (see below) |
| `W_pos` | zeroed | for the whole loop, including the checkpointed backward recomputation |

Three things are new relative to cell 68:

1. **`ExactSemanticHeadForwardPass.forward_pass_kwargs(L_prompt)`** is the single source of truth for
   the physics. The same dict drives the exploration engine and the gradient forward passes, so the two
   phases provably cannot drift apart.
2. **Exploration uses `MetastableKGramEngine`** instead of a top-k rollout: pruning happens on *fully
   aggregated* flux, so states reached by many weak paths survive. One power iteration per step, plus a
   fully converged run every `explore_every` steps.
3. **`pi` is the far-field measure**, not the whole-context unigram -- the local window and the BOS sink
   are already counted exactly once by `M_local` and `M_sink`.

`L_PROMPT` is not a nuisance parameter to be faked: it is *the* physical parameter of the problem, since
it sets `L_global`, the weight of the mean field against the sink and the local window. `L -> infinity`
is the pure mean-field limit. With `W_pos = 0` there is no `n_ctx` ceiling on it. **Sweep it.**


In [ ]:
### EXACT SINGLE-SEMANTIC-HEAD FIXED-POINT SEARCH -- SETUP ###
#
# This lane finds a fixed point pi = pi P_pi of a 1-layer, attention-only,
# POSITION-FREE model restricted to ONE semantic head, using the exact same
# parameterization that `ExactSemanticHeadForwardPass` uses to reproduce the concrete
# model to ~1e-12 JSD. Nothing here is a profiled estimate: C_far == 1 exactly,
# L_ctx is the prompt length exactly, and LayerNorm is exact per token.
#
# WHAT pi MEANS HERE (this is the one semantic change vs. cell 68)
# ---------------------------------------------------------------
# pi is the FAR-FIELD measure: the unigram (K-gram) law of the L_PROMPT - K - 1 tokens
# that are neither the BOS sink nor inside the explicit local window. It is NOT the
# whole-context unigram. Consequences, all of which this cell enforces:
#   * BOS and PAD are forbidden from pi's support (M_sink already contributes exactly
#     one BOS; counting it again in pi double-counts the sink).
#   * pi handed to the forward pass must sum to exactly 1, because L_global * pi_c is
#     the expected COUNT of state c in the far field. A truncated, unnormalized pi
#     silently shrinks the effective context to L * sum(pi).
#   * The fixed point equation becomes pi = normalize(pi P_pi restricted to non-dead
#     states): "the law of a token far from the query equals the law of the emitted
#     token". Do not compare pi* against the raw unigram of a generated sample without
#     stripping special tokens first.
#
# PRECONDITIONS
#   * model loaded with fold_ln=True  (the TransformerLens default) -- this is the
#     OPPOSITE of what the rest of the notebook wants (see the developer guide 4.2).
#     `single_layer_forward` applies gamma but never beta, so beta must be folded away.
#   * model.cfg.n_layers == 1, attn_only, learned absolute positional embeddings.
#   * cells 9, 10, 11, 100 and 105 have been run (helpers, engine, jsd_dense, the class).
#   * `corpus` (a List[str]) exists -- cell 102 builds one.

import io
import math
import contextlib
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

torch.set_grad_enabled(True)          # cell 15 disables this globally
model.cfg.use_attn_result = False     # cell 15 turns it on; large and unused here
for _p in model.parameters():
    _p.requires_grad = False

# `compute_aligned_jsd` (the cell-60 redefinition) appends to this module-level list.
if "N_union_list" not in globals():
    N_union_list = []

# =====================================================================
# 1. CONFIG
# =====================================================================
ACTIVE_HEAD      = 3        # the single semantic head being modelled
K_WINDOW         = 1        # K_list[0] and S_init. Any value is exact; 1 is cheapest.
L_PROMPT         = 800.0    # total context length: 1 BOS + K local + (L-K-1) far
EXACT_LAYERNORM  = True     # True -> exact per-token sigma table; False -> frozen scalar

# --- optimization ---
N_active         = 256                   # queries per forward pass
N_tracking       = model.cfg.d_vocab // 4  # hard cap on |support(pi)|
K_pruning        = 512                   # successors kept per query inside pi P
N_output         = N_active              # states kept in pi P
M_iterations     = 1                     # power iterations per gradient step
n_iterations     = 3000
lr_max, lr_min   = 1e0, 1e-1
loss_scale       = 1e5 / model.cfg.d_vocab
print_iterations = 20

# --- exploration (MetastableKGramEngine) ---
explore_every         = 50      # how often to run a DEEP (converged) exploration
explore_deep_iters    = 30      # max power iterations in a deep exploration
explore_shallow_iters = 1       # power iterations on every other step
eta_explore           = 0.00    # damped-Picard seed: pi <- normalize(pi + eta * flux)
epsilon_power         = 1e-6    # POST-AGGREGATION flux guillotine
frontier_threshold    = 1e-4
leakage_threshold     = 1e-2
min_stable_steps      = 1
explore_query_batch   = 1024
explore_suffix_chunk  = 2048    # dense [chunk, vocab] fp32 accumulator (~411 MB)
verbose_explore       = True    # print the 3-part audit on deep explorations only

# --- seeding ---
SEED_TOKENS  = 700
N_SEED_RANDOM    = 0            # random padding k-grams; 0 = let the engine discover

# =====================================================================
# 2. THE EXACT DRIVER AND ITS KWARGS
# =====================================================================
exact_driver = ExactSemanticHeadForwardPass(
    model,
    active_head=ACTIVE_HEAD,
    K=K_WINDOW,
    exact_layernorm=EXACT_LAYERNORM,
    mass_in_L_ctx=True,
    temperature=1.0,
    top_p=1.0,
)

# The single source of truth for the physics. The SAME dict drives the exploration
# engine and the gradient forward passes, so the two phases cannot drift apart.
fp_kwargs = exact_driver.forward_pass_kwargs(L_PROMPT)

device   = model.cfg.device
S_init   = exact_driver.S_init                     # == K_WINDOW for a 1-layer model
dead_ids = exact_driver.dead_ids                   # BOS + PAD
eps      = 1e-10
N_far    = L_PROMPT - K_WINDOW - 1.0               # == L_global

print(f"S_init        = {S_init}")
print(f"L_ctx         = {fp_kwargs['L_ctx_list'][0].item()}  ->  L_global = N_far = {N_far}")
print(f"C_far         = {fp_kwargs['C_far_list'][0].item()}")
print(f"ln sigma      = {'per-token table [d_vocab]' if EXACT_LAYERNORM else fp_kwargs['ln_avg_sigma_list'][0]}")
print(f"dead ids      = {dead_ids.tolist()} (excluded from pi and from pi P)")

# =====================================================================
# 3. FAR-FIELD BOOKKEEPING
# =====================================================================
def drop_dead_states(vals, keys, dead_ids, renormalize=True):
    """
    Restrict a sparse measure to the far field: drop every state containing BOS/PAD.

    pi is the law of the FAR tokens only, and BOS is already carried by M_sink, so a
    dead state has no meaning here. Applying this to BOTH pi and pi P keeps the
    fixed-point equation self-consistent -- filtering only one side would leave a
    permanent JSD floor and a gradient that forever pulls pi toward BOS.

    Differentiable in `vals`.
    """
    alive = ~torch.isin(keys, dead_ids).any(dim=1)
    if not bool(alive.any()):
        raise ValueError("Every state in the measure is a dead (BOS/PAD) state.")
    if not bool(alive.all()):
        vals = vals[alive]
        keys = keys[alive]
    if renormalize:
        vals = vals / vals.sum().clamp(min=1e-30)
    return vals, keys

# =====================================================================
# 4. SEED pi FROM A REAL CONTEXT (far field only)
# =====================================================================
seed_tokens = model.to_tokens(corpus[0])[0,:SEED_TOKENS]
seed_vals, seed_keys = get_kgram_distribution_from_tokens(
    seed_tokens, K=S_init, offset=S_init, device=str(device)
)
pi_vals, pi_keys = drop_dead_states(seed_vals.to(device), seed_keys.to(device), dead_ids)

if N_SEED_RANDOM > 0:
    rand_keys = torch.randint(0, model.cfg.d_vocab, (N_SEED_RANDOM, S_init), device=device)
    keep = ~torch.isin(rand_keys, dead_ids).any(dim=1)
    cat_keys = torch.cat([pi_keys, rand_keys[keep]], dim=0)
    unique_keys, inv = torch.unique(cat_keys, dim=0, return_inverse=True)
    padded = torch.full((unique_keys.size(0),), eps, device=device, dtype=pi_vals.dtype)
    padded.scatter_(0, inv[: pi_keys.size(0)], pi_vals)
    pi_vals, pi_keys = padded / padded.sum(), unique_keys

print(f"\nseeded pi: {pi_keys.size(0)} far-field states, mass {pi_vals.sum().item():.6f}")

# =====================================================================
# 5. THE EXPLORATION ENGINE (same forward_pass_kwargs as the gradient phase)
# =====================================================================
explore_engine = MetastableKGramEngine(
    model=model,
    k_value=S_init,
    frozen_context_vals=pi_vals,
    frozen_context_keys=pi_keys,
    forward_pass_kwargs=fp_kwargs,
    epsilon_power_method=epsilon_power,
    query_batch_size=explore_query_batch,
    suffix_chunk_size=explore_suffix_chunk,
    device=str(device),
)

losses_exact        = []
N_union_history     = []
top_mass_history    = []
explore_size_history = []


In [ ]:
### ACCEPTANCE TEST -- is the BATCHED path really the exact forward pass? ###
#
# Run this BEFORE the optimization. It answers three separate questions:
#
#   (1) Does `forward_pass_kwargs` + the per-token sigma table reproduce
#       `ExactSemanticHeadForwardPass.predict`? These are two INDEPENDENT exact
#       constructions -- predict() rescales W_E for one query token, the batched path
#       divides by a per-token sigma table for every row at once -- so agreement at
#       float32 round-off validates both. Expect ~1e-12.
#   (2) What does treating pi as the far field (option a) buy over the naive
#       whole-context unigram? This is the O((K+1)/L) double-count, measured.
#   (3) What does the frozen-sigma linearization still cost? (~4.9e-3 on attn-only-1l.)
#
# Cell 107 remains the ground truth against the CONCRETE model; this cell only checks
# that the batched lane agrees with the already-validated per-position lane.

test_tokens = model.to_tokens(corpus[0])[0][: int(L_PROMPT)]
assert test_tokens.numel() == int(L_PROMPT), (
    f"seed text too short: {test_tokens.numel()} tokens < L_PROMPT={int(L_PROMPT)}"
)
test_position = test_tokens.numel() - 1

with torch.no_grad():
    # (1) reference: the class's own per-position exact pass (W_E rescaling trick)
    p_ref, info = exact_driver.predict(test_tokens, test_position)
    print(f"n_far = {info['n_far']:.0f}, L_ctx = {info['L_ctx']:.0f}, "
          f"N_c = {info['n_context_states']}")

    ctx_vals_ref = info["context_vals"].to(device)
    ctx_keys_ref = info["context_keys"].to(device)
    qry_keys_ref = info["query_keys"].to(device)

    with exact_driver.position_free():
        # (1) the batched path this optimization uses
        p_batched = abstract_forward_pass(
            context_vals=ctx_vals_ref, context_keys=ctx_keys_ref,
            query_keys=qry_keys_ref, model=model,
            **exact_driver.forward_pass_kwargs(info["L_ctx"]),
        )[0]

        # (3) the frozen-sigma ablation, same context
        p_frozen = abstract_forward_pass(
            context_vals=ctx_vals_ref, context_keys=ctx_keys_ref,
            query_keys=qry_keys_ref, model=model,
            **exact_driver.forward_pass_kwargs(info["L_ctx"], exact_layernorm=False),
        )[0]

        # (2) the naive whole-context unigram at the SAME L_ctx
        counts = torch.bincount(
            test_tokens.to(device), minlength=model.cfg.d_vocab
        ).to(model.W_E.dtype)
        counts[dead_ids] = 0.0
        ids = counts.nonzero(as_tuple=True)[0]
        keys_full = torch.full(
            (ids.numel(), S_init), int(exact_driver.fill_token_id),
            dtype=torch.long, device=device,
        )
        keys_full[:, -1] = ids
        vals_full = counts[ids] / counts[ids].sum()
        p_full_pi = abstract_forward_pass(
            context_vals=vals_full, context_keys=keys_full,
            query_keys=qry_keys_ref, model=model,
            **exact_driver.forward_pass_kwargs(L_PROMPT),
        )[0]

print(f"\n(1) batched kwargs   vs predict()   JSD = {jsd_dense(p_ref, p_batched):.3e}   <- must be ~1e-12")
print(f"(2) whole-context pi vs far-field pi JSD = {jsd_dense(p_ref, p_full_pi):.3e}   <- price of option (b)")
print(f"(3) frozen sigma     vs exact LN     JSD = {jsd_dense(p_ref, p_frozen):.3e}   <- price of EXACT_LAYERNORM=False")


In [ ]:
### EXACT SINGLE-SEMANTIC-HEAD FIXED-POINT SEARCH -- THE LOOP ###
#
# WHY THE WHOLE LOOP SITS INSIDE `position_free()`
# ------------------------------------------------
# `abstract_forward_pass` reads `model.W_pos` DIRECTLY (not through a hook), and
# `compute_stationary_distribution` wraps each step in torch.utils.checkpoint, which
# RECOMPUTES the forward pass during loss.backward(). If the context manager only
# covered the forward call, the backward recomputation would run with the real W_pos
# restored and the gradients would be computed for a different model. So the manager
# wraps everything, including backward.
#
# Side effect: for the duration of this cell the model is globally position-free.
# Do not run any other model code concurrently (it is also not thread-safe).

with exact_driver.position_free():
    for i_iteration in tqdm(range(n_iterations)):

        # -----------------------------------------------------------------
        # PHASE 1: EXPLORATION -- MetastableKGramEngine, no gradients
        #
        # Replaces cell 68's single top-k rollout. The engine prunes on FULLY
        # AGGREGATED flux (sum over all parents) instead of top-k per query, so states
        # reached by many weak paths -- exactly the frontier a mean-field operator
        # produces -- survive instead of being cut.
        #
        # Cadence: 1 power iteration per step (cheap, same cost class as cell 68's
        # rollout), and a fully converged run every `explore_every` steps.
        # -----------------------------------------------------------------
        n_seed = min(N_active, pi_vals.numel())
        top_pi_vals, top_pi_inds = pi_vals.topk(n_seed)
        top_pi_keys = pi_keys[top_pi_inds]

        deep = (i_iteration % explore_every == 0)

        with torch.no_grad():
            # Rebind rather than reconstruct: pi changes every step, and the engine's
            # constructor does nothing but store references.
            explore_engine.frozen_context_vals = (pi_vals / pi_vals.sum()).detach()
            explore_engine.frozen_context_keys = pi_keys

            def _run_exploration():
                return explore_engine.run_pruned_power_iteration(
                    initial_kgrams=top_pi_keys,
                    # Warm start from the current pi. Without this the engine seeds a
                    # UNIFORM measure over the seeds, and the flux it returns would not
                    # be a pi-weighted flux -- which would make the eta blend below
                    # meaningless.
                    initial_measure=(top_pi_vals / top_pi_vals.sum()).detach(),
                    max_iterations=(explore_deep_iters if deep else explore_shallow_iters),
                    frontier_threshold=frontier_threshold,
                    leakage_threshold=leakage_threshold,
                    min_stable_steps=min_stable_steps,
                )

            if deep and verbose_explore:
                explore_result = _run_exploration()
            else:
                with contextlib.redirect_stdout(io.StringIO()):
                    explore_result = _run_exploration()

            explore_keys = explore_result["closed_support_keys"]
            explore_flux = explore_result["stationary_measure"]

            # pi is the far field: no BOS/PAD states, ever.
            explore_flux, explore_keys = drop_dead_states(explore_flux, explore_keys, dead_ids)

            # The engine's support only grows; cap it.
            if explore_keys.size(0) > N_tracking:
                keep = explore_flux.topk(N_tracking).indices
                explore_keys = explore_keys[keep]
                explore_flux = explore_flux[keep] / explore_flux[keep].sum()

            # ---- union support, seeded with the engine's flux (damped Picard) ----
            cat_keys = torch.cat([pi_keys, explore_keys], dim=0)
            unique_keys, inverse_indices = torch.unique(cat_keys, dim=0, return_inverse=True)
            N_union = unique_keys.size(0)

            pi_vals_union = torch.full(
                (N_union,), 2 * eps, device=device, dtype=pi_vals.dtype
            )
            pi_vals_union.scatter_(0, inverse_indices[: pi_keys.size(0)], pi_vals)

            flux_union = torch.zeros(N_union, device=device, dtype=pi_vals.dtype)
            flux_union.scatter_(
                0, inverse_indices[pi_keys.size(0):], explore_flux.to(pi_vals.dtype)
            )

            # Newly discovered states enter with eta * their flux instead of 2*eps, so
            # the JSD gradient does not have to drag them up from nothing. NOTE: this
            # makes the method a hybrid -- gradient descent on JSD plus a damped Picard
            # step. Set eta_explore = 0 to recover pure gradient descent.
            pi_vals_union = pi_vals_union + eta_explore * flux_union
            pi_vals_union = pi_vals_union / pi_vals_union.sum()

            N_union_history.append(N_union)
            explore_size_history.append(int(explore_keys.size(0)))

        # -----------------------------------------------------------------
        # PHASE 2: OPTIMIZATION PASS (with gradients)
        # -----------------------------------------------------------------
        pi_logits_union = nn.Parameter(torch.log(pi_vals_union.clamp(min=1e-30)))

        current_lr = lr_min + 0.5 * (lr_max - lr_min) * (
            1 + math.cos(math.pi * i_iteration / n_iterations)
        )
        optimizer = torch.optim.SGD([pi_logits_union], lr=current_lr, momentum=0)
        optimizer.zero_grad()

        pi_vals_active = F.softmax(pi_logits_union, dim=-1)

        n_query = min(N_active, N_union)
        top_pi_vals_active, top_pi_active_inds = pi_vals_active.topk(n_query)
        top_pi_active_keys = unique_keys[top_pi_active_inds]

        # CRITICAL for exactness: L_global * pi_c is the expected COUNT of state c in
        # the far field, so the measure fed to the forward pass must sum to 1. Cell 68
        # passed the raw top-k slice (sum < 1), which silently rescales the effective
        # context length to L * top_mass. `top_mass` is logged below -- if it drifts
        # away from 1, raise N_active.
        top_mass = top_pi_vals_active.sum()
        ctx_vals = top_pi_vals_active / top_mass

        pi_P_vals, pi_P_keys = compute_stationary_distribution(
            ctx_vals, top_pi_active_keys, N_output, K_pruning, model,
            M_iterations, **fp_kwargs,
        )

        # Restrict pi P to the far field too, so both sides of the JSD live on the
        # same space. (The power iteration can still emit BOS/PAD successors and spend
        # output slots on them; with N_output = 256 that is at most a couple of rows.)
        pi_P_vals, pi_P_keys = drop_dead_states(pi_P_vals, pi_P_keys, dead_ids)

        loss = compute_aligned_jsd(
            pi_vals_active, unique_keys, pi_P_vals.clamp(1e-10), pi_P_keys
        ) * loss_scale
        losses_exact.append(loss.item())
        top_mass_history.append(top_mass.item())

        loss.backward()

        # -----------------------------------------------------------------
        # PHASE 3: NATURAL GRADIENT ON THE SIMPLEX (Sherman-Morrison)
        # -----------------------------------------------------------------
        with torch.no_grad():
            g = pi_logits_union.grad
            lmbda = 1e-10

            A_inv     = 1.0 / (pi_vals_active + lmbda)
            A_inv_g   = A_inv * g
            A_inv_p   = A_inv * pi_vals_active

            p_A_inv_g = torch.sum(pi_vals_active * A_inv_g)
            p_A_inv_p = torch.sum(pi_vals_active * A_inv_p)

            numerator   = A_inv_p * p_A_inv_g
            denominator = 1.0 - p_A_inv_p

            pi_logits_union.grad = A_inv_g + (numerator / denominator.clamp(min=1e-10))
            torch.nn.utils.clip_grad_norm_([pi_logits_union], max_norm=1.0)

        optimizer.step()

        # -----------------------------------------------------------------
        # PHASE 4: PRUNING & MIGRATION (stay on the far field, stay normalized)
        # -----------------------------------------------------------------
        with torch.no_grad():
            updated_pi_vals = F.softmax(pi_logits_union, dim=-1)
            top_vals, top_idx = torch.topk(updated_pi_vals, k=min(N_tracking, N_union))
            pi_vals, pi_keys = drop_dead_states(top_vals, unique_keys[top_idx], dead_ids)

        # -----------------------------------------------------------------
        # PHASE 5: LOGGING
        # -----------------------------------------------------------------
        if i_iteration % print_iterations == 0:
            grad_mean = pi_logits_union.grad.abs().mean().item()
            grad_max  = pi_logits_union.grad.abs().max().item()
            print(
                f"\nIter {i_iteration} | lr {current_lr:.2e} | Grad mean {grad_mean:.2e} "
                f"max {grad_max:.2e}"
            )
            print(
                f"Loss: {loss.item():.4e} | N_union: {N_union} | |support(pi)|: "
                f"{pi_keys.size(0)} | top-{n_query} mass: {top_mass.item():.4f}"
            )
            if deep and explore_result["history_frontier_rates"]:
                print(
                    f"explore: converged={explore_result['is_converged']} "
                    f"iters={explore_result['total_iterations']} "
                    f"frontier={explore_result['history_frontier_rates'][-1]*100:.2f}% "
                    f"leakage={explore_result['history_leakage_rates'][-1]*100:.2f}%"
                )

            show = min(5, pi_keys.size(0))
            tv, ti = pi_vals.topk(show)
            pv, pidx = pi_P_vals.topk(min(show, pi_P_vals.numel()))
            print("pi   :", [f"{v:.4f}" for v in tv.tolist()],
                  [repr(model.to_string(k)) for k in pi_keys[ti].tolist()])
            print("pi P :", [f"{v:.4f}" for v in pv.tolist()],
                  [repr(model.to_string(k)) for k in pi_P_keys[pidx].tolist()])

print("\nDone. Final |support(pi)| =", pi_keys.size(0), "| final loss =", losses_exact[-1])


In [ ]:
### EXACT SINGLE-SEMANTIC-HEAD FIXED POINT -- DIAGNOSTICS ###

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(np.arange(1, len(losses_exact) + 1), losses_exact)
axes[0, 0].set_yscale("log")
axes[0, 0].set_xlabel("Iteration"); axes[0, 0].set_ylabel("JSD loss (scaled)")
axes[0, 0].set_title("Loss vs. iteration"); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(N_union_history, label="|union support|")
axes[0, 1].plot(explore_size_history, label="|engine support|")
axes[0, 1].set_xlabel("Iteration"); axes[0, 1].set_ylabel("states")
axes[0, 1].set_title("Support size vs. iteration"); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# top_mass is the fraction of pi captured by the N_active queries actually fed to the
# forward pass. It is the effective context-length rescaling: L_eff = L_PROMPT * top_mass
# BEFORE the renormalization in phase 2. If it sits well below 1, raise N_active.
axes[1, 0].plot(top_mass_history)
axes[1, 0].axhline(1.0, color="k", ls="--", lw=1)
axes[1, 0].set_xlabel("Iteration"); axes[1, 0].set_ylabel(f"mass in top-{N_active}")
axes[1, 0].set_title("Truncation mass (want ~1)"); axes[1, 0].grid(alpha=0.3)

top_to_display = min(30, pi_keys.size(0))
tv, ti = pi_vals.topk(top_to_display)
labels = [repr(model.to_string(k)) for k in pi_keys[ti].tolist()]
axes[1, 1].barh(labels[::-1], tv.cpu().numpy()[::-1])
axes[1, 1].set_xlabel("Probability")
axes[1, 1].set_title(f"Top-{top_to_display} far-field states in $\\pi_{{final}}$")
axes[1, 1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Sample an 800-token prompt from the final optimized distribution.
final_pi = torch.zeros(
    model.cfg.d_vocab,
    device=pi_vals.device,
    dtype=pi_vals.dtype,
)
final_pi.scatter_add_(0, pi_keys[:, -1], pi_vals)
final_pi = final_pi / final_pi.sum()

sampled_tokens = context_from_pi(final_pi, N=800).to(device)
sampled_prompt = model.to_string(sampled_tokens)

# Keep only head 3 in every layer and remove positional embeddings.
hooks = [
    (
        f"blocks.{layer_idx}.attn.hook_z",
        partial(
            zero_head_hook,
            head_idx=[head for head in range(model.cfg.n_heads) if head != 3],
            min_position=0,
        ),
    )
    for layer_idx in range(model.cfg.n_layers)
]
hooks.append(("hook_pos_embed", remove_pos_embed_hook))

with torch.no_grad():
    with model.hooks(fwd_hooks=hooks):
        generated_tokens = model.generate(
            sampled_prompt,
            max_new_tokens=200,
            do_sample=True,
            temperature=1.0,
            top_p=1.0,
            prepend_bos=True,
            stop_at_eos=True,
            return_type="tokens",
        )

print(model.to_string(generated_tokens[0]))